In [1]:
# CELL 1 — ZERO-INSTALL G4 SETUP: mount Drive, import existing packages, configure paths/constants
"""
G4 / RTX PRO 6000 Blackwell — modern sequence-mixer baselines.

RUNS
----
4 already-tokenized corpora from Google Drive:
    FineWeb-Edu
    arXiv
    PG-19
    WikiText-103

3 methods:
    Mamba-2 (Hugging Face native; preferred)
    Mamba (Hugging Face native; automatic fallback only)
    Hyena
    RetNet

IMPORTANT
---------
* NO tokenizer training.
* NO dataset downloading.
* NO tokenization.
* Uses the SAME C4-trained 32k SentencePiece token IDs already created
  for the TransWave experiments.
* Effective batch follows the completed TransWave protocol per corpus.
* Sequence length = 512.
* Same deterministic MLM corruption used by the G4 TransWave notebooks.
* Same Adam / 1% warmup then constant LR protocol used by the G4 notebooks.
* BF16 + TF32 on Blackwell.
* Fused Adam when supported.
* Activation checkpointing OFF.
* Physical microbatch chosen by exhaustive measured full-update throughput over every divisor of the effective batch.
* Data are copied from Drive to local SSD one dataset at a time.
* Persistent Drive resume checkpoints.
* Aggressive CPU/GPU cleanup AFTER EVERY METHOD and AFTER EVERY DATASET.
* NEVER asks for or requires a runtime restart.
* Hugging Face Transformers only for Mamba; no mamba_ssm/causal_conv1d binary dependency.

For masked-LM comparability, causal mixers are adapted bidirectionally with
SHARED parameters: forward(x) and reverse(forward(reverse(x))) are averaged.
This changes compute, not parameter count, and must be stated in the paper.
"""

from __future__ import annotations

import os
import sys
import gc
import json
import math
import time
import random
import shutil
import ctypes
import hashlib
import urllib.request
import subprocess
import importlib
import importlib.util
import importlib.metadata
import inspect
import re

CELL1_VERSION = "FIX2-HF-AUTO-MAMBA-NO-TOKENIZER-FILE-2026-10-09"
print("CELL 1 VERSION:", CELL1_VERSION)

# Configure the CUDA allocator BEFORE importing torch.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# ---------------------------------------------------------------------
# MOUNT GOOGLE DRIVE
# ---------------------------------------------------------------------

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# ---------------------------------------------------------------------
# HUGGING FACE-ONLY MAMBA SETUP
#
# Rules:
# * Never install mamba_ssm / causal_conv1d / selective_scan CUDA wheels.
# * Prefer Hugging Face Mamba-2.
# * If the installed Transformers lacks Mamba-2, upgrade Transformers only.
# * If Mamba-2 cannot pass a REAL CUDA forward/backward test, fall back to
#   Hugging Face Mamba.
# * No source build, no GitHub install, no CUDA-wheel ABI dependency.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F


def _installed_distribution_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def _purge_modules(prefixes):
    for _name in list(sys.modules):
        if any(_name == p or _name.startswith(p + ".") for p in prefixes):
            sys.modules.pop(_name, None)


# Previous failed Colab attempts may have left ABI-incompatible optional
# Mamba CUDA packages installed.  HF native Mamba does not need them; remove
# them before Transformers is imported so its availability checks cannot pick
# up a broken selective_scan_cuda binary.
_external = [
    p for p in ("mamba-ssm", "causal-conv1d")
    if _installed_distribution_version(p) is not None
]
if _external:
    print("Removing incompatible optional Mamba CUDA packages:", ", ".join(_external))
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", *_external],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    _purge_modules(("mamba_ssm", "causal_conv1d", "selective_scan_cuda"))
    importlib.invalidate_caches()


def _import_hf_mamba_public_api():
    """Return public HF Mamba classes without importing any external kernels."""
    try:
        import transformers as _tf
        _m2_cfg = getattr(_tf, "Mamba2Config", None)
        _m2_model = getattr(_tf, "Mamba2Model", None)
        _m1_cfg = getattr(_tf, "MambaConfig", None)
        _m1_model = getattr(_tf, "MambaModel", None)
        return {
            "transformers": _tf,
            "version": getattr(_tf, "__version__", "unknown"),
            "mamba2_config": _m2_cfg,
            "mamba2_model": _m2_model,
            "mamba_config": _m1_cfg,
            "mamba_model": _m1_model,
            "error": None,
        }
    except Exception as exc:
        return {
            "transformers": None,
            "version": _installed_distribution_version("transformers"),
            "mamba2_config": None,
            "mamba2_model": None,
            "mamba_config": None,
            "mamba_model": None,
            "error": repr(exc),
        }


_hf = _import_hf_mamba_public_api()
print("Transformers initially:", _hf["version"])

# Upgrade only when Mamba-2 is unavailable.  This is a pure Python/HF package
# upgrade; Torch/CUDA are left untouched.  If networking happens to fail but
# the installed version has Mamba, the fallback remains usable.
if _hf["mamba2_config"] is None or _hf["mamba2_model"] is None:
    print("HF Mamba-2 unavailable in the installed Transformers; upgrading Transformers only...")
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--upgrade",
            "transformers>=5.0,<6",
        ])
    except subprocess.CalledProcessError as exc:
        print("Transformers upgrade failed; will test any already-installed HF Mamba fallback.")
        print("pip return code:", exc.returncode)

    _purge_modules(("transformers",))
    importlib.invalidate_caches()
    _hf = _import_hf_mamba_public_api()

HF_TRANSFORMERS_VERSION = _hf["version"]
HF_MAMBA2_CONFIG = _hf["mamba2_config"]
HF_MAMBA2_MODEL = _hf["mamba2_model"]
HF_MAMBA_CONFIG = _hf["mamba_config"]
HF_MAMBA_MODEL = _hf["mamba_model"]

if HF_MAMBA2_MODEL is None and HF_MAMBA_MODEL is None:
    raise RuntimeError(
        "Hugging Face Transformers provides neither Mamba-2 nor Mamba on this runtime. "
        f"Transformers={HF_TRANSFORMERS_VERSION}; initial import error={_hf['error']}"
    )

print("Transformers selected:", HF_TRANSFORMERS_VERSION)
print("HF Mamba-2 public API:", "YES" if HF_MAMBA2_MODEL is not None else "NO")
print("HF Mamba public API  :", "YES" if HF_MAMBA_MODEL is not None else "NO")

# ---------------------------------------------------------------------
# PROJECT / DRIVE PATHS
# ---------------------------------------------------------------------

PROTOCOL_VERSION = "splinet_g4_hf_independent_tokenizers_v1"
IMPLEMENTATION_VERSION = "splinet_g4_hf_independent_tokenizers_v1_stacked_bidir_masked_head"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/spliNet"
)
CHECKPOINT_ROOT = PROJECT_ROOT / "checkpoints"
RESULTS_ROOT = PROJECT_ROOT / "results"
CONFIG_ROOT = PROJECT_ROOT / "configs"

for p in [PROJECT_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT, CONFIG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

TOKENIZER_ROOT = PROJECT_ROOT / "tokenizers"
TOKEN_ROOT = PROJECT_ROOT / "tokens"
for p in [TOKENIZER_ROOT, TOKEN_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

LOCAL_ROOT = Path("/content/spliNet")
LOCAL_STAGE_ROOT = LOCAL_ROOT / "staged"
LOCAL_TMP_ROOT = LOCAL_ROOT / "tmp"

for p in [LOCAL_ROOT, LOCAL_STAGE_ROOT, LOCAL_TMP_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# These experiments consume already-tokenized NumPy arrays.
# No tokenizer file is required at runtime.  Do not probe Drive for one.
TOKENIZER_MODEL = None
TOKENIZER_SHA256 = None
TOKENIZER_METADATA_STATUS = "not required: using pre-tokenized 32k-ID arrays"
print("Tokenizer file: NOT REQUIRED (using existing pre-tokenized arrays)")

# Existing token IDs already produced by the TransWave notebooks.
FINEWEB_ROOT = Path(
    "/content/drive/MyDrive/TransWave/"
    "scratch_pretraining/fineweb_edu_100m_sp32k"
)

ARXIV_ROOT = Path(
    "/content/drive/MyDrive/TransWave_ArXiv_G4/"
    "data/arxiv_100m_sp32k"
)

PG19_ROOT = Path(
    "/content/drive/MyDrive/TransWave_PG19_G4/"
    "data/pg19_100m_sp32k"
)

WIKITEXT_ROOT = Path(
    "/content/drive/MyDrive/TransWave_WikiText103_G4/"
    "data/wikitext103_100m_sp32k"
)

# ---------------------------------------------------------------------
# EXACT EXPERIMENT CONSTANTS FROM THE G4 TRANSWAVE RUNS
# ---------------------------------------------------------------------

SEQ_LENGTH = 512
VOCAB_SIZE = 32_000

EXPECTED_TRAIN_BLOCKS = 193_359
EXPECTED_VALID_BLOCKS = 1_953

EFFECTIVE_BATCH_BY_CORPUS = {
    "fineweb_edu": 64,      # matches the completed FineWeb-Edu sweep
    "arxiv": 128,           # matches the G4 arXiv run
    "pg19": 128,            # matches the G4 PG-19 run
    "wikitext103": 128,     # matches the G4 WikiText-103 run
}
LEARNING_RATE = 1.0e-4
ADAM_BETAS = (0.9, 0.999)
ADAM_EPS = 1.0e-8
WEIGHT_DECAY = 0.0

MLM_PROBABILITY = 0.15
MAX_PREDICTIONS = 80

GLOBAL_SEED = 2026
DATA_ORDER_SEED = 7301
MASK_SEED = 912431

SAVE_EVERY = 500
SAVE_EVERY_MINUTES = 20
LOG_EVERY = 25

SPECIAL_IDS = (0, 1, 2, 3, 4, 5, 6)
MASK_ID = 6
PAD_ID = 3

HIDDEN_SIZE = 768
NUM_LAYERS = 12
NUM_HEADS = 12
HEAD_DIM = 64
INTERMEDIATE_SIZE = 3072
TYPE_VOCAB_SIZE = 4

HIDDEN_DROPOUT = 0.1
LAYER_NORM_EPS = 1.0e-12
INITIALIZER_RANGE = 0.02

# Save the final weights only if desired. Leaving this False is faster and
# greatly reduces Drive traffic; resumable training_latest.pt is still kept
# while a run is in progress.
SAVE_FINAL_MODEL = True

RETENTION_CHUNK = 64

# Throughput autotuning is versioned independently of the training protocol.
# This forces a fresh exhaustive G4 tune without invalidating training checkpoints.
AUTOTUNE_VERSION = "splinet_exhaustive_90gib_v1"
AUTOTUNE_WARMUP_UPDATES = 1
AUTOTUNE_TIMED_UPDATES = 3
AUTOTUNE_TARGET_RESERVED_GB = 90.0
AUTOTUNE_VRAM_FRACTION = None  # set after GPU VRAM is known

# Re-benchmark the physical microbatch when resuming an older compatible
# checkpoint. Effective batch and optimizer state remain unchanged.
# True ensures partial runs created with the older heuristic tuner adopt the
# exhaustively measured partition on the next resume.
REAUTOTUNE_ON_RESUME = True

# Exact MLM optimization: positions with label -100 have zero loss, so the
# expensive 32k-vocabulary projection is evaluated only at masked positions.
MASKED_ONLY_MLM_HEAD = True


# ---------------------------------------------------------------------
# GPU / BLACKWELL SETTINGS
# ---------------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
AUTOTUNE_VRAM_FRACTION = min(0.96, AUTOTUNE_TARGET_RESERVED_GB / VRAM_GB)
AUTOTUNE_EFFECTIVE_RESERVED_GB = AUTOTUNE_VRAM_FRACTION * VRAM_GB

BF16 = bool(torch.cuda.is_bf16_supported())
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
AMP_NAME = "bfloat16" if BF16 else "float16"
USE_SCALER = not BF16

CPU_THREADS = min(8, os.cpu_count() or 8)
torch.set_num_threads(CPU_THREADS)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = True
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = True
except Exception:
    pass


try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

for fn in [
    "enable_flash_sdp",
    "enable_mem_efficient_sdp",
    "enable_math_sdp",
]:
    if hasattr(torch.backends.cuda, fn):
        try:
            getattr(torch.backends.cuda, fn)(True)
        except Exception:
            pass

# ---------------------------------------------------------------------
# PREPARE + TEST HUGGING FACE MAMBA BACKEND ON THE ACTUAL G4
# ---------------------------------------------------------------------

_HF_MIXER_CONFIG = None
MAMBA_KIND = None
MAMBA_BACKEND = None
MAMBA_BACKEND_FAILURES = {}


def _mamba_config(kind, num_layers=1):
    if kind == "mamba2":
        # Mamba-2 expands 768 -> 1536 internally. 24 heads x 64 = 1536.
        return HF_MAMBA2_CONFIG(
            vocab_size=VOCAB_SIZE,
            hidden_size=HIDDEN_SIZE,
            state_size=64,
            num_hidden_layers=int(num_layers),
            expand=2,
            conv_kernel=4,
            num_heads=(HIDDEN_SIZE * 2) // HEAD_DIM,
            head_dim=HEAD_DIM,
            n_groups=1,
            use_cache=False,
            residual_in_fp32=False,
            rms_norm=False,
            chunk_size=256,
            tie_word_embeddings=False,
        )

    if kind == "mamba":
        return HF_MAMBA_CONFIG(
            vocab_size=VOCAB_SIZE,
            hidden_size=HIDDEN_SIZE,
            state_size=16,
            num_hidden_layers=int(num_layers),
            expand=2,
            conv_kernel=4,
            use_cache=False,
            residual_in_fp32=False,
            use_mambapy=False,
            use_associative_scan=True,
            tie_word_embeddings=False,
        )

    raise ValueError(kind)


def _public_model_class(kind):
    if kind == "mamba2":
        return HF_MAMBA2_MODEL
    if kind == "mamba":
        return HF_MAMBA_MODEL
    raise ValueError(kind)


def _extract_mixers(model, kind):
    """Extract raw HF mixer modules from a public Mamba model, in layer order."""
    wanted = "mamba2" if kind == "mamba2" else "mamba"
    candidates = []
    for name, module in model.named_modules():
        if module is model:
            continue
        clsname = module.__class__.__name__.lower()
        if "mixer" in clsname and wanted in clsname:
            candidates.append((name, module))

    if not candidates:
        for name, module in model.named_modules():
            if module is model:
                continue
            clsname = module.__class__.__name__.lower()
            if "mamba" in clsname and "mixer" in clsname:
                candidates.append((name, module))

    if not candidates:
        raise RuntimeError(
            f"Could not locate raw {kind} mixer modules inside {model.__class__.__name__}."
        )

    return [module for _, module in candidates]


def _fresh_hf_mixer_stack(kind, num_layers):
    """
    Build through the stable public HF model API so HF performs the correct
    version-specific initialization, then keep only the raw mixers. The unused
    token embedding/final norm die with the temporary parent.
    """
    Model = _public_model_class(kind)
    if Model is None:
        raise RuntimeError(f"HF public model for {kind} is unavailable")
    config = _mamba_config(kind, num_layers=num_layers)
    temp = Model(config)
    mixers = _extract_mixers(temp, kind)
    if len(mixers) != int(num_layers):
        raise RuntimeError(
            f"HF {kind}: expected {num_layers} raw mixers, found {len(mixers)}"
        )
    return mixers, config


def _prepare_hf_kind(kind):
    """Verify public HF construction plus an actual CUDA mixer fwd/bwd."""
    global _HF_MIXER_CONFIG

    if kind == "mamba2" and HF_MAMBA2_MODEL is None:
        raise RuntimeError("HF Mamba-2 public API unavailable")
    if kind == "mamba" and HF_MAMBA_MODEL is None:
        raise RuntimeError("HF Mamba public API unavailable")

    mixers, config = _fresh_hf_mixer_stack(kind, 1)
    mixer = mixers[0]
    x = None
    y = None
    loss = None
    try:
        gc.collect()
        torch.cuda.empty_cache()
        mixer = mixer.to(DEVICE)
        mixer.train()
        x = torch.randn(
            2, 256, HIDDEN_SIZE,
            device=DEVICE,
            dtype=torch.float32,
            requires_grad=True,
        )
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            y = mixer(x)
            if not torch.is_tensor(y):
                if isinstance(y, (tuple, list)):
                    y = y[0]
                elif hasattr(y, "last_hidden_state"):
                    y = y.last_hidden_state
            if not torch.is_tensor(y):
                raise RuntimeError(f"{kind} mixer returned {type(y)}")
            if tuple(y.shape) != tuple(x.shape):
                raise RuntimeError(
                    f"{kind} shape violation: input {tuple(x.shape)} -> output {tuple(y.shape)}"
                )
            loss = y.float().square().mean()
        loss.backward()
        torch.cuda.synchronize()
        _HF_MIXER_CONFIG = config
        return True
    finally:
        mixer = x = y = loss = mixers = None
        gc.collect()
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

# Preferred architecture: Mamba-2.  Fall back to Mamba only after Mamba-2
# fails a REAL CUDA forward/backward test on this exact runtime.
for _candidate in ("mamba2", "mamba"):
    if _candidate == "mamba2" and HF_MAMBA2_MODEL is None:
        MAMBA_BACKEND_FAILURES[_candidate] = "public API unavailable"
        continue
    if _candidate == "mamba" and HF_MAMBA_MODEL is None:
        MAMBA_BACKEND_FAILURES[_candidate] = "public API unavailable"
        continue

    print(f"Testing Hugging Face {_candidate} native mixer on this GPU...")
    try:
        _prepare_hf_kind(_candidate)
        MAMBA_KIND = _candidate
        MAMBA_BACKEND = f"huggingface_native_{_candidate}"
        print(f"HF {_candidate} CUDA forward/backward: PASS")
        break
    except Exception as exc:
        MAMBA_BACKEND_FAILURES[_candidate] = repr(exc)
        print(f"HF {_candidate} failed; trying fallback if available:")
        print(" ", repr(exc))

if MAMBA_KIND is None:
    raise RuntimeError(
        "Neither Hugging Face Mamba-2 nor Hugging Face Mamba passed an actual "
        f"CUDA forward/backward test. Failures={MAMBA_BACKEND_FAILURES}"
    )

print("=" * 80)
print("GPU              :", GPU_NAME)
print("VRAM             :", f"{VRAM_GB:.2f} GB")
print("PyTorch          :", torch.__version__)
print("CUDA             :", torch.version.cuda)
print("AMP              :", AMP_NAME)
print("Protocol         :", PROTOCOL_VERSION)
print("Tokenizer metadata:", TOKENIZER_METADATA_STATUS if TOKENIZER_SHA256 is None else TOKENIZER_SHA256)
print("Runtime restart  : NEVER REQUIRED")
print("Transformers      :", HF_TRANSFORMERS_VERSION)
print("Mamba architecture:", MAMBA_KIND)
print("Mamba backend     :", MAMBA_BACKEND)
print("External Mamba CUDA packages: NONE")
print("=" * 80)

# ---------------------------------------------------------------------
# AGGRESSIVE MEMORY CLEANUP
# ---------------------------------------------------------------------

_RETNET_CACHE = {}

def _malloc_trim():
    try:
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except Exception:
        pass

def clear_architecture_caches():
    _RETNET_CACHE.clear()

    fast_cache = globals().get("_RETNET_FAST_CACHE")
    if isinstance(fast_cache, dict):
        fast_cache.clear()

def hard_cleanup(label=""):
    """
    Release references/caches without restarting the runtime.
    Safe between every architecture and every dataset.
    """
    clear_architecture_caches()

    try:
        torch.cuda.synchronize()
    except Exception:
        pass

    for _ in range(3):
        gc.collect()

    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

    _malloc_trim()

    try:
        torch.cuda.reset_peak_memory_stats()
    except Exception:
        pass

    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        reserv = torch.cuda.memory_reserved() / 1024**3
        print(
            f"[cleanup] {label} | "
            f"allocated={alloc:.3f} GB | reserved={reserv:.3f} GB"
        )

# ---- merged from former CELL 2 — Validate existing Drive tokens, stage datasets locally, deterministic MLM
# ---------------------------------------------------------------------
# DRIVE TOKEN SOURCES
# ---------------------------------------------------------------------

CORPORA = [
    {
        "name": "fineweb_edu",
        "mode": "shards",
        "train_dir": FINEWEB_ROOT / "train",
        "valid_dir": FINEWEB_ROOT / "validation",
    },
    {
        "name": "arxiv",
        "mode": "files",
        "train_file": ARXIV_ROOT / "train_blocks.npy",
        "valid_file": ARXIV_ROOT / "validation_blocks.npy",
    },
    {
        "name": "pg19",
        "mode": "files",
        "train_file": PG19_ROOT / "train_blocks.npy",
        "valid_file": PG19_ROOT / "validation_blocks.npy",
    },
    {
        "name": "wikitext103",
        "mode": "files",
        "train_file": WIKITEXT_ROOT / "train_blocks.npy",
        "valid_file": WIKITEXT_ROOT / "validation_blocks.npy",
    },
]

METHODS = [MAMBA_KIND, "hyena", "retnet"]

def _assert_array(path: Path, expected_rows: int):
    if not path.exists():
        raise FileNotFoundError(path)
    a = np.load(path, mmap_mode="r")
    try:
        if a.ndim != 2 or a.shape[1] != SEQ_LENGTH:
            raise RuntimeError(
                f"{path}: expected [N,{SEQ_LENGTH}], found {a.shape}"
            )
        if int(a.shape[0]) != int(expected_rows):
            raise RuntimeError(
                f"{path}: expected {expected_rows:,} rows, "
                f"found {a.shape[0]:,}"
            )
        if int(a.min()) < 0 or int(a.max()) >= VOCAB_SIZE:
            raise RuntimeError(
                f"{path}: token IDs outside [0,{VOCAB_SIZE-1}]"
            )
    finally:
        if hasattr(a, "_mmap") and a._mmap is not None:
            a._mmap.close()
        del a

def _validate_shards(paths, expected_rows, label):
    if not paths:
        raise FileNotFoundError(f"No shards found for {label}")

    rows = 0
    for p in paths:
        a = np.load(p, mmap_mode="r")
        try:
            if a.ndim != 2 or a.shape[1] != SEQ_LENGTH:
                raise RuntimeError(
                    f"{p}: expected [N,{SEQ_LENGTH}], found {a.shape}"
                )
            rows += int(a.shape[0])
        finally:
            if hasattr(a, "_mmap") and a._mmap is not None:
                a._mmap.close()
            del a

    if rows != expected_rows:
        raise RuntimeError(
            f"{label}: expected {expected_rows:,} rows, found {rows:,}"
        )

def validate_drive_tokens():
    print("\nValidating PRETOKENIZED Google Drive datasets...")

    for spec in CORPORA:
        name = spec["name"]

        if spec["mode"] == "files":
            _assert_array(spec["train_file"], EXPECTED_TRAIN_BLOCKS)
            _assert_array(spec["valid_file"], EXPECTED_VALID_BLOCKS)
            print(f"  {name:14s}: PASS")

        else:
            train_shards = sorted(spec["train_dir"].glob("shard_*.npy"))
            valid_shards = sorted(spec["valid_dir"].glob("shard_*.npy"))

            _validate_shards(
                train_shards,
                EXPECTED_TRAIN_BLOCKS,
                f"{name}/train",
            )
            _validate_shards(
                valid_shards,
                EXPECTED_VALID_BLOCKS,
                f"{name}/validation",
            )
            print(
                f"  {name:14s}: PASS "
                f"({len(train_shards)} train shards, "
                f"{len(valid_shards)} validation shards)"
            )

    print("No corpus will be downloaded or tokenized.")

# ---------------------------------------------------------------------
# LOCAL SSD STAGING
# ---------------------------------------------------------------------

def _copy_file_if_needed(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)

    if (
        dst.exists()
        and dst.stat().st_size == src.stat().st_size
    ):
        return

    tmp = dst.with_suffix(dst.suffix + ".tmp")
    tmp.unlink(missing_ok=True)
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

def _load_one_shard(path):
    # Full read is intentional: 48 FineWeb shards total only ~200 MB,
    # while this machine has ~177 GB system RAM. Parallel Drive reads
    # avoid the huge latency of opening/copying 48 shards serially.
    return np.load(
        path,
        allow_pickle=False,
    )

def _consolidate_shards_to_local(shards, dst: Path, expected_rows: int):
    """
    FAST FineWeb staging:
      Google Drive shards -> parallel RAM reads -> one local SSD .npy.

    No serial mmap-copy loop.
    """
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists():
        try:
            a = np.load(dst, mmap_mode="r")
            ok = (a.shape == (expected_rows, SEQ_LENGTH))
            if hasattr(a, "_mmap") and a._mmap is not None:
                a._mmap.close()
            del a
            if ok:
                print("  local cache already exists:", dst)
                return
        except Exception:
            pass
        dst.unlink(missing_ok=True)

    workers = min(8, len(shards))
    t0 = time.perf_counter()

    print(
        f"  parallel-reading {len(shards)} Drive shards "
        f"with {workers} workers..."
    )

    with ThreadPoolExecutor(max_workers=workers) as ex:
        arrays = list(ex.map(_load_one_shard, shards))

    rows = sum(int(a.shape[0]) for a in arrays)

    if rows != expected_rows:
        del arrays
        gc.collect()
        raise RuntimeError(
            f"Expected {expected_rows:,} rows, read {rows:,}"
        )

    merged = np.concatenate(arrays, axis=0)
    del arrays
    gc.collect()

    if merged.shape != (expected_rows, SEQ_LENGTH):
        raise RuntimeError(
            f"Unexpected merged shape: {merged.shape}"
        )

    tmp = dst.with_suffix(dst.suffix + ".tmp.npy")
    tmp.unlink(missing_ok=True)

    np.save(
        tmp,
        merged,
        allow_pickle=False,
    )

    del merged
    gc.collect()

    os.replace(tmp, dst)

    elapsed = time.perf_counter() - t0

    print(
        f"  FineWeb shard consolidation complete in "
        f"{elapsed:.1f} s"
    )

def stage_dataset(spec):
    """
    Pull existing tokens from Drive to local SSD ONCE.

    The local SSD cache is intentionally KEPT for the lifetime of the
    current Colab runtime. Re-running Cell 7 does not stage the dataset again.
    """
    name = spec["name"]
    local_dir = LOCAL_STAGE_ROOT / name
    local_dir.mkdir(parents=True, exist_ok=True)

    local_train = local_dir / "train_blocks.npy"
    local_valid = local_dir / "validation_blocks.npy"

    if (
        local_train.exists()
        and local_valid.exists()
    ):
        try:
            _assert_array(local_train, EXPECTED_TRAIN_BLOCKS)
            _assert_array(local_valid, EXPECTED_VALID_BLOCKS)
            print(f"\n{name}: using existing LOCAL SSD cache.")
            return local_train, local_valid, local_dir
        except Exception:
            local_train.unlink(missing_ok=True)
            local_valid.unlink(missing_ok=True)

    print(f"\nStaging {name}: Drive -> local SSD...")

    if spec["mode"] == "files":
        t0 = time.perf_counter()

        # One sequential Drive file per split; no extra metadata copy.
        _copy_file_if_needed(spec["train_file"], local_train)
        _copy_file_if_needed(spec["valid_file"], local_valid)

        print(
            f"  single-file staging complete in "
            f"{time.perf_counter() - t0:.1f} s"
        )

    else:
        train_shards = sorted(spec["train_dir"].glob("shard_*.npy"))
        valid_shards = sorted(spec["valid_dir"].glob("shard_*.npy"))

        _consolidate_shards_to_local(
            train_shards,
            local_train,
            EXPECTED_TRAIN_BLOCKS,
        )
        _consolidate_shards_to_local(
            valid_shards,
            local_valid,
            EXPECTED_VALID_BLOCKS,
        )

    _assert_array(local_train, EXPECTED_TRAIN_BLOCKS)
    _assert_array(local_valid, EXPECTED_VALID_BLOCKS)

    print("  local train:", local_train)
    print("  local valid:", local_valid)

    return local_train, local_valid, local_dir


# ---------------------------------------------------------------------
# DETERMINISTIC MLM — SAME LOGIC AS THE G4 TRANSWAVE NOTEBOOKS
# ---------------------------------------------------------------------

def make_mlm_batch(clean_np, seed_index, validation=False):
    clean = torch.from_numpy(
        np.asarray(clean_np, dtype=np.int64).copy()
    )

    ids = clean.clone()
    labels = torch.full_like(ids, -100)

    seed = MASK_SEED + int(seed_index)
    if validation:
        seed += 1_000_000_000

    g = torch.Generator(device="cpu")
    g.manual_seed(seed)

    eligible = torch.ones_like(ids, dtype=torch.bool)
    for sid in SPECIAL_IDS:
        eligible &= ids.ne(sid)

    scores = torch.rand(ids.shape, generator=g)
    selected = eligible & scores.lt(MLM_PROBABILITY)

    ranked_scores = scores.masked_fill(
        ~selected,
        float("inf"),
    )

    keep_index = torch.topk(
        ranked_scores,
        k=MAX_PREDICTIONS,
        dim=1,
        largest=False,
        sorted=False,
    ).indices

    keep = torch.zeros_like(selected)
    keep.scatter_(1, keep_index, True)
    selected &= keep

    labels[selected] = ids[selected]

    replacement = torch.rand(
        ids.shape,
        generator=g,
    )

    mask_positions = (
        selected
        &
        replacement.lt(0.80)
    )

    random_positions = (
        selected
        &
        replacement.ge(0.80)
        &
        replacement.lt(0.90)
    )

    ids[mask_positions] = MASK_ID

    random_ids = torch.randint(
        7,
        VOCAB_SIZE,
        ids.shape,
        generator=g,
        dtype=torch.long,
    )

    ids[random_positions] = random_ids[random_positions]

    return ids.pin_memory(), labels.pin_memory()

# ---- merged from former CELL 3 — FAST mixers + stacked bidirectionality + masked-only MLM head
# ---------------------------------------------------------------------
# SHARED FNET-LIKE NON-MIXER SCAFFOLD
# ---------------------------------------------------------------------

def gelu_new(x):
    return (
        0.5
        * x
        * (
            1.0
            + torch.tanh(
                math.sqrt(2.0 / math.pi)
                * (
                    x
                    + 0.044715 * x.pow(3)
                )
            )
        )
    )

class CommonEmbeddings(nn.Module):
    def __init__(self):
        super().__init__()

        self.word_embeddings = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE,
            padding_idx=PAD_ID,
        )

        self.position_embeddings = nn.Embedding(
            SEQ_LENGTH,
            HIDDEN_SIZE,
        )

        self.token_type_embeddings = nn.Embedding(
            TYPE_VOCAB_SIZE,
            HIDDEN_SIZE,
        )

        self.LayerNorm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

        self.dropout = nn.Dropout(
            HIDDEN_DROPOUT
        )

        self.register_buffer(
            "position_ids",
            torch.arange(
                SEQ_LENGTH
            ).unsqueeze(0),
            persistent=False,
        )

    def forward(self, input_ids):
        B, L = input_ids.shape

        pos = self.position_ids[:, :L]

        token_type = torch.zeros(
            (B, L),
            dtype=torch.long,
            device=input_ids.device,
        )

        x = (
            self.word_embeddings(input_ids)
            + self.position_embeddings(pos)
            + self.token_type_embeddings(token_type)
        )

        x = self.LayerNorm(x)
        return self.dropout(x)

class CommonFFN(nn.Module):
    def __init__(self):
        super().__init__()

        self.dense_in = nn.Linear(
            HIDDEN_SIZE,
            INTERMEDIATE_SIZE,
        )

        self.dense_out = nn.Linear(
            INTERMEDIATE_SIZE,
            HIDDEN_SIZE,
        )

        self.dropout = nn.Dropout(
            HIDDEN_DROPOUT
        )

        self.LayerNorm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

    def forward(self, x):
        y = self.dense_in(x)
        y = gelu_new(y)
        y = self.dense_out(y)
        y = self.dropout(y)

        return self.LayerNorm(
            x + y
        )

class BidirectionalSharedMixer(nn.Module):
    """
    Shared-weight bidirectional adaptation for MLM.

    FAST: forward and reversed sequences are concatenated on the batch axis,
    sent through the mixer in ONE call, then split/realigned. This preserves
    the same shared-parameter operator while removing one full sequence of
    mixer kernel launches per layer and presenting twice as much parallel
    work to the GPU.
    """
    def __init__(self, mixer):
        super().__init__()
        self.mixer = mixer

    def forward(self, x):
        B = x.shape[0]

        both = torch.cat(
            (x, torch.flip(x, dims=(1,))),
            dim=0,
        )

        mixed = self.mixer(both)

        forward_y = mixed[:B]
        reverse_y = torch.flip(
            mixed[B:],
            dims=(1,),
        )

        return (forward_y + reverse_y) * 0.5

# ---------------------------------------------------------------------
# HUGGING FACE NATIVE MAMBA-2 / MAMBA MIXER
# ---------------------------------------------------------------------

class HFMambaMixer(nn.Module):
    def __init__(self, kind, raw_mixer):
        super().__init__()
        if kind != MAMBA_KIND:
            raise RuntimeError(
                f"Requested {kind}, but Cell 1 selected {MAMBA_KIND}."
            )
        self.kind = kind
        self.block = raw_mixer

    def forward(self, x):
        y = self.block(x)
        if not torch.is_tensor(y):
            if isinstance(y, (tuple, list)):
                y = y[0]
            elif hasattr(y, "last_hidden_state"):
                y = y.last_hidden_state
        if not torch.is_tensor(y):
            raise RuntimeError(f"{self.kind} returned unexpected output {type(y)}")
        if y.shape != x.shape:
            raise RuntimeError(
                f"{self.kind} shape violation: input {tuple(x.shape)} -> output {tuple(y.shape)}"
            )
        return y

# ---------------------------------------------------------------------
# HYENA
# ---------------------------------------------------------------------

class SinActivation(nn.Module):
    def __init__(self, width, w=1.0):
        super().__init__()
        self.freq = nn.Parameter(
            torch.full(
                (1, width),
                float(w),
            )
        )

    def forward(self, x):
        return torch.sin(
            self.freq * x
        )

class HyenaImplicitFilter(nn.Module):
    def __init__(
        self,
        channels,
        l_max=SEQ_LENGTH,
        width=64,
    ):
        super().__init__()

        t = torch.linspace(
            0.0,
            1.0,
            l_max,
        ).view(1, l_max, 1)

        idx = torch.arange(
            l_max,
            dtype=torch.float32,
        ).view(1, l_max, 1)

        z = torch.cat(
            [
                t,
                torch.cos(
                    2.0
                    * math.pi
                    * idx
                    / l_max
                ),
                torch.sin(
                    2.0
                    * math.pi
                    * idx
                    / l_max
                ),
            ],
            dim=-1,
        )

        self.register_buffer(
            "z",
            z,
            persistent=False,
        )

        self.net = nn.Sequential(
            nn.Linear(3, width),
            SinActivation(width),
            nn.Linear(width, width),
            SinActivation(width),
            nn.Linear(
                width,
                channels,
                bias=False,
            ),
        )

        self.bias = nn.Parameter(
            torch.randn(channels)
            * 0.02
        )

        decay = torch.linspace(
            math.log(1.0e-2) / 1.5,
            math.log(1.0e-2) / 0.3,
            channels,
        ).abs().view(
            1,
            1,
            channels,
        )

        self.register_buffer(
            "decay",
            decay,
            persistent=False,
        )

    def forward(
        self,
        length,
        dtype,
        device,
    ):
        z = self.z[
            :,
            :length,
        ].to(
            device=device
        )

        h = self.net(z)

        t = z[..., :1]

        h = (
            h
            *
            torch.exp(
                -t
                *
                self.decay.to(
                    device=device
                )
            )
        )

        return h.to(
            dtype=dtype
        )

def fft_long_conv(
    u,
    k,
    diagonal,
):
    """
    u: [B,C,L]
    k: [C,L]
    """
    L = u.shape[-1]
    fft_len = 2 * L

    # Exactly as with the FNet experiments: FFT itself stays FP32.
    with torch.autocast(
        device_type="cuda",
        enabled=False,
    ):
        u_fft = torch.fft.rfft(
            u.float(),
            n=fft_len,
        )

        k_fft = torch.fft.rfft(
            k.float(),
            n=fft_len,
        )

        y = torch.fft.irfft(
            u_fft
            *
            k_fft.unsqueeze(0),
            n=fft_len,
        )[..., :L]

        y = (
            y
            +
            u.float()
            *
            diagonal.float().view(
                1,
                -1,
                1,
            )
        )

    return y.to(
        dtype=u.dtype
    )

class HyenaMixer(nn.Module):
    def __init__(self, order=2):
        super().__init__()

        self.order = int(order)

        total = (
            self.order + 1
        ) * HIDDEN_SIZE

        self.in_proj = nn.Linear(
            HIDDEN_SIZE,
            total,
        )

        # padding=2 with kernel 3, followed by [:L], implements the
        # causal short filter used by the forward branch.
        self.short = nn.Conv1d(
            total,
            total,
            kernel_size=3,
            padding=2,
            groups=total,
            bias=True,
        )

        self.filters = nn.ModuleList(
            [
                HyenaImplicitFilter(
                    HIDDEN_SIZE,
                    SEQ_LENGTH,
                    width=64,
                )
                for _ in range(
                    self.order - 1
                )
            ]
        )

        self.out_proj = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE,
        )

    def forward(self, x):
        _, L, _ = x.shape

        u = self.in_proj(
            x
        ).transpose(
            1,
            2,
        )

        u = self.short(
            u
        )[..., :L]

        chunks = u.chunk(
            self.order + 1,
            dim=1,
        )

        xs = list(
            chunks[:-1]
        )

        v = chunks[-1]

        for i, xi in enumerate(
            reversed(
                xs[1:]
            )
        ):
            v = v * xi

            k = self.filters[i](
                L,
                v.dtype,
                v.device,
            )[0].transpose(
                0,
                1,
            )

            v = fft_long_conv(
                v,
                k,
                self.filters[i].bias,
            )

        y = (
            v
            *
            xs[0]
        ).transpose(
            1,
            2,
        )

        return self.out_proj(
            y
        )

# ---------------------------------------------------------------------
# RETNET — EXACT CHUNKWISE-RECURRENT MULTISCALE RETENTION
#
# No [B,H,512,512] retention tensor.  At L=512, chunk=64 this evaluates
# eight 64x64 local blocks plus a recurrent [B,H,D,D] cross-chunk state.
# It is algebraically the same causal exponential-retention operator up to
# normal floating-point roundoff.
# ---------------------------------------------------------------------

_RETNET_FAST_CACHE = {}

def _retnet_fast_constants(length, chunk, dtype, device):
    key = (
        int(length),
        int(chunk),
        dtype,
        device.type,
        device.index,
    )

    if key in _RETNET_FAST_CACHE:
        return _RETNET_FAST_CACHE[key]

    if length % chunk != 0:
        raise ValueError(
            f"length={length} must be divisible by chunk={chunk}"
        )

    log_decay = torch.log(
        1.0
        -
        2.0 ** (
            -5.0
            -
            torch.arange(
                NUM_HEADS,
                device=device,
                dtype=torch.float32,
            )
        )
    )
    gamma = torch.exp(log_decay)

    freq = 1.0 / (
        10000.0 ** torch.linspace(
            0.0,
            1.0,
            HEAD_DIM // 2,
            device=device,
            dtype=torch.float32,
        )
    )
    pos = torch.arange(
        length,
        device=device,
        dtype=torch.float32,
    )
    phase = pos[:, None] * freq[None, :]
    cos = phase.cos().to(dtype)
    sin = phase.sin().to(dtype)

    r = torch.arange(
        chunk,
        device=device,
        dtype=torch.long,
    )
    delta = r[:, None] - r[None, :]
    lower = delta >= 0

    within = (
        gamma[:, None, None]
        ** delta.clamp_min(0).float()[None, :, :]
    )
    within = within.masked_fill(
        ~lower[None, :, :],
        0.0,
    ).to(dtype)

    cross = (
        gamma[:, None]
        ** (r.float()[None, :] + 1.0)
    ).to(dtype)

    tail = (
        gamma[:, None]
        ** (
            float(chunk - 1)
            -
            r.float()[None, :]
        )
    ).to(dtype)

    chunk_decay = (
        gamma ** float(chunk)
    ).to(dtype)

    _RETNET_FAST_CACHE[key] = (
        cos,
        sin,
        within,
        cross,
        tail,
        chunk_decay,
    )

    return _RETNET_FAST_CACHE[key]


def _rope_half_fast(x, cos, sin):
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]

    c = cos[None, None, :, :]
    s = sin[None, None, :, :]

    return torch.stack(
        (
            x1 * c - x2 * s,
            x1 * s + x2 * c,
        ),
        dim=-1,
    ).flatten(-2)


class MultiScaleRetention(nn.Module):
    def __init__(self):
        super().__init__()

        # Four projections in one GEMM; same parameter count.
        self.qkvg = nn.Linear(
            HIDDEN_SIZE,
            4 * HIDDEN_SIZE,
            bias=False,
        )

        self.out = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE,
            bias=False,
        )

        self.group_norm = nn.GroupNorm(
            NUM_HEADS,
            HIDDEN_SIZE,
        )

    def forward(self, x):
        B, L, _ = x.shape
        C = RETENTION_CHUNK

        qkvg = self.qkvg(x).view(
            B,
            L,
            4,
            NUM_HEADS,
            HEAD_DIM,
        )

        q = qkvg[:, :, 0].transpose(1, 2)
        k = qkvg[:, :, 1].transpose(1, 2)
        v = qkvg[:, :, 2].transpose(1, 2)

        gate = F.silu(
            qkvg[:, :, 3].reshape(
                B,
                L,
                HIDDEN_SIZE,
            )
        )

        (
            cos,
            sin,
            within_decay,
            cross_decay,
            tail_decay,
            chunk_decay,
        ) = _retnet_fast_constants(
            L,
            C,
            q.dtype,
            q.device,
        )

        q = _rope_half_fast(q, cos, sin)
        k = _rope_half_fast(k, cos, sin)

        N = L // C

        qch = q.view(
            B, NUM_HEADS, N, C, HEAD_DIM
        )
        kch = k.view(
            B, NUM_HEADS, N, C, HEAD_DIM
        )
        vch = v.view(
            B, NUM_HEADS, N, C, HEAD_DIM
        )

        local_score = torch.matmul(
            qch,
            kch.transpose(-1, -2),
        )

        local_score = (
            local_score
            *
            within_decay[
                None, :, None, :, :
            ]
            /
            math.sqrt(HEAD_DIM)
        )

        y = torch.matmul(
            local_score,
            vch,
        )
        del local_score

        state = torch.zeros(
            (
                B,
                NUM_HEADS,
                HEAD_DIM,
                HEAD_DIM,
            ),
            dtype=q.dtype,
            device=q.device,
        )

        inv_sqrt_d = 1.0 / math.sqrt(HEAD_DIM)

        # Only 8 iterations at 512/64; no token-by-token Python recurrence.
        for n in range(N):
            cross_y = torch.matmul(
                qch[:, :, n],
                state,
            )

            y[:, :, n].add_(
                cross_y
                *
                cross_decay[
                    None, :, :, None
                ]
                *
                inv_sqrt_d
            )

            chunk_state = torch.einsum(
                "bhcd,bhce,hc->bhde",
                kch[:, :, n],
                vch[:, :, n],
                tail_decay,
            )

            state = (
                state
                *
                chunk_decay[
                    None, :, None, None
                ]
                +
                chunk_state
            )

        y = y.reshape(
            B,
            NUM_HEADS,
            L,
            HEAD_DIM,
        )

        y = (
            y
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                L,
                HIDDEN_SIZE,
            )
        )

        y = (
            self.group_norm(
                y.transpose(1, 2)
            )
            .transpose(1, 2)
        )

        return self.out(
            y * gate
        )


class RetNetMixer(nn.Module):
    def __init__(self):
        super().__init__()
        self.retention = MultiScaleRetention()

    def forward(self, x):
        return self.retention(x)

# ---------------------------------------------------------------------
# COMMON ENCODER LAYER + MLM HEAD
# ---------------------------------------------------------------------

class MixerLayer(nn.Module):
    def __init__(
        self,
        kind,
        layer_idx=0,
        mamba_raw_mixer=None,
    ):
        super().__init__()

        if kind in ("mamba2", "mamba"):
            if mamba_raw_mixer is None:
                raise RuntimeError("Mamba layer missing HF raw mixer")
            raw_mixer = HFMambaMixer(kind, mamba_raw_mixer)

        elif kind == "hyena":
            raw_mixer = HyenaMixer(
                order=2
            )

        elif kind == "retnet":
            raw_mixer = RetNetMixer()

        else:
            raise ValueError(kind)

        self.mixer = BidirectionalSharedMixer(
            raw_mixer
        )

        self.mixer_norm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

        self.ffn = CommonFFN()

    def forward(self, x):
        mixed = self.mixer(
            x
        )

        x = self.mixer_norm(
            x
            +
            mixed
        )

        return self.ffn(
            x
        )

class MLMHead(nn.Module):
    def __init__(
        self,
        embeddings,
    ):
        super().__init__()

        self.dense = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE,
        )

        self.LayerNorm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

        self.bias = nn.Parameter(
            torch.zeros(VOCAB_SIZE)
        )

        self.embeddings = embeddings

    def forward(self, x, labels=None):
        # Exact optimization for ignore-index MLM: unmasked positions have
        # zero loss, so do the costly 32k-vocabulary projection only for the
        # positions that actually contribute to the objective.
        if labels is not None and MASKED_ONLY_MLM_HEAD:
            selected = labels.ne(-100)
            x = x[selected]
            targets = labels[selected]
        else:
            targets = labels

        x = self.dense(x)
        x = gelu_new(x)
        x = self.LayerNorm(x)

        logits = F.linear(
            x,
            self.embeddings.weight,
            self.bias,
        )

        return logits, targets

class ModernMixerMLM(nn.Module):
    def __init__(
        self,
        kind,
    ):
        super().__init__()

        self.kind = kind

        self.embeddings = (
            CommonEmbeddings()
        )

        if kind in ("mamba2", "mamba"):
            mamba_mixers, _ = _fresh_hf_mixer_stack(kind, NUM_LAYERS)
        else:
            mamba_mixers = [None] * NUM_LAYERS

        self.layers = nn.ModuleList(
            [
                MixerLayer(
                    kind,
                    layer_idx=idx,
                    mamba_raw_mixer=mamba_mixers[idx],
                )
                for idx in range(
                    NUM_LAYERS
                )
            ]
        )

        self.mlm = MLMHead(
            self.embeddings.word_embeddings
        )

        self._reset_common_parameters()

    @staticmethod
    def _normal_tensor(
        tensor,
        seed,
        std=INITIALIZER_RANGE,
    ):
        g = torch.Generator(
            device="cpu"
        )
        g.manual_seed(
            int(seed)
        )

        values = torch.randn(
            tensor.shape,
            generator=g,
            dtype=torch.float32,
        ) * std

        with torch.no_grad():
            tensor.copy_(
                values.to(
                    dtype=tensor.dtype
                )
            )

    @staticmethod
    def _zero_bias(module):
        if (
            hasattr(module, "bias")
            and module.bias is not None
        ):
            nn.init.zeros_(
                module.bias
            )

    def _reset_common_parameters(self):
        """
        Force ALL non-mixer weights to be identical across the selected Mamba baseline,
        Hyena and RetNet regardless of how many random numbers their
        mixer constructors consume.
        """
        # embeddings
        self._normal_tensor(
            self.embeddings.word_embeddings.weight,
            10001,
        )
        with torch.no_grad():
            self.embeddings.word_embeddings.weight[
                PAD_ID
            ].zero_()

        self._normal_tensor(
            self.embeddings.position_embeddings.weight,
            10002,
        )

        self._normal_tensor(
            self.embeddings.token_type_embeddings.weight,
            10003,
        )

        nn.init.ones_(
            self.embeddings.LayerNorm.weight
        )
        nn.init.zeros_(
            self.embeddings.LayerNorm.bias
        )

        # layers
        for idx, layer in enumerate(
            self.layers
        ):
            base = (
                20000
                +
                idx * 100
            )

            nn.init.ones_(
                layer.mixer_norm.weight
            )
            nn.init.zeros_(
                layer.mixer_norm.bias
            )

            self._normal_tensor(
                layer.ffn.dense_in.weight,
                base + 1,
            )
            self._zero_bias(
                layer.ffn.dense_in
            )

            self._normal_tensor(
                layer.ffn.dense_out.weight,
                base + 2,
            )
            self._zero_bias(
                layer.ffn.dense_out
            )

            nn.init.ones_(
                layer.ffn.LayerNorm.weight
            )
            nn.init.zeros_(
                layer.ffn.LayerNorm.bias
            )

        # MLM transform; decoder is tied to embeddings.
        self._normal_tensor(
            self.mlm.dense.weight,
            30001,
        )
        self._zero_bias(
            self.mlm.dense
        )

        nn.init.ones_(
            self.mlm.LayerNorm.weight
        )
        nn.init.zeros_(
            self.mlm.LayerNorm.bias
        )

        nn.init.zeros_(
            self.mlm.bias
        )

    def forward(
        self,
        input_ids,
        labels=None,
    ):
        x = self.embeddings(
            input_ids
        )

        for layer in self.layers:
            x = layer(x)

        return self.mlm(
            x,
            labels,
        )

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

def upgrade_model_state_dict_for_fast_impl(kind, state_dict):
    if kind != "retnet":
        return state_dict

    sd = dict(state_dict)

    for i in range(NUM_LAYERS):
        prefix = (
            f"layers.{i}."
            f"mixer.mixer.retention."
        )

        old_keys = [
            prefix + "q.weight",
            prefix + "k.weight",
            prefix + "v.weight",
            prefix + "g.weight",
        ]

        new_key = prefix + "qkvg.weight"

        if (
            new_key not in sd
            and all(k in sd for k in old_keys)
        ):
            sd[new_key] = torch.cat(
                [sd[k] for k in old_keys],
                dim=0,
            )

            for k in old_keys:
                del sd[k]

    return sd


def build_model(kind):
    set_all_seeds(
        GLOBAL_SEED
    )

    model = ModernMixerMLM(
        kind
    )

    return model

def parameter_counts(model):
    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return int(total), int(trainable)

# ---- merged from former CELL 4 — Optimizer, scheduler, full-update microbatch autotuning, persistent checkpoints
# ---------------------------------------------------------------------
# OPTIMIZER / SCHEDULER / SCALER — SAME AS G4 TRANSWAVE NOTEBOOKS
# ---------------------------------------------------------------------

def make_optimizer(model):
    kwargs = dict(
        lr=LEARNING_RATE,
        betas=ADAM_BETAS,
        eps=ADAM_EPS,
        weight_decay=WEIGHT_DECAY,
    )

    try:
        return (
            torch.optim.Adam(
                model.parameters(),
                fused=True,
                **kwargs,
            ),
            True,
        )

    except Exception:
        return (
            torch.optim.Adam(
                model.parameters(),
                **kwargs,
            ),
            False,
        )

def make_scheduler(
    optimizer,
    total_updates,
):
    warmup = max(
        1,
        round(
            0.01
            *
            total_updates
        ),
    )

    def lr_lambda(step):
        if step < warmup:
            return (
                float(step + 1)
                /
                float(warmup)
            )

        return 1.0

    return (
        torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda,
        ),
        warmup,
    )

def make_scaler():
    return torch.amp.GradScaler(
        "cuda",
        enabled=USE_SCALER,
    )

# ---------------------------------------------------------------------
# AUTOTUNE PHYSICAL MICRO-BATCH BY REAL FULL-UPDATE THROUGHPUT
# ---------------------------------------------------------------------

def _benchmark_candidate(
    kind,
    micro_batch,
    effective_batch,
):
    grad_accum = (
        effective_batch
        //
        micro_batch
    )

    hard_cleanup(
        f"before probe {kind} mb={micro_batch}"
    )

    model = None
    optimizer = None
    scaler = None
    ids = None
    labels = None

    try:
        model = build_model(
            kind
        ).to(
            DEVICE
        )

        model.train()

        optimizer, _ = (
            make_optimizer(
                model
            )
        )

        scaler = make_scaler()

        ids = torch.randint(
            7,
            VOCAB_SIZE,
            (
                micro_batch,
                SEQ_LENGTH,
            ),
            device=DEVICE,
            dtype=torch.long,
        )

        labels = torch.full_like(
            ids,
            -100,
        )

        labels[
            :,
            1:81
        ] = ids[
            :,
            1:81
        ]

        def one_update():
            optimizer.zero_grad(
                set_to_none=True
            )

            for _ in range(
                grad_accum
            ):
                with torch.autocast(
                    "cuda",
                    dtype=AMP_DTYPE,
                ):
                    logits, targets = model(
                        ids,
                        labels,
                    )

                    loss = F.cross_entropy(
                        logits,
                        targets,
                    )

                    loss = (
                        loss
                        /
                        grad_accum
                    )

                scaler.scale(
                    loss
                ).backward()

                del logits, targets, loss

            scaler.step(
                optimizer
            )

            scaler.update()

        torch.cuda.reset_peak_memory_stats()

        # Warmup includes optimizer-state allocation + fused/Triton/cuDNN kernel
        # initialization.  Throughput is then measured over multiple complete
        # effective-batch updates to reduce one-update timing noise.
        for _ in range(AUTOTUNE_WARMUP_UPDATES):
            one_update()
        torch.cuda.synchronize()

        t0 = time.perf_counter()

        for _ in range(AUTOTUNE_TIMED_UPDATES):
            one_update()

        torch.cuda.synchronize()

        elapsed = (
            time.perf_counter()
            -
            t0
        )

        peak = (
            torch.cuda.max_memory_allocated()
            /
            1024**3
        )
        peak_reserved = (
            torch.cuda.max_memory_reserved()
            /
            1024**3
        )

        tokens_per_second = (
            AUTOTUNE_TIMED_UPDATES
            *
            effective_batch
            *
            SEQ_LENGTH
            /
            elapsed
        )

        # Use nearly all of the G4 while preserving a small workspace margin.
        # Reserved memory is the safer quantity for CUDA OOM protection.
        safe = (
            peak_reserved
            <=
            AUTOTUNE_VRAM_FRACTION
            *
            VRAM_GB
        )

        return {
            "micro_batch":
                int(micro_batch),

            "grad_accum":
                int(grad_accum),

            "tokens_per_second":
                float(tokens_per_second),

            "peak_allocated_gb":
                float(peak),

            "peak_reserved_gb":
                float(peak_reserved),

            "safe":
                bool(safe),
        }

    except (
        torch.cuda.OutOfMemoryError,
        RuntimeError,
    ) as exc:
        msg = str(exc).lower()

        if (
            not isinstance(
                exc,
                torch.cuda.OutOfMemoryError,
            )
            and
            "out of memory" not in msg
        ):
            raise

        return {
            "micro_batch":
                int(micro_batch),

            "grad_accum":
                int(grad_accum),

            "tokens_per_second":
                0.0,

            "peak_allocated_gb":
                float("inf"),

            "peak_reserved_gb":
                float("inf"),

            "safe":
                False,
        }

    finally:
        ids = None
        labels = None
        scaler = None
        optimizer = None
        model = None

        hard_cleanup(
            f"after probe {kind} mb={micro_batch}"
        )

def autotune_microbatch(
    kind,
    effective_batch,
):
    gpu_tag = re.sub(r"[^A-Za-z0-9]+", "_", GPU_NAME).strip("_")
    cache_path = (
        CONFIG_ROOT
        / f"autotune_{PROTOCOL_VERSION}_{AUTOTUNE_VERSION}_{gpu_tag}_{kind}_{MAMBA_BACKEND if kind in ('mamba2','mamba') else 'native'}_eb{effective_batch}.json"
    )

    if cache_path.exists():
        try:
            cached = json.loads(cache_path.read_text())
            if (
                cached.get("protocol_version") == PROTOCOL_VERSION
                and cached.get("autotune_version") == AUTOTUNE_VERSION
                and cached.get("gpu") == GPU_NAME
                and int(cached.get("effective_batch")) == int(effective_batch)
                and cached.get("kind") == kind
            ):
                print(
                    f"Reusing persistent EXHAUSTIVE G4 autotune: {kind} | "
                    f"micro={cached['micro_batch']} x {cached['grad_accum']} | "
                    f"{cached['tokens_per_second']:,.0f} tok/s"
                )
                return cached
        except Exception:
            pass

    # Benchmark EVERY divisor of the effective batch.  For the protocol's
    # effective batches (64 and 128), this is 64..1 / 128..1 respectively.
    # No early stopping: the fastest safe candidate wins by measurement.
    candidates = [
        b
        for b in range(effective_batch, 0, -1)
        if effective_batch % b == 0
    ]

    results = []

    print(
        f"\nEXHAUSTIVE autotuning {kind} on "
        f"{GPU_NAME} / {VRAM_GB:.2f} GB..."
    )
    print(
        f"  candidates={candidates} | "
        f"warmup={AUTOTUNE_WARMUP_UPDATES} effective update(s) | "
        f"timed={AUTOTUNE_TIMED_UPDATES} effective update(s) each | "
        f"VRAM ceiling={AUTOTUNE_VRAM_FRACTION * VRAM_GB:.2f} GiB reserved ({100.0 * AUTOTUNE_VRAM_FRACTION:.1f}%)"
    )

    for mb in candidates:
        r = _benchmark_candidate(
            kind,
            mb,
            effective_batch,
        )

        results.append(r)

        if r["safe"]:
            print(
                f"  mb={mb:3d} x {r['grad_accum']:3d} | PASS | "
                f"{r['tokens_per_second']:,.0f} tok/s | "
                f"alloc {r['peak_allocated_gb']:.2f} / "
                f"reserved {r['peak_reserved_gb']:.2f} GB"
            )
        else:
            print(
                f"  mb={mb:3d} x {r['grad_accum']:3d} | REJECT/OOM"
            )

    safe = [r for r in results if r["safe"]]

    if not safe:
        raise RuntimeError(
            f"No safe physical batch found for {kind}."
        )

    best = max(
        safe,
        key=lambda r: r["tokens_per_second"],
    )

    print(
        f"Chosen {kind}: "
        f"microbatch={best['micro_batch']}, "
        f"accum={best['grad_accum']}, "
        f"measured={best['tokens_per_second']:,.0f} tok/s"
    )

    best = dict(best)
    best.update({
        "protocol_version": PROTOCOL_VERSION,
        "autotune_version": AUTOTUNE_VERSION,
        "gpu": GPU_NAME,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "kind": kind,
        "mamba_backend": MAMBA_BACKEND if kind in ("mamba2", "mamba") else None,
        "transformers": HF_TRANSFORMERS_VERSION,
        "effective_batch": int(effective_batch),
        "warmup_updates": int(AUTOTUNE_WARMUP_UPDATES),
        "timed_updates": int(AUTOTUNE_TIMED_UPDATES),
        "vram_fraction": float(AUTOTUNE_VRAM_FRACTION),
        "all_candidates": results,
    })
    cache_path.write_text(json.dumps(best, indent=2))
    return best

# ---------------------------------------------------------------------
# PERSISTENT DRIVE CHECKPOINT HELPERS
# ---------------------------------------------------------------------

def atomic_torch_save(
    payload,
    destination,
):
    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    local_tmp = (
        LOCAL_TMP_ROOT
        /
        (
            destination.name
            +
            ".tmp"
        )
    )

    drive_tmp = (
        destination.parent
        /
        (
            destination.name
            +
            ".tmp"
        )
    )

    local_tmp.unlink(
        missing_ok=True
    )

    drive_tmp.unlink(
        missing_ok=True
    )

    torch.save(
        payload,
        local_tmp,
    )

    # Check locally before copying to Drive.
    check = torch.load(
        local_tmp,
        map_location="cpu",
        weights_only=False,
    )
    del check

    shutil.copy2(
        local_tmp,
        drive_tmp,
    )

    os.replace(
        drive_tmp,
        destination,
    )

    local_tmp.unlink(
        missing_ok=True
    )

def capture_rng():
    return {
        "python":
            random.getstate(),

        "numpy":
            np.random.get_state(),

        "torch":
            torch.get_rng_state(),

        "cuda":
            torch.cuda.get_rng_state_all(),
    }

def restore_rng(state):
    random.setstate(
        state["python"]
    )

    np.random.set_state(
        state["numpy"]
    )

    torch.set_rng_state(
        state["torch"]
    )

    torch.cuda.set_rng_state_all(
        state["cuda"]
    )

def optimizer_to_gpu(
    optimizer,
):
    for state in optimizer.state.values():
        for k, v in list(
            state.items()
        ):
            if torch.is_tensor(
                v
            ):
                state[k] = v.to(
                    DEVICE,
                    non_blocking=True,
                )

# ---- merged from former CELL 5 — Optimized resumable training + full validation
# ---------------------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------------------

def evaluate(
    model,
    valid,
    micro_batch,
    effective_batch,
):
    model.eval()

    weighted_loss = 0.0
    target_count = 0

    n_effective = math.ceil(
        len(valid)
        /
        effective_batch
    )

    with torch.inference_mode():
        for e in range(
            n_effective
        ):
            start = (
                e
                *
                effective_batch
            )

            clean = valid[
                start:
                min(
                    start
                    +
                    effective_batch,
                    len(valid),
                )
            ]

            ids_cpu, labels_cpu = (
                make_mlm_batch(
                    clean,
                    e,
                    validation=True,
                )
            )

            for mstart in range(
                0,
                len(ids_cpu),
                micro_batch,
            ):
                ids_slice = ids_cpu[
                    mstart:
                    mstart
                    +
                    micro_batch
                ]

                labels_slice = labels_cpu[
                    mstart:
                    mstart
                    +
                    micro_batch
                ]

                n_targets = int(
                    labels_slice
                    .ne(-100)
                    .sum()
                    .item()
                )

                ids = ids_slice.to(
                    DEVICE,
                    non_blocking=True,
                )

                labels = labels_slice.to(
                    DEVICE,
                    non_blocking=True,
                )

                with torch.autocast(
                    "cuda",
                    dtype=AMP_DTYPE,
                ):
                    logits, targets = model(
                        ids,
                        labels,
                    )

                    loss_sum = F.cross_entropy(
                        logits,
                        targets,
                        reduction="sum",
                    )

                weighted_loss += float(
                    loss_sum
                    .detach()
                    .float()
                    .cpu()
                )

                target_count += (
                    n_targets
                )

                del (
                    ids_slice,
                    labels_slice,
                    ids,
                    labels,
                    logits,
                    targets,
                    loss_sum,
                )

            del (
                ids_cpu,
                labels_cpu,
            )

    model.train()

    return (
        weighted_loss
        /
        max(
            target_count,
            1,
        )
    )

# ---------------------------------------------------------------------
# TRAIN ONE METHOD ON ONE DATASET
# ---------------------------------------------------------------------

def train_one(
    corpus_name,
    kind,
    train_path,
    valid_path,
):
    run_dir = (
        CHECKPOINT_ROOT
        /
        corpus_name
        /
        kind
    )

    run_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    latest_file = (
        run_dir
        /
        "training_latest.pt"
    )

    final_file = (
        run_dir
        /
        "trained_model_final.pt"
    )

    complete_file = (
        run_dir
        /
        "TRAINING_COMPLETE.json"
    )

    log_file = (
        run_dir
        /
        "training_log.jsonl"
    )

    if complete_file.exists():
        result = json.loads(
            complete_file.read_text()
        )

        if (
            result.get(
                "protocol_version"
            )
            !=
            PROTOCOL_VERSION
        ):
            raise RuntimeError(
                f"{corpus_name}/{kind}: existing completion manifest "
                f"belongs to protocol "
                f"{result.get('protocol_version')!r}, not "
                f"{PROTOCOL_VERSION!r}. Refusing to mix protocols."
            )

        print(
            f"{corpus_name}/{kind}: "
            f"already COMPLETE — skipping."
        )

        return result

    train = None
    valid = None
    model = None
    optimizer = None
    scheduler = None
    scaler = None
    executor = None

    hard_cleanup(
        f"before {corpus_name}/{kind}"
    )

    try:
        train = np.load(
            train_path,
            mmap_mode="r",
        )

        valid = np.load(
            valid_path,
            mmap_mode="r",
        )

        effective_batch = int(
            EFFECTIVE_BATCH_BY_CORPUS[corpus_name]
        )

        usable_rows = (
            len(train)
            //
            effective_batch
        ) * effective_batch

        total_updates = (
            usable_rows
            //
            effective_batch
        )

        actual_train_tokens = (
            total_updates
            *
            effective_batch
            *
            SEQ_LENGTH
        )

        rng = np.random.default_rng(
            DATA_ORDER_SEED
        )

        permutation = rng.permutation(
            len(train)
        )[
            :usable_rows
        ]

        # Resume first; do not autotune again after a partial run.
        resume = None

        if latest_file.exists():
            resume = torch.load(
                latest_file,
                map_location="cpu",
                weights_only=False,
            )

            if (
                resume.get(
                    "protocol_version"
                )
                !=
                PROTOCOL_VERSION
            ):
                raise RuntimeError(
                    f"{corpus_name}/{kind}: recovery checkpoint "
                    f"uses incompatible protocol."
                )

            old_micro = int(
                resume["micro_batch"]
            )
            old_accum = int(
                resume["gradient_accumulation"]
            )

            print(
                f"{corpus_name}/{kind}: "
                f"resume update {resume['next_update']} | "
                f"old micro={old_micro} x {old_accum}"
            )

            if REAUTOTUNE_ON_RESUME:
                tune = autotune_microbatch(
                    kind,
                    effective_batch,
                )
                micro_batch = int(
                    tune["micro_batch"]
                )
                grad_accum = int(
                    tune["grad_accum"]
                )
                print(
                    f"  optimized resume partition: "
                    f"micro={micro_batch} x {grad_accum}"
                )
            else:
                micro_batch = old_micro
                grad_accum = old_accum

        else:
            tune = autotune_microbatch(
                kind,
                effective_batch,
            )

            micro_batch = int(
                tune[
                    "micro_batch"
                ]
            )

            grad_accum = int(
                tune[
                    "grad_accum"
                ]
            )

        if (
            micro_batch
            *
            grad_accum
            !=
            effective_batch
        ):
            raise RuntimeError(
                "Effective batch mismatch."
            )

        # Fresh deterministic construction.
        set_all_seeds(
            GLOBAL_SEED
        )

        model = build_model(
            kind
        ).to(
            DEVICE
        )

        total_params, trainable_params = (
            parameter_counts(
                model
            )
        )

        model.train()

        optimizer, fused_adam = (
            make_optimizer(
                model
            )
        )

        scheduler, warmup_steps = (
            make_scheduler(
                optimizer,
                total_updates,
            )
        )

        scaler = make_scaler()

        next_update = 0

        if resume is not None:
            resume_model_state = (
                upgrade_model_state_dict_for_fast_impl(
                    kind,
                    resume["model_state"],
                )
            )

            model.load_state_dict(
                resume_model_state,
                strict=True,
            )
            del resume_model_state

            resume_impl = resume.get(
                "implementation_version"
            )

            if (
                kind == "retnet"
                and
                resume_impl != IMPLEMENTATION_VERSION
            ):
                raise RuntimeError(
                    "Old partial RetNet checkpoint uses the previous q/k/v/g "
                    "optimizer layout. Delete ONLY that RetNet training_latest.pt; "
                    "no runtime restart is required."
                )

            optimizer.load_state_dict(
                resume["optimizer_state"]
            )

            optimizer_to_gpu(
                optimizer
            )

            scheduler.load_state_dict(
                resume[
                    "scheduler_state"
                ]
            )

            scaler.load_state_dict(
                resume[
                    "scaler_state"
                ]
            )

            next_update = int(
                resume[
                    "next_update"
                ]
            )

            restore_rng(
                resume[
                    "rng_state"
                ]
            )

            del resume
            hard_cleanup(
                f"post-resume CPU cleanup {corpus_name}/{kind}"
            )

        def save_recovery(
            next_step,
        ):
            atomic_torch_save(
                {
                    "protocol_version":
                        PROTOCOL_VERSION,

                    "implementation_version":
                        IMPLEMENTATION_VERSION,

                    "corpus":
                        corpus_name,

                    "kind":
                        kind,

                    "model_state":
                        model.state_dict(),

                    "optimizer_state":
                        optimizer.state_dict(),

                    "scheduler_state":
                        scheduler.state_dict(),

                    "scaler_state":
                        scaler.state_dict(),

                    "next_update":
                        int(next_step),

                    "micro_batch":
                        int(micro_batch),

                    "gradient_accumulation":
                        int(grad_accum),

                    "effective_batch":
                        effective_batch,

                    "amp_dtype":
                        AMP_NAME,

                    "rng_state":
                        capture_rng(),
                },
                latest_file,
            )

        def append_log(
            record,
        ):
            with open(
                log_file,
                "a",
                encoding="utf-8",
            ) as f:
                f.write(
                    json.dumps(
                        record
                    )
                    +
                    "\n"
                )

        def prepare_update(
            update_index,
        ):
            idx = permutation[
                update_index
                *
                effective_batch
                :
                (
                    update_index
                    +
                    1
                )
                *
                effective_batch
            ]

            return make_mlm_batch(
                train[idx],
                update_index,
                validation=False,
            )

        print(
            "\n"
            +
            "=" * 80
        )

        print(
            f"RUN: {corpus_name} / {kind}"
        )

        print(
            "GPU                  :",
            GPU_NAME,
        )

        print(
            "AMP                  :",
            AMP_NAME,
        )

        print(
            "Activation checkpoint:",
            "OFF",
        )

        print(
            "Fused Adam           :",
            fused_adam,
        )

        print(
            "Parameters           :",
            f"{total_params:,}",
        )

        print(
            "Trainable parameters :",
            f"{trainable_params:,}",
        )

        print(
            "Microbatch           :",
            micro_batch,
        )

        print(
            "Gradient accumulation:",
            grad_accum,
        )

        print(
            "Effective batch       :",
            effective_batch,
        )

        print(
            "Updates               :",
            total_updates,
        )

        print(
            "Training tokens       :",
            f"{actual_train_tokens:,}",
        )

        print(
            "Warmup updates        :",
            warmup_steps,
        )

        print(
            "=" * 80
        )

        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

        session_start = (
            time.perf_counter()
        )

        start_update = (
            next_update
        )

        running_loss_gpu = torch.zeros(
            (),
            dtype=torch.float32,
            device=DEVICE,
        )

        running_count = 0
        last_completed = next_update
        last_checkpoint_wall = time.monotonic()

        # CPU SSD-read + vectorized MLM preparation overlaps GPU compute.
        executor = ThreadPoolExecutor(
            max_workers=1
        )

        future = (
            executor.submit(
                prepare_update,
                next_update,
            )
            if next_update
            <
            total_updates
            else None
        )

        try:
            for update_index in range(
                next_update,
                total_updates,
            ):
                ids_cpu, labels_cpu = (
                    future.result()
                )

                if (
                    update_index + 1
                    <
                    total_updates
                ):
                    future = executor.submit(
                        prepare_update,
                        update_index + 1,
                    )
                else:
                    future = None

                optimizer.zero_grad(
                    set_to_none=True
                )

                for mstart in range(
                    0,
                    effective_batch,
                    micro_batch,
                ):
                    ids = ids_cpu[
                        mstart:
                        mstart
                        +
                        micro_batch
                    ].to(
                        DEVICE,
                        non_blocking=True,
                    )

                    labels = labels_cpu[
                        mstart:
                        mstart
                        +
                        micro_batch
                    ].to(
                        DEVICE,
                        non_blocking=True,
                    )

                    with torch.autocast(
                        "cuda",
                        dtype=AMP_DTYPE,
                    ):
                        logits, targets = model(
                            ids,
                            labels,
                        )

                        loss = F.cross_entropy(
                            logits,
                            targets,
                        )

                        scaled_loss = (
                            loss
                            /
                            grad_accum
                        )

                    scaler.scale(
                        scaled_loss
                    ).backward()

                    running_loss_gpu += (
                        loss.detach().float()
                        /
                        grad_accum
                    )

                    del (
                        ids,
                        labels,
                        logits,
                        targets,
                        loss,
                        scaled_loss,
                    )

                del (
                    ids_cpu,
                    labels_cpu,
                )

                scale_before = (
                    scaler.get_scale()
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                scale_after = (
                    scaler.get_scale()
                )

                if (
                    USE_SCALER
                    and
                    scale_after
                    <
                    scale_before
                ):
                    raise FloatingPointError(
                        "AMP skipped an optimizer update "
                        "because of non-finite gradients."
                    )

                scheduler.step()

                completed = (
                    update_index
                    +
                    1
                )

                last_completed = (
                    completed
                )

                running_count += 1

                if (
                    completed
                    %
                    LOG_EVERY
                    ==
                    0
                    or
                    completed
                    ==
                    total_updates
                ):
                    torch.cuda.synchronize()

                    elapsed = (
                        time.perf_counter()
                        -
                        session_start
                    )

                    session_updates = (
                        completed
                        -
                        start_update
                    )

                    tps = (
                        session_updates
                        *
                        effective_batch
                        *
                        SEQ_LENGTH
                        /
                        max(
                            elapsed,
                            1.0e-9,
                        )
                    )

                    eta_hours = (
                        (
                            total_updates
                            -
                            completed
                        )
                        *
                        effective_batch
                        *
                        SEQ_LENGTH
                        /
                        max(
                            tps,
                            1.0e-9,
                        )
                        /
                        3600.0
                    )

                    peak = (
                        torch.cuda
                        .max_memory_allocated()
                        /
                        1024**3
                    )

                    avg_loss = (
                        float(
                            running_loss_gpu
                            .detach()
                            .cpu()
                        )
                        /
                        max(
                            running_count,
                            1,
                        )
                    )

                    print(
                        f"{corpus_name:12s} | "
                        f"{kind:7s} | "
                        f"{completed:4d}/{total_updates} | "
                        f"loss {avg_loss:.4f} | "
                        f"{tps:,.0f} tok/s | "
                        f"VRAM {peak:.2f} GB | "
                        f"ETA {eta_hours:.2f} h"
                    )

                    append_log(
                        {
                            "update":
                                completed,

                            "loss":
                                avg_loss,

                            "tokens_per_second":
                                tps,

                            "peak_vram_gb":
                                peak,

                            "eta_hours":
                                eta_hours,

                            "micro_batch":
                                micro_batch,

                            "amp_dtype":
                                AMP_NAME,
                        }
                    )

                    running_loss_gpu.zero_()
                    running_count = 0

                now_wall = time.monotonic()
                if (
                    completed % SAVE_EVERY == 0
                    or (now_wall - last_checkpoint_wall) >= SAVE_EVERY_MINUTES * 60
                ):
                    save_recovery(completed)
                    last_checkpoint_wall = time.monotonic()
                    print(
                        f"  persistent Drive recovery checkpoint saved at update {completed}."
                    )

            executor.shutdown(
                wait=True
            )
            executor = None

        except BaseException:
            if executor is not None:
                executor.shutdown(
                    wait=False,
                    cancel_futures=True,
                )
                executor = None

            try:
                save_recovery(
                    last_completed
                )
                print(
                    f"Emergency recovery saved at "
                    f"update {last_completed}."
                )
            except Exception as save_exc:
                print(
                    "Emergency recovery save failed:",
                    save_exc,
                )

            raise

        # Critical restart boundary: if the runtime dies during validation,
        # resumption starts at the end of training instead of replaying updates.
        save_recovery(total_updates)
        last_checkpoint_wall = time.monotonic()

        print(
            f"\n{corpus_name}/{kind}: "
            f"running FULL validation..."
        )

        # Validation has no backward graph, so start aggressively and halve only on OOM.
        validation_micro_batch = effective_batch
        while True:
            try:
                final_val = evaluate(
                    model,
                    valid,
                    validation_micro_batch,
                    effective_batch,
                )
                break
            except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
                if "out of memory" not in str(exc).lower() and not isinstance(exc, torch.cuda.OutOfMemoryError):
                    raise
                if validation_micro_batch <= 1:
                    raise
                validation_micro_batch = max(1, validation_micro_batch // 2)
                torch.cuda.empty_cache()
                print(f"Validation OOM; retrying with microbatch={validation_micro_batch}")

        final_ppl = math.exp(
            min(
                final_val,
                20.0,
            )
        )

        peak = (
            torch.cuda
            .max_memory_allocated()
            /
            1024**3
        )

        torch.cuda.synchronize()

        train_elapsed = (
            time.perf_counter()
            -
            session_start
        )

        training_tps = (
            (
                total_updates
                -
                start_update
            )
            *
            effective_batch
            *
            SEQ_LENGTH
            /
            max(
                train_elapsed,
                1.0e-9,
            )
        )

        result = {
            "status":
                "complete",

            "protocol_version":
                PROTOCOL_VERSION,

            "implementation_version":
                IMPLEMENTATION_VERSION,

            "corpus":
                corpus_name,

            "model":
                kind,

            "objective":
                "masked_language_modeling",

            "bidirectional_shared_mixer":
                True,

            "stacked_bidirectional_execution":
                True,

            "masked_only_mlm_head":
                MASKED_ONLY_MLM_HEAD,

            "retention_chunk":
                RETENTION_CHUNK if kind == "retnet" else None,

            "training_tokens":
                int(
                    actual_train_tokens
                ),

            "optimizer_updates":
                int(
                    total_updates
                ),

            "validation_loss":
                float(
                    final_val
                ),

            "validation_perplexity":
                float(
                    final_ppl
                ),

            "micro_batch":
                int(
                    micro_batch
                ),

            "gradient_accumulation":
                int(
                    grad_accum
                ),

            "effective_batch":
                effective_batch,

            "amp_dtype":
                AMP_NAME,

            "transformers_version":
                HF_TRANSFORMERS_VERSION,

            "mamba_backend":
                MAMBA_BACKEND if kind in ("mamba2", "mamba") else None,

            "activation_checkpointing":
                False,

            "training_tokens_per_second":
                float(
                    training_tps
                ),

            "peak_training_vram_gb":
                float(
                    peak
                ),

            "parameters":
                int(
                    total_params
                ),

            "trainable_parameters":
                int(
                    trainable_params
                ),

            "gpu":
                GPU_NAME,

            "torch":
                torch.__version__,

            "cuda":
                torch.version.cuda,

            "tokenizer_sha256":
                TOKENIZER_SHA256,

            "pretrained_weights":
                False,

            "save_final_model":
                bool(
                    SAVE_FINAL_MODEL
                ),
        }

        # Completion manifest FIRST: this is what controls skip-on-rerun.
        complete_file.write_text(
            json.dumps(
                result,
                indent=2,
            )
        )

        if SAVE_FINAL_MODEL:
            atomic_torch_save(
                {
                    **result,
                    "model_state":
                        model.state_dict(),
                },
                final_file,
            )

        # Once the manifest exists, a recovery checkpoint is no longer needed.
        latest_file.unlink(
            missing_ok=True
        )

        print(
            f"COMPLETE {corpus_name}/{kind} | "
            f"val={final_val:.6f} | "
            f"ppl={final_ppl:.3f} | "
            f"{training_tps:,.0f} tok/s"
        )

        return result

    finally:
        if executor is not None:
            try:
                executor.shutdown(
                    wait=False,
                    cancel_futures=True,
                )
            except Exception:
                pass

        executor = None
        scaler = None
        scheduler = None
        optimizer = None
        model = None

        if train is not None:
            try:
                if (
                    hasattr(
                        train,
                        "_mmap",
                    )
                    and
                    train._mmap
                    is not None
                ):
                    train._mmap.close()
            except Exception:
                pass

        if valid is not None:
            try:
                if (
                    hasattr(
                        valid,
                        "_mmap",
                    )
                    and
                    valid._mmap
                    is not None
                ):
                    valid._mmap.close()
            except Exception:
                pass

        train = None
        valid = None

        hard_cleanup(
            f"AFTER METHOD {corpus_name}/{kind}"
        )

# ---- merged from former CELL 6 — Consolidated result collection and saving
# ---------------------------------------------------------------------
# RESULTS
# ---------------------------------------------------------------------

def collect_all_results():
    rows = []

    for spec in CORPORA:
        for kind in METHODS:
            p = (
                CHECKPOINT_ROOT
                /
                spec["name"]
                /
                kind
                /
                "TRAINING_COMPLETE.json"
            )

            if p.exists():
                d = json.loads(
                    p.read_text()
                )
                rows.append(
                    d
                )
            else:
                rows.append(
                    {
                        "status":
                            "missing",

                        "protocol_version":
                            PROTOCOL_VERSION,

                        "corpus":
                            spec[
                                "name"
                            ],

                        "model":
                            kind,
                    }
                )

    return rows

def save_summary():
    rows = collect_all_results()

    df = pd.DataFrame(
        rows
    )

    csv_path = (
        RESULTS_ROOT
        /
        "spliNet_FINAL.csv"
    )

    json_path = (
        RESULTS_ROOT
        /
        "spliNet_FINAL.json"
    )

    df.to_csv(
        csv_path,
        index=False,
    )

    json_path.write_text(
        json.dumps(
            rows,
            indent=2,
        )
    )

    print(
        "\n"
        +
        "=" * 140
    )

    print(
        "CURRENT CONSOLIDATED RESULTS"
    )

    print(
        "=" * 140
    )

    columns = [
        c
        for c in [
            "corpus",
            "model",
            "status",
            "validation_loss",
            "validation_perplexity",
            "micro_batch",
            "gradient_accumulation",
            "effective_batch",
            "training_tokens_per_second",
            "peak_training_vram_gb",
            "parameters",
            "gpu",
        ]
        if c in df.columns
    ]

    print(
        df[
            columns
        ].to_string(
            index=False
        )
    )

    print(
        "\nSaved:"
    )

    print(
        csv_path
    )

    print(
        json_path
    )


# ---------------------------------------------------------------------
# RESTART-SAFE CORPUS ENTRYPOINTS
# ---------------------------------------------------------------------

def _corpus_spec(name):
    for spec in CORPORA:
        if spec["name"] == name:
            return spec
    raise KeyError(name)

def run_corpus(name):
    """Run/resume one corpus. Safe to call after a fresh runtime using Cell 1 only."""
    spec = _corpus_spec(name)
    local_dir = None
    try:
        local_train, local_valid, local_dir = stage_dataset(spec)
        for kind in METHODS:
            train_one(name, kind, local_train, local_valid)
            save_summary()
            hard_cleanup(f"METHOD BOUNDARY {name}/{kind}")
    finally:
        hard_cleanup(f"AFTER DATASET {name}")
        print(f"Local SSD cache retained for this runtime: {local_dir}")
    save_summary()

# ---------------------------------------------------------------------
# STARTUP STUBS — actual G4 kernels, tiny workload, catches shape/backward errors
# ---------------------------------------------------------------------

def run_startup_stubs():
    print("\\nRunning G4 startup stubs (forward + backward + optimizer step)...")
    stub_len = SEQ_LENGTH
    for kind in METHODS:
        hard_cleanup(f"before startup stub {kind}")
        model = None
        optimizer = None
        try:
            model = build_model(kind).to(DEVICE)
            model.train()
            optimizer, _ = make_optimizer(model)
            ids = torch.randint(7, VOCAB_SIZE, (1, stub_len), device=DEVICE, dtype=torch.long)
            labels = torch.full_like(ids, -100)
            labels[:, 1:9] = ids[:, 1:9]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits, targets = model(ids, labels)
                if logits.shape != (8, VOCAB_SIZE):
                    raise RuntimeError(f"{kind} stub logits shape {tuple(logits.shape)}")
                loss = F.cross_entropy(logits, targets)
            loss.backward()
            optimizer.step()
            torch.cuda.synchronize()
            print(f"  {kind:7s}: PASS | logits={tuple(logits.shape)} | loss={float(loss.detach()):.4f}")
        finally:
            model = optimizer = None
            hard_cleanup(f"after startup stub {kind}")
    print("Startup stubs: ALL PASS")

run_startup_stubs()
print("\\nRESTART CONTRACT: after a disconnect, run ONLY Cell 1, then rerun the interrupted corpus cell.")


CELL 1 VERSION: FIX2-HF-AUTO-MAMBA-NO-TOKENIZER-FILE-2026-10-09
Mounted at /content/drive
Transformers initially: 5.18.0
Transformers selected: 5.18.0
HF Mamba-2 public API: YES
HF Mamba public API  : YES
Tokenizer file: NOT REQUIRED (using existing pre-tokenized arrays)
Testing Hugging Face mamba2 native mixer on this GPU...


[transformers] `mamba_split_conv1d_scan_combined` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `mamba_chunk_scan_combined` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.


HF mamba2 CUDA forward/backward: PASS
GPU              : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM             : 94.97 GB
PyTorch          : 2.11.0+cu130
CUDA             : 13.0
AMP              : bfloat16
Protocol         : splinet_g4_hf_independent_tokenizers_v1
Tokenizer metadata: not required: using pre-tokenized 32k-ID arrays
Runtime restart  : NEVER REQUIRED
Transformers      : 5.18.0
Mamba architecture: mamba2
Mamba backend     : huggingface_native_mamba2
External Mamba CUDA packages: NONE
\nRunning G4 startup stubs (forward + backward + optimizer step)...
[cleanup] before startup stub mamba2 | allocated=0.016 GB | reserved=0.039 GB
  mamba2 : PASS | logits=(8, 32000) | loss=10.3984
[cleanup] after startup stub mamba2 | allocated=0.958 GB | reserved=1.217 GB
[cleanup] before startup stub hyena | allocated=0.958 GB | reserved=1.217 GB
  hyena  : PASS | logits=(8, 32000) | loss=10.5938
[cleanup] after startup stub hyena | allocated=0.847 GB | reserved=1.143 GB
[cleanup] be

In [14]:
# CELL 2 — spliNet CLEAN, RAM-EFFICIENT DATA + TOKENIZER PREPARATION
#
# What this replacement cell does:
#   * Keeps the already-downloaded RAW TEXT caches (expensive to recreate).
#   * ONCE deletes all older spliNet tokenizers, token arrays, local token staging,
#     checkpoints and results because they are incompatible with a clean rebuild.
#   * Trains ONE independent 32k SentencePiece tokenizer per method.
#   * Tokenizes with a STRICTLY BOUNDED-RAM streaming encoder:
#       - never loads a corpus into RAM
#       - never loads a giant book/article line into RAM as one token list
#       - reads bounded text chunks
#       - small bounded SentencePiece batches
#       - writes blocks directly to a local SSD .npy memmap
#       - copies the finished array atomically to Google Drive
#   * Reuses completed NEW artifacts after a disconnect; the destructive cleanup is
#     guarded by a one-time rebuild marker.
#   * Includes the WikiText-103 short-validation and finite-train fixes.
#
# Run after the working spliNet Cell 1.

from __future__ import annotations

import os
import sys
import gc
import json
import time
import shutil
import hashlib
import subprocess
import importlib
import ctypes
from pathlib import Path

import numpy as np

# -----------------------------------------------------------------------------
# REQUIRE WORKING CELL 1
# -----------------------------------------------------------------------------
EXPECTED_CELL1_VERSION = "FIX2-HF-AUTO-MAMBA-NO-TOKENIZER-FILE-2026-10-09"
if globals().get("CELL1_VERSION") != EXPECTED_CELL1_VERSION:
    raise RuntimeError(
        "Run the working spliNet Cell 1 first. "
        f"Expected {EXPECTED_CELL1_VERSION!r}, got {globals().get('CELL1_VERSION')!r}."
    )

for _required in (
    "torch", "DEVICE", "GPU_NAME", "METHODS",
    "SEQ_LENGTH", "VOCAB_SIZE", "MASK_ID", "PAD_ID",
    "EXPECTED_TRAIN_BLOCKS", "EXPECTED_VALID_BLOCKS",
):
    if _required not in globals():
        raise RuntimeError(f"Cell 1 is incomplete: missing {_required}")

DATA_METHODS = list(dict.fromkeys(METHODS))
if len(DATA_METHODS) != 3 or not {"hyena", "retnet"}.issubset(DATA_METHODS):
    raise RuntimeError(f"Unexpected method set from Cell 1: {DATA_METHODS}")
if not any(m in DATA_METHODS for m in ("mamba2", "mamba")):
    raise RuntimeError(f"No HF Mamba backend selected by Cell 1: {DATA_METHODS}")

# -----------------------------------------------------------------------------
# spliNet PATHS
# -----------------------------------------------------------------------------
DRIVE_HOME = Path("/content/drive/MyDrive")
SPLINET_ROOT = DRIVE_HOME / "spliNet"
HF_CACHE_ROOT = SPLINET_ROOT / "hf_cache"
RAW_ROOT = SPLINET_ROOT / "raw_text"
TOKENIZER_ROOT = SPLINET_ROOT / "tokenizers"
TOKEN_ROOT = SPLINET_ROOT / "tokens"
CHECKPOINT_ROOT = SPLINET_ROOT / "checkpoints"
RESULTS_ROOT = SPLINET_ROOT / "results"
CONFIG_ROOT = SPLINET_ROOT / "configs"

LOCAL_ROOT = Path("/content/spliNet")
LOCAL_STAGE_ROOT = LOCAL_ROOT / "staged"
LOCAL_TRAIN_STAGE_ROOT = LOCAL_ROOT / "staged_training"
LOCAL_TMP_ROOT = LOCAL_ROOT / "tmp"
TOKENIZE_WORK = Path("/content/spliNet_tokenize_work")

for _p in (SPLINET_ROOT, HF_CACHE_ROOT, RAW_ROOT, CONFIG_ROOT, LOCAL_ROOT):
    _p.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT = SPLINET_ROOT
os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE_ROOT / "datasets")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_CACHE_ROOT / "hub")
os.environ["TOKENIZERS_PARALLELISM"] = "true"

# -----------------------------------------------------------------------------
# PACKAGE CHECK — HF/PyPI ONLY
# -----------------------------------------------------------------------------
def _pkg_version(name):
    try:
        import importlib.metadata as _md
        return _md.version(name)
    except Exception:
        return None

_install = []
if _pkg_version("datasets") is None:
    _install.append("datasets>=4,<5")
if _pkg_version("sentencepiece") is None:
    _install.append("sentencepiece>=0.2,<0.3")
if _pkg_version("huggingface-hub") is None:
    _install.append("huggingface-hub>=0.34")
if _pkg_version("psutil") is None:
    _install.append("psutil>=5.9")

if _install:
    print("Installing missing compatible packages:", _install)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *_install
    ])
    importlib.invalidate_caches()

import datasets
import sentencepiece as spm
import psutil

PROCESS = psutil.Process(os.getpid())
PHYSICAL_CORES = psutil.cpu_count(logical=False) or (os.cpu_count() or 1)

print("spliNet methods :", DATA_METHODS)
print("Mamba backend   :", globals().get("MAMBA_BACKEND", "unknown"))
print("datasets        :", datasets.__version__)
print("sentencepiece   :", spm.__version__)
print("torch           :", torch.__version__)
print("CUDA            :", torch.version.cuda)
print("physical cores  :", PHYSICAL_CORES)

# -----------------------------------------------------------------------------
# PROTOCOL
# -----------------------------------------------------------------------------
TRAIN_BLOCKS = int(EXPECTED_TRAIN_BLOCKS)
VALID_BLOCKS = int(EXPECTED_VALID_BLOCKS)
if int(SEQ_LENGTH) != 512 or int(VOCAB_SIZE) != 32_000:
    raise RuntimeError(f"Unexpected protocol: seq={SEQ_LENGTH}, vocab={VOCAB_SIZE}")

C4_TOKENIZER_BYTES = 256 * 1024**2
TRAIN_RAW_BYTES = 520 * 1024**2
VALID_RAW_BYTES = 16 * 1024**2
WIKI_MIN_TRAIN_BYTES = 450 * 1024**2

# RAM-bounded tokenizer settings.  These are intentionally conservative.
# A giant PG-19 book is NEVER encoded as one Python list.
TOKENIZE_TEXT_CHARS = 64 * 1024        # <= 64K characters per text chunk
TOKENIZE_READ_CHARS = 512 * 1024       # <= 512K characters per file read
TOKENIZE_WORKERS = max(1, min(8, int(PHYSICAL_CORES)))
TOKENIZE_BATCH_CHUNKS = max(2, min(8, TOKENIZE_WORKERS))
TOKEN_DTYPE = np.uint16                 # 0..31999 fits exactly; halves I/O/RAM vs int32
EOS_ID = 2

DATASETS = {
    "fineweb_edu": {
        "repo": "HuggingFaceFW/fineweb-edu",
        "configs": ["sample-10BT", None],
        "text_column": "text",
        "train_split": "train",
        "valid_split": None,
        "test_split": None,
    },
    "arxiv": {
        "repo": "ccdv/arxiv-summarization",
        "configs": ["document", "section", None],
        "text_column": "article",
        "train_split": "train",
        "valid_split": "validation",
        "test_split": "test",
    },
    "pg19": {
        "repo": "emozilla/pg19",
        "configs": [None],
        "text_column": "text",
        "train_split": "train",
        "valid_split": "validation",
        "test_split": "test",
    },
    "wikitext103": {
        "repo": "Salesforce/wikitext",
        "configs": ["wikitext-103-raw-v1"],
        "text_column": "text",
        "train_split": "train",
        "valid_split": "validation",
        "test_split": "test",
    },
}

# -----------------------------------------------------------------------------
# ATOMIC / MEMORY HELPERS
# -----------------------------------------------------------------------------
def _rss_gib():
    return PROCESS.memory_info().rss / 1024**3


def _malloc_trim():
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


def _clean_memory():
    gc.collect()
    _malloc_trim()


def sha256_file(path, block_size=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(block_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def atomic_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    os.replace(tmp, path)


def _clean_text(value):
    if value is None:
        return ""
    text = str(value).replace("\x00", " ").replace("\r", " ").replace("\n", " ")
    return " ".join(text.split())

# -----------------------------------------------------------------------------
# ONE-TIME CLEAN REBUILD
# -----------------------------------------------------------------------------
# The marker is written only after old tokenizer-dependent artifacts are deleted.
# Therefore: first run = clean rebuild; later reruns after disconnect = resume safely.
RESET_MARKER = CONFIG_ROOT / "ram_efficient_tokenization_rebuild_v1.done.json"

if not RESET_MARKER.exists():
    print("\nCLEAN REBUILD: deleting ALL previous tokenizer-dependent spliNet artifacts...")
    print("RAW text and HF download cache are PRESERVED.")

    # New tokenizers change the token-ID mapping, therefore old token arrays,
    # training staging, checkpoints and results are all incompatible.
    for _p in (
        TOKENIZER_ROOT,
        TOKEN_ROOT,
        CHECKPOINT_ROOT,
        RESULTS_ROOT,
        LOCAL_STAGE_ROOT,
        LOCAL_TRAIN_STAGE_ROOT,
        LOCAL_TMP_ROOT,
        TOKENIZE_WORK,
    ):
        if _p.exists():
            print("  deleting:", _p)
            shutil.rmtree(_p, ignore_errors=False)

    for _old in (
        CONFIG_ROOT / "data_and_tokenizer_audit.json",
        CONFIG_ROOT / "autotune",
    ):
        if _old.is_dir():
            shutil.rmtree(_old, ignore_errors=True)
        elif _old.exists():
            _old.unlink(missing_ok=True)

    for _p in (
        TOKENIZER_ROOT, TOKEN_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT,
        LOCAL_STAGE_ROOT, LOCAL_TRAIN_STAGE_ROOT, LOCAL_TMP_ROOT, TOKENIZE_WORK,
    ):
        _p.mkdir(parents=True, exist_ok=True)

    atomic_json({
        "rebuild": "ram_efficient_tokenization_v1",
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "deleted_old_tokenizers": True,
        "deleted_old_token_arrays": True,
        "deleted_old_checkpoints": True,
        "deleted_old_results": True,
        "preserved_raw_text": True,
        "preserved_hf_cache": True,
    }, RESET_MARKER)

    print("Old tokenizer-dependent artifacts deleted. Clean rebuild marker written.")
else:
    print("RAM-efficient rebuild marker already present: resuming NEW artifacts; no deletion.")

# -----------------------------------------------------------------------------
# HF STREAMING WITH RETRY
# -----------------------------------------------------------------------------
def _load_stream(spec, split, attempts=4):
    from datasets import load_dataset
    errors = []
    for cfg in spec["configs"]:
        for attempt in range(1, attempts + 1):
            try:
                kwargs = {"path": spec["repo"], "split": split, "streaming": True}
                if cfg is not None:
                    kwargs["name"] = cfg
                return load_dataset(**kwargs), cfg
            except Exception as exc:
                errors.append(f"cfg={cfg!r} attempt={attempt}: {exc!r}")
                if attempt < attempts:
                    time.sleep(min(5 * attempt, 20))
    raise RuntimeError(
        f"Could not stream {spec['repo']} split={split}.\n" + "\n".join(errors[-12:])
    )


def _write_from_stream(
    stream, text_column, path, target_bytes, skip_records=0,
    *, require_full=True, minimum_bytes=0,
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.unlink(missing_ok=True)

    target_bytes = int(target_bytes)
    skip_records = int(skip_records)
    minimum_bytes = int(minimum_bytes)
    written = used = seen = 0
    next_report = 64 * 1024**2

    with tmp.open("w", encoding="utf-8") as out:
        for row in stream:
            if seen < skip_records:
                seen += 1
                continue
            seen += 1
            text = _clean_text(row.get(text_column, ""))
            if not text:
                continue
            line = text + "\n"
            out.write(line)
            written += len(line.encode("utf-8"))
            used += 1

            if written >= next_report:
                print(
                    f"    raw text: {written / 1024**2:.1f} MiB / "
                    f"{target_bytes / 1024**2:.1f} MiB | RSS={_rss_gib():.2f} GiB"
                )
                next_report += 64 * 1024**2
            if written >= target_bytes:
                break

    complete = written >= target_bytes
    if require_full and not complete:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(
            f"Stream ended after {written / 1024**2:.1f} MiB; "
            f"need {target_bytes / 1024**2:.1f} MiB."
        )
    if written < minimum_bytes:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(
            f"Stream ended after only {written / 1024**2:.1f} MiB; "
            f"minimum accepted is {minimum_bytes / 1024**2:.1f} MiB."
        )

    os.replace(tmp, path)
    return {
        "bytes": int(written),
        "records_written": int(used),
        "records_seen": int(seen),
        "complete": bool(complete),
    }

# -----------------------------------------------------------------------------
# RAW DATA — PRESERVED / REUSED
# -----------------------------------------------------------------------------
def ensure_c4_tokenizer_source():
    path = RAW_ROOT / "tokenizer_c4_256mb.txt"
    manifest = RAW_ROOT / "tokenizer_c4_256mb.json"
    if path.exists() and manifest.exists() and path.stat().st_size >= C4_TOKENIZER_BYTES:
        print("C4 tokenizer source: cached")
        return path

    print("Streaming C4 tokenizer-training source from Hugging Face...")
    spec = {"repo": "allenai/c4", "configs": ["en"]}
    ds, cfg = _load_stream(spec, "train")
    stats = _write_from_stream(ds, "text", path, C4_TOKENIZER_BYTES, require_full=True)
    atomic_json({
        "repo": "allenai/c4", "config": cfg, "split": "train",
        "target_bytes": C4_TOKENIZER_BYTES, **stats, "sha256": sha256_file(path),
    }, manifest)
    return path


def ensure_raw_corpus(corpus_name):
    spec = DATASETS[corpus_name]
    root = RAW_ROOT / corpus_name
    train_path = root / "train.txt"
    valid_path = root / "validation.txt"
    manifest_path = root / "manifest.json"

    if train_path.exists() and valid_path.exists() and manifest_path.exists():
        if corpus_name == "wikitext103":
            train_ok = train_path.stat().st_size >= WIKI_MIN_TRAIN_BYTES
        else:
            train_ok = train_path.stat().st_size >= TRAIN_RAW_BYTES
        valid_ok = valid_path.stat().st_size >= VALID_RAW_BYTES
        if train_ok and valid_ok:
            print(
                f"{corpus_name}: raw cache present | "
                f"train={train_path.stat().st_size / 1024**2:.1f} MiB | "
                f"valid={valid_path.stat().st_size / 1024**2:.1f} MiB"
            )
            return train_path, valid_path

    root.mkdir(parents=True, exist_ok=True)
    train_path.unlink(missing_ok=True)
    valid_path.unlink(missing_ok=True)
    manifest_path.unlink(missing_ok=True)
    for stale in root.glob("_valid_*.txt"):
        stale.unlink(missing_ok=True)

    print(f"\nPreparing raw corpus: {corpus_name}")
    holdout_train_records = 0
    valid_bytes = 0
    valid_parts = []
    valid_sources = []

    # Keep short official validation/test splits instead of failing.
    for split_key in ("valid_split", "test_split"):
        split = spec.get(split_key)
        if not split or valid_bytes >= VALID_RAW_BYTES:
            continue
        remaining = VALID_RAW_BYTES - valid_bytes
        try:
            ds, cfg = _load_stream(spec, split)
            part = root / f"_valid_{split}.txt"
            stats = _write_from_stream(
                ds, spec["text_column"], part, remaining,
                require_full=False, minimum_bytes=0,
            )
        except Exception as exc:
            print(f"    {split}: unavailable ({type(exc).__name__}: {exc})")
            continue

        if stats["bytes"] > 0:
            valid_parts.append(part)
            valid_sources.append({"split": split, "config": cfg, **stats})
            valid_bytes += stats["bytes"]
            print(
                f"    {split}: kept {stats['bytes'] / 1024**2:.1f} MiB; "
                f"validation total={valid_bytes / 1024**2:.1f} MiB"
            )
        else:
            part.unlink(missing_ok=True)

    # Top up from beginning of train and skip exactly those records in training.
    if valid_bytes < VALID_RAW_BYTES:
        remaining = VALID_RAW_BYTES - valid_bytes
        print(f"    topping validation up from train: {remaining / 1024**2:.1f} MiB")
        ds, cfg = _load_stream(spec, spec["train_split"])
        part = root / "_valid_from_train.txt"
        stats = _write_from_stream(
            ds, spec["text_column"], part, remaining, require_full=True
        )
        valid_parts.append(part)
        valid_sources.append({"split": "train-holdout", "config": cfg, **stats})
        valid_bytes += stats["bytes"]
        holdout_train_records = int(stats["records_seen"])

    valid_tmp = valid_path.with_suffix(valid_path.suffix + ".tmp")
    valid_tmp.unlink(missing_ok=True)
    with valid_tmp.open("wb") as out:
        for part in valid_parts:
            with part.open("rb") as inp:
                shutil.copyfileobj(inp, out, length=8 << 20)
            part.unlink(missing_ok=True)
    os.replace(valid_tmp, valid_path)

    if valid_path.stat().st_size < VALID_RAW_BYTES:
        raise RuntimeError(
            f"{corpus_name}: validation cache is only "
            f"{valid_path.stat().st_size / 1024**2:.1f} MiB; need "
            f"{VALID_RAW_BYTES / 1024**2:.1f} MiB."
        )

    ds, cfg_used = _load_stream(spec, spec["train_split"])
    if corpus_name == "wikitext103":
        train_stats = _write_from_stream(
            ds, spec["text_column"], train_path, TRAIN_RAW_BYTES,
            skip_records=holdout_train_records,
            require_full=False,
            minimum_bytes=WIKI_MIN_TRAIN_BYTES,
        )
        if not train_stats["complete"]:
            print(
                f"    WikiText-103 finite train split accepted at "
                f"{train_stats['bytes'] / 1024**2:.1f} MiB. "
                "Exact token-block target will still be enforced per tokenizer."
            )
    else:
        train_stats = _write_from_stream(
            ds, spec["text_column"], train_path, TRAIN_RAW_BYTES,
            skip_records=holdout_train_records, require_full=True,
        )

    atomic_json({
        "corpus": corpus_name,
        "repo": spec["repo"],
        "config_used_for_train": cfg_used,
        "train_split": spec["train_split"],
        "holdout_train_records": holdout_train_records,
        "validation_sources": valid_sources,
        "requested_train_raw_bytes": TRAIN_RAW_BYTES,
        "actual_train_bytes": train_path.stat().st_size,
        "validation_bytes": valid_path.stat().st_size,
        "train_stream_reached_raw_cap": bool(train_stats["complete"]),
        "train_records_written": train_stats["records_written"],
        "train_sha256": sha256_file(train_path),
        "validation_sha256": sha256_file(valid_path),
    }, manifest_path)

    print(
        f"{corpus_name}: raw cache complete | "
        f"train={train_path.stat().st_size / 1024**2:.1f} MiB | "
        f"valid={valid_path.stat().st_size / 1024**2:.1f} MiB"
    )
    return train_path, valid_path

# -----------------------------------------------------------------------------
# INDEPENDENT TOKENIZERS
# -----------------------------------------------------------------------------
def tokenizer_paths(method):
    root = TOKENIZER_ROOT / method
    return {
        "root": root,
        "model": root / "spiece.model",
        "vocab": root / "spiece.vocab",
        "manifest": root / "manifest.json",
    }


def ensure_method_tokenizer(method):
    paths = tokenizer_paths(method)
    paths["root"].mkdir(parents=True, exist_ok=True)

    if paths["model"].exists() and paths["manifest"].exists():
        sp = spm.SentencePieceProcessor(model_file=str(paths["model"]))
        if (
            sp.get_piece_size() == VOCAB_SIZE
            and sp.piece_to_id("[MASK]") == MASK_ID
            and sp.pad_id() == PAD_ID
        ):
            print(f"{method}: tokenizer cached | sha256={sha256_file(paths['model'])[:16]}")
            return paths["model"]

    paths["model"].unlink(missing_ok=True)
    paths["vocab"].unlink(missing_ok=True)
    paths["manifest"].unlink(missing_ok=True)

    source = ensure_c4_tokenizer_source()
    prefix = paths["root"] / "spiece"
    print(f"Training independent SentencePiece tokenizer for {method} ...")
    print(f"    RSS before tokenizer training: {_rss_gib():.2f} GiB")

    spm.SentencePieceTrainer.train(
        input=str(source),
        model_prefix=str(prefix),
        vocab_size=VOCAB_SIZE,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0,
        bos_id=1,
        eos_id=2,
        pad_id=3,
        user_defined_symbols=["[CLS]", "[SEP]", "[MASK]"],
        shuffle_input_sentence=False,
        num_threads=max(1, min(8, int(PHYSICAL_CORES))),
        hard_vocab_limit=True,
        train_extremely_large_corpus=True,
    )

    sp = spm.SentencePieceProcessor(model_file=str(paths["model"]))
    if sp.get_piece_size() != VOCAB_SIZE:
        raise RuntimeError(f"{method}: vocab={sp.get_piece_size()}, expected {VOCAB_SIZE}")
    if sp.piece_to_id("[MASK]") != MASK_ID:
        raise RuntimeError(f"{method}: [MASK] id={sp.piece_to_id('[MASK]')}, expected {MASK_ID}")
    if sp.pad_id() != PAD_ID:
        raise RuntimeError(f"{method}: PAD id={sp.pad_id()}, expected {PAD_ID}")

    # Small functional stub.
    ids = sp.encode("spliNet tokenizer functional test", out_type=int)
    if not ids or min(ids) < 0 or max(ids) >= VOCAB_SIZE:
        raise RuntimeError(f"{method}: tokenizer functional stub failed")

    atomic_json({
        "method": method,
        "trained_independently": True,
        "training_source": str(source),
        "training_source_sha256": sha256_file(source),
        "model_type": "unigram",
        "vocab_size": VOCAB_SIZE,
        "model_sha256": sha256_file(paths["model"]),
        "mask_id": MASK_ID,
        "pad_id": PAD_ID,
        "ram_efficient_rebuild": True,
    }, paths["manifest"])

    del sp
    _clean_memory()
    print(f"    RSS after tokenizer training: {_rss_gib():.2f} GiB")
    return paths["model"]

# -----------------------------------------------------------------------------
# RAM-BOUNDED TEXT CHUNKER
# -----------------------------------------------------------------------------
def _iter_bounded_document_chunks(
    text_path,
    max_chars=TOKENIZE_TEXT_CHARS,
    read_chars=TOKENIZE_READ_CHARS,
):
    """
    Yields (text_chunk, end_of_document) while keeping the text buffer bounded.

    Raw cache format is one cleaned document per line.  Huge PG-19 books are split
    at whitespace near max_chars.  A newline generates end_of_document=True, so EOS
    is still inserted once per original document.  No giant line is ever converted
    into one giant Python token list.
    """
    max_chars = int(max_chars)
    read_chars = int(read_chars)
    buf = ""
    doc_has_content = False

    with open(text_path, "r", encoding="utf-8", errors="ignore") as f:
        eof = False
        while not eof:
            chunk = f.read(read_chars)
            if chunk:
                buf += chunk
            else:
                eof = True

            while True:
                nl = buf.find("\n")

                # Complete document fits in current bounded region.
                if nl >= 0 and nl <= max_chars:
                    piece = buf[:nl]
                    buf = buf[nl + 1:]
                    if piece or doc_has_content:
                        yield piece, True
                    doc_has_content = False
                    continue

                # Document is larger than the bound: split near whitespace.
                if len(buf) > max_chars:
                    cut = buf.rfind(" ", 0, max_chars + 1)
                    if cut < max_chars // 2:
                        cut = max_chars
                    piece = buf[:cut]
                    buf = buf[cut:]
                    if piece:
                        yield piece, False
                        doc_has_content = True
                    continue

                break

        # EOF: flush the final document, still bounded.
        while len(buf) > max_chars:
            cut = buf.rfind(" ", 0, max_chars + 1)
            if cut < max_chars // 2:
                cut = max_chars
            piece = buf[:cut]
            buf = buf[cut:]
            if piece:
                yield piece, False
                doc_has_content = True

        if buf or doc_has_content:
            yield buf, True

# -----------------------------------------------------------------------------
# METHOD-SPECIFIC TOKEN ARRAYS
# -----------------------------------------------------------------------------
def token_paths(method, corpus_name):
    root = TOKEN_ROOT / method / corpus_name
    return {
        "root": root,
        "train": root / "train_blocks.npy",
        "valid": root / "validation_blocks.npy",
        "manifest": root / "manifest.json",
    }


def _array_ok(path, rows):
    path = Path(path)
    if not path.is_file():
        return False
    a = None
    try:
        a = np.load(path, mmap_mode="r", allow_pickle=False)
        if not (
            a.ndim == 2
            and tuple(a.shape) == (int(rows), int(SEQ_LENGTH))
            and np.issubdtype(a.dtype, np.integer)
        ):
            return False
        # Tiny range stub without scanning the full file.
        sample = np.asarray(a[[0, int(rows)//2, int(rows)-1]])
        return int(sample.min()) >= 0 and int(sample.max()) < VOCAB_SIZE
    except Exception:
        return False
    finally:
        if a is not None:
            try:
                if getattr(a, "_mmap", None) is not None:
                    a._mmap.close()
            except Exception:
                pass
            del a


def _encode_text_file_to_blocks_low_ram(text_path, sp, target_blocks, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    method = destination.parent.parent.name
    corpus = destination.parent.name
    split = destination.stem

    TOKENIZE_WORK.mkdir(parents=True, exist_ok=True)
    local_tmp = TOKENIZE_WORK / f"{method}__{corpus}__{split}.npy"
    local_tmp.unlink(missing_ok=True)

    target_blocks = int(target_blocks)
    out = np.lib.format.open_memmap(
        local_tmp,
        mode="w+",
        dtype=TOKEN_DTYPE,
        shape=(target_blocks, int(SEQ_LENGTH)),
    )

    block = np.empty((int(SEQ_LENGTH),), dtype=TOKEN_DTYPE)
    fill = 0
    row = 0
    chunks_seen = 0
    next_report = max(1, target_blocks // 10)
    next_report_row = next_report

    # SentencePiece outputs Python ints, but only for a SMALL bounded batch.
    batch = []

    def push_ids(ids):
        nonlocal row, fill
        if row >= target_blocks:
            return
        if ids:
            arr = np.asarray(ids, dtype=TOKEN_DTYPE)
            pos = 0
            n_ids = int(arr.size)
            while pos < n_ids and row < target_blocks:
                n = min(int(SEQ_LENGTH) - fill, n_ids - pos)
                block[fill:fill+n] = arr[pos:pos+n]
                fill += n
                pos += n
                if fill == int(SEQ_LENGTH):
                    out[row] = block
                    row += 1
                    fill = 0
            del arr

    def push_one(token_id):
        nonlocal row, fill
        if row >= target_blocks:
            return
        block[fill] = token_id
        fill += 1
        if fill == int(SEQ_LENGTH):
            out[row] = block
            row += 1
            fill = 0

    def consume_batch(items):
        nonlocal row, chunks_seen, next_report_row
        if not items or row >= target_blocks:
            return

        texts = [x[0] for x in items]
        eods = [bool(x[1]) for x in items]

        encoded = sp.encode(
            texts,
            out_type=int,
            enable_sampling=False,
            num_threads=TOKENIZE_WORKERS,
        )

        for ids, eod in zip(encoded, eods):
            push_ids(ids)
            if eod and row < target_blocks:
                push_one(EOS_ID)
            if row >= target_blocks:
                break

        chunks_seen += len(items)
        del encoded, texts, eods

        if row >= next_report_row or row >= target_blocks:
            print(
                f"    {method}/{corpus}/{split}: {row:,}/{target_blocks:,} blocks | "
                f"RSS={_rss_gib():.2f} GiB"
            )
            while next_report_row <= row:
                next_report_row += next_report

    try:
        for item in _iter_bounded_document_chunks(text_path):
            batch.append(item)
            if len(batch) >= TOKENIZE_BATCH_CHUNKS:
                consume_batch(batch)
                batch.clear()
                if row >= target_blocks:
                    break

        if row < target_blocks and batch:
            consume_batch(batch)
            batch.clear()

        out.flush()
        del out
        out = None

        if row < target_blocks:
            local_tmp.unlink(missing_ok=True)
            raise RuntimeError(
                f"{text_path}: produced only {row:,} blocks; need {target_blocks:,}. "
                "The raw-text cache is genuinely too small for this tokenizer."
            )

        # Atomic Drive write.  The large .npy lived on local SSD during tokenization.
        drive_tmp = destination.with_suffix(destination.suffix + ".tmp")
        drive_tmp.unlink(missing_ok=True)
        print(
            f"    copying completed {method}/{corpus}/{split} to Drive "
            f"({local_tmp.stat().st_size / 1024**2:.1f} MiB)..."
        )
        shutil.copyfile(local_tmp, drive_tmp)
        os.replace(drive_tmp, destination)
        local_tmp.unlink(missing_ok=True)

        if not _array_ok(destination, target_blocks):
            destination.unlink(missing_ok=True)
            raise RuntimeError(f"Post-write validation failed: {destination}")

    finally:
        if out is not None:
            try:
                out.flush()
            except Exception:
                pass
            del out
        batch.clear()
        del block
        _clean_memory()


def ensure_method_corpus_tokens(method, corpus_name):
    paths = token_paths(method, corpus_name)
    paths["root"].mkdir(parents=True, exist_ok=True)

    if (
        _array_ok(paths["train"], TRAIN_BLOCKS)
        and _array_ok(paths["valid"], VALID_BLOCKS)
        and paths["manifest"].exists()
    ):
        print(f"{method}/{corpus_name}: NEW RAM-efficient token arrays cached")
        return paths["train"], paths["valid"]

    train_text, valid_text = ensure_raw_corpus(corpus_name)
    model_path = ensure_method_tokenizer(method)
    sp = spm.SentencePieceProcessor(model_file=str(model_path))

    print(f"\nTokenizing {corpus_name} independently for {method} ...")
    print(
        f"    bounded chunks={TOKENIZE_TEXT_CHARS//1024} Ki chars | "
        f"batch={TOKENIZE_BATCH_CHUNKS} | threads={TOKENIZE_WORKERS} | "
        f"dtype={np.dtype(TOKEN_DTYPE).name} | RSS={_rss_gib():.2f} GiB"
    )

    if not _array_ok(paths["train"], TRAIN_BLOCKS):
        paths["train"].unlink(missing_ok=True)
        _encode_text_file_to_blocks_low_ram(
            train_text, sp, TRAIN_BLOCKS, paths["train"]
        )

    if not _array_ok(paths["valid"], VALID_BLOCKS):
        paths["valid"].unlink(missing_ok=True)
        _encode_text_file_to_blocks_low_ram(
            valid_text, sp, VALID_BLOCKS, paths["valid"]
        )

    atomic_json({
        "method": method,
        "corpus": corpus_name,
        "tokenizer_model": str(model_path),
        "tokenizer_sha256": sha256_file(model_path),
        "train_shape": [TRAIN_BLOCKS, int(SEQ_LENGTH)],
        "valid_shape": [VALID_BLOCKS, int(SEQ_LENGTH)],
        "token_dtype": np.dtype(TOKEN_DTYPE).name,
        "ram_bounded_tokenization": True,
        "text_chunk_chars": TOKENIZE_TEXT_CHARS,
        "batch_chunks": TOKENIZE_BATCH_CHUNKS,
        "tokenize_threads": TOKENIZE_WORKERS,
        "train_raw_sha256": sha256_file(train_text),
        "valid_raw_sha256": sha256_file(valid_text),
    }, paths["manifest"])

    del sp
    _clean_memory()
    print(f"    completed {method}/{corpus_name} | RSS={_rss_gib():.2f} GiB")
    return paths["train"], paths["valid"]

# -----------------------------------------------------------------------------
# PREPARE EVERYTHING
# -----------------------------------------------------------------------------
def prepare_all_splinet_data_ram_efficient():
    print("\n" + "=" * 92)
    print("spliNet CLEAN RAM-EFFICIENT DATA PREPARATION")
    print("=" * 92)
    print("Methods             :", DATA_METHODS)
    print("Corpora              :", list(DATASETS))
    print("Storage              :", SPLINET_ROOT)
    print("Raw caches           : PRESERVED / REUSED")
    print("Old tokenizers/tokens: DELETED ONCE")
    print("Each method          : OWN tokenizer + OWN token arrays")
    print("Tokenizer RAM policy : bounded 64K-char chunks; small batches; direct memmap")
    print("Token array dtype    :", np.dtype(TOKEN_DTYPE).name)
    print("Starting RSS         :", f"{_rss_gib():.2f} GiB")
    print("=" * 92)

    # Raw data first; already-complete caches are reused.
    ensure_c4_tokenizer_source()
    for corpus_name in DATASETS:
        ensure_raw_corpus(corpus_name)

    # Independent tokenizer per method.
    tokenizer_hashes = {}
    for method in DATA_METHODS:
        model_path = ensure_method_tokenizer(method)
        tokenizer_hashes[method] = sha256_file(model_path)

    # Every method gets its own arrays for every corpus.
    for method in DATA_METHODS:
        for corpus_name in DATASETS:
            ensure_method_corpus_tokens(method, corpus_name)
            _clean_memory()

    # Final audit.
    audit = {
        "project": "spliNet",
        "rebuild": "ram_efficient_tokenization_v1",
        "methods": DATA_METHODS,
        "corpora": list(DATASETS),
        "tokenizer_hashes": tokenizer_hashes,
        "separate_tokenizer_training_per_method": True,
        "tokenizer_reuse_across_methods": False,
        "pretrained_model_weights": False,
        "sequence_length": int(SEQ_LENGTH),
        "vocab_size": int(VOCAB_SIZE),
        "train_blocks": TRAIN_BLOCKS,
        "validation_blocks": VALID_BLOCKS,
        "token_dtype": np.dtype(TOKEN_DTYPE).name,
        "ram_bounded_tokenization": True,
        "text_chunk_chars": TOKENIZE_TEXT_CHARS,
        "tokenize_batch_chunks": TOKENIZE_BATCH_CHUNKS,
        "tokenize_threads": TOKENIZE_WORKERS,
        "python": sys.version,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "datasets": datasets.__version__,
        "sentencepiece": spm.__version__,
        "gpu": GPU_NAME,
    }
    atomic_json(audit, CONFIG_ROOT / "data_and_tokenizer_audit.json")

    print("\n" + "=" * 92)
    print("spliNet RAM-EFFICIENT DATA PREPARATION COMPLETE")
    print("=" * 92)
    for method in DATA_METHODS:
        print(f"{method:8s} tokenizer sha256: {tokenizer_hashes[method]}")
    print("Token root :", TOKEN_ROOT)
    print("Final RSS  :", f"{_rss_gib():.2f} GiB")
    print("=" * 92)


prepare_all_splinet_data_ram_efficient()


spliNet methods : ['mamba2', 'hyena', 'retnet']
Mamba backend   : huggingface_native_mamba2
datasets        : 4.8.5
sentencepiece   : 0.2.2
torch           : 2.11.0+cu130
CUDA            : 13.0
physical cores  : 24

CLEAN REBUILD: deleting ALL previous tokenizer-dependent spliNet artifacts...
RAW text and HF download cache are PRESERVED.
  deleting: /content/drive/MyDrive/spliNet/tokenizers
  deleting: /content/drive/MyDrive/spliNet/tokens
  deleting: /content/drive/MyDrive/spliNet/checkpoints
  deleting: /content/drive/MyDrive/spliNet/results
  deleting: /content/spliNet/staged
  deleting: /content/spliNet/tmp
  deleting: /content/spliNet_tokenize_work
Old tokenizer-dependent artifacts deleted. Clean rebuild marker written.

spliNet CLEAN RAM-EFFICIENT DATA PREPARATION
Methods             : ['mamba2', 'hyena', 'retnet']
Corpora              : ['fineweb_edu', 'arxiv', 'pg19', 'wikitext103']
Storage              : /content/drive/MyDrive/spliNet
Raw caches           : PRESERVED / REUSED


In [16]:
# CELL 3 — spliNet TRAINING v3 — G4 HIGH-VRAM MAX THROUGHPUT, DRIVE CHECKPOINTS
# Run only after Cell 1 and the clean spliNet data/tokenizer preparation cell finish.
# After a runtime disconnect: run Cell 1, then THIS cell. Completed runs skip; partial runs resume.

from pathlib import Path
import os
import gc
import json
import time
import shutil
import hashlib
import numpy as np

EXPECTED_CELL1_VERSION = "FIX2-HF-AUTO-MAMBA-NO-TOKENIZER-FILE-2026-10-09"
if globals().get("CELL1_VERSION") != EXPECTED_CELL1_VERSION:
    raise RuntimeError(
        "Run the corrected spliNet Cell 1 first. "
        f"Expected {EXPECTED_CELL1_VERSION!r}, got {globals().get('CELL1_VERSION')!r}."
    )

_REQUIRED = (
    "torch", "DEVICE", "GPU_NAME", "VRAM_GB", "METHODS", "MAMBA_BACKEND",
    "SEQ_LENGTH", "VOCAB_SIZE", "EXPECTED_TRAIN_BLOCKS", "EXPECTED_VALID_BLOCKS",
    "EFFECTIVE_BATCH_BY_CORPUS", "train_one", "save_summary", "hard_cleanup",
    "autotune_microbatch", "PROTOCOL_VERSION", "IMPLEMENTATION_VERSION",
)
_missing = [x for x in _REQUIRED if x not in globals()]
if _missing:
    raise RuntimeError(f"Cell 1 is incomplete; missing definitions: {_missing}")

# -----------------------------------------------------------------------------
# spliNet paths — Drive is persistent; training arrays are staged to local SSD.
# -----------------------------------------------------------------------------
SPLINET_ROOT = Path("/content/drive/MyDrive/spliNet")
TOKENIZER_ROOT = SPLINET_ROOT / "tokenizers"
TOKEN_ROOT = SPLINET_ROOT / "tokens"
CHECKPOINT_ROOT = SPLINET_ROOT / "checkpoints"
RESULTS_ROOT = SPLINET_ROOT / "results"
CONFIG_ROOT = SPLINET_ROOT / "configs"
LOCAL_TRAIN_ROOT = Path("/content/spliNet/staged_training")

for _p in (SPLINET_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT, CONFIG_ROOT, LOCAL_TRAIN_ROOT):
    _p.mkdir(parents=True, exist_ok=True)

# Keep Cell-1 helpers pointed at spliNet.
PROJECT_ROOT = SPLINET_ROOT

CORPUS_ORDER = ["fineweb_edu", "arxiv", "pg19", "wikitext103"]
TRAIN_METHODS = list(dict.fromkeys(METHODS))

if len(TRAIN_METHODS) != 3 or not {"hyena", "retnet"}.issubset(TRAIN_METHODS):
    raise RuntimeError(f"Unexpected method set: {TRAIN_METHODS}")
if not any(m in TRAIN_METHODS for m in ("mamba2", "mamba")):
    raise RuntimeError(f"No HF Mamba method selected: {TRAIN_METHODS}")

# save_summary()/collect_all_results() only use the names from CORPORA.
CORPORA = [{"name": c} for c in CORPUS_ORDER]

# -----------------------------------------------------------------------------
# G4 HIGH-VRAM throughput policy.
#
# Objective requested:
#     maximize measured end-to-end training throughput
#     SUBJECT TO using roughly 82--90 GiB allocated VRAM when achievable.
#
# Reserved memory gets a separate hard cap to preserve CUDA workspace headroom.
# -----------------------------------------------------------------------------
AUTOTUNE_MIN_ALLOCATED_GB = min(82.0, max(1.0, float(VRAM_GB) - 12.0))
AUTOTUNE_MAX_ALLOCATED_GB = min(90.0, max(1.0, float(VRAM_GB) - 4.0))
AUTOTUNE_MAX_RESERVED_GB = min(92.0, max(1.0, float(VRAM_GB) - 2.5))

AUTOTUNE_VERSION = "splinet_high_vram_82_90gib_v3"
AUTOTUNE_WARMUP_UPDATES = 1
AUTOTUNE_TIMED_UPDATES = 3
REAUTOTUNE_ON_RESUME = True

# Old low-memory autotune files are intentionally invalid for this policy.
for _old_tune in CONFIG_ROOT.glob(
    f"autotune_{PROTOCOL_VERSION}_splinet_*90gib_v[12]_*"
):
    try:
        _old_tune.unlink()
    except Exception:
        pass

# Persistent recovery to Drive: whichever occurs first.
SAVE_EVERY = 500
SAVE_EVERY_MINUTES = 20
SAVE_FINAL_MODEL = True

# Host-side MLM preparation is tiny compared with model compute, but do not
# artificially restrict the large Colab host to eight CPU threads.
CPU_THREADS = max(1, min(32, os.cpu_count() or 8))
torch.set_num_threads(CPU_THREADS)

# Blackwell fast settings (idempotent if Cell 1 already set them).
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = True
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = True
except Exception:
    pass
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass



# =============================================================================
# CORRECTED HIGH-VRAM AUTOTUNER + ARBITRARY-MICROBATCH TRAINER
# =============================================================================

def _benchmark_candidate(
    kind,
    micro_batch,
    effective_batch,
):
    micro_batch = int(micro_batch)
    effective_batch = int(effective_batch)

    if micro_batch < 1 or micro_batch > effective_batch:
        raise ValueError(
            f"Invalid micro_batch={micro_batch} for effective_batch={effective_batch}"
        )

    grad_accum = int(
        math.ceil(
            effective_batch
            /
            micro_batch
        )
    )

    hard_cleanup(
        f"before probe {kind} mb={micro_batch}"
    )

    model = None
    optimizer = None
    scaler = None
    ids = None
    labels = None

    try:
        model = build_model(
            kind
        ).to(
            DEVICE
        )

        model.train()

        optimizer, _ = make_optimizer(
            model
        )

        scaler = make_scaler()

        # Allocate at the candidate's PEAK physical microbatch.
        ids = torch.randint(
            7,
            VOCAB_SIZE,
            (
                micro_batch,
                SEQ_LENGTH,
            ),
            device=DEVICE,
            dtype=torch.long,
        )

        labels = torch.full_like(
            ids,
            -100,
        )

        # Exactly 80 targets / row so arbitrary final chunks can be
        # weighted exactly by row count during the synthetic benchmark.
        labels[
            :,
            1:81
        ] = ids[
            :,
            1:81
        ]

        def one_update():
            optimizer.zero_grad(
                set_to_none=True
            )

            processed = 0

            while processed < effective_batch:
                current = min(
                    micro_batch,
                    effective_batch - processed,
                )

                weight = (
                    float(current)
                    /
                    float(effective_batch)
                )

                with torch.autocast(
                    "cuda",
                    dtype=AMP_DTYPE,
                ):
                    logits, targets = model(
                        ids[:current],
                        labels[:current],
                    )

                    loss = F.cross_entropy(
                        logits,
                        targets,
                    )

                    scaled_loss = (
                        loss
                        *
                        weight
                    )

                scaler.scale(
                    scaled_loss
                ).backward()

                del (
                    logits,
                    targets,
                    loss,
                    scaled_loss,
                )

                processed += current

            scaler.step(
                optimizer
            )

            scaler.update()

        # Warmup includes optimizer-state allocation and kernel init.
        for _ in range(
            AUTOTUNE_WARMUP_UPDATES
        ):
            one_update()

        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

        t0 = time.perf_counter()

        for _ in range(
            AUTOTUNE_TIMED_UPDATES
        ):
            one_update()

        torch.cuda.synchronize()

        elapsed = (
            time.perf_counter()
            -
            t0
        )

        peak_allocated = (
            torch.cuda.max_memory_allocated()
            /
            1024**3
        )

        peak_reserved = (
            torch.cuda.max_memory_reserved()
            /
            1024**3
        )

        tokens_per_second = (
            AUTOTUNE_TIMED_UPDATES
            *
            effective_batch
            *
            SEQ_LENGTH
            /
            max(
                elapsed,
                1.0e-9,
            )
        )

        safe = (
            peak_allocated
            <=
            AUTOTUNE_MAX_ALLOCATED_GB
            and
            peak_reserved
            <=
            AUTOTUNE_MAX_RESERVED_GB
        )

        in_target_window = (
            safe
            and
            peak_allocated
            >=
            AUTOTUNE_MIN_ALLOCATED_GB
        )

        return {
            "micro_batch":
                int(micro_batch),

            "grad_accum":
                int(grad_accum),

            "tokens_per_second":
                float(tokens_per_second),

            "peak_allocated_gb":
                float(peak_allocated),

            "peak_reserved_gb":
                float(peak_reserved),

            "safe":
                bool(safe),

            "in_target_window":
                bool(in_target_window),
        }

    except (
        torch.cuda.OutOfMemoryError,
        RuntimeError,
    ) as exc:
        msg = str(exc).lower()

        if (
            not isinstance(
                exc,
                torch.cuda.OutOfMemoryError,
            )
            and
            "out of memory"
            not in msg
        ):
            raise

        return {
            "micro_batch":
                int(micro_batch),

            "grad_accum":
                int(grad_accum),

            "tokens_per_second":
                0.0,

            "peak_allocated_gb":
                float("inf"),

            "peak_reserved_gb":
                float("inf"),

            "safe":
                False,

            "in_target_window":
                False,
        }

    finally:
        ids = None
        labels = None
        scaler = None
        optimizer = None
        model = None

        hard_cleanup(
            f"after probe {kind} mb={micro_batch}"
        )

def autotune_microbatch(
    kind,
    effective_batch,
):
    gpu_tag = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        GPU_NAME,
    ).strip("_")

    cache_path = (
        CONFIG_ROOT
        /
        (
            f"autotune_{PROTOCOL_VERSION}_"
            f"{AUTOTUNE_VERSION}_"
            f"{gpu_tag}_{kind}_"
            f"{MAMBA_BACKEND if kind in ('mamba2','mamba') else 'native'}_"
            f"eb{effective_batch}.json"
        )
    )

    if cache_path.exists():
        try:
            cached = json.loads(
                cache_path.read_text()
            )

            if (
                cached.get("protocol_version")
                ==
                PROTOCOL_VERSION
                and
                cached.get("autotune_version")
                ==
                AUTOTUNE_VERSION
                and
                cached.get("gpu")
                ==
                GPU_NAME
                and
                int(
                    cached.get(
                        "effective_batch"
                    )
                )
                ==
                int(
                    effective_batch
                )
                and
                cached.get("kind")
                ==
                kind
            ):
                print(
                    f"Reusing G4 high-VRAM autotune: "
                    f"{kind} | "
                    f"micro={cached['micro_batch']} x "
                    f"{cached['grad_accum']} | "
                    f"{cached['tokens_per_second']:,.0f} tok/s | "
                    f"alloc/res="
                    f"{cached['peak_allocated_gb']:.2f}/"
                    f"{cached['peak_reserved_gb']:.2f} GiB"
                )

                return cached

        except Exception:
            pass

    effective_batch = int(
        effective_batch
    )

    # IMPORTANT:
    # Every integer microbatch is legal.  The last accumulation chunk can
    # be shorter, and train_one weights it exactly by masked-target count.
    candidates = list(
        range(
            effective_batch,
            0,
            -1,
        )
    )

    results = []
    preferred = []
    safe = []

    below_floor_streak = 0

    print(
        f"\nG4 HIGH-VRAM autotuning {kind} on "
        f"{GPU_NAME} / {VRAM_GB:.2f} GiB..."
    )

    print(
        f"  objective: maximum measured throughput "
        f"with allocated VRAM in "
        f"[{AUTOTUNE_MIN_ALLOCATED_GB:.1f}, "
        f"{AUTOTUNE_MAX_ALLOCATED_GB:.1f}] GiB"
    )

    print(
        f"  hard reserved-memory cap: "
        f"{AUTOTUNE_MAX_RESERVED_GB:.1f} GiB"
    )

    print(
        f"  candidate microbatches: "
        f"{effective_batch}..1 "
        f"(arbitrary final short chunk allowed)"
    )

    print(
        f"  warmup={AUTOTUNE_WARMUP_UPDATES}; "
        f"timed={AUTOTUNE_TIMED_UPDATES} full effective update(s)"
    )

    for mb in candidates:
        r = _benchmark_candidate(
            kind,
            mb,
            effective_batch,
        )

        results.append(
            r
        )

        if r["safe"]:
            safe.append(
                r
            )

            if r[
                "in_target_window"
            ]:
                preferred.append(
                    r
                )

            print(
                f"  mb={mb:3d} x "
                f"{r['grad_accum']:3d} | "
                f"{'TARGET' if r['in_target_window'] else 'SAFE  '} | "
                f"{r['tokens_per_second']:,.0f} tok/s | "
                f"alloc/res "
                f"{r['peak_allocated_gb']:.2f}/"
                f"{r['peak_reserved_gb']:.2f} GiB"
            )

            if (
                r[
                    "peak_allocated_gb"
                ]
                <
                AUTOTUNE_MIN_ALLOCATED_GB
            ):
                below_floor_streak += 1
            else:
                below_floor_streak = 0

        else:
            print(
                f"  mb={mb:3d} x "
                f"{r['grad_accum']:3d} | "
                f"REJECT/OOM"
            )

            below_floor_streak = 0

        # Microbatches are scanned from large -> small. Once several
        # consecutive safe candidates are already below the target VRAM
        # floor, smaller candidates will not help satisfy the user's
        # high-VRAM constraint.  We have benchmarked the complete target
        # window plus a guard band below it.
        if (
            below_floor_streak
            >=
            3
            and
            (
                preferred
                or
                safe
            )
        ):
            print(
                "  target-memory region fully traversed; "
                "stopping lower-memory probes."
            )
            break

    if not safe:
        raise RuntimeError(
            f"No safe physical batch found for {kind}."
        )

    if preferred:
        # EXACT requested policy:
        # maximize throughput subject to high G4 memory occupancy.
        best = max(
            preferred,
            key=lambda r:
                r[
                    "tokens_per_second"
                ],
        )

        selection_reason = (
            "fastest measured candidate "
            "inside target allocated-VRAM window"
        )

    else:
        # If the fixed effective batch itself cannot reach the requested
        # memory floor (possible for a lighter model), use the highest
        # physically achievable safe occupancy; throughput breaks ties.
        best = max(
            safe,
            key=lambda r: (
                r[
                    "peak_allocated_gb"
                ],
                r[
                    "tokens_per_second"
                ],
            ),
        )

        selection_reason = (
            "target VRAM floor unreachable under fixed effective batch; "
            "selected highest safe allocated VRAM"
        )

    print(
        f"Chosen {kind}: "
        f"microbatch={best['micro_batch']}, "
        f"accum={best['grad_accum']}, "
        f"{best['tokens_per_second']:,.0f} tok/s, "
        f"alloc/res="
        f"{best['peak_allocated_gb']:.2f}/"
        f"{best['peak_reserved_gb']:.2f} GiB"
    )

    print(
        "  selection:",
        selection_reason,
    )

    best = dict(
        best
    )

    best.update({
        "protocol_version":
            PROTOCOL_VERSION,

        "autotune_version":
            AUTOTUNE_VERSION,

        "gpu":
            GPU_NAME,

        "torch":
            torch.__version__,

        "cuda":
            torch.version.cuda,

        "kind":
            kind,

        "mamba_backend":
            MAMBA_BACKEND
            if kind in (
                "mamba2",
                "mamba",
            )
            else None,

        "transformers":
            HF_TRANSFORMERS_VERSION,

        "effective_batch":
            int(
                effective_batch
            ),

        "warmup_updates":
            int(
                AUTOTUNE_WARMUP_UPDATES
            ),

        "timed_updates":
            int(
                AUTOTUNE_TIMED_UPDATES
            ),

        "target_allocated_min_gb":
            float(
                AUTOTUNE_MIN_ALLOCATED_GB
            ),

        "target_allocated_max_gb":
            float(
                AUTOTUNE_MAX_ALLOCATED_GB
            ),

        "hard_reserved_cap_gb":
            float(
                AUTOTUNE_MAX_RESERVED_GB
            ),

        "selection_reason":
            selection_reason,

        "all_candidates":
            results,
    })

    tmp = cache_path.with_suffix(
        cache_path.suffix
        +
        ".tmp"
    )

    tmp.write_text(
        json.dumps(
            best,
            indent=2,
        )
    )

    os.replace(
        tmp,
        cache_path,
    )

    return best

def train_one(
    corpus_name,
    kind,
    train_path,
    valid_path,
):
    run_dir = (
        CHECKPOINT_ROOT
        /
        corpus_name
        /
        kind
    )

    run_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    latest_file = (
        run_dir
        /
        "training_latest.pt"
    )

    final_file = (
        run_dir
        /
        "trained_model_final.pt"
    )

    complete_file = (
        run_dir
        /
        "TRAINING_COMPLETE.json"
    )

    log_file = (
        run_dir
        /
        "training_log.jsonl"
    )

    if complete_file.exists():
        result = json.loads(
            complete_file.read_text()
        )

        if (
            result.get(
                "protocol_version"
            )
            !=
            PROTOCOL_VERSION
        ):
            raise RuntimeError(
                f"{corpus_name}/{kind}: existing completion manifest "
                f"belongs to protocol "
                f"{result.get('protocol_version')!r}, not "
                f"{PROTOCOL_VERSION!r}. Refusing to mix protocols."
            )

        print(
            f"{corpus_name}/{kind}: "
            f"already COMPLETE — skipping."
        )

        return result

    train = None
    valid = None
    model = None
    optimizer = None
    scheduler = None
    scaler = None
    executor = None

    hard_cleanup(
        f"before {corpus_name}/{kind}"
    )

    try:
        train = np.load(
            train_path,
            mmap_mode="r",
        )

        valid = np.load(
            valid_path,
            mmap_mode="r",
        )

        effective_batch = int(
            EFFECTIVE_BATCH_BY_CORPUS[corpus_name]
        )

        usable_rows = (
            len(train)
            //
            effective_batch
        ) * effective_batch

        total_updates = (
            usable_rows
            //
            effective_batch
        )

        actual_train_tokens = (
            total_updates
            *
            effective_batch
            *
            SEQ_LENGTH
        )

        rng = np.random.default_rng(
            DATA_ORDER_SEED
        )

        permutation = rng.permutation(
            len(train)
        )[
            :usable_rows
        ]

        # Resume first; do not autotune again after a partial run.
        resume = None

        if latest_file.exists():
            resume = torch.load(
                latest_file,
                map_location="cpu",
                weights_only=False,
            )

            if (
                resume.get(
                    "protocol_version"
                )
                !=
                PROTOCOL_VERSION
            ):
                raise RuntimeError(
                    f"{corpus_name}/{kind}: recovery checkpoint "
                    f"uses incompatible protocol."
                )

            old_micro = int(
                resume["micro_batch"]
            )
            old_accum = int(
                resume["gradient_accumulation"]
            )

            print(
                f"{corpus_name}/{kind}: "
                f"resume update {resume['next_update']} | "
                f"old micro={old_micro} x {old_accum}"
            )

            if REAUTOTUNE_ON_RESUME:
                tune = autotune_microbatch(
                    kind,
                    effective_batch,
                )
                micro_batch = int(
                    tune["micro_batch"]
                )
                grad_accum = int(
                    tune["grad_accum"]
                )
                print(
                    f"  optimized resume partition: "
                    f"micro={micro_batch} x {grad_accum}"
                )
            else:
                micro_batch = old_micro
                grad_accum = old_accum

        else:
            tune = autotune_microbatch(
                kind,
                effective_batch,
            )

            micro_batch = int(
                tune[
                    "micro_batch"
                ]
            )

            grad_accum = int(
                tune[
                    "grad_accum"
                ]
            )

        expected_accum = int(
            math.ceil(
                effective_batch
                /
                micro_batch
            )
        )

        if grad_accum != expected_accum:
            raise RuntimeError(
                "Microbatch plan mismatch: "
                f"micro={micro_batch}, accum={grad_accum}, "
                f"effective={effective_batch}."
            )

        # Fresh deterministic construction.
        set_all_seeds(
            GLOBAL_SEED
        )

        model = build_model(
            kind
        ).to(
            DEVICE
        )

        total_params, trainable_params = (
            parameter_counts(
                model
            )
        )

        model.train()

        optimizer, fused_adam = (
            make_optimizer(
                model
            )
        )

        scheduler, warmup_steps = (
            make_scheduler(
                optimizer,
                total_updates,
            )
        )

        scaler = make_scaler()

        next_update = 0

        if resume is not None:
            resume_model_state = (
                upgrade_model_state_dict_for_fast_impl(
                    kind,
                    resume["model_state"],
                )
            )

            model.load_state_dict(
                resume_model_state,
                strict=True,
            )
            del resume_model_state

            resume_impl = resume.get(
                "implementation_version"
            )

            if (
                kind == "retnet"
                and
                resume_impl != IMPLEMENTATION_VERSION
            ):
                raise RuntimeError(
                    "Old partial RetNet checkpoint uses the previous q/k/v/g "
                    "optimizer layout. Delete ONLY that RetNet training_latest.pt; "
                    "no runtime restart is required."
                )

            optimizer.load_state_dict(
                resume["optimizer_state"]
            )

            optimizer_to_gpu(
                optimizer
            )

            scheduler.load_state_dict(
                resume[
                    "scheduler_state"
                ]
            )

            scaler.load_state_dict(
                resume[
                    "scaler_state"
                ]
            )

            next_update = int(
                resume[
                    "next_update"
                ]
            )

            restore_rng(
                resume[
                    "rng_state"
                ]
            )

            del resume
            hard_cleanup(
                f"post-resume CPU cleanup {corpus_name}/{kind}"
            )

        def save_recovery(
            next_step,
        ):
            atomic_torch_save(
                {
                    "protocol_version":
                        PROTOCOL_VERSION,

                    "implementation_version":
                        IMPLEMENTATION_VERSION,

                    "corpus":
                        corpus_name,

                    "kind":
                        kind,

                    "model_state":
                        model.state_dict(),

                    "optimizer_state":
                        optimizer.state_dict(),

                    "scheduler_state":
                        scheduler.state_dict(),

                    "scaler_state":
                        scaler.state_dict(),

                    "next_update":
                        int(next_step),

                    "micro_batch":
                        int(micro_batch),

                    "gradient_accumulation":
                        int(grad_accum),

                    "effective_batch":
                        effective_batch,

                    "amp_dtype":
                        AMP_NAME,

                    "rng_state":
                        capture_rng(),
                },
                latest_file,
            )

        def append_log(
            record,
        ):
            with open(
                log_file,
                "a",
                encoding="utf-8",
            ) as f:
                f.write(
                    json.dumps(
                        record
                    )
                    +
                    "\n"
                )

        def prepare_update(
            update_index,
        ):
            idx = permutation[
                update_index
                *
                effective_batch
                :
                (
                    update_index
                    +
                    1
                )
                *
                effective_batch
            ]

            return make_mlm_batch(
                train[idx],
                update_index,
                validation=False,
            )

        print(
            "\n"
            +
            "=" * 80
        )

        print(
            f"RUN: {corpus_name} / {kind}"
        )

        print(
            "GPU                  :",
            GPU_NAME,
        )

        print(
            "AMP                  :",
            AMP_NAME,
        )

        print(
            "Activation checkpoint:",
            "OFF",
        )

        print(
            "Fused Adam           :",
            fused_adam,
        )

        print(
            "Parameters           :",
            f"{total_params:,}",
        )

        print(
            "Trainable parameters :",
            f"{trainable_params:,}",
        )

        print(
            "Microbatch           :",
            micro_batch,
        )

        print(
            "Gradient accumulation:",
            grad_accum,
        )

        print(
            "Effective batch       :",
            effective_batch,
        )

        print(
            "Updates               :",
            total_updates,
        )

        print(
            "Training tokens       :",
            f"{actual_train_tokens:,}",
        )

        print(
            "Warmup updates        :",
            warmup_steps,
        )

        print(
            "=" * 80
        )

        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

        session_start = (
            time.perf_counter()
        )

        start_update = (
            next_update
        )

        running_loss_gpu = torch.zeros(
            (),
            dtype=torch.float32,
            device=DEVICE,
        )

        running_count = 0
        last_completed = next_update
        last_checkpoint_wall = time.monotonic()

        # CPU SSD-read + vectorized MLM preparation overlaps GPU compute.
        executor = ThreadPoolExecutor(
            max_workers=1
        )

        future = (
            executor.submit(
                prepare_update,
                next_update,
            )
            if next_update
            <
            total_updates
            else None
        )

        try:
            for update_index in range(
                next_update,
                total_updates,
            ):
                ids_cpu, labels_cpu = (
                    future.result()
                )

                if (
                    update_index + 1
                    <
                    total_updates
                ):
                    future = executor.submit(
                        prepare_update,
                        update_index + 1,
                    )
                else:
                    future = None

                optimizer.zero_grad(
                    set_to_none=True
                )

                # Exact arbitrary-microbatch accumulation.  Weight each
                # chunk by the number of actual masked MLM targets so a
                # short final chunk is equivalent to one full update.
                row_masked_counts = (
                    labels_cpu
                    .ne(-100)
                    .sum(dim=1)
                )
                total_masked_targets = int(
                    row_masked_counts
                    .sum()
                    .item()
                )

                if total_masked_targets <= 0:
                    raise RuntimeError(
                        "MLM update contains zero masked targets."
                    )

                for mstart in range(
                    0,
                    effective_batch,
                    micro_batch,
                ):
                    mend = min(
                        mstart + micro_batch,
                        effective_batch,
                    )

                    chunk_masked_targets = int(
                        row_masked_counts[
                            mstart:mend
                        ]
                        .sum()
                        .item()
                    )

                    chunk_weight = (
                        float(chunk_masked_targets)
                        /
                        float(total_masked_targets)
                    )
                    ids = ids_cpu[
                        mstart:mend
                    ].to(
                        DEVICE,
                        non_blocking=True,
                    )

                    labels = labels_cpu[
                        mstart:mend
                    ].to(
                        DEVICE,
                        non_blocking=True,
                    )

                    with torch.autocast(
                        "cuda",
                        dtype=AMP_DTYPE,
                    ):
                        logits, targets = model(
                            ids,
                            labels,
                        )

                        loss = F.cross_entropy(
                            logits,
                            targets,
                        )

                        scaled_loss = (
                            loss
                            *
                            chunk_weight
                        )

                    scaler.scale(
                        scaled_loss
                    ).backward()

                    running_loss_gpu += (
                        loss.detach().float()
                        *
                        chunk_weight
                    )

                    del (
                        ids,
                        labels,
                        logits,
                        targets,
                        loss,
                        scaled_loss,
                    )

                del (
                    ids_cpu,
                    labels_cpu,
                    row_masked_counts,
                )

                scale_before = (
                    scaler.get_scale()
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                scale_after = (
                    scaler.get_scale()
                )

                if (
                    USE_SCALER
                    and
                    scale_after
                    <
                    scale_before
                ):
                    raise FloatingPointError(
                        "AMP skipped an optimizer update "
                        "because of non-finite gradients."
                    )

                scheduler.step()

                completed = (
                    update_index
                    +
                    1
                )

                last_completed = (
                    completed
                )

                running_count += 1

                if (
                    completed
                    %
                    LOG_EVERY
                    ==
                    0
                    or
                    completed
                    ==
                    total_updates
                ):
                    torch.cuda.synchronize()

                    elapsed = (
                        time.perf_counter()
                        -
                        session_start
                    )

                    session_updates = (
                        completed
                        -
                        start_update
                    )

                    tps = (
                        session_updates
                        *
                        effective_batch
                        *
                        SEQ_LENGTH
                        /
                        max(
                            elapsed,
                            1.0e-9,
                        )
                    )

                    eta_hours = (
                        (
                            total_updates
                            -
                            completed
                        )
                        *
                        effective_batch
                        *
                        SEQ_LENGTH
                        /
                        max(
                            tps,
                            1.0e-9,
                        )
                        /
                        3600.0
                    )

                    peak = (
                        torch.cuda
                        .max_memory_allocated()
                        /
                        1024**3
                    )

                    peak_reserved = (
                        torch.cuda
                        .max_memory_reserved()
                        /
                        1024**3
                    )

                    avg_loss = (
                        float(
                            running_loss_gpu
                            .detach()
                            .cpu()
                        )
                        /
                        max(
                            running_count,
                            1,
                        )
                    )

                    print(
                        f"{corpus_name:12s} | "
                        f"{kind:7s} | "
                        f"{completed:4d}/{total_updates} | "
                        f"loss {avg_loss:.4f} | "
                        f"{tps:,.0f} tok/s | "
                        f"VRAM alloc/res "
                        f"{peak:.2f}/{peak_reserved:.2f} GB | "
                        f"ETA {eta_hours:.2f} h"
                    )

                    append_log(
                        {
                            "update":
                                completed,

                            "loss":
                                avg_loss,

                            "tokens_per_second":
                                tps,

                            "peak_vram_gb":
                                peak,

                            "peak_reserved_vram_gb":
                                peak_reserved,

                            "eta_hours":
                                eta_hours,

                            "micro_batch":
                                micro_batch,

                            "amp_dtype":
                                AMP_NAME,
                        }
                    )

                    running_loss_gpu.zero_()
                    running_count = 0

                now_wall = time.monotonic()
                if (
                    completed % SAVE_EVERY == 0
                    or (now_wall - last_checkpoint_wall) >= SAVE_EVERY_MINUTES * 60
                ):
                    save_recovery(completed)
                    last_checkpoint_wall = time.monotonic()
                    print(
                        f"  persistent Drive recovery checkpoint saved at update {completed}."
                    )

            executor.shutdown(
                wait=True
            )
            executor = None

        except BaseException:
            if executor is not None:
                executor.shutdown(
                    wait=False,
                    cancel_futures=True,
                )
                executor = None

            try:
                save_recovery(
                    last_completed
                )
                print(
                    f"Emergency recovery saved at "
                    f"update {last_completed}."
                )
            except Exception as save_exc:
                print(
                    "Emergency recovery save failed:",
                    save_exc,
                )

            raise

        # Critical restart boundary: if the runtime dies during validation,
        # resumption starts at the end of training instead of replaying updates.
        save_recovery(total_updates)
        last_checkpoint_wall = time.monotonic()

        print(
            f"\n{corpus_name}/{kind}: "
            f"running FULL validation..."
        )

        # Validation has no backward graph, so start aggressively and halve only on OOM.
        validation_micro_batch = effective_batch
        while True:
            try:
                final_val = evaluate(
                    model,
                    valid,
                    validation_micro_batch,
                    effective_batch,
                )
                break
            except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
                if "out of memory" not in str(exc).lower() and not isinstance(exc, torch.cuda.OutOfMemoryError):
                    raise
                if validation_micro_batch <= 1:
                    raise
                validation_micro_batch = max(1, validation_micro_batch // 2)
                torch.cuda.empty_cache()
                print(f"Validation OOM; retrying with microbatch={validation_micro_batch}")

        final_ppl = math.exp(
            min(
                final_val,
                20.0,
            )
        )

        peak = (
            torch.cuda
            .max_memory_allocated()
            /
            1024**3
        )

        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            /
            1024**3
        )

        torch.cuda.synchronize()

        train_elapsed = (
            time.perf_counter()
            -
            session_start
        )

        training_tps = (
            (
                total_updates
                -
                start_update
            )
            *
            effective_batch
            *
            SEQ_LENGTH
            /
            max(
                train_elapsed,
                1.0e-9,
            )
        )

        result = {
            "status":
                "complete",

            "protocol_version":
                PROTOCOL_VERSION,

            "implementation_version":
                IMPLEMENTATION_VERSION,

            "corpus":
                corpus_name,

            "model":
                kind,

            "objective":
                "masked_language_modeling",

            "bidirectional_shared_mixer":
                True,

            "stacked_bidirectional_execution":
                True,

            "masked_only_mlm_head":
                MASKED_ONLY_MLM_HEAD,

            "retention_chunk":
                RETENTION_CHUNK if kind == "retnet" else None,

            "training_tokens":
                int(
                    actual_train_tokens
                ),

            "optimizer_updates":
                int(
                    total_updates
                ),

            "validation_loss":
                float(
                    final_val
                ),

            "validation_perplexity":
                float(
                    final_ppl
                ),

            "micro_batch":
                int(
                    micro_batch
                ),

            "gradient_accumulation":
                int(
                    grad_accum
                ),

            "effective_batch":
                effective_batch,

            "amp_dtype":
                AMP_NAME,

            "transformers_version":
                HF_TRANSFORMERS_VERSION,

            "mamba_backend":
                MAMBA_BACKEND if kind in ("mamba2", "mamba") else None,

            "activation_checkpointing":
                False,

            "training_tokens_per_second":
                float(
                    training_tps
                ),

            "peak_training_vram_gb":
                float(
                    peak
                ),

            "peak_training_reserved_vram_gb":
                float(
                    peak_reserved
                ),

            "parameters":
                int(
                    total_params
                ),

            "trainable_parameters":
                int(
                    trainable_params
                ),

            "gpu":
                GPU_NAME,

            "torch":
                torch.__version__,

            "cuda":
                torch.version.cuda,

            "tokenizer_sha256":
                TOKENIZER_SHA256,

            "pretrained_weights":
                False,

            "save_final_model":
                bool(
                    SAVE_FINAL_MODEL
                ),
        }

        # Completion manifest FIRST: this is what controls skip-on-rerun.
        complete_file.write_text(
            json.dumps(
                result,
                indent=2,
            )
        )

        if SAVE_FINAL_MODEL:
            atomic_torch_save(
                {
                    **result,
                    "model_state":
                        model.state_dict(),
                },
                final_file,
            )

        # Once the manifest exists, a recovery checkpoint is no longer needed.
        latest_file.unlink(
            missing_ok=True
        )

        print(
            f"COMPLETE {corpus_name}/{kind} | "
            f"val={final_val:.6f} | "
            f"ppl={final_ppl:.3f} | "
            f"{training_tps:,.0f} tok/s"
        )

        return result

    finally:
        if executor is not None:
            try:
                executor.shutdown(
                    wait=False,
                    cancel_futures=True,
                )
            except Exception:
                pass

        executor = None
        scaler = None
        scheduler = None
        optimizer = None
        model = None

        if train is not None:
            try:
                if (
                    hasattr(
                        train,
                        "_mmap",
                    )
                    and
                    train._mmap
                    is not None
                ):
                    train._mmap.close()
            except Exception:
                pass

        if valid is not None:
            try:
                if (
                    hasattr(
                        valid,
                        "_mmap",
                    )
                    and
                    valid._mmap
                    is not None
                ):
                    valid._mmap.close()
            except Exception:
                pass

        train = None
        valid = None

        hard_cleanup(
            f"AFTER METHOD {corpus_name}/{kind}"
        )

# -----------------------------------------------------------------------------
# Input validation / method-specific tokenizer binding.
# -----------------------------------------------------------------------------
def _sha256(path: Path, block=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(block)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _token_paths(method: str, corpus: str):
    root = TOKEN_ROOT / method / corpus
    return root / "train_blocks.npy", root / "validation_blocks.npy", root / "manifest.json"


def _array_probe(path: Path, rows: int, label: str):
    path = Path(path)
    if not path.is_file():
        return False, f"missing: {path}"
    a = None
    try:
        a = np.load(path, mmap_mode="r", allow_pickle=False)
        expected = (int(rows), int(SEQ_LENGTH))
        if tuple(a.shape) != expected:
            return False, f"{label}: shape={tuple(a.shape)}, expected={expected}"
        if not np.issubdtype(a.dtype, np.integer):
            return False, f"{label}: dtype={a.dtype}, expected integer"
        sample = np.asarray(a[[0, rows // 2, rows - 1]])
        lo, hi = int(sample.min()), int(sample.max())
        if lo < 0 or hi >= int(VOCAB_SIZE):
            return False, f"{label}: sampled token range [{lo},{hi}] outside [0,{VOCAB_SIZE-1}]"
        return True, f"{label}: PASS | shape={expected} | dtype={a.dtype}"
    except Exception as exc:
        return False, f"{label}: {type(exc).__name__}: {exc}"
    finally:
        if a is not None:
            try:
                mm = getattr(a, "_mmap", None)
                if mm is not None:
                    mm.close()
            except Exception:
                pass
            del a


def _method_input_fingerprint(method: str, corpus: str):
    tok = TOKENIZER_ROOT / method / "spiece.model"
    train, valid, manifest = _token_paths(method, corpus)
    if not tok.is_file():
        raise FileNotFoundError(tok)
    if not manifest.is_file():
        raise FileNotFoundError(manifest)
    meta = json.loads(manifest.read_text())
    tok_sha = _sha256(tok)
    manifest_tok_sha = meta.get("tokenizer_sha256")
    if manifest_tok_sha and manifest_tok_sha != tok_sha:
        raise RuntimeError(
            f"{method}/{corpus}: tokenizer SHA mismatch between spiece.model and token manifest."
        )
    manifest_sha = _sha256(manifest)
    payload = {
        "method": method,
        "corpus": corpus,
        "tokenizer_sha256": tok_sha,
        "token_manifest_sha256": manifest_sha,
        "train_shape": [int(EXPECTED_TRAIN_BLOCKS), int(SEQ_LENGTH)],
        "valid_shape": [int(EXPECTED_VALID_BLOCKS), int(SEQ_LENGTH)],
        "vocab_size": int(VOCAB_SIZE),
    }
    canonical = json.dumps(payload, sort_keys=True).encode("utf-8")
    payload["input_fingerprint"] = hashlib.sha256(canonical).hexdigest()
    return payload


def _bind_run_fingerprint(method: str, corpus: str, payload: dict):
    run_dir = CHECKPOINT_ROOT / corpus / method
    run_dir.mkdir(parents=True, exist_ok=True)
    fp_file = run_dir / "INPUT_FINGERPRINT.json"
    latest = run_dir / "training_latest.pt"
    complete = run_dir / "TRAINING_COMPLETE.json"

    if fp_file.exists():
        old = json.loads(fp_file.read_text())
        if old.get("input_fingerprint") != payload["input_fingerprint"]:
            raise RuntimeError(
                f"{corpus}/{method}: token data/tokenizer changed since the existing checkpoint. "
                f"Refusing to resume incompatible optimizer/model state."
            )
    elif latest.exists() or complete.exists():
        raise RuntimeError(
            f"{corpus}/{method}: checkpoint exists without an spliNet input fingerprint. "
            f"The clean data rebuild should have removed old checkpoints; refusing to mix runs."
        )
    else:
        tmp = fp_file.with_suffix(".json.tmp")
        tmp.write_text(json.dumps(payload, indent=2))
        os.replace(tmp, fp_file)


# -----------------------------------------------------------------------------
# Local SSD staging — never train by repeatedly reading Google Drive.
# -----------------------------------------------------------------------------
def _stage_array(src: Path, dst: Path, rows: int, label: str):
    ok, _ = _array_probe(dst, rows, label + " local")
    if ok and dst.stat().st_size == src.stat().st_size:
        print(f"  local SSD cache: {label}")
        return dst

    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    tmp.unlink(missing_ok=True)
    print(f"  staging {label}: Drive -> local SSD")
    shutil.copyfile(src, tmp)
    os.replace(tmp, dst)

    ok, msg = _array_probe(dst, rows, label + " local")
    if not ok:
        dst.unlink(missing_ok=True)
        raise RuntimeError(f"Local staging failed: {msg}")
    return dst


def _stage_method_corpus(method: str, corpus: str):
    drive_train, drive_valid, _ = _token_paths(method, corpus)
    local_root = LOCAL_TRAIN_ROOT / method / corpus
    local_train = _stage_array(
        drive_train, local_root / "train_blocks.npy",
        EXPECTED_TRAIN_BLOCKS, f"{method}/{corpus}/train"
    )
    local_valid = _stage_array(
        drive_valid, local_root / "validation_blocks.npy",
        EXPECTED_VALID_BLOCKS, f"{method}/{corpus}/validation"
    )
    return local_train, local_valid


# -----------------------------------------------------------------------------
# Full preflight before burning GPU hours.
# -----------------------------------------------------------------------------
print("=" * 96)
print("spliNet G4 TRAINING PREFLIGHT")
print("=" * 96)
print("GPU                     :", GPU_NAME)
print("VRAM                    :", f"{VRAM_GB:.2f} GiB")
print("Autotune VRAM ceiling   :", f"{AUTOTUNE_EFFECTIVE_RESERVED_GB:.2f} GiB reserved")
print("Methods                 :", TRAIN_METHODS)
print("Corpora                 :", CORPUS_ORDER)
print("Mamba backend           :", MAMBA_BACKEND)
print("CPU prep threads        :", CPU_THREADS)
print("Training arrays         : local SSD")
print("Recovery checkpoints    :", CHECKPOINT_ROOT)
print("Checkpoint cadence      : 500 updates OR 20 minutes")
print("Activation checkpointing: OFF")
print("Final model weights     : SAVED")
print("=" * 96)

_INPUTS = {}
_errors = []
for _method in TRAIN_METHODS:
    _tok = TOKENIZER_ROOT / _method / "spiece.model"
    if not _tok.is_file():
        _errors.append(f"{_method}: missing tokenizer {_tok}")
        continue
    for _corpus in CORPUS_ORDER:
        _tr, _va, _man = _token_paths(_method, _corpus)
        ok1, msg1 = _array_probe(_tr, EXPECTED_TRAIN_BLOCKS, f"{_method}/{_corpus}/train")
        ok2, msg2 = _array_probe(_va, EXPECTED_VALID_BLOCKS, f"{_method}/{_corpus}/validation")
        if not ok1:
            _errors.append(msg1)
        if not ok2:
            _errors.append(msg2)
        if not _man.is_file():
            _errors.append(f"{_method}/{_corpus}: missing manifest {_man}")
        if ok1 and ok2 and _man.is_file():
            try:
                _INPUTS[(_method, _corpus)] = _method_input_fingerprint(_method, _corpus)
            except Exception as exc:
                _errors.append(f"{_method}/{_corpus}: {type(exc).__name__}: {exc}")

if _errors:
    print("\nTRAINING NOT STARTED — data preparation is incomplete or inconsistent:")
    for _e in _errors:
        print("  -", _e)
    raise RuntimeError("Finish the clean spliNet tokenizer/token-array preparation first.")

print(f"Preflight PASS: {len(TRAIN_METHODS)} independent tokenizers × {len(CORPUS_ORDER)} corpora.")

# -----------------------------------------------------------------------------
# Train/resume all runs.
# -----------------------------------------------------------------------------
for _corpus in CORPUS_ORDER:
    for _method in TRAIN_METHODS:
        print("\n" + "#" * 96)
        print(f"spliNet TRAIN/RESUME: {_corpus} / {_method}")
        print("#" * 96)

        _fingerprint = _INPUTS[(_method, _corpus)]
        _bind_run_fingerprint(_method, _corpus, _fingerprint)

        # train_one writes this into the completion manifest.
        TOKENIZER_SHA256 = _fingerprint["tokenizer_sha256"]

        _local_train, _local_valid = _stage_method_corpus(_method, _corpus)

        # Cell 1 handles:
        #   * exhaustive real forward/backward/optimizer autotuning
        #   * <= ~90 GiB reserved-VRAM candidate acceptance
        #   * BF16 + TF32
        #   * fused Adam when PyTorch supports it
        #   * fixed effective batch + gradient accumulation
        #   * deterministic MLM
        #   * Google Drive recovery checkpoint every 500 updates or 20 min
        #   * emergency checkpoint on exception
        #   * exact optimizer/scheduler/scaler/RNG resume
        #   * full validation and final model persistence
        train_one(_corpus, _method, _local_train, _local_valid)
        save_summary()
        hard_cleanup(f"spliNet boundary {_corpus}/{_method}")

save_summary()
print("\n" + "=" * 96)
print("spliNet ALL TRAINING RUNS COMPLETE")
print("Results     :", RESULTS_ROOT)
print("Checkpoints :", CHECKPOINT_ROOT)
print("=" * 96)


spliNet G4 TRAINING PREFLIGHT
GPU                     : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM                    : 94.97 GiB
Autotune VRAM ceiling   : 90.00 GiB reserved
Methods                 : ['mamba2', 'hyena', 'retnet']
Corpora                 : ['fineweb_edu', 'arxiv', 'pg19', 'wikitext103']
Mamba backend           : huggingface_native_mamba2
CPU prep threads        : 32
Training arrays         : local SSD
Recovery checkpoints    : /content/drive/MyDrive/spliNet/checkpoints
Checkpoint cadence      : 500 updates OR 20 minutes
Activation checkpointing: OFF
Final model weights     : SAVED
Preflight PASS: 3 independent tokenizers × 4 corpora.

################################################################################################
spliNet TRAIN/RESUME: fineweb_edu / mamba2
################################################################################################
  local SSD cache: mamba2/fineweb_edu/train
  local SSD cache: mamba2/fineweb_edu/validation
[cl

In [9]:
# CELL 1 — Meyer-TransWave long-context experiment: common setup
# Standalone Meyer experiment only. Baseline runs are not touched.

from __future__ import annotations

import os, sys, gc, json, math, time, random, shutil, ctypes, hashlib
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    raise RuntimeError("This cell is intended for Google Colab.") from exc

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required.")

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = bool(torch.cuda.is_bf16_supported())
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
AMP_NAME = "bfloat16" if BF16 else "float16"
USE_SCALER = not BF16

# Same base architecture scale as the baseline experiment.
VOCAB_SIZE = 32_000
HIDDEN_SIZE = 768
NUM_LAYERS = 12
INTERMEDIATE_SIZE = 3072
HIDDEN_DROPOUT = 0.1
LAYER_NORM_EPS = 1.0e-12
INITIALIZER_RANGE = 0.02

PAD_ID = 3
MASK_ID = 6
SPECIAL_IDS = (0, 1, 2, 3, 4, 5, 6)

MLM_PROBABILITY = 0.15
MAX_PREDICTIONS = 80

LEARNING_RATE = 1.0e-4
ADAM_BETAS = (0.9, 0.999)
ADAM_EPS = 1.0e-8
WEIGHT_DECAY = 0.0

GLOBAL_SEED = 2026
DATA_ORDER_SEED = 7301
MASK_SEED = 912431

LOG_EVERY = 25
SAVE_EVERY = 500
SAVE_EVERY_MINUTES = 20
SAVE_FINAL_MODEL = True

CORPORA = (
    "fineweb_edu",
    "arxiv",
    "pg19",
    "wikitext103",
)

# Flat-stream sizes matching the existing ~100M-token experiment.
SOURCE_TRAIN_TOKENS = 193_359 * 512
SOURCE_VALID_TOKENS = 1_953 * 512

# Start at 512 and keep doubling until batch=1 no longer fits.
# The upper bound comes from available training data, not from an
# arbitrary model-context cap.
MIN_CONTEXT = 512
MAX_CONTEXT_PROBE = (
    1
    <<
    int(
        math.floor(
            math.log2(
                SOURCE_TRAIN_TOKENS
            )
        )
    )
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/TransWave_Meyer_Context"
)

RAW_ROOT = (
    PROJECT_ROOT
    /
    "raw_text"
)

TOKENIZER_ROOT = (
    PROJECT_ROOT
    /
    "tokenizer"
)

STREAM_ROOT = (
    PROJECT_ROOT
    /
    "streams"
)

CHECKPOINT_ROOT = (
    PROJECT_ROOT
    /
    "checkpoints"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    /
    "results"
)

CONFIG_ROOT = (
    PROJECT_ROOT
    /
    "configs"
)

LOCAL_ROOT = Path(
    "/content/TransWave_Meyer_Context"
)

LOCAL_STAGE_ROOT = (
    LOCAL_ROOT
    /
    "staged"
)

LOCAL_TMP_ROOT = (
    LOCAL_ROOT
    /
    "tmp"
)

for p in (
    PROJECT_ROOT,
    RAW_ROOT,
    TOKENIZER_ROOT,
    STREAM_ROOT,
    CHECKPOINT_ROOT,
    RESULTS_ROOT,
    CONFIG_ROOT,
    LOCAL_ROOT,
    LOCAL_STAGE_ROOT,
    LOCAL_TMP_ROOT,
):
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

CPU_THREADS = max(
    1,
    min(
        32,
        os.cpu_count() or 8,
    ),
)

torch.set_num_threads(
    CPU_THREADS
)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

try:
    torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = True
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = True
    torch.set_float32_matmul_precision(
        "high"
    )
except Exception:
    pass


def set_all_seeds(
    seed: int,
) -> None:

    seed = int(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    torch.cuda.manual_seed_all(
        seed
    )


def _malloc_trim() -> None:

    try:

        ctypes.CDLL(
            "libc.so.6"
        ).malloc_trim(
            0
        )

    except Exception:

        pass


def hard_cleanup(
    label: str = "",
) -> None:

    try:
        torch.cuda.synchronize()
    except Exception:
        pass

    for _ in range(
        3
    ):
        gc.collect()

    try:

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

    except Exception:

        pass

    _malloc_trim()

    if label:

        print(
            f"[cleanup] {label} | "
            f"alloc="
            f"{torch.cuda.memory_allocated()/1024**3:.3f} GiB | "
            f"reserved="
            f"{torch.cuda.memory_reserved()/1024**3:.3f} GiB"
        )


set_all_seeds(
    GLOBAL_SEED
)


def make_mlm_batch(
    clean_np,
    seed_index: int,
    validation: bool = False,
):

    """
    Deterministic BERT-style MLM:
    15% selection, capped at 80 targets/row.
    """

    clean = torch.from_numpy(
        np.asarray(
            clean_np,
            dtype=np.int64,
        ).copy()
    )

    if clean.ndim != 2:

        raise ValueError(
            f"Expected [B,L], got {tuple(clean.shape)}"
        )

    ids = clean.clone()

    labels = torch.full_like(
        ids,
        -100,
    )

    g = torch.Generator(
        device="cpu"
    )

    g.manual_seed(
        MASK_SEED
        +
        int(
            seed_index
        )
        +
        (
            1_000_000_000
            if validation
            else 0
        )
    )

    eligible = torch.ones_like(
        ids,
        dtype=torch.bool,
    )

    for sid in SPECIAL_IDS:

        eligible &= ids.ne(
            sid
        )

    scores = torch.rand(
        ids.shape,
        generator=g,
    )

    selected = (
        eligible
        &
        scores.lt(
            MLM_PROBABILITY
        )
    )

    # Preserve the original experiment protocol:
    # no more than 80 MLM predictions per sequence.
    #
    # This is important for the long-context hardware test:
    # the sequence operator should be what scales with L,
    # not a gigantic 32k-vocabulary output matrix.
    k = min(
        MAX_PREDICTIONS,
        ids.shape[
            1
        ],
    )

    ranked = scores.masked_fill(
        ~selected,
        float(
            "inf"
        ),
    )

    keep_idx = torch.topk(
        ranked,
        k=k,
        dim=1,
        largest=False,
        sorted=False,
    ).indices

    keep = torch.zeros_like(
        selected
    )

    keep.scatter_(
        1,
        keep_idx,
        True,
    )

    selected &= keep

    # Defensive: at least one target if an eligible token exists.
    empty = (
        selected.sum(
            dim=1
        )
        .eq(
            0
        )
    )

    if bool(
        empty.any()
    ):

        fallback_scores = (
            torch.rand(
                ids.shape,
                generator=g,
            )
            .masked_fill(
                ~eligible,
                -1.0,
            )
        )

        best = fallback_scores.argmax(
            dim=1
        )

        rows = torch.where(
            empty
        )[
            0
        ]

        valid_rows = (
            eligible[
                rows
            ]
            .any(
                dim=1
            )
        )

        rows = rows[
            valid_rows
        ]

        if rows.numel():

            selected[
                rows,
                best[
                    rows
                ],
            ] = True

    labels[
        selected
    ] = ids[
        selected
    ]

    replacement = torch.rand(
        ids.shape,
        generator=g,
    )

    mask_pos = (
        selected
        &
        replacement.lt(
            0.80
        )
    )

    rand_pos = (
        selected
        &
        replacement.ge(
            0.80
        )
        &
        replacement.lt(
            0.90
        )
    )

    ids[
        mask_pos
    ] = MASK_ID

    random_ids = torch.randint(
        7,
        VOCAB_SIZE,
        ids.shape,
        generator=g,
        dtype=torch.long,
    )

    ids[
        rand_pos
    ] = random_ids[
        rand_pos
    ]

    return (
        ids.pin_memory(),
        labels.pin_memory(),
    )


def gelu_new(
    x,
):

    return (
        0.5
        *
        x
        *
        (
            1.0
            +
            torch.tanh(
                math.sqrt(
                    2.0
                    /
                    math.pi
                )
                *
                (
                    x
                    +
                    0.044715
                    *
                    x.pow(
                        3
                    )
                )
            )
        )
    )


class CommonEmbeddings(
    nn.Module
):

    """
    Token embeddings only.
    There is deliberately no learned fixed-length absolute
    positional embedding table.
    """

    def __init__(
        self,
    ):

        super().__init__()

        self.word_embeddings = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE,
            padding_idx=PAD_ID,
        )

        self.norm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

        self.dropout = nn.Dropout(
            HIDDEN_DROPOUT
        )

    def forward(
        self,
        input_ids,
    ):

        return self.dropout(
            self.norm(
                self.word_embeddings(
                    input_ids
                )
            )
        )


class CommonFFN(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()

        self.fc1 = nn.Linear(
            HIDDEN_SIZE,
            INTERMEDIATE_SIZE,
        )

        self.fc2 = nn.Linear(
            INTERMEDIATE_SIZE,
            HIDDEN_SIZE,
        )

        self.dropout = nn.Dropout(
            HIDDEN_DROPOUT
        )

        self.norm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

    def forward(
        self,
        x,
    ):

        y = self.dropout(
            self.fc2(
                gelu_new(
                    self.fc1(
                        x
                    )
                )
            )
        )

        return self.norm(
            x
            +
            y
        )


class MLMHead(
    nn.Module
):

    """
    Only masked positions receive the expensive 32k projection.
    """

    def __init__(
        self,
        word_embeddings,
    ):

        super().__init__()

        self.fc = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE,
        )

        self.norm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

        self.bias = nn.Parameter(
            torch.zeros(
                VOCAB_SIZE
            )
        )

        self.word_embeddings = (
            word_embeddings
        )

    def forward(
        self,
        x,
        labels=None,
    ):

        if labels is not None:

            selected = labels.ne(
                -100
            )

            x = x[
                selected
            ]

            targets = labels[
                selected
            ]

        else:

            targets = None

        x = self.norm(
            gelu_new(
                self.fc(
                    x
                )
            )
        )

        logits = F.linear(
            x,
            self.word_embeddings.weight,
            self.bias,
        )

        return (
            logits,
            targets,
        )


def parameter_counts(
    model,
):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return (
        int(
            total
        ),
        int(
            trainable
        ),
    )


def make_optimizer(
    model,
):

    kwargs = dict(
        lr=LEARNING_RATE,
        betas=ADAM_BETAS,
        eps=ADAM_EPS,
        weight_decay=WEIGHT_DECAY,
    )

    try:

        return (
            torch.optim.Adam(
                model.parameters(),
                fused=True,
                **kwargs,
            ),
            True,
        )

    except Exception:

        return (
            torch.optim.Adam(
                model.parameters(),
                **kwargs,
            ),
            False,
        )


def make_scheduler(
    optimizer,
    total_updates: int,
):

    total_updates = int(
        total_updates
    )

    if total_updates <= 0:

        raise ValueError(
            "total_updates must be positive"
        )

    warmup = max(
        1,
        round(
            0.01
            *
            total_updates
        ),
    )

    def factor(
        step,
    ):

        if step < warmup:

            return (
                float(
                    step
                    +
                    1
                )
                /
                float(
                    warmup
                )
            )

        return 1.0

    return (
        torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            factor,
        ),
        warmup,
    )


def make_scaler():

    return torch.amp.GradScaler(
        "cuda",
        enabled=USE_SCALER,
    )


def optimizer_to_gpu(
    optimizer,
):

    for state in optimizer.state.values():

        for key, value in list(
            state.items()
        ):

            if torch.is_tensor(
                value
            ):

                state[
                    key
                ] = value.to(
                    DEVICE,
                    non_blocking=True,
                )


def capture_rng():

    return {
        "python":
            random.getstate(),

        "numpy":
            np.random.get_state(),

        "torch":
            torch.get_rng_state(),

        "cuda":
            torch.cuda.get_rng_state_all(),
    }


def restore_rng(
    state,
):

    random.setstate(
        state[
            "python"
        ]
    )

    np.random.set_state(
        state[
            "numpy"
        ]
    )

    torch.set_rng_state(
        state[
            "torch"
        ]
    )

    torch.cuda.set_rng_state_all(
        state[
            "cuda"
        ]
    )


def atomic_torch_save(
    payload,
    destination,
):

    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    stamp = (
        f"{os.getpid()}_"
        f"{time.time_ns()}"
    )

    local_tmp = (
        LOCAL_TMP_ROOT
        /
        f"{destination.name}."
        f"{stamp}.tmp"
    )

    drive_tmp = (
        destination.parent
        /
        f"{destination.name}."
        f"{stamp}.tmp"
    )

    local_tmp.unlink(
        missing_ok=True
    )

    drive_tmp.unlink(
        missing_ok=True
    )

    try:

        torch.save(
            payload,
            local_tmp,
        )

        probe = torch.load(
            local_tmp,
            map_location="cpu",
            weights_only=False,
        )

        del probe

        shutil.copyfile(
            local_tmp,
            drive_tmp,
        )

        os.replace(
            drive_tmp,
            destination,
        )

    finally:

        local_tmp.unlink(
            missing_ok=True
        )

        drive_tmp.unlink(
            missing_ok=True
        )


def atomic_json(
    payload,
    destination,
):

    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = destination.with_suffix(
        destination.suffix
        +
        ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    os.replace(
        tmp,
        destination,
    )


def sha256_file(
    path,
    block=8 << 20,
):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            b = f.read(
                block
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


print(
    "=" * 96
)

print(
    "MEYER-TRANSWAVE LONG-CONTEXT CELL 1 READY"
)

print(
    "GPU                :",
    GPU_NAME,
)

print(
    "VRAM               :",
    f"{VRAM_GB:.2f} GiB",
)

print(
    "PyTorch / CUDA     :",
    torch.__version__,
    "/",
    torch.version.cuda,
)

print(
    "AMP                :",
    AMP_NAME,
)

print(
    "Context search     :",
    f"{MIN_CONTEXT:,} then x2 until batch=1 fails",
)

print(
    "Probe upper bound  :",
    f"{MAX_CONTEXT_PROBE:,}",
)

print(
    "Train stream tokens:",
    f"{SOURCE_TRAIN_TOKENS:,}",
)

print(
    "Valid stream tokens:",
    f"{SOURCE_VALID_TOKENS:,}",
)

print(
    "Absolute positions : NONE"
)

print(
    "Batch size         : NOT FIXED; "
    "Cell 4 searches it on the G4"
)

print(
    "=" * 96
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MEYER-TRANSWAVE LONG-CONTEXT CELL 1 READY
GPU                : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM               : 94.97 GiB
PyTorch / CUDA     : 2.11.0+cu130 / 13.0
AMP                : bfloat16
Context search     : 512 then x2 until batch=1 fails
Probe upper bound  : 67,108,864
Train stream tokens: 98,999,808
Valid stream tokens: 999,936
Absolute positions : NONE
Batch size         : NOT FIXED; Cell 4 searches it on the G4


In [10]:
# CELL 2 — Dynamic-context fixed Meyer operator + TransWave model

from __future__ import annotations

import math
import contextlib

import torch
import torch.nn as nn


TRANSWAVE_KIND = (
    "transwave_meyer"
)

TRANSWAVE_OPERATOR_VERSION = (
    "periodic_meyer_WstarDW_dynamic_v2"
)

TRANSWAVE_MEYER_TRANSITION = (
    1.0
    /
    3.0
)


for _name in (
    "HIDDEN_SIZE",
    "NUM_LAYERS",
    "LAYER_NORM_EPS",
    "INITIALIZER_RANGE",
    "PAD_ID",
    "CommonEmbeddings",
    "CommonFFN",
    "MLMHead",
    "GLOBAL_SEED",
    "set_all_seeds",
    "MIN_CONTEXT",
):

    if _name not in globals():

        raise RuntimeError(
            f"Run Cell 1 first; missing {_name}"
        )


def _meyer_nu(
    t,
):

    t = t.clamp(
        0.0,
        1.0,
    )

    return (
        t**4
        *
        (
            35.0
            -
            84.0
            *
            t
            +
            70.0
            *
            t**2
            -
            20.0
            *
            t**3
        )
    )


def _meyer_lowpass_power(
    f,
    cutoff,
    transition,
):

    lo = max(
        0.0,
        float(
            cutoff
        )
        *
        (
            1.0
            -
            float(
                transition
            )
        ),
    )

    hi = min(
        1.0,
        float(
            cutoff
        )
        *
        (
            1.0
            +
            float(
                transition
            )
        ),
    )

    if hi <= lo:

        return (
            f
            <=
            float(
                cutoff
            )
        ).to(
            torch.float32
        )

    out = torch.ones_like(
        f,
        dtype=torch.float32,
    )

    out = torch.where(
        f
        >=
        hi,
        torch.zeros_like(
            out
        ),
        out,
    )

    mid = (
        f
        >
        lo
    ) & (
        f
        <
        hi
    )

    if bool(
        mid.any()
    ):

        u = (
            f[
                mid
            ]
            -
            lo
        ) / (
            hi
            -
            lo
        )

        out[
            mid
        ] = torch.cos(
            0.5
            *
            math.pi
            *
            _meyer_nu(
                u
            )
        ).square()

    return out


def build_periodic_meyer_response(
    length: int,
):

    """
    Fixed finite periodic realization of

        T_L = W_L^* D_L W_L.

    The entire multiscale analysis, fixed scale translations,
    and synthesis collapse to one Fourier multiplier H_L.

    Runtime:
        RFFT -> H_L multiply -> IRFFT
    """

    L = int(
        length
    )

    if (
        L
        <
        512

        or

        (
            L
            &
            (
                L
                -
                1
            )
        )
        !=
        0
    ):

        raise ValueError(
            "Context length must be a "
            "power of two >= 512"
        )

    # Automatically increases with context.
    num_scales = (
        int(
            math.log2(
                L
            )
        )
        -
        1
    )

    freq = torch.fft.rfftfreq(
        L,
        d=1.0,
    ).to(
        torch.float32
    )

    f_norm = (
        freq
        /
        0.5
    ).clamp(
        0.0,
        1.0,
    )

    omega = (
        2.0
        *
        math.pi
        *
        freq
    )

    cumulative = torch.ones_like(
        f_norm
    )

    power_windows = []

    shifts = []

    for j in range(
        1,
        num_scales
        +
        1,
    ):

        low = _meyer_lowpass_power(
            f_norm,
            cutoff=2.0 ** (-j),
            transition=TRANSWAVE_MEYER_TRANSITION,
        )

        q = (
            cumulative
            -
            low
        ).clamp_min(
            0.0
        )

        power_windows.append(
            q
        )

        magnitude = min(
            2 ** (j - 1),
            L // 4,
        )

        shifts.append(
            magnitude
            if (
                j
                %
                2
                ==
                1
            )
            else -magnitude
        )

        cumulative = (
            low
        )

    # Coarsest scaling band.
    power_windows.append(
        cumulative.clamp_min(
            0.0
        )
    )

    shifts.append(
        L
        //
        2
    )

    Q = torch.stack(
        power_windows,
        dim=0,
    )

    # Exact numerical partition of unity.
    Q = (
        Q
        /
        Q.sum(
            dim=0
        )
        .clamp_min(
            1.0e-12
        )
        .unsqueeze(
            0
        )
    )

    H = torch.zeros(
        freq.numel(),
        dtype=torch.complex64,
    )

    for q, tau in zip(
        Q,
        shifts,
    ):

        phase = torch.exp(
            (
                -1j
                *
                omega
                *
                float(
                    tau
                )
            ).to(
                torch.complex64
            )
        )

        H += (
            q.to(
                torch.complex64
            )
            *
            phase
        )

    # Real-valued RFFT endpoints.
    H[
        0
    ] = torch.complex(
        torch.tensor(
            1.0
        ),
        torch.tensor(
            0.0
        ),
    )

    if (
        L
        %
        2
        ==
        0
    ):

        H[
            -1
        ] = torch.complex(
            H[
                -1
            ].real,
            torch.tensor(
                0.0
            ),
        )

    partition_error = float(
        (
            Q.sum(
                dim=0
            )
            -
            1.0
        )
        .abs()
        .max()
    )

    max_response = float(
        H.abs().max()
    )

    if (
        partition_error
        >
        1.0e-6

        or

        max_response
        >
        1.00001
    ):

        raise RuntimeError(
            "Invalid Meyer response: "
            f"partition_error="
            f"{partition_error}, "
            f"max|H|="
            f"{max_response}"
        )

    return (
        H.contiguous(),
        {
            "length":
                L,

            "num_scales":
                num_scales,

            "num_bands":
                int(
                    Q.shape[
                        0
                    ]
                ),

            "shifts":
                shifts,

            "partition_error":
                partition_error,

            "max_abs_response":
                max_response,
        },
    )


class MeyerWaveletOperator(
    nn.Module
):

    """
    Fixed Meyer sequence operator.

    Trainable parameter count: exactly zero.
    """

    def __init__(
        self,
        context_length: int,
    ):

        super().__init__()

        self.context_length = int(
            context_length
        )

        (
            H,
            diagnostics,
        ) = build_periodic_meyer_response(
            self.context_length
        )

        self.diagnostics = (
            diagnostics
        )

        self.register_buffer(
            "response",
            H,
            persistent=True,
        )

    def forward(
        self,
        x,
    ):

        if (
            x.ndim
            !=
            3

            or

            int(
                x.shape[
                    1
                ]
            )
            !=
            self.context_length
        ):

            raise RuntimeError(
                f"Expected "
                f"[B,{self.context_length},D], "
                f"got {tuple(x.shape)}"
            )

        original_dtype = (
            x.dtype
        )

        # [B,L,D] -> [B,D,L]
        z = (
            x.transpose(
                1,
                2,
            )
            .contiguous()
        )

        ctx = (
            torch.autocast(
                device_type="cuda",
                enabled=False,
            )

            if z.is_cuda

            else contextlib.nullcontext()
        )

        with ctx:

            Z = torch.fft.rfft(
                z.float(),
                n=self.context_length,
                dim=-1,
            )

            y = torch.fft.irfft(
                Z
                *
                self.response.view(
                    1,
                    1,
                    -1,
                ),
                n=self.context_length,
                dim=-1,
            )

        return (
            y.transpose(
                1,
                2,
            )
            .contiguous()
            .to(
                original_dtype
            )
        )


class TransWaveMeyerLayer(
    nn.Module
):

    def __init__(
        self,
        context_length: int,
    ):

        super().__init__()

        self.mixer = (
            MeyerWaveletOperator(
                context_length
            )
        )

        self.mixer_norm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

        self.ffn = (
            CommonFFN()
        )

    def forward(
        self,
        x,
    ):

        x = self.mixer_norm(
            x
            +
            self.mixer(
                x
            )
        )

        return self.ffn(
            x
        )


class TransWaveMeyerMLM(
    nn.Module
):

    def __init__(
        self,
        context_length: int,
    ):

        super().__init__()

        self.kind = (
            TRANSWAVE_KIND
        )

        self.context_length = int(
            context_length
        )

        self.embeddings = (
            CommonEmbeddings()
        )

        self.layers = nn.ModuleList(
            [
                TransWaveMeyerLayer(
                    self.context_length
                )

                for _ in range(
                    NUM_LAYERS
                )
            ]
        )

        self.mlm = MLMHead(
            self.embeddings
            .word_embeddings
        )

        self._reset_parameters()

    def _reset_parameters(
        self,
    ):

        # Only neural components are learned.
        # Meyer responses remain fixed buffers.
        for module in self.modules():

            if isinstance(
                module,
                nn.Linear,
            ):

                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=INITIALIZER_RANGE,
                )

                if (
                    module.bias
                    is not None
                ):

                    nn.init.zeros_(
                        module.bias
                    )

            elif isinstance(
                module,
                nn.Embedding,
            ):

                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=INITIALIZER_RANGE,
                )

                if (
                    module.padding_idx
                    is not None
                ):

                    with torch.no_grad():

                        module.weight[
                            module.padding_idx
                        ].zero_()

            elif isinstance(
                module,
                nn.LayerNorm,
            ):

                nn.init.ones_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

        nn.init.zeros_(
            self.mlm.bias
        )

    def forward(
        self,
        input_ids,
        labels=None,
    ):

        x = self.embeddings(
            input_ids
        )

        for layer in self.layers:

            x = layer(
                x
            )

        return self.mlm(
            x,
            labels,
        )


def build_model(
    context_length: int,
):

    set_all_seeds(
        GLOBAL_SEED
    )

    return TransWaveMeyerMLM(
        int(
            context_length
        )
    )


print(
    "=" * 96
)

print(
    "MEYER-TRANSWAVE CELL 2 READY"
)

print(
    "Operator: T_L = W_L* D_L W_L"
)

print(
    "GPU runtime: "
    "RFFT -> fixed Meyer multiplier -> IRFFT"
)

print(
    "Trainable sequence-mixer parameters: 0"
)

_H, _d = build_periodic_meyer_response(
    MIN_CONTEXT
)

print(
    f"Initial L="
    f"{MIN_CONTEXT:,} | "
    f"scales="
    f"{_d['num_scales']} | "
    f"bands="
    f"{_d['num_bands']} | "
    f"max|H|="
    f"{_d['max_abs_response']:.6f}"
)

print(
    "Larger responses are constructed "
    "on demand by Cell 4."
)

print(
    "=" * 96
)

MEYER-TRANSWAVE CELL 2 READY
Operator: T_L = W_L* D_L W_L
GPU runtime: RFFT -> fixed Meyer multiplier -> IRFFT
Trainable sequence-mixer parameters: 0
Initial L=512 | scales=8 | bands=9 | max|H|=1.000000
Larger responses are constructed on demand by Cell 4.


In [11]:
# CELL 3 — Meyer-related learned multiresolution tokenizer + flat streams
#
# One 32k tokenizer for Meyer-TransWave.
#
# Difference from ordinary SentencePiece:
#   - BPE rather than unigram
#   - trained on ~1 GiB balanced TRAINING-ONLY text
#   - merges are allowed across whitespace
#   - learned pieces may span up to 64 characters
#   - therefore vocabulary contains a hierarchy:
#       chars/subwords -> words -> short phrases
#
# The output for each corpus is ONE flat token stream.
# No context length is imposed here.
#
# IMPORTANT:
# We tokenize the entire fixed raw corpus.
# We DO NOT repeat raw text to force an arbitrary token count.

from __future__ import annotations

import os
import gc
import json
import shutil
import subprocess
import sys
import hashlib
from pathlib import Path

import numpy as np


# =============================================================================
# SENTENCEPIECE
# =============================================================================

try:
    import sentencepiece as spm

except Exception:

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "sentencepiece>=0.2.0",
    ])

    import sentencepiece as spm


# =============================================================================
# PRECONDITIONS
# =============================================================================

_REQUIRED = (
    "CORPORA",
    "VOCAB_SIZE",
    "PAD_ID",
    "MASK_ID",
    "RAW_ROOT",
    "TOKENIZER_ROOT",
    "STREAM_ROOT",
    "LOCAL_ROOT",
    "TRANSWAVE_OPERATOR_VERSION",
)

_missing = [
    name
    for name in _REQUIRED
    if name not in globals()
]

if _missing:

    raise RuntimeError(
        "Run Cells 1 and 2 first; "
        f"missing definitions: {_missing}"
    )


if int(VOCAB_SIZE) != 32000:

    raise RuntimeError(
        f"Expected VOCAB_SIZE=32000, got {VOCAB_SIZE}"
    )


if int(PAD_ID) != 3 or int(MASK_ID) != 6:

    raise RuntimeError(
        "Expected PAD_ID=3 and MASK_ID=6; "
        f"got {PAD_ID}, {MASK_ID}"
    )


# =============================================================================
# TOKENIZER PROTOCOL
# =============================================================================

TOKENIZER_VERSION = (
    "meyer_multiresolution_bpe_32k_v1"
)

TOKEN_DTYPE = np.uint16


# Approximate balanced training corpus:
#
#   C4                  256 MiB
#   FineWeb-Edu         192 MiB
#   arXiv               192 MiB
#   PG-19               192 MiB
#   WikiText-103        192 MiB
#
# ≈ 1 GiB total.
#
# NO validation text is used to learn the tokenizer.

MIB = 1024 * 1024

TOKENIZER_SOURCE_QUOTAS = {
    "c4":
        256 * MIB,

    "fineweb_edu":
        192 * MIB,

    "arxiv":
        192 * MIB,

    "pg19":
        192 * MIB,

    "wikitext103":
        192 * MIB,
}


# Re-wrap training text into bounded lines before SentencePiece training.
TOKENIZER_TRAIN_LINE_CHARS = 16_384


# Learned BPE pieces can be long enough to represent short phrases.
MAX_LEARNED_PIECE_CHARS = 64


TOKENIZE_WORKERS = max(
    1,
    min(
        32,
        os.cpu_count() or 8,
    ),
)


ENCODE_BATCH_LINES = 256


# =============================================================================
# PATHS
# =============================================================================

TOKENIZER_ROOT = Path(
    TOKENIZER_ROOT
)

STREAM_ROOT = Path(
    STREAM_ROOT
)

RAW_ROOT = Path(
    RAW_ROOT
)

LOCAL_ROOT = Path(
    LOCAL_ROOT
)


TOKENIZER_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

STREAM_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


WORK_ROOT = (
    LOCAL_ROOT
    /
    "multiresolution_tokenizer"
)

WORK_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


TRAINING_SOURCE = (
    WORK_ROOT
    /
    "meyer_tokenizer_training.txt"
)


SP_MODEL = (
    TOKENIZER_ROOT
    /
    "spiece.model"
)

SP_VOCAB = (
    TOKENIZER_ROOT
    /
    "spiece.vocab"
)

SP_MANIFEST = (
    TOKENIZER_ROOT
    /
    "manifest.json"
)


# =============================================================================
# GENERIC HELPERS
# =============================================================================

def _sha256_file(
    path: Path,
    block_size: int = 8 << 20,
) -> str:

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            b = f.read(
                block_size
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


def _atomic_json(
    payload,
    path: Path,
) -> None:

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    os.replace(
        tmp,
        path,
    )


# =============================================================================
# PREPARED RAW-TEXT CACHE
# =============================================================================

def _required_raw_files(
    root: Path,
):

    out = [
        root
        /
        "tokenizer_c4_256mb.txt"
    ]

    for corpus in CORPORA:

        out.extend([
            root
            /
            corpus
            /
            "train.txt",

            root
            /
            corpus
            /
            "validation.txt",
        ])

    return out


def _raw_complete(
    root: Path,
) -> bool:

    return all(
        p.is_file()
        and
        p.stat().st_size > 0

        for p in _required_raw_files(
            root
        )
    )


if not _raw_complete(
    RAW_ROOT
):

    # Bounded known locations only.
    # No recursive Google Drive search.

    candidates = [
        Path(
            "/content/drive/MyDrive/"
            "TransWave_Meyer/raw_text"
        ),

        Path(
            "/content/drive/MyDrive/"
            "spliNet/raw_text"
        ),
    ]

    source_root = next(
        (
            p
            for p in candidates
            if _raw_complete(
                p
            )
        ),
        None,
    )

    if source_root is None:

        raise FileNotFoundError(
            "Prepared raw-text cache not found.\n"
            "Checked only:\n  - "
            +
            "\n  - ".join(
                str(
                    x
                )
                for x in candidates
            )
        )

    print(
        "Copying prepared raw text once ..."
    )

    print(
        "source:",
        source_root
    )

    print(
        "dest  :",
        RAW_ROOT
    )

    for src in _required_raw_files(
        source_root
    ):

        rel = src.relative_to(
            source_root
        )

        dst = (
            RAW_ROOT
            /
            rel
        )

        dst.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if (
            dst.is_file()
            and
            dst.stat().st_size
            ==
            src.stat().st_size
        ):

            continue

        tmp = dst.with_suffix(
            dst.suffix + ".tmp"
        )

        tmp.unlink(
            missing_ok=True
        )

        shutil.copyfile(
            src,
            tmp,
        )

        os.replace(
            tmp,
            dst,
        )


if not _raw_complete(
    RAW_ROOT
):

    raise RuntimeError(
        "Raw-text cache remains incomplete."
    )


# =============================================================================
# BUILD ~1 GiB BALANCED TOKENIZER-TRAINING CORPUS
# =============================================================================

def _write_bounded_training_text(
    src: Path,
    dst_handle,
    byte_budget: int,
    line_chars: int,
) -> int:

    """
    Copy up to byte_budget bytes from a TRAINING source while
    rewrapping long records into bounded SentencePiece training lines.

    Returns the number of source bytes consumed.
    """

    byte_budget = int(
        byte_budget
    )

    consumed = 0

    with open(
        src,
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as f:

        while consumed < byte_budget:

            line = f.readline()

            if not line:
                break

            consumed += len(
                line.encode(
                    "utf-8",
                    errors="ignore",
                )
            )

            text = line.strip()

            if not text:
                continue

            start = 0

            while start < len(
                text
            ):

                end = min(
                    len(
                        text
                    ),
                    start
                    +
                    line_chars,
                )

                if end < len(
                    text
                ):

                    cut = text.rfind(
                        " ",
                        start,
                        end,
                    )

                    if (
                        cut
                        >
                        start
                        +
                        line_chars
                        //
                        2
                    ):

                        end = cut

                piece = text[
                    start:end
                ].strip()

                if piece:

                    dst_handle.write(
                        piece
                    )

                    dst_handle.write(
                        "\n"
                    )

                start = max(
                    end,
                    start + 1,
                )

    return consumed


def _build_tokenizer_training_source():

    source_map = {
        "c4":
            RAW_ROOT
            /
            "tokenizer_c4_256mb.txt",

        "fineweb_edu":
            RAW_ROOT
            /
            "fineweb_edu"
            /
            "train.txt",

        "arxiv":
            RAW_ROOT
            /
            "arxiv"
            /
            "train.txt",

        "pg19":
            RAW_ROOT
            /
            "pg19"
            /
            "train.txt",

        "wikitext103":
            RAW_ROOT
            /
            "wikitext103"
            /
            "train.txt",
    }

    tmp = TRAINING_SOURCE.with_suffix(
        ".txt.tmp"
    )

    tmp.unlink(
        missing_ok=True
    )

    stats = {}

    with open(
        tmp,
        "w",
        encoding="utf-8",
    ) as out:

        for name in (
            "c4",
            "fineweb_edu",
            "arxiv",
            "pg19",
            "wikitext103",
        ):

            src = source_map[
                name
            ]

            quota = TOKENIZER_SOURCE_QUOTAS[
                name
            ]

            print(
                f"Tokenizer source {name:14s}: "
                f"target {quota/MIB:.0f} MiB"
            )

            used = _write_bounded_training_text(
                src,
                out,
                quota,
                TOKENIZER_TRAIN_LINE_CHARS,
            )

            stats[
                name
            ] = int(
                used
            )

            print(
                f"  consumed "
                f"{used/MIB:.1f} MiB"
            )

    os.replace(
        tmp,
        TRAINING_SOURCE,
    )

    if (
        not TRAINING_SOURCE.is_file()
        or
        TRAINING_SOURCE.stat().st_size
        <
        512
        *
        MIB
    ):

        raise RuntimeError(
            "Tokenizer training corpus is unexpectedly small."
        )

    return stats


# =============================================================================
# TOKENIZER VALIDATION
# =============================================================================

def _tokenizer_valid() -> bool:

    if (
        not SP_MODEL.is_file()
        or
        not SP_MANIFEST.is_file()
    ):

        return False

    try:

        processor = (
            spm.SentencePieceProcessor(
                model_file=str(
                    SP_MODEL
                )
            )
        )

        meta = json.loads(
            SP_MANIFEST.read_text(
                encoding="utf-8"
            )
        )

        return (
            processor.get_piece_size()
            ==
            VOCAB_SIZE

            and

            processor.unk_id()
            ==
            0

            and

            processor.bos_id()
            ==
            1

            and

            processor.eos_id()
            ==
            2

            and

            processor.pad_id()
            ==
            PAD_ID

            and

            processor.piece_to_id(
                "[CLS]"
            )
            ==
            4

            and

            processor.piece_to_id(
                "[SEP]"
            )
            ==
            5

            and

            processor.piece_to_id(
                "[MASK]"
            )
            ==
            MASK_ID

            and

            meta.get(
                "version"
            )
            ==
            TOKENIZER_VERSION

            and

            meta.get(
                "operator_version"
            )
            ==
            TRANSWAVE_OPERATOR_VERSION

            and

            meta.get(
                "model_sha256"
            )
            ==
            _sha256_file(
                SP_MODEL
            )
        )

    except Exception:

        return False


# =============================================================================
# LEARN MULTIRESOLUTION BPE TOKENIZER
# =============================================================================

if not _tokenizer_valid():

    SP_MODEL.unlink(
        missing_ok=True
    )

    SP_VOCAB.unlink(
        missing_ok=True
    )

    SP_MANIFEST.unlink(
        missing_ok=True
    )

    source_stats = (
        _build_tokenizer_training_source()
    )

    print(
        "\nTraining Meyer-related "
        "multiresolution 32k BPE tokenizer ..."
    )

    print(
        "Cross-whitespace merges : ENABLED"
    )

    print(
        "Maximum learned piece   :",
        MAX_LEARNED_PIECE_CHARS,
        "characters"
    )

    spm.SentencePieceTrainer.train(

        input=str(
            TRAINING_SOURCE
        ),

        model_prefix=str(
            TOKENIZER_ROOT
            /
            "spiece"
        ),

        vocab_size=VOCAB_SIZE,

        # Hierarchical merge vocabulary.
        model_type="bpe",

        character_coverage=1.0,

        unk_id=0,
        bos_id=1,
        eos_id=2,
        pad_id=3,

        user_defined_symbols=[
            "[CLS]",
            "[SEP]",
            "[MASK]",
        ],

        # Critical for learned phrase-scale units.
        split_by_whitespace=False,

        # Allow words and short multiword phrases,
        # but prevent pathological giant pieces.
        max_sentencepiece_length=(
            MAX_LEARNED_PIECE_CHARS
        ),

        max_sentence_length=(
            TOKENIZER_TRAIN_LINE_CHARS
            *
            2
        ),

        shuffle_input_sentence=True,

        num_threads=TOKENIZE_WORKERS,

        hard_vocab_limit=True,

        train_extremely_large_corpus=True,
    )

    processor = (
        spm.SentencePieceProcessor(
            model_file=str(
                SP_MODEL
            )
        )
    )

    special_checks = {
        "vocab":
            processor.get_piece_size()
            ==
            VOCAB_SIZE,

        "unk":
            processor.unk_id()
            ==
            0,

        "bos":
            processor.bos_id()
            ==
            1,

        "eos":
            processor.eos_id()
            ==
            2,

        "pad":
            processor.pad_id()
            ==
            PAD_ID,

        "cls":
            processor.piece_to_id(
                "[CLS]"
            )
            ==
            4,

        "sep":
            processor.piece_to_id(
                "[SEP]"
            )
            ==
            5,

        "mask":
            processor.piece_to_id(
                "[MASK]"
            )
            ==
            MASK_ID,
    }

    if not all(
        special_checks.values()
    ):

        raise RuntimeError(
            "Tokenizer special-ID validation failed: "
            f"{special_checks}"
        )

    # -------------------------------------------------------------
    # Check that the learned vocabulary is genuinely multiresolution.
    #
    # SentencePiece represents whitespace as ▁.
    # A piece containing >=2 ▁ markers spans multiple words.
    # -------------------------------------------------------------

    pieces = [
        processor.id_to_piece(
            i
        )
        for i in range(
            processor.get_piece_size()
        )
    ]

    multiword_pieces = [
        piece
        for piece in pieces[
            7:
        ]
        if piece.count(
            "▁"
        )
        >=
        2
    ]

    piece_lengths = np.asarray(
        [
            len(
                piece.replace(
                    "▁",
                    " ",
                )
            )
            for piece in pieces[
                7:
            ]
        ],
        dtype=np.int32,
    )

    if len(
        multiword_pieces
    ) == 0:

        raise RuntimeError(
            "The BPE vocabulary learned no "
            "cross-word pieces. "
            "Multiresolution-tokenizer check failed."
        )

    tokenizer_manifest = {
        "version":
            TOKENIZER_VERSION,

        "operator_version":
            TRANSWAVE_OPERATOR_VERSION,

        "model_type":
            "bpe",

        "multiresolution":
            True,

        "cross_whitespace_merges":
            True,

        "max_learned_piece_chars":
            MAX_LEARNED_PIECE_CHARS,

        "vocab_size":
            VOCAB_SIZE,

        "model_sha256":
            _sha256_file(
                SP_MODEL
            ),

        "training_source_bytes":
            int(
                TRAINING_SOURCE.stat().st_size
            ),

        "training_source_components":
            source_stats,

        "validation_text_used":
            False,

        "multiword_piece_count":
            int(
                len(
                    multiword_pieces
                )
            ),

        "mean_piece_char_length":
            float(
                piece_lengths.mean()
            ),

        "median_piece_char_length":
            float(
                np.median(
                    piece_lengths
                )
            ),

        "max_observed_piece_char_length":
            int(
                piece_lengths.max()
            ),

        "special_ids":
            {
                "unk":
                    0,

                "bos":
                    1,

                "eos":
                    2,

                "pad":
                    3,

                "cls":
                    4,

                "sep":
                    5,

                "mask":
                    6,
            },
    }

    _atomic_json(
        tokenizer_manifest,
        SP_MANIFEST,
    )

    print(
        "\nTokenizer multiresolution diagnostics"
    )

    print(
        "multiword pieces :",
        f"{len(multiword_pieces):,}"
    )

    print(
        "mean chars/piece :",
        f"{piece_lengths.mean():.2f}"
    )

    print(
        "median           :",
        f"{np.median(piece_lengths):.1f}"
    )

    print(
        "max              :",
        int(
            piece_lengths.max()
        )
    )

    print(
        "example phrase pieces:"
    )

    for piece in multiword_pieces[
        :10
    ]:

        print(
            "   ",
            repr(
                piece.replace(
                    "▁",
                    " ",
                )
            )
        )


if not _tokenizer_valid():

    raise RuntimeError(
        "Tokenizer final validation failed."
    )


# Local temporary 1-GiB training source is no longer required.
TRAINING_SOURCE.unlink(
    missing_ok=True
)


sp = (
    spm.SentencePieceProcessor(
        model_file=str(
            SP_MODEL
        )
    )
)

TOKENIZER_SHA256 = (
    _sha256_file(
        SP_MODEL
    )
)

EOS_ID = int(
    sp.eos_id()
)


print(
    "\nTokenizer SHA256:",
    TOKENIZER_SHA256
)


# =============================================================================
# COMPLETE RAW DOCUMENT ITERATOR
# =============================================================================

def _iter_documents(
    path: Path,
):

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as f:

        for line in f:

            text = line.strip()

            if text:

                yield text


# =============================================================================
# FLAT-STREAM WRITER
#
# We do NOT know the number of output tokens in advance because the whole
# point of the new tokenizer is improved linguistic compression.
#
# Therefore:
#   1. tokenize to temporary chunk files on local SSD,
#   2. count actual tokens,
#   3. create exact-size .npy,
#   4. concatenate chunks,
#   5. persist to Drive.
# =============================================================================

def _stream_valid(
    path: Path,
    expected_tokens: int | None = None,
) -> bool:

    path = Path(
        path
    )

    if not path.is_file():

        return False

    array = None

    try:

        array = np.load(
            path,
            mmap_mode="r",
            allow_pickle=False,
        )

        if (
            array.ndim != 1
            or
            array.dtype
            !=
            TOKEN_DTYPE
        ):

            return False

        if (
            expected_tokens is not None
            and
            len(
                array
            )
            !=
            int(
                expected_tokens
            )
        ):

            return False

        if len(
            array
        ) <= 0:

            return False

        idx = sorted(
            set([
                0,
                len(
                    array
                )
                //
                2,
                len(
                    array
                )
                -
                1,
            ])
        )

        sample = np.asarray(
            array[
                idx
            ]
        )

        return (
            int(
                sample.min()
            )
            >=
            0

            and

            int(
                sample.max()
            )
            <
            VOCAB_SIZE
        )

    except Exception:

        return False

    finally:

        if array is not None:

            try:

                if (
                    getattr(
                        array,
                        "_mmap",
                        None,
                    )
                    is not None
                ):

                    array._mmap.close()

            except Exception:

                pass


def _encode_entire_raw_file(
    text_path: Path,
    destination: Path,
) -> int:

    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    scratch = (
        WORK_ROOT
        /
        (
            destination.parent.name
            +
            "__"
            +
            destination.stem
        )
    )

    shutil.rmtree(
        scratch,
        ignore_errors=True,
    )

    scratch.mkdir(
        parents=True,
        exist_ok=True,
    )

    chunk_paths = []

    chunk_index = 0

    total_tokens = 0

    batch = []

    batch_token_lists = []


    def flush_batch():

        nonlocal batch
        nonlocal batch_token_lists
        nonlocal chunk_index
        nonlocal total_tokens

        if not batch:

            return

        encoded = sp.encode(
            batch,
            out_type=int,
            enable_sampling=False,
            num_threads=TOKENIZE_WORKERS,
        )

        flattened = []

        for ids in encoded:

            flattened.extend(
                ids
            )

            # Explicit document boundary.
            flattened.append(
                EOS_ID
            )

        if flattened:

            arr = np.asarray(
                flattened,
                dtype=TOKEN_DTYPE,
            )

            chunk_path = (
                scratch
                /
                f"chunk_{chunk_index:07d}.npy"
            )

            np.save(
                chunk_path,
                arr,
                allow_pickle=False,
            )

            chunk_paths.append(
                chunk_path
            )

            total_tokens += int(
                arr.size
            )

            chunk_index += 1

        batch.clear()

        batch_token_lists.clear()


    for document in _iter_documents(
        text_path
    ):

        # Extremely long documents are still given to SentencePiece
        # in bounded textual chunks, but EOS is added only at the
        # actual document boundary.

        if len(
            document
        ) <= 64_000:

            batch.append(
                document
            )

            if (
                len(
                    batch
                )
                >=
                ENCODE_BATCH_LINES
            ):

                flush_batch()

        else:

            # Flush ordinary batch first.
            flush_batch()

            start = 0

            doc_ids = []

            while start < len(
                document
            ):

                end = min(
                    len(
                        document
                    ),
                    start
                    +
                    64_000,
                )

                if end < len(
                    document
                ):

                    cut = document.rfind(
                        " ",
                        start,
                        end,
                    )

                    if cut > start + 32_000:

                        end = cut

                piece = document[
                    start:end
                ]

                ids = sp.encode(
                    piece,
                    out_type=int,
                    enable_sampling=False,
                )

                doc_ids.extend(
                    ids
                )

                start = max(
                    end,
                    start + 1,
                )

            doc_ids.append(
                EOS_ID
            )

            arr = np.asarray(
                doc_ids,
                dtype=TOKEN_DTYPE,
            )

            chunk_path = (
                scratch
                /
                f"chunk_{chunk_index:07d}.npy"
            )

            np.save(
                chunk_path,
                arr,
                allow_pickle=False,
            )

            chunk_paths.append(
                chunk_path
            )

            total_tokens += int(
                arr.size
            )

            chunk_index += 1


    flush_batch()


    if total_tokens <= 0:

        raise RuntimeError(
            f"No tokens generated from {text_path}"
        )


    local_final = (
        WORK_ROOT
        /
        (
            destination.parent.name
            +
            "__"
            +
            destination.name
        )
    )

    local_final.unlink(
        missing_ok=True
    )


    output = np.lib.format.open_memmap(
        local_final,
        mode="w+",
        dtype=TOKEN_DTYPE,
        shape=(
            total_tokens,
        ),
    )


    cursor = 0

    for i, chunk_path in enumerate(
        chunk_paths
    ):

        chunk = np.load(
            chunk_path,
            mmap_mode="r",
            allow_pickle=False,
        )

        n = int(
            chunk.size
        )

        output[
            cursor:
            cursor
            +
            n
        ] = chunk

        cursor += n

        if (
            i
            %
            100
            ==
            0
            or
            i
            +
            1
            ==
            len(
                chunk_paths
            )
        ):

            print(
                f"    assembled "
                f"{cursor:,}/"
                f"{total_tokens:,} tokens"
            )


    output.flush()

    del output


    if cursor != total_tokens:

        raise RuntimeError(
            "Flat-stream assembly count mismatch."
        )


    drive_tmp = destination.with_suffix(
        destination.suffix
        +
        ".tmp"
    )

    drive_tmp.unlink(
        missing_ok=True
    )

    shutil.copyfile(
        local_final,
        drive_tmp,
    )

    os.replace(
        drive_tmp,
        destination,
    )

    local_final.unlink(
        missing_ok=True
    )

    shutil.rmtree(
        scratch,
        ignore_errors=True,
    )


    if not _stream_valid(
        destination,
        total_tokens,
    ):

        destination.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "Final flat-stream validation failed: "
            f"{destination}"
        )


    return int(
        total_tokens
    )


# =============================================================================
# TOKENIZE THE FOUR FIXED RAW CORPORA
# =============================================================================

STREAM_COUNTS = {}


for corpus in CORPORA:

    root = (
        STREAM_ROOT
        /
        corpus
    )

    root.mkdir(
        parents=True,
        exist_ok=True,
    )

    train_out = (
        root
        /
        "train_tokens.npy"
    )

    valid_out = (
        root
        /
        "validation_tokens.npy"
    )

    manifest = (
        root
        /
        "manifest.json"
    )


    cache_ok = False

    meta = None


    if (
        train_out.is_file()
        and
        valid_out.is_file()
        and
        manifest.is_file()
    ):

        try:

            meta = json.loads(
                manifest.read_text(
                    encoding="utf-8"
                )
            )

            cache_ok = (
                meta.get(
                    "tokenizer_version"
                )
                ==
                TOKENIZER_VERSION

                and

                meta.get(
                    "tokenizer_sha256"
                )
                ==
                TOKENIZER_SHA256

                and

                meta.get(
                    "operator_version"
                )
                ==
                TRANSWAVE_OPERATOR_VERSION

                and

                _stream_valid(
                    train_out,
                    int(
                        meta[
                            "train_tokens"
                        ]
                    ),
                )

                and

                _stream_valid(
                    valid_out,
                    int(
                        meta[
                            "validation_tokens"
                        ]
                    ),
                )
            )

        except Exception:

            cache_ok = False


    print(
        "\n"
        +
        "=" * 92
    )

    print(
        "Multiresolution tokenization:",
        corpus
    )

    print(
        "=" * 92
    )


    if cache_ok:

        train_count = int(
            meta[
                "train_tokens"
            ]
        )

        valid_count = int(
            meta[
                "validation_tokens"
            ]
        )

        print(
            "    cached train tokens     :",
            f"{train_count:,}"
        )

        print(
            "    cached validation tokens:",
            f"{valid_count:,}"
        )

    else:

        train_out.unlink(
            missing_ok=True
        )

        valid_out.unlink(
            missing_ok=True
        )

        manifest.unlink(
            missing_ok=True
        )


        print(
            "  encoding complete TRAIN raw corpus ..."
        )

        train_count = (
            _encode_entire_raw_file(
                RAW_ROOT
                /
                corpus
                /
                "train.txt",

                train_out,
            )
        )


        print(
            "  encoding complete VALIDATION raw corpus ..."
        )

        valid_count = (
            _encode_entire_raw_file(
                RAW_ROOT
                /
                corpus
                /
                "validation.txt",

                valid_out,
            )
        )


        _atomic_json(
            {
                "tokenizer_version":
                    TOKENIZER_VERSION,

                "tokenizer_sha256":
                    TOKENIZER_SHA256,

                "operator_version":
                    TRANSWAVE_OPERATOR_VERSION,

                "corpus":
                    corpus,

                "train_tokens":
                    int(
                        train_count
                    ),

                "validation_tokens":
                    int(
                        valid_count
                    ),

                "dtype":
                    np.dtype(
                        TOKEN_DTYPE
                    ).name,

                "raw_train_bytes":
                    int(
                        (
                            RAW_ROOT
                            /
                            corpus
                            /
                            "train.txt"
                        )
                        .stat()
                        .st_size
                    ),

                "raw_validation_bytes":
                    int(
                        (
                            RAW_ROOT
                            /
                            corpus
                            /
                            "validation.txt"
                        )
                        .stat()
                        .st_size
                    ),
            },
            manifest,
        )


    STREAM_COUNTS[
        corpus
    ] = {
        "train":
            train_count,

        "validation":
            valid_count,
    }


# =============================================================================
# SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "MEYER MULTIRESOLUTION TOKENIZER COMPLETE"
)

print(
    "Vocabulary       : 32,000 learned BPE pieces"
)

print(
    "Hierarchy        : chars/subwords -> words -> phrases"
)

print(
    "Cross-word BPE   : YES"
)

print(
    "Max piece span   :",
    MAX_LEARNED_PIECE_CHARS,
    "characters"
)

print(
    "Training text    : ~1 GiB, balanced, TRAINING ONLY"
)

print(
    "Flat token streams:"
)

for corpus in CORPORA:

    print(
        f"  {corpus:12s} | "
        f"train "
        f"{STREAM_COUNTS[corpus]['train']:,} | "
        f"valid "
        f"{STREAM_COUNTS[corpus]['validation']:,}"
    )

print(
    "No context length has been imposed."
)

print(
    "=" * 100
)

Copying prepared raw text once ...
source: /content/drive/MyDrive/TransWave_Meyer/raw_text
dest  : /content/drive/MyDrive/TransWave_Meyer_Context/raw_text
Tokenizer source c4            : target 256 MiB
  consumed 256.0 MiB
Tokenizer source fineweb_edu   : target 192 MiB
  consumed 192.0 MiB
Tokenizer source arxiv         : target 192 MiB
  consumed 192.0 MiB
Tokenizer source pg19          : target 192 MiB
  consumed 192.5 MiB
Tokenizer source wikitext103   : target 192 MiB
  consumed 192.0 MiB

Training Meyer-related multiresolution 32k BPE tokenizer ...
Cross-whitespace merges : ENABLED
Maximum learned piece   : 64 characters

Tokenizer multiresolution diagnostics
multiword pieces : 5,987
mean chars/piece : 5.76
median           : 5.0
max              : 64
example phrase pieces:
    ' of the'
    ' in the'
    ' to the'
    ' the s'
    ' and the'
    ' on the'
    ' for the'
    ' = ='
    ' the f'
    ' the p'

Tokenizer SHA256: 58f1985d0e2a2509aa1f3bf33c7ed7d30ac1a2367dc29c7d1d608

In [15]:
# CELL 4 — FIXED Meyer-TransWave training
#
# Hard-coded from the completed hardware search:
#
#   context length = 4096
#   physical batch = 155
#   activation checkpointing = ON
#
# No context tuning.
# No batch tuning.
# No probing.
#
# Trains the four corpora only.

from __future__ import annotations

import os
import gc
import json
import math
import time
import types
import shutil
import hashlib
from pathlib import Path

import numpy as np

import torch
import torch.nn.functional as F

from torch.utils.checkpoint import checkpoint as torch_checkpoint


# =============================================================================
# PRECONDITIONS
# =============================================================================

REQ = (
    "CORPORA",
    "VOCAB_SIZE",
    "TOKENIZER_ROOT",
    "STREAM_ROOT",
    "CHECKPOINT_ROOT",
    "RESULTS_ROOT",
    "LOCAL_STAGE_ROOT",
    "DEVICE",
    "GPU_NAME",
    "AMP_DTYPE",
    "USE_SCALER",
    "GLOBAL_SEED",
    "DATA_ORDER_SEED",
    "LOG_EVERY",
    "SAVE_EVERY",
    "SAVE_EVERY_MINUTES",
    "SAVE_FINAL_MODEL",
    "TRANSWAVE_OPERATOR_VERSION",
    "build_model",
    "make_optimizer",
    "make_scheduler",
    "make_scaler",
    "make_mlm_batch",
    "parameter_counts",
    "optimizer_to_gpu",
    "capture_rng",
    "restore_rng",
    "atomic_torch_save",
    "hard_cleanup",
    "set_all_seeds",
)

missing = [
    x
    for x in REQ
    if x not in globals()
]

if missing:

    raise RuntimeError(
        f"Run Cells 1 and 2 first; missing: {missing}"
    )


# =============================================================================
# FIXED VALUES FROM THE COMPLETED TUNING RUN
# =============================================================================

L = 4096

BATCH = 155

USE_CKPT = True

PROTOCOL = (
    "meyer_fixed_L4096_B155_v1"
)


# =============================================================================
# CLEAR INTERRUPTED TUNER STATE ONCE
# =============================================================================

gc.collect()

try:

    torch.cuda.synchronize()

    torch.cuda.empty_cache()

    torch.cuda.ipc_collect()

except Exception:

    pass


hard_cleanup(
    "before fixed Meyer training"
)


# =============================================================================
# OUTPUT PATHS
# =============================================================================

RUN_ROOT = (
    Path(CHECKPOINT_ROOT)
    /
    PROTOCOL
)

RES_ROOT = (
    Path(RESULTS_ROOT)
    /
    PROTOCOL
)


RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

Path(
    LOCAL_STAGE_ROOT
).mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# SMALL HELPERS
# =============================================================================

def atomic_json(
    obj,
    path,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )

    tmp.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    os.replace(
        tmp,
        path,
    )


def sha256(
    path,
    block=8 << 20,
):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            z = f.read(
                block
            )

            if not z:
                break

            h.update(
                z
            )

    return h.hexdigest()


# Do not save the deterministic Meyer response 12 times.
def model_state(
    model,
):

    return {
        k:
            v

        for k, v
        in model.state_dict().items()

        if not k.endswith(
            ".mixer.response"
        )
    }


def load_model_state(
    model,
    state,
):

    missing, unexpected = (
        model.load_state_dict(
            state,
            strict=False,
        )
    )

    bad = [
        k
        for k in missing
        if not k.endswith(
            ".mixer.response"
        )
    ]

    if (
        bad
        or
        unexpected
    ):

        raise RuntimeError(
            "State mismatch: "
            f"missing={bad}, "
            f"unexpected={list(unexpected)}"
        )


def enable_checkpointing(
    model,
):

    def forward(
        self,
        input_ids,
        labels=None,
    ):

        x = self.embeddings(
            input_ids
        )

        if (
            self.training
            and
            torch.is_grad_enabled()
        ):

            for layer in self.layers:

                x = torch_checkpoint(
                    layer,
                    x,
                    use_reentrant=False,
                )

        else:

            for layer in self.layers:

                x = layer(
                    x
                )

        return self.mlm(
            x,
            labels,
        )

    model.forward = types.MethodType(
        forward,
        model,
    )

    return model


# =============================================================================
# READ CELL-3 TOKENIZER
# =============================================================================

SP_MODEL = (
    Path(TOKENIZER_ROOT)
    /
    "spiece.model"
)

SP_MANIFEST = (
    Path(TOKENIZER_ROOT)
    /
    "manifest.json"
)


if (
    not SP_MODEL.is_file()
    or
    not SP_MANIFEST.is_file()
):

    raise FileNotFoundError(
        "Run Cell 3 first."
    )


TOKENIZER_SHA = sha256(
    SP_MODEL
)


tokmeta = json.loads(
    SP_MANIFEST.read_text(
        encoding="utf-8"
    )
)


if (
    tokmeta.get(
        "model_sha256"
    )
    !=
    TOKENIZER_SHA
):

    raise RuntimeError(
        "Tokenizer SHA mismatch."
    )


if (
    tokmeta.get(
        "operator_version"
    )
    !=
    TRANSWAVE_OPERATOR_VERSION
):

    raise RuntimeError(
        "Tokenizer/operator mismatch."
    )


# =============================================================================
# READ ACTUAL FLAT-STREAM SIZES FROM CELL 3
# =============================================================================

META = {}


for corpus in CORPORA:

    root = (
        Path(STREAM_ROOT)
        /
        corpus
    )

    manifest = (
        root
        /
        "manifest.json"
    )

    meta = json.loads(
        manifest.read_text(
            encoding="utf-8"
        )
    )


    if (
        meta.get(
            "tokenizer_sha256"
        )
        !=
        TOKENIZER_SHA
    ):

        raise RuntimeError(
            f"{corpus}: tokenizer mismatch"
        )


    if (
        meta.get(
            "operator_version"
        )
        !=
        TRANSWAVE_OPERATOR_VERSION
    ):

        raise RuntimeError(
            f"{corpus}: operator mismatch"
        )


    META[
        corpus
    ] = {

        "train_n":
            int(
                meta[
                    "train_tokens"
                ]
            ),

        "valid_n":
            int(
                meta[
                    "validation_tokens"
                ]
            ),

        "train":
            root
            /
            "train_tokens.npy",

        "valid":
            root
            /
            "validation_tokens.npy",
    }


# =============================================================================
# STREAM VALIDATION / LOCAL SSD STAGING
# =============================================================================

def stream_ok(
    path,
    n,
):

    a = None

    try:

        a = np.load(
            path,
            mmap_mode="r",
            allow_pickle=False,
        )

        return (
            a.ndim
            ==
            1

            and

            len(
                a
            )
            ==
            n

            and

            np.issubdtype(
                a.dtype,
                np.integer,
            )
        )

    except Exception:

        return False

    finally:

        if a is not None:

            try:

                if (
                    getattr(
                        a,
                        "_mmap",
                        None,
                    )
                    is not None
                ):

                    a._mmap.close()

            except Exception:

                pass


def stage(
    corpus,
    split,
):

    m = META[
        corpus
    ]

    if split == "train":

        src = Path(
            m[
                "train"
            ]
        )

        n = m[
            "train_n"
        ]

    else:

        src = Path(
            m[
                "valid"
            ]
        )

        n = m[
            "valid_n"
        ]


    if not stream_ok(
        src,
        n,
    ):

        raise RuntimeError(
            f"Bad stream: {src}"
        )


    dst = (
        Path(LOCAL_STAGE_ROOT)
        /
        corpus
        /
        src.name
    )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    if (
        dst.is_file()

        and

        dst.stat().st_size
        ==
        src.stat().st_size

        and

        stream_ok(
            dst,
            n,
        )
    ):

        return dst


    tmp = dst.with_suffix(
        dst.suffix
        +
        ".tmp"
    )

    tmp.unlink(
        missing_ok=True
    )


    print(
        "staging Drive -> local SSD:",
        src,
    )


    shutil.copyfile(
        src,
        tmp,
    )

    os.replace(
        tmp,
        dst,
    )


    if not stream_ok(
        dst,
        n,
    ):

        raise RuntimeError(
            f"Staging failed: {dst}"
        )


    return dst


LOCAL = {

    corpus: {

        "train":
            stage(
                corpus,
                "train",
            ),

        "valid":
            stage(
                corpus,
                "valid",
            ),
    }

    for corpus in CORPORA
}


# =============================================================================
# FIXED 4096-TOKEN VIEWS
# =============================================================================

def open_blocks(
    path,
    n,
):

    usable = (
        n
        //
        L
        *
        L
    )


    if usable < L:

        raise RuntimeError(
            f"{path}: fewer than {L} tokens"
        )


    stream = np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )


    blocks = (
        stream[
            :usable
        ]
        .reshape(
            -1,
            L,
        )
    )


    return (
        stream,
        blocks,
        usable,
    )


# =============================================================================
# VALIDATION
# =============================================================================

def evaluate(
    model,
    blocks,
    batch,
    seed,
):

    model.eval()

    total_loss = 0.0

    total_targets = 0


    vb = max(
        1,
        min(
            batch,
            len(
                blocks
            ),
        ),
    )


    try:

        with torch.inference_mode():

            for start in range(
                0,
                len(
                    blocks
                ),
                vb,
            ):

                (
                    ids_cpu,
                    labels_cpu,
                ) = make_mlm_batch(

                    blocks[
                        start:
                        start
                        +
                        vb
                    ],

                    seed
                    +
                    start,

                    validation=True,
                )


                ids = ids_cpu.to(
                    DEVICE,
                    non_blocking=True,
                )

                labels = labels_cpu.to(
                    DEVICE,
                    non_blocking=True,
                )


                with torch.autocast(
                    "cuda",
                    dtype=AMP_DTYPE,
                ):

                    (
                        logits,
                        targets,
                    ) = model(
                        ids,
                        labels,
                    )


                    loss_sum = (
                        F.cross_entropy(
                            logits,
                            targets,
                            reduction="sum",
                        )
                    )


                total_loss += float(
                    loss_sum
                    .detach()
                    .float()
                    .cpu()
                )

                total_targets += int(
                    targets.numel()
                )


                del (
                    ids_cpu,
                    labels_cpu,
                    ids,
                    labels,
                    logits,
                    targets,
                    loss_sum,
                )


    finally:

        model.train()


    if (
        total_targets
        ==
        0
    ):

        raise RuntimeError(
            "Validation produced zero targets"
        )


    return (
        total_loss
        /
        total_targets
    )


# =============================================================================
# TRAIN ONE CORPUS
# =============================================================================

def train_one(
    corpus,
):

    m = META[
        corpus
    ]


    (
        train_stream,
        train,
        train_tokens,
    ) = open_blocks(
        LOCAL[
            corpus
        ][
            "train"
        ],
        m[
            "train_n"
        ],
    )


    (
        valid_stream,
        valid,
        valid_tokens,
    ) = open_blocks(
        LOCAL[
            corpus
        ][
            "valid"
        ],
        m[
            "valid_n"
        ],
    )


    if (
        len(
            train
        )
        <
        BATCH
    ):

        raise RuntimeError(
            f"{corpus}: only "
            f"{len(train)} sequences; "
            f"need batch {BATCH}"
        )


    nseq = len(
        train
    )

    updates = math.ceil(
        nseq
        /
        BATCH
    )

    corpus_index = (
        CORPORA.index(
            corpus
        )
    )


    run_dir = (
        RUN_ROOT
        /
        corpus
    )

    run_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    latest = (
        run_dir
        /
        "training_latest.pt"
    )

    final = (
        run_dir
        /
        "trained_model_final.pt"
    )

    complete = (
        run_dir
        /
        "TRAINING_COMPLETE.json"
    )

    result_path = (
        RES_ROOT
        /
        f"{corpus}.json"
    )


    fingerprint = hashlib.sha256(
        (
            f"{PROTOCOL}|"
            f"{TRANSWAVE_OPERATOR_VERSION}|"
            f"{TOKENIZER_SHA}|"
            f"{corpus}|"
            f"{L}|"
            f"{BATCH}|"
            f"{train_tokens}|"
            f"{valid_tokens}"
        )
        .encode()
    ).hexdigest()


    if complete.is_file():

        done = json.loads(
            complete.read_text(
                encoding="utf-8"
            )
        )


        if (
            done.get(
                "fingerprint"
            )
            !=
            fingerprint
        ):

            raise RuntimeError(
                f"Completion fingerprint mismatch: "
                f"{corpus}"
            )


        print(
            "SKIP complete:",
            corpus,
        )


        return done


    permutation = (
        np.random.default_rng(
            DATA_ORDER_SEED
            +
            1000
            *
            corpus_index
        )
        .permutation(
            nseq
        )
    )


    hard_cleanup(
        f"before training {corpus}"
    )


    set_all_seeds(
        GLOBAL_SEED
    )


    model = (
        build_model(
            L
        )
        .to(
            DEVICE
        )
    )


    model = enable_checkpointing(
        model
    )


    model.train()


    (
        optimizer,
        fused_adam,
    ) = make_optimizer(
        model
    )


    (
        scheduler,
        warmup,
    ) = make_scheduler(
        optimizer,
        updates,
    )


    scaler = (
        make_scaler()
    )


    (
        total_params,
        trainable_params,
    ) = parameter_counts(
        model
    )


    cursor = 0

    update = 0

    prior_seconds = 0.0


    # =========================================================================
    # RESUME
    # =========================================================================

    if latest.is_file():

        ckpt = torch.load(
            latest,
            map_location="cpu",
            weights_only=False,
        )


        expected = {

            "fingerprint":
                fingerprint,

            "context_length":
                L,

            "batch_size":
                BATCH,

            "checkpoint_layers":
                True,

            "corpus":
                corpus,
        }


        for key, value in expected.items():

            if (
                ckpt.get(
                    key
                )
                !=
                value
            ):

                raise RuntimeError(
                    f"Resume mismatch {key}: "
                    f"{ckpt.get(key)!r} "
                    f"!= "
                    f"{value!r}"
                )


        load_model_state(
            model,
            ckpt[
                "model_state"
            ],
        )


        optimizer.load_state_dict(
            ckpt[
                "optimizer_state"
            ]
        )


        optimizer_to_gpu(
            optimizer
        )


        scheduler.load_state_dict(
            ckpt[
                "scheduler_state"
            ]
        )


        scaler.load_state_dict(
            ckpt[
                "scaler_state"
            ]
        )


        restore_rng(
            ckpt[
                "rng_state"
            ]
        )


        cursor = int(
            ckpt[
                "next_cursor"
            ]
        )


        update = int(
            ckpt[
                "update_index"
            ]
        )


        prior_seconds = float(
            ckpt.get(
                "training_seconds",
                0.0,
            )
        )


        del ckpt


        print(
            f"RESUME {corpus}: "
            f"{cursor:,}/"
            f"{nseq:,} sequences"
        )


    session_start = (
        time.perf_counter()
    )


    def elapsed():

        return (
            prior_seconds
            +
            (
                time.perf_counter()
                -
                session_start
            )
        )


    def save_recovery(
        cur,
        upd,
    ):

        atomic_torch_save(
            {

                "fingerprint":
                    fingerprint,

                "protocol_version":
                    PROTOCOL,

                "operator_version":
                    TRANSWAVE_OPERATOR_VERSION,

                "tokenizer_sha256":
                    TOKENIZER_SHA,

                "corpus":
                    corpus,

                "context_length":
                    L,

                "batch_size":
                    BATCH,

                "checkpoint_layers":
                    True,

                "next_cursor":
                    int(
                        cur
                    ),

                "update_index":
                    int(
                        upd
                    ),

                "training_seconds":
                    float(
                        elapsed()
                    ),

                "model_state":
                    model_state(
                        model
                    ),

                "optimizer_state":
                    optimizer.state_dict(),

                "scheduler_state":
                    scheduler.state_dict(),

                "scaler_state":
                    scaler.state_dict(),

                "rng_state":
                    capture_rng(),
            },
            latest,
        )


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        f"MEYER TRAIN | "
        f"{corpus} | "
        f"L={L:,} | "
        f"B={BATCH} | "
        f"checkpointing=ON"
    )

    print(
        f"train tokens="
        f"{train_tokens:,} | "
        f"valid tokens="
        f"{valid_tokens:,} | "
        f"sequences="
        f"{nseq:,} | "
        f"updates="
        f"{updates:,}"
    )

    print(
        f"params="
        f"{total_params:,} | "
        f"trainable="
        f"{trainable_params:,} | "
        f"fused Adam="
        f"{fused_adam} | "
        f"warmup="
        f"{warmup}"
    )

    print(
        "=" * 100
    )


    torch.cuda.reset_peak_memory_stats()


    last_save = (
        time.monotonic()
    )

    last_cursor = (
        cursor
    )

    last_update = (
        update
    )


    try:

        while (
            cursor
            <
            nseq
        ):

            end = min(
                cursor
                +
                BATCH,
                nseq,
            )


            idx = permutation[
                cursor:
                end
            ]


            (
                ids_cpu,
                labels_cpu,
            ) = make_mlm_batch(

                train[
                    idx
                ],

                seed_index=(
                    corpus_index
                    *
                    1_000_000
                    +
                    update
                ),

                validation=False,
            )


            ids = ids_cpu.to(
                DEVICE,
                non_blocking=True,
            )


            labels = labels_cpu.to(
                DEVICE,
                non_blocking=True,
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.autocast(
                "cuda",
                dtype=AMP_DTYPE,
            ):

                (
                    logits,
                    targets,
                ) = model(
                    ids,
                    labels,
                )


                loss = (
                    F.cross_entropy(
                        logits,
                        targets,
                    )
                )


            scale_before = (
                scaler.get_scale()
            )


            scaler.scale(
                loss
            ).backward()


            scaler.step(
                optimizer
            )


            scaler.update()


            if (
                USE_SCALER

                and

                scaler.get_scale()
                <
                scale_before
            ):

                raise FloatingPointError(
                    "AMP skipped optimizer update"
                )


            scheduler.step()


            cursor = (
                end
            )

            update += 1

            last_cursor = (
                cursor
            )

            last_update = (
                update
            )


            if (
                update
                %
                LOG_EVERY
                ==
                0

                or

                cursor
                ==
                nseq
            ):

                torch.cuda.synchronize()


                seconds = (
                    elapsed()
                )


                done = min(
                    cursor
                    *
                    L,
                    train_tokens,
                )


                tokens_per_second = (
                    done
                    /
                    max(
                        seconds,
                        1.0e-9,
                    )
                )


                eta = (
                    (
                        train_tokens
                        -
                        done
                    )
                    /
                    max(
                        tokens_per_second,
                        1.0e-9,
                    )
                    /
                    3600.0
                )


                allocated = (
                    torch.cuda
                    .max_memory_allocated()
                    /
                    1024**3
                )


                reserved = (
                    torch.cuda
                    .max_memory_reserved()
                    /
                    1024**3
                )


                print(
                    f"{corpus:12s} | "
                    f"{cursor:7,d}/"
                    f"{nseq:,} seq | "
                    f"loss "
                    f"{float(loss.detach()):.4f} | "
                    f"{tokens_per_second:,.0f} tok/s | "
                    f"VRAM "
                    f"{allocated:.2f}/"
                    f"{reserved:.2f} GiB | "
                    f"ETA "
                    f"{eta:.2f}h"
                )


            del (
                ids_cpu,
                labels_cpu,
                ids,
                labels,
                logits,
                targets,
                loss,
            )


            if (
                update
                %
                SAVE_EVERY
                ==
                0

                or

                time.monotonic()
                -
                last_save
                >=
                SAVE_EVERY_MINUTES
                *
                60
            ):

                save_recovery(
                    cursor,
                    update,
                )

                last_save = (
                    time.monotonic()
                )


                print(
                    f"  recovery saved at "
                    f"sequence {cursor:,}"
                )


    except BaseException:

        try:

            save_recovery(
                last_cursor,
                last_update,
            )

            print(
                "Emergency recovery checkpoint saved"
            )

        except Exception as exc:

            print(
                "Emergency save failed:",
                exc,
            )

        raise


    torch.cuda.synchronize()


    training_seconds = (
        elapsed()
    )


    # Save fully trained state before validation.
    save_recovery(
        nseq,
        update,
    )


    # =========================================================================
    # VALIDATION
    # =========================================================================

    validation_loss = evaluate(
        model,
        valid,
        BATCH,
        50_000_000
        +
        corpus_index
        *
        100_000,
    )


    result = {

        "status":
            "complete",

        "fingerprint":
            fingerprint,

        "protocol_version":
            PROTOCOL,

        "operator_version":
            TRANSWAVE_OPERATOR_VERSION,

        "wavelet_family":
            "Meyer",

        "operator_trainable_parameters":
            0,

        "tokenizer_sha256":
            TOKENIZER_SHA,

        "tokenizer_type":
            "multiresolution_cross_whitespace_bpe",

        "corpus":
            corpus,

        "context_length":
            L,

        "batch_size":
            BATCH,

        "checkpoint_layers":
            True,

        "usable_train_tokens":
            int(
                train_tokens
            ),

        "usable_validation_tokens":
            int(
                valid_tokens
            ),

        "training_sequences":
            int(
                nseq
            ),

        "optimizer_updates":
            int(
                updates
            ),

        "training_seconds":
            float(
                training_seconds
            ),

        "training_tokens_per_second":
            float(
                train_tokens
                /
                max(
                    training_seconds,
                    1.0e-9,
                )
            ),

        "validation_loss":
            float(
                validation_loss
            ),

        "validation_perplexity":
            float(
                math.exp(
                    min(
                        float(
                            validation_loss
                        ),
                        20.0,
                    )
                )
            ),

        "peak_allocated_vram_gb":
            float(
                torch.cuda
                .max_memory_allocated()
                /
                1024**3
            ),

        "peak_reserved_vram_gb":
            float(
                torch.cuda
                .max_memory_reserved()
                /
                1024**3
            ),

        "parameters":
            int(
                total_params
            ),

        "trainable_parameters":
            int(
                trainable_params
            ),

        "gpu":
            GPU_NAME,

        "torch":
            torch.__version__,

        "cuda":
            torch.version.cuda,
    }


    if SAVE_FINAL_MODEL:

        atomic_torch_save(
            {

                **result,

                "model_state":
                    model_state(
                        model
                    ),
            },
            final,
        )


    atomic_json(
        result,
        result_path,
    )


    # Completion marker LAST.
    atomic_json(
        result,
        complete,
    )


    latest.unlink(
        missing_ok=True
    )


    for stream in (
        train_stream,
        valid_stream,
    ):

        try:

            if (
                getattr(
                    stream,
                    "_mmap",
                    None,
                )
                is not None
            ):

                stream._mmap.close()

        except Exception:

            pass


    model = None

    optimizer = None

    scheduler = None

    scaler = None


    hard_cleanup(
        f"finished {corpus}"
    )


    print(
        f"COMPLETE | "
        f"{corpus} | "
        f"L={L:,} | "
        f"B={BATCH} | "
        f"val="
        f"{validation_loss:.6f} | "
        f"ppl="
        f"{result['validation_perplexity']:.3f} | "
        f"{result['training_tokens_per_second']:,.0f} tok/s"
    )


    return result


# =============================================================================
# TRAIN ONLY — NO AUTOTUNING
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "MEYER-TRANSWAVE FIXED TRAINING ONLY"
)

print(
    "Context length : 4,096"
)

print(
    "Batch size     : 155"
)

print(
    "Checkpointing  : ON"
)

print(
    "Measured tuning: "
    "B=155 gave 95,586 tok/s"
)

print(
    "L=8,192/B=128 : failed"
)

print(
    "No context search. No batch search."
)

print(
    "=" * 100
)


results = []


for corpus in CORPORA:

    results.append(
        train_one(
            corpus
        )
    )


    atomic_json(
        results,
        RES_ROOT
        /
        "all_results.json",
    )


print(
    "\n"
    +
    "=" * 100
)

print(
    "ALL FOUR MEYER RUNS COMPLETE"
)

print(
    "Results:",
    RES_ROOT
    /
    "all_results.json",
)

print(
    "=" * 100
)

[cleanup] before fixed Meyer training | alloc=1.372 GiB | reserved=1.510 GiB

MEYER-TRANSWAVE FIXED TRAINING ONLY
Context length : 4,096
Batch size     : 155
Checkpointing  : ON
Measured tuning: B=155 gave 95,586 tok/s
L=8,192/B=128 : failed
No context search. No batch search.
[cleanup] before training fineweb_edu | alloc=1.372 GiB | reserved=1.510 GiB

MEYER TRAIN | fineweb_edu | L=4,096 | B=155 | checkpointing=ON
train tokens=107,638,784 | valid tokens=3,350,528 | sequences=26,279 | updates=170
params=81,907,712 | trainable=81,907,712 | fused Adam=True | warmup=2
fineweb_edu  |   3,875/26,279 seq | loss 8.9872 | 95,474 tok/s | VRAM 80.80/93.04 GiB | ETA 0.27h
fineweb_edu  |   7,750/26,279 seq | loss 8.6273 | 95,514 tok/s | VRAM 80.80/93.04 GiB | ETA 0.22h
fineweb_edu  |  11,625/26,279 seq | loss 8.5475 | 95,496 tok/s | VRAM 80.80/93.04 GiB | ETA 0.17h
fineweb_edu  |  15,500/26,279 seq | loss 8.5816 | 95,512 tok/s | VRAM 80.80/93.04 GiB | ETA 0.13h
fineweb_edu  |  19,375/26,279 seq | 

In [2]:
# CELL 1 — TransWave v2 common setup: true 15% MLM supervision
# Replaces the previous Meyer Cell 1. Existing Cell-3 tokenizer/streams are preserved.

from __future__ import annotations

import os, gc, json, math, time, random, shutil, ctypes, hashlib
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    raise RuntimeError("This cell is intended for Google Colab.") from exc

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required.")

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = bool(torch.cuda.is_bf16_supported())
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
AMP_NAME = "bfloat16" if BF16 else "float16"
USE_SCALER = not BF16

VOCAB_SIZE = 32_000
HIDDEN_SIZE = 768
NUM_LAYERS = 12
INTERMEDIATE_SIZE = 3072
HIDDEN_DROPOUT = 0.1
LAYER_NORM_EPS = 1.0e-12
INITIALIZER_RANGE = 0.02

PAD_ID = 3
MASK_ID = 6
SPECIAL_IDS = (0, 1, 2, 3, 4, 5, 6)

MLM_PROBABILITY = 0.15

LEARNING_RATE = 1.0e-4
ADAM_BETAS = (0.9, 0.999)
ADAM_EPS = 1.0e-8
WEIGHT_DECAY = 0.0

GLOBAL_SEED = 2026
DATA_ORDER_SEED = 7301
MASK_SEED = 912431

LOG_EVERY = 25
SAVE_EVERY = 500
SAVE_EVERY_MINUTES = 20
SAVE_FINAL_MODEL = True

CORPORA = (
    "fineweb_edu",
    "arxiv",
    "pg19",
    "wikitext103",
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/TransWave_Meyer_Context"
)

RAW_ROOT = (
    PROJECT_ROOT
    /
    "raw_text"
)

TOKENIZER_ROOT = (
    PROJECT_ROOT
    /
    "tokenizer"
)

STREAM_ROOT = (
    PROJECT_ROOT
    /
    "streams"
)

CHECKPOINT_ROOT = (
    PROJECT_ROOT
    /
    "checkpoints"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    /
    "results"
)

CONFIG_ROOT = (
    PROJECT_ROOT
    /
    "configs"
)

LOCAL_ROOT = Path(
    "/content/TransWave_Meyer_Context"
)

LOCAL_STAGE_ROOT = (
    LOCAL_ROOT
    /
    "staged"
)

LOCAL_TMP_ROOT = (
    LOCAL_ROOT
    /
    "tmp"
)

for p in (
    PROJECT_ROOT,
    RAW_ROOT,
    TOKENIZER_ROOT,
    STREAM_ROOT,
    CHECKPOINT_ROOT,
    RESULTS_ROOT,
    CONFIG_ROOT,
    LOCAL_ROOT,
    LOCAL_STAGE_ROOT,
    LOCAL_TMP_ROOT,
):
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

CPU_THREADS = max(
    1,
    min(
        32,
        os.cpu_count() or 8,
    ),
)

torch.set_num_threads(
    CPU_THREADS
)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

try:
    torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = True
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = True
    torch.set_float32_matmul_precision(
        "high"
    )
except Exception:
    pass


def set_all_seeds(
    seed: int,
) -> None:

    seed = int(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    torch.cuda.manual_seed_all(
        seed
    )


def _malloc_trim() -> None:

    try:

        ctypes.CDLL(
            "libc.so.6"
        ).malloc_trim(
            0
        )

    except Exception:

        pass


def hard_cleanup(
    label: str = "",
) -> None:

    try:
        torch.cuda.synchronize()
    except Exception:
        pass

    gc.collect()

    try:

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

    except Exception:

        pass

    _malloc_trim()

    if label:

        print(
            f"[cleanup] {label} | "
            f"alloc="
            f"{torch.cuda.memory_allocated()/1024**3:.3f} GiB | "
            f"reserved="
            f"{torch.cuda.memory_reserved()/1024**3:.3f} GiB"
        )


set_all_seeds(
    GLOBAL_SEED
)


def make_mlm_batch(
    clean_np,
    seed_index: int,
    validation: bool = False,
):

    """
    Deterministic BERT-style MLM with an actual 15% target density.

    For each row, exactly round(0.15 * eligible_tokens) positions
    are selected.

    There is NO 80-target cap.

    Selected tokens use the standard:
        80% -> [MASK]
        10% -> random token
        10% -> unchanged
    """

    clean = torch.from_numpy(
        np.asarray(
            clean_np,
            dtype=np.int64,
        ).copy()
    )

    if clean.ndim != 2:

        raise ValueError(
            f"Expected [B,L], got {tuple(clean.shape)}"
        )

    ids = clean.clone()

    labels = torch.full_like(
        ids,
        -100,
    )

    g = torch.Generator(
        device="cpu"
    )

    g.manual_seed(
        MASK_SEED
        +
        int(
            seed_index
        )
        +
        (
            1_000_000_000
            if validation
            else 0
        )
    )

    eligible = torch.ones_like(
        ids,
        dtype=torch.bool,
    )

    for sid in SPECIAL_IDS:

        eligible &= ids.ne(
            sid
        )

    # At B=155,L=4096 this CPU score matrix is small.
    # Top-k makes the supervision density exactly 15% per row.
    scores = torch.rand(
        ids.shape,
        generator=g,
    )

    eligible_counts = eligible.sum(
        dim=1
    )

    target_counts = torch.round(
        eligible_counts.float()
        *
        MLM_PROBABILITY
    ).long()

    target_counts = torch.where(
        eligible_counts.gt(
            0
        ),
        target_counts.clamp_min(
            1
        ),
        target_counts,
    )

    selected = torch.zeros_like(
        eligible
    )

    max_k = (
        int(
            target_counts.max().item()
        )
        if target_counts.numel()
        else 0
    )

    if max_k:

        ranked = scores.masked_fill(
            ~eligible,
            float(
                "inf"
            ),
        )

        idx = torch.topk(
            ranked,
            k=max_k,
            dim=1,
            largest=False,
            sorted=True,
        ).indices

        ranks = torch.arange(
            max_k
        ).view(
            1,
            -1,
        )

        keep = (
            ranks
            <
            target_counts.view(
                -1,
                1,
            )
        )

        selected.scatter_(
            1,
            idx,
            keep,
        )

        selected &= (
            eligible
        )

    labels[
        selected
    ] = ids[
        selected
    ]

    replacement = torch.rand(
        ids.shape,
        generator=g,
    )

    mask_pos = (
        selected
        &
        replacement.lt(
            0.80
        )
    )

    rand_pos = (
        selected
        &
        replacement.ge(
            0.80
        )
        &
        replacement.lt(
            0.90
        )
    )

    ids[
        mask_pos
    ] = MASK_ID

    n_rand = int(
        rand_pos.sum().item()
    )

    if n_rand:

        ids[
            rand_pos
        ] = torch.randint(
            7,
            VOCAB_SIZE,
            (
                n_rand,
            ),
            generator=g,
            dtype=torch.long,
        )

    return (
        ids.pin_memory(),
        labels.pin_memory(),
    )


def gelu_new(
    x,
):

    return (
        0.5
        *
        x
        *
        (
            1.0
            +
            torch.tanh(
                math.sqrt(
                    2.0
                    /
                    math.pi
                )
                *
                (
                    x
                    +
                    0.044715
                    *
                    x.pow(
                        3
                    )
                )
            )
        )
    )


class CommonEmbeddings(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()

        self.word_embeddings = nn.Embedding(
            VOCAB_SIZE,
            HIDDEN_SIZE,
            padding_idx=PAD_ID,
        )

        self.norm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

        self.dropout = nn.Dropout(
            HIDDEN_DROPOUT
        )

    def forward(
        self,
        input_ids,
    ):

        return self.dropout(
            self.norm(
                self.word_embeddings(
                    input_ids
                )
            )
        )


class CommonFFN(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()

        self.fc1 = nn.Linear(
            HIDDEN_SIZE,
            INTERMEDIATE_SIZE,
        )

        self.fc2 = nn.Linear(
            INTERMEDIATE_SIZE,
            HIDDEN_SIZE,
        )

        self.dropout = nn.Dropout(
            HIDDEN_DROPOUT
        )

        self.norm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

    def forward(
        self,
        x,
    ):

        y = self.dropout(
            self.fc2(
                gelu_new(
                    self.fc1(
                        x
                    )
                )
            )
        )

        return self.norm(
            x
            +
            y
        )


class MLMHead(
    nn.Module
):

    """
    Masked-only head.

    Projection can be deferred and chunked by Cell 4.
    """

    def __init__(
        self,
        word_embeddings,
    ):

        super().__init__()

        self.fc = nn.Linear(
            HIDDEN_SIZE,
            HIDDEN_SIZE,
        )

        self.norm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

        self.bias = nn.Parameter(
            torch.zeros(
                VOCAB_SIZE
            )
        )

        self.word_embeddings = (
            word_embeddings
        )

    def prepare(
        self,
        x,
        labels=None,
    ):

        targets = None

        if labels is not None:

            selected = labels.ne(
                -100
            )

            x = x[
                selected
            ]

            targets = labels[
                selected
            ]

        x = self.norm(
            gelu_new(
                self.fc(
                    x
                )
            )
        )

        return (
            x,
            targets,
        )

    def project(
        self,
        x,
    ):

        return F.linear(
            x,
            self.word_embeddings.weight,
            self.bias,
        )

    def forward(
        self,
        x,
        labels=None,
        project: bool = True,
    ):

        x, targets = self.prepare(
            x,
            labels,
        )

        if not project:

            return (
                x,
                targets,
            )

        return (
            self.project(
                x
            ),
            targets,
        )


def parameter_counts(
    model,
):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return (
        int(
            total
        ),
        int(
            trainable
        ),
    )


def make_optimizer(
    model,
):

    kwargs = dict(
        lr=LEARNING_RATE,
        betas=ADAM_BETAS,
        eps=ADAM_EPS,
        weight_decay=WEIGHT_DECAY,
    )

    try:

        return (
            torch.optim.Adam(
                model.parameters(),
                fused=True,
                **kwargs,
            ),
            True,
        )

    except Exception:

        return (
            torch.optim.Adam(
                model.parameters(),
                **kwargs,
            ),
            False,
        )


def make_scheduler(
    optimizer,
    total_updates: int,
):

    total_updates = int(
        total_updates
    )

    if total_updates <= 0:

        raise ValueError(
            "total_updates must be positive"
        )

    warmup = max(
        1,
        round(
            0.01
            *
            total_updates
        ),
    )

    def factor(
        step,
    ):

        if step < warmup:

            return (
                float(
                    step
                    +
                    1
                )
                /
                float(
                    warmup
                )
            )

        return 1.0

    return (
        torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            factor,
        ),
        warmup,
    )


def make_scaler():

    return torch.amp.GradScaler(
        "cuda",
        enabled=USE_SCALER,
    )


def optimizer_to_gpu(
    optimizer,
):

    for state in optimizer.state.values():

        for key, value in list(
            state.items()
        ):

            if torch.is_tensor(
                value
            ):

                state[
                    key
                ] = value.to(
                    DEVICE,
                    non_blocking=True,
                )


def capture_rng():

    return {
        "python":
            random.getstate(),

        "numpy":
            np.random.get_state(),

        "torch":
            torch.get_rng_state(),

        "cuda":
            torch.cuda.get_rng_state_all(),
    }


def restore_rng(
    state,
):

    random.setstate(
        state[
            "python"
        ]
    )

    np.random.set_state(
        state[
            "numpy"
        ]
    )

    torch.set_rng_state(
        state[
            "torch"
        ]
    )

    torch.cuda.set_rng_state_all(
        state[
            "cuda"
        ]
    )


def atomic_torch_save(
    payload,
    destination,
):

    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    stamp = (
        f"{os.getpid()}_"
        f"{time.time_ns()}"
    )

    local_tmp = (
        LOCAL_TMP_ROOT
        /
        f"{destination.name}."
        f"{stamp}.tmp"
    )

    drive_tmp = (
        destination.parent
        /
        f"{destination.name}."
        f"{stamp}.tmp"
    )

    local_tmp.unlink(
        missing_ok=True
    )

    drive_tmp.unlink(
        missing_ok=True
    )

    try:

        torch.save(
            payload,
            local_tmp,
        )

        probe = torch.load(
            local_tmp,
            map_location="cpu",
            weights_only=False,
        )

        del probe

        shutil.copyfile(
            local_tmp,
            drive_tmp,
        )

        os.replace(
            drive_tmp,
            destination,
        )

    finally:

        local_tmp.unlink(
            missing_ok=True
        )

        drive_tmp.unlink(
            missing_ok=True
        )


def atomic_json(
    payload,
    destination,
):

    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = destination.with_suffix(
        destination.suffix
        +
        ".tmp"
    )

    tmp.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    os.replace(
        tmp,
        destination,
    )


def sha256_file(
    path,
    block=8 << 20,
):

    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:

        while True:

            b = f.read(
                block
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


print(
    "=" * 96
)

print(
    "TRANSWAVE v2 CELL 1 READY"
)

print(
    "GPU             :",
    GPU_NAME,
)

print(
    "VRAM            :",
    f"{VRAM_GB:.2f} GiB",
)

print(
    "AMP             :",
    AMP_NAME,
)

print(
    "MLM supervision : EXACT 15% of eligible tokens per sequence; NO 80-token cap"
)

print(
    "At L=4096       : ~614 targets/sequence before special-token adjustment"
)

print(
    "Cell-3 artifacts: PRESERVED; do not rerun tokenizer/stream generation"
)

print(
    "=" * 96
)

Mounted at /content/drive
TRANSWAVE v2 CELL 1 READY
GPU             : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM            : 94.97 GiB
AMP             : bfloat16
MLM supervision : EXACT 15% of eligible tokens per sequence; NO 80-token cap
At L=4096       : ~614 targets/sequence before special-token adjustment
Cell-3 artifacts: PRESERVED; do not rerun tokenizer/stream generation


In [3]:
# CELL 2 — TransWave v2: learned Meyer scale mixing
# Fixed Meyer basis, learned scale gains, global FFT mixing.

from __future__ import annotations

import math
import contextlib

import torch
import torch.nn as nn

from torch.utils.checkpoint import checkpoint as torch_checkpoint


TRANSWAVE_KIND = (
    "transwave_meyer_learned_scales"
)

TRANSWAVE_OPERATOR_VERSION = (
    "periodic_meyer_learned_scale_groups_v1"
)

TRANSWAVE_MEYER_TRANSITION = (
    1.0
    /
    3.0
)

# 768 / 12 = 64 hidden channels per scale-weight group.
MIXER_GROUPS = 12


for _name in (
    "HIDDEN_SIZE",
    "NUM_LAYERS",
    "LAYER_NORM_EPS",
    "INITIALIZER_RANGE",
    "CommonEmbeddings",
    "CommonFFN",
    "MLMHead",
    "GLOBAL_SEED",
    "set_all_seeds",
):

    if _name not in globals():

        raise RuntimeError(
            f"Run replacement Cell 1 first; missing {_name}"
        )


if (
    HIDDEN_SIZE
    %
    MIXER_GROUPS
    !=
    0
):

    raise RuntimeError(
        "HIDDEN_SIZE must be divisible by MIXER_GROUPS"
    )


def _meyer_nu(
    t,
):

    t = t.clamp(
        0.0,
        1.0,
    )

    return (
        t**4
        *
        (
            35.0
            -
            84.0
            *
            t
            +
            70.0
            *
            t**2
            -
            20.0
            *
            t**3
        )
    )


def _meyer_lowpass_power(
    f,
    cutoff,
    transition,
):

    lo = max(
        0.0,
        float(
            cutoff
        )
        *
        (
            1.0
            -
            float(
                transition
            )
        ),
    )

    hi = min(
        1.0,
        float(
            cutoff
        )
        *
        (
            1.0
            +
            float(
                transition
            )
        ),
    )

    if hi <= lo:

        return (
            f
            <=
            float(
                cutoff
            )
        ).to(
            torch.float32
        )

    out = torch.ones_like(
        f,
        dtype=torch.float32,
    )

    out = torch.where(
        f
        >=
        hi,
        torch.zeros_like(
            out
        ),
        out,
    )

    mid = (
        f
        >
        lo
    ) & (
        f
        <
        hi
    )

    if bool(
        mid.any()
    ):

        u = (
            f[
                mid
            ]
            -
            lo
        ) / (
            hi
            -
            lo
        )

        out[
            mid
        ] = torch.cos(
            0.5
            *
            math.pi
            *
            _meyer_nu(
                u
            )
        ).square()

    return out


def build_periodic_meyer_basis(
    length: int,
):

    """
    Fixed Meyer band responses

        B_j(w) = Q_j(w) exp(-i w tau_j).

    With every learned gain equal to one,

        sum_j B_j(w)

    exactly reproduces the previous fixed Meyer operator.
    """

    L = int(
        length
    )

    if (
        L
        <
        512

        or

        (
            L
            &
            (
                L
                -
                1
            )
        )
        !=
        0
    ):

        raise ValueError(
            "Context length must be a power of two >= 512"
        )

    num_scales = (
        int(
            math.log2(
                L
            )
        )
        -
        1
    )

    freq = torch.fft.rfftfreq(
        L,
        d=1.0,
    ).to(
        torch.float32
    )

    f_norm = (
        freq
        /
        0.5
    ).clamp(
        0.0,
        1.0,
    )

    omega = (
        2.0
        *
        math.pi
        *
        freq
    )

    cumulative = torch.ones_like(
        f_norm
    )

    windows = []

    shifts = []

    for j in range(
        1,
        num_scales
        +
        1,
    ):

        low = _meyer_lowpass_power(
            f_norm,
            cutoff=2.0 ** (-j),
            transition=TRANSWAVE_MEYER_TRANSITION,
        )

        windows.append(
            (
                cumulative
                -
                low
            ).clamp_min(
                0.0
            )
        )

        magnitude = min(
            2 ** (j - 1),
            L // 4,
        )

        shifts.append(
            magnitude
            if j % 2 == 1
            else -magnitude
        )

        cumulative = (
            low
        )

    windows.append(
        cumulative.clamp_min(
            0.0
        )
    )

    shifts.append(
        L
        //
        2
    )

    Q = torch.stack(
        windows,
        dim=0,
    )

    Q = (
        Q
        /
        Q.sum(
            dim=0
        )
        .clamp_min(
            1.0e-12
        )
        .unsqueeze(
            0
        )
    )

    basis = []

    for q, tau in zip(
        Q,
        shifts,
    ):

        phase = torch.exp(
            (
                -1j
                *
                omega
                *
                float(
                    tau
                )
            ).to(
                torch.complex64
            )
        )

        b = (
            q.to(
                torch.complex64
            )
            *
            phase
        )

        b = b.clone()

        b[
            0
        ] = torch.complex(
            b[
                0
            ].real,
            torch.zeros_like(
                b[
                    0
                ].real
            ),
        )

        if (
            L
            %
            2
            ==
            0
        ):

            b[
                -1
            ] = torch.complex(
                b[
                    -1
                ].real,
                torch.zeros_like(
                    b[
                        -1
                    ].real
                ),
            )

        basis.append(
            b
        )

    basis = torch.stack(
        basis,
        dim=0,
    ).contiguous()

    partition_error = float(
        (
            Q.sum(
                dim=0
            )
            -
            1.0
        )
        .abs()
        .max()
    )

    fixed_response = basis.sum(
        dim=0
    )

    return (
        basis,
        {
            "length":
                L,

            "num_scales":
                num_scales,

            "num_bands":
                int(
                    basis.shape[
                        0
                    ]
                ),

            "shifts":
                shifts,

            "partition_error":
                partition_error,

            "fixed_response_max_abs":
                float(
                    fixed_response
                    .abs()
                    .max()
                ),
        },
    )


class LearnedMeyerWaveletOperator(
    nn.Module
):

    """
    Fixed Meyer basis; learned scale weights.

    Each channel group learns:

        H_g(w)
        =
        sum_j a[g,j] B_j(w).

    Still only:

        RFFT -> multiplier -> IRFFT.
    """

    def __init__(
        self,
        context_length: int,
    ):

        super().__init__()

        self.context_length = int(
            context_length
        )

        (
            basis,
            diagnostics,
        ) = build_periodic_meyer_basis(
            self.context_length
        )

        self.diagnostics = (
            diagnostics
        )

        self.num_bands = int(
            basis.shape[
                0
            ]
        )

        self.group_width = (
            HIDDEN_SIZE
            //
            MIXER_GROUPS
        )

        # Deterministic basis is reconstructed from L.
        # It is deliberately not written into checkpoints.
        self.register_buffer(
            "response_basis",
            basis,
            persistent=False,
        )

        self.scale_gains = nn.Parameter(
            torch.ones(
                MIXER_GROUPS,
                self.num_bands,
            )
        )

        group_index = (
            torch.arange(
                HIDDEN_SIZE,
                dtype=torch.long,
            )
            //
            self.group_width
        )

        self.register_buffer(
            "group_index",
            group_index,
            persistent=False,
        )

    def _channel_response(
        self,
    ):

        gains = self.scale_gains.float()

        br = self.response_basis.real

        bi = self.response_basis.imag

        rr = torch.matmul(
            gains,
            br,
        )

        ri = torch.matmul(
            gains,
            bi,
        )

        group_response = torch.complex(
            rr,
            ri,
        )

        return group_response.index_select(
            0,
            self.group_index,
        )

    def forward(
        self,
        x,
    ):

        if (
            x.ndim
            !=
            3

            or

            int(
                x.shape[
                    1
                ]
            )
            !=
            self.context_length
        ):

            raise RuntimeError(
                f"Expected "
                f"[B,{self.context_length},D], "
                f"got {tuple(x.shape)}"
            )

        if (
            int(
                x.shape[
                    2
                ]
            )
            !=
            HIDDEN_SIZE
        ):

            raise RuntimeError(
                f"Expected hidden size "
                f"{HIDDEN_SIZE}, "
                f"got {x.shape[2]}"
            )

        original_dtype = (
            x.dtype
        )

        z = (
            x.transpose(
                1,
                2,
            )
            .contiguous()
        )

        ctx = (
            torch.autocast(
                device_type="cuda",
                enabled=False,
            )

            if z.is_cuda

            else contextlib.nullcontext()
        )

        with ctx:

            Z = torch.fft.rfft(
                z.float(),
                n=self.context_length,
                dim=-1,
            )

            H = self._channel_response()

            y = torch.fft.irfft(
                Z
                *
                H.unsqueeze(
                    0
                ),
                n=self.context_length,
                dim=-1,
            )

        return (
            y.transpose(
                1,
                2,
            )
            .contiguous()
            .to(
                original_dtype
            )
        )


class TransWaveMeyerLayer(
    nn.Module
):

    def __init__(
        self,
        context_length: int,
    ):

        super().__init__()

        self.mixer = (
            LearnedMeyerWaveletOperator(
                context_length
            )
        )

        self.mixer_norm = nn.LayerNorm(
            HIDDEN_SIZE,
            eps=LAYER_NORM_EPS,
        )

        self.ffn = (
            CommonFFN()
        )

    def forward(
        self,
        x,
    ):

        x = self.mixer_norm(
            x
            +
            self.mixer(
                x
            )
        )

        return self.ffn(
            x
        )


class TransWaveMeyerMLM(
    nn.Module
):

    def __init__(
        self,
        context_length: int,
    ):

        super().__init__()

        self.kind = (
            TRANSWAVE_KIND
        )

        self.context_length = int(
            context_length
        )

        self.embeddings = (
            CommonEmbeddings()
        )

        self.layers = nn.ModuleList(
            [
                TransWaveMeyerLayer(
                    self.context_length
                )

                for _ in range(
                    NUM_LAYERS
                )
            ]
        )

        self.mlm = MLMHead(
            self.embeddings
            .word_embeddings
        )

        self._reset_parameters()

    def _reset_parameters(
        self,
    ):

        for module in self.modules():

            if isinstance(
                module,
                nn.Linear,
            ):

                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=INITIALIZER_RANGE,
                )

                if (
                    module.bias
                    is not None
                ):

                    nn.init.zeros_(
                        module.bias
                    )

            elif isinstance(
                module,
                nn.Embedding,
            ):

                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=INITIALIZER_RANGE,
                )

                if (
                    module.padding_idx
                    is not None
                ):

                    with torch.no_grad():

                        module.weight[
                            module.padding_idx
                        ].zero_()

            elif isinstance(
                module,
                nn.LayerNorm,
            ):

                nn.init.ones_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

        nn.init.zeros_(
            self.mlm.bias
        )

        # Starts exactly at the old fixed Meyer response.
        for layer in self.layers:

            nn.init.ones_(
                layer
                .mixer
                .scale_gains
            )

    def forward(
        self,
        input_ids,
        labels=None,
        project: bool = True,
        checkpoint_layers: bool = False,
    ):

        x = self.embeddings(
            input_ids
        )

        if (
            checkpoint_layers
            and
            self.training
            and
            torch.is_grad_enabled()
        ):

            for layer in self.layers:

                x = torch_checkpoint(
                    layer,
                    x,
                    use_reentrant=False,
                )

        else:

            for layer in self.layers:

                x = layer(
                    x
                )

        return self.mlm(
            x,
            labels,
            project=project,
        )


def build_model(
    context_length: int,
):

    set_all_seeds(
        GLOBAL_SEED
    )

    return TransWaveMeyerMLM(
        int(
            context_length
        )
    )


_probe_basis, _probe_diag = (
    build_periodic_meyer_basis(
        4096
    )
)

print(
    "=" * 96
)

print(
    "TRANSWAVE v2 CELL 2 READY"
)

print(
    "Operator          : fixed Meyer basis + LEARNED scale gains"
)

print(
    "Context target    : 4,096"
)

print(
    "Meyer bands       :",
    _probe_diag[
        "num_bands"
    ],
)

print(
    "Channel groups    :",
    MIXER_GROUPS,
    f"({HIDDEN_SIZE // MIXER_GROUPS} channels/group)",
)

print(
    "Mixer parameters  :",
    NUM_LAYERS
    *
    MIXER_GROUPS
    *
    _probe_diag[
        "num_bands"
    ],
    "total",
)

print(
    "Initialization    : all gains = 1, reproducing old fixed response"
)

print(
    "Runtime           : RFFT -> learned grouped Meyer multiplier -> IRFFT"
)

print(
    "=" * 96
)

TRANSWAVE v2 CELL 2 READY
Operator          : fixed Meyer basis + LEARNED scale gains
Context target    : 4,096
Meyer bands       : 12
Channel groups    : 12 (64 channels/group)
Mixer parameters  : 1728 total
Initialization    : all gains = 1, reproducing old fixed response
Runtime           : RFFT -> learned grouped Meyer multiplier -> IRFFT


In [10]:
# =============================================================================
# CELL 3 — TransWave learned lifting tokenizer + raw-byte data
#
# arXiv ONLY.
#
# NO BPE.
# NO SentencePiece.
# NO tokenizer-training preprocessing job.
# NO token shards.
#
# Raw UTF-8 bytes are the atomic tokens.
# The tokenizer itself is a learned GPU module trained end-to-end with TransWave.
#
# This cell:
#   1. deletes ONLY the abandoned "more-token" shard experiments;
#   2. finds the existing prepared arXiv train/validation text;
#   3. uses those files directly as uint8 token streams;
#   4. writes a small manifest;
#   5. defines the learned 3-level lifting tokenizer;
#   6. defines exact 15% GPU byte-block masking.
#
# It PRESERVES:
#   - old canonical BPE streams
#   - old BPE tokenizer
#   - previous checkpoints/results/models
# =============================================================================

from __future__ import annotations

import os
import json
import math
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================================
# MOUNT DRIVE
# =============================================================================

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    raise RuntimeError("This cell is intended for Google Colab.") from exc


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/TransWave_Meyer_Context"
)

PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# REMOVE THE ABANDONED LONG-TOKEN SHARDS
# =============================================================================

# ONLY these known experimental directories are removed.
# /streams is NOT touched.

_OBSOLETE_DRIVE_NAMES = {
    "streams_more_tokens_transwave_v1",
    "streams_more_tokens_v2",
    "streams_more_tokens_v1",
}

for name in sorted(_OBSOLETE_DRIVE_NAMES):

    path = (
        PROJECT_ROOT
        /
        name
    )

    # Hard safety guard.
    if (
        path.parent != PROJECT_ROOT
        or
        path.name not in _OBSOLETE_DRIVE_NAMES
    ):
        raise RuntimeError(
            f"Unsafe deletion refused: {path}"
        )

    if path.exists():

        print(
            "Removing obsolete shard tree:",
            path,
        )

        shutil.rmtree(
            path
        )


for path in (
    Path(
        "/content/"
        "TransWave_Meyer_Context_more_tokens_transwave_v1"
    ),
    Path(
        "/content/"
        "TransWave_Meyer_Context_more_tokens_v2"
    ),
    Path(
        "/content/"
        "TransWave_Meyer_Context_more_tokens_v1"
    ),
):

    if path.exists():

        print(
            "Removing obsolete local scratch:",
            path,
        )

        shutil.rmtree(
            path
        )


# =============================================================================
# ARXIV RAW DATA
# =============================================================================

CORPUS = "arxiv"

RAW_ROOT_CANDIDATES = (

    PROJECT_ROOT
    /
    "raw_text",

    Path(
        "/content/drive/MyDrive/"
        "TransWave_Meyer/raw_text"
    ),

    Path(
        "/content/drive/MyDrive/"
        "spliNet/raw_text"
    ),
)


def find_raw_file(
    corpus: str,
    filename: str,
) -> Path:

    checked = []

    for root in RAW_ROOT_CANDIDATES:

        path = (
            root
            /
            corpus
            /
            filename
        )

        checked.append(
            path
        )

        if (
            path.is_file()
            and
            path.stat().st_size > 0
        ):

            return path

    raise FileNotFoundError(
        f"Could not find {corpus}/{filename}.\n"
        "Checked only:\n  - "
        +
        "\n  - ".join(
            str(p)
            for p in checked
        )
    )


RAW_TRAIN = find_raw_file(
    CORPUS,
    "train.txt",
)

RAW_VALID = find_raw_file(
    CORPUS,
    "validation.txt",
)


# =============================================================================
# NEW TOKENIZER GEOMETRY
# =============================================================================

# IDs:
#
#   0..255 = ordinary UTF-8 bytes
#   256    = MASK
#   257    = PAD
#
# No BOS/EOS is required here because newline byte 0x0A is already retained
# in the prepared raw text and therefore exposes document boundaries.

BYTE_VOCAB = 256

MASK_BYTE_ID = 256

PAD_BYTE_ID = 257

INPUT_VOCAB = 258


# Three dyadic lifting stages:
#
#       8 raw bytes
#           ↓
#       4 D0 slots
#       2 D1 slots
#       1 D2 slot
#       1 S3 slot
#
#       total = 8 slots

LIFTING_LEVELS = 3

BYTES_PER_MODEL_TOKEN = (
    2
    **
    LIFTING_LEVELS
)

assert BYTES_PER_MODEL_TOKEN == 8


# Critically sampled width:
#
#   8L × 96 = L × 768

BYTE_DIM = 96

MODEL_DIM = 768

assert (
    BYTE_DIM
    *
    BYTES_PER_MODEL_TOKEN
    ==
    MODEL_DIM
)


MODEL_CONTEXT = 4096

RAW_BYTES_PER_SEQUENCE = (
    MODEL_CONTEXT
    *
    BYTES_PER_MODEL_TOKEN
)

assert (
    RAW_BYTES_PER_SEQUENCE
    ==
    32_768
)


# =============================================================================
# BYTE "TOKENIZATION"
# =============================================================================

# There is deliberately no encoding pass.
#
# train.txt is ALREADY a byte stream on disk.
#
# np.memmap(path, dtype=np.uint8) maps it directly as token IDs 0..255.
#
# Therefore we avoid:
#   - BPE training
#   - SentencePiece
#   - CPU tokenization
#   - token shards
#   - another multi-hundred-MB/GB copy on Drive

TRAIN_BYTES = int(
    RAW_TRAIN.stat().st_size
)

VALID_BYTES = int(
    RAW_VALID.stat().st_size
)


USABLE_TRAIN_BYTES = (
    TRAIN_BYTES
    //
    RAW_BYTES_PER_SEQUENCE
    *
    RAW_BYTES_PER_SEQUENCE
)

USABLE_VALID_BYTES = (
    VALID_BYTES
    //
    RAW_BYTES_PER_SEQUENCE
    *
    RAW_BYTES_PER_SEQUENCE
)


TRAIN_SEQUENCES = (
    USABLE_TRAIN_BYTES
    //
    RAW_BYTES_PER_SEQUENCE
)

VALID_SEQUENCES = (
    USABLE_VALID_BYTES
    //
    RAW_BYTES_PER_SEQUENCE
)


if TRAIN_SEQUENCES <= 0:

    raise RuntimeError(
        "arXiv train.txt contains fewer than "
        f"{RAW_BYTES_PER_SEQUENCE:,} bytes."
    )


if VALID_SEQUENCES <= 0:

    raise RuntimeError(
        "arXiv validation.txt contains fewer than "
        f"{RAW_BYTES_PER_SEQUENCE:,} bytes."
    )


# Real-data byte sanity check.
# This reads only 64 KiB; it does NOT tokenize the corpus.

with RAW_TRAIN.open(
    "rb"
) as f:

    probe = f.read(
        min(
            65_536,
            TRAIN_BYTES,
        )
    )


if not probe:

    raise RuntimeError(
        "arXiv training file is empty."
    )


probe_arr = np.frombuffer(
    probe,
    dtype=np.uint8,
)


if not (
    int(
        probe_arr.min()
    )
    >=
    0
    and
    int(
        probe_arr.max()
    )
    <=
    255
):

    raise RuntimeError(
        "Raw-byte probe failed."
    )


# =============================================================================
# BYTE-STREAM MANIFEST
# =============================================================================

BYTE_STREAM_ROOT = (
    PROJECT_ROOT
    /
    "byte_streams_lifting_v1"
)

BYTE_STREAM_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


MANIFEST_PATH = (
    BYTE_STREAM_ROOT
    /
    "arxiv.json"
)


def atomic_json(
    obj,
    path,
):

    path = Path(
        path
    )

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )

    tmp.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    os.replace(
        tmp,
        path,
    )


BYTE_MANIFEST = {

    "version":
        "transwave_learned_lifting_bytes_v1",

    "corpus":
        CORPUS,

    "tokenization":
        "identity_utf8_bytes",

    "tokenizer_training":
        (
            "none; lifting tokenizer "
            "is learned end-to-end"
        ),

    "train_path":
        str(
            RAW_TRAIN
        ),

    "validation_path":
        str(
            RAW_VALID
        ),

    "train_bytes":
        TRAIN_BYTES,

    "validation_bytes":
        VALID_BYTES,

    "usable_train_bytes":
        USABLE_TRAIN_BYTES,

    "usable_validation_bytes":
        USABLE_VALID_BYTES,

    "train_sequences":
        TRAIN_SEQUENCES,

    "validation_sequences":
        VALID_SEQUENCES,

    "byte_vocab":
        BYTE_VOCAB,

    "input_vocab":
        INPUT_VOCAB,

    "mask_byte_id":
        MASK_BYTE_ID,

    "pad_byte_id":
        PAD_BYTE_ID,

    "lifting_levels":
        LIFTING_LEVELS,

    "bytes_per_model_token":
        BYTES_PER_MODEL_TOKEN,

    "byte_dim":
        BYTE_DIM,

    "model_dim":
        MODEL_DIM,

    "model_context":
        MODEL_CONTEXT,

    "raw_bytes_per_sequence":
        RAW_BYTES_PER_SEQUENCE,

    "document_boundary":
        "newline byte 0x0A",

    "created_token_shards":
        False,
}


atomic_json(
    BYTE_MANIFEST,
    MANIFEST_PATH,
)


# =============================================================================
# CHEAP LEARNED LOCAL OPERATOR
# =============================================================================

class LocalBlockDelta(nn.Module):

    """
    GPU-cheap nonlinear correction.

    Sequence mixing:
        depthwise 3-tap convolution

    Channel mixing:
        grouped 1x1
        -> SiLU
        -> grouped 1x1

    The final projection starts at zero.

    Therefore the lifting transform starts EXACTLY at the
    Haar-like initialization rather than approximately there.
    """

    def __init__(
        self,
        dim=BYTE_DIM,
        groups=12,
        expansion=2,
    ):

        super().__init__()

        if dim % groups != 0:

            raise ValueError(
                "dim must be divisible by groups"
            )

        hidden = (
            dim
            *
            expansion
        )

        if hidden % groups != 0:

            raise ValueError(
                "expanded dimension must be divisible by groups"
            )


        self.depthwise = nn.Conv1d(
            dim,
            dim,
            kernel_size=3,
            padding=1,
            groups=dim,
            bias=True,
        )


        self.mix_in = nn.Conv1d(
            dim,
            hidden,
            kernel_size=1,
            groups=groups,
            bias=True,
        )


        self.mix_out = nn.Conv1d(
            hidden,
            dim,
            kernel_size=1,
            groups=groups,
            bias=True,
        )


        # Zero residual branch initially.
        nn.init.zeros_(
            self.mix_out.weight
        )

        nn.init.zeros_(
            self.mix_out.bias
        )


    def forward(
        self,
        x,
    ):

        # [B,N,C] -> [B,C,N]
        y = x.transpose(
            1,
            2,
        )

        y = self.depthwise(
            y
        )

        y = F.silu(
            y
        )

        y = self.mix_in(
            y
        )

        y = F.silu(
            y
        )

        y = self.mix_out(
            y
        )

        return y.transpose(
            1,
            2,
        )


# =============================================================================
# ONE LEARNED LIFTING STAGE
# =============================================================================

class LearnedLiftingStage(nn.Module):

    """
    Forward:

        D = O - P(E)
        S = E + U(D)

    Initialization:

        P(E) = E
        U(D) = 0.5 D

    therefore:

        D = O - E
        S = (E + O)/2

    Exact nonlinear inverse:

        E = S - U(D)
        O = D + P(E)
    """

    def __init__(
        self,
        dim=BYTE_DIM,
        groups=12,
        expansion=2,
    ):

        super().__init__()


        self.predict_delta = (
            LocalBlockDelta(
                dim=dim,
                groups=groups,
                expansion=expansion,
            )
        )


        self.update_delta = (
            LocalBlockDelta(
                dim=dim,
                groups=groups,
                expansion=expansion,
            )
        )


    def predict(
        self,
        even,
    ):

        return (
            even
            +
            self.predict_delta(
                even
            )
        )


    def update(
        self,
        detail,
    ):

        return (
            0.5
            *
            detail
            +
            self.update_delta(
                detail
            )
        )


    def forward(
        self,
        x,
    ):

        if (
            x.ndim != 3
            or
            x.shape[1] % 2 != 0
        ):

            raise ValueError(
                "Expected [B,N,C] with even N."
            )


        even = x[
            :,
            0::2,
            :,
        ]


        odd = x[
            :,
            1::2,
            :,
        ]


        detail = (
            odd
            -
            self.predict(
                even
            )
        )


        coarse = (
            even
            +
            self.update(
                detail
            )
        )


        return (
            coarse,
            detail,
        )


    def inverse(
        self,
        coarse,
        detail,
    ):

        even = (
            coarse
            -
            self.update(
                detail
            )
        )


        odd = (
            detail
            +
            self.predict(
                even
            )
        )


        out = torch.empty(
            (
                even.shape[0],
                even.shape[1] * 2,
                even.shape[2],
            ),
            device=even.device,
            dtype=even.dtype,
        )


        out[
            :,
            0::2,
            :,
        ] = even


        out[
            :,
            1::2,
            :,
        ] = odd


        return out


# =============================================================================
# EXPLICIT LEARNED CROSS-SCALE MIXER
# =============================================================================

class LearnedCrossScaleMixer(nn.Module):

    """
    Input/output:
        [B,L,8,96]

    96 channels are divided into 12 groups of 8.

    Each group learns an independent 8x8 matrix across the
    four fine + two medium + one coarse-detail + one coarse slots.

    It also learns content-dependent gates.

    Initialization is exactly identity:
        scale matrix = I
        gate gain    = 1
    """

    def __init__(
        self,
        scales=BYTES_PER_MODEL_TOKEN,
        dim=BYTE_DIM,
        groups=12,
    ):

        super().__init__()


        if dim % groups != 0:

            raise ValueError(
                "dim must be divisible by groups"
            )


        self.scales = int(
            scales
        )

        self.dim = int(
            dim
        )

        self.groups = int(
            groups
        )

        self.channels_per_group = (
            self.dim
            //
            self.groups
        )


        # M_g = I + Delta_g
        #
        # 12 × 8 × 8 parameters.

        self.scale_delta = nn.Parameter(
            torch.zeros(
                self.groups,
                self.scales,
                self.scales,
            )
        )


        # Content-sensitive gate for every
        # (channel group, scale slot).

        self.gate_weight = nn.Parameter(
            torch.zeros(
                self.groups,
                self.scales,
                self.channels_per_group,
            )
        )


        self.gate_bias = nn.Parameter(
            torch.zeros(
                self.groups,
                self.scales,
            )
        )


    def forward(
        self,
        z,
    ):

        if (
            z.ndim != 4
            or
            z.shape[2] != self.scales
            or
            z.shape[3] != self.dim
        ):

            raise ValueError(
                f"Expected [B,L,{self.scales},{self.dim}]"
            )


        b, l, s, _ = z.shape


        zg = z.reshape(
            b,
            l,
            s,
            self.groups,
            self.channels_per_group,
        )


        # -------------------------------------------------------------
        # CONTENT-DEPENDENT SCALE GATING
        # -------------------------------------------------------------

        # [B,L,S,G]

        gate_logits = torch.einsum(
            "blsgc,gsc->blsg",
            zg,
            self.gate_weight,
        )


        gate_logits = (
            gate_logits
            /
            math.sqrt(
                self.channels_per_group
            )
            +
            self.gate_bias
            .transpose(
                0,
                1,
            )
        )


        # At initialization logits=0:
        #
        # 2 * sigmoid(0) = 1

        gain = (
            2.0
            *
            torch.sigmoid(
                gate_logits
            )
        )


        zg = (
            zg
            *
            gain
            .to(
                dtype=zg.dtype
            )
            .unsqueeze(
                -1
            )
        )


        # -------------------------------------------------------------
        # EXPLICIT SCALE-TO-SCALE MIXING
        # -------------------------------------------------------------

        eye = torch.eye(
            self.scales,
            device=z.device,
            dtype=z.dtype,
        ).unsqueeze(
            0
        )


        mix = (
            eye
            +
            self.scale_delta.to(
                dtype=z.dtype
            )
        )


        # g = channel group
        # s = input scale
        # t = output scale
        #
        # out[t] = Σ_s M[t,s] in[s]

        mixed = torch.einsum(
            "blsgc,gts->bltgc",
            zg,
            mix,
        )


        return mixed.reshape(
            b,
            l,
            s,
            self.dim,
        )


# =============================================================================
# FULL LEARNED TOKENIZER
# =============================================================================

class LearnedLiftingTokenizer(nn.Module):

    """
    Input
    -----
        byte IDs [B,8L]

    Embedding
    ---------
        [B,8L,96]

    Learned lifting pyramid
    -----------------------
        D0 : [B,4L,96]
        D1 : [B,2L,96]
        D2 : [B, L,96]
        S3 : [B, L,96]

    Packed scale representation
    ---------------------------
        D0 -> four slots
        D1 -> two slots
        D2 -> one slot
        S3 -> one slot

        [B,L,8,96]

    Learned cross-scale mixer
    -------------------------
        [B,L,8,96]

    Flat interface, when required
    -----------------------------
        [B,L,768]
    """

    def __init__(
        self,
        byte_dim=BYTE_DIM,
        model_dim=MODEL_DIM,
        levels=LIFTING_LEVELS,
        lifting_groups=12,
        lifting_expansion=2,
        scale_groups=12,
    ):

        super().__init__()


        ratio = (
            2
            **
            int(
                levels
            )
        )


        if (
            ratio
            *
            byte_dim
            !=
            model_dim
        ):

            raise ValueError(
                "Need 2**levels * byte_dim == model_dim."
            )


        self.byte_dim = int(
            byte_dim
        )

        self.model_dim = int(
            model_dim
        )

        self.levels = int(
            levels
        )

        self.ratio = int(
            ratio
        )


        # -------------------------------------------------------------
        # ATOMIC BYTE EMBEDDING
        # -------------------------------------------------------------

        self.byte_embedding = nn.Embedding(
            INPUT_VOCAB,
            self.byte_dim,
        )


        nn.init.normal_(
            self.byte_embedding.weight,
            mean=0.0,
            std=(
                self.byte_dim
                **
                -0.5
            ),
        )


        # -------------------------------------------------------------
        # THREE LEARNED LIFTING LEVELS
        # -------------------------------------------------------------

        self.stages = nn.ModuleList(
            [

                LearnedLiftingStage(
                    dim=self.byte_dim,
                    groups=lifting_groups,
                    expansion=lifting_expansion,
                )

                for _ in range(
                    self.levels
                )
            ]
        )


        # -------------------------------------------------------------
        # LEARNED EXPLICIT CROSS-SCALE COMMUNICATION
        # -------------------------------------------------------------

        self.cross_scale = (
            LearnedCrossScaleMixer(
                scales=self.ratio,
                dim=self.byte_dim,
                groups=scale_groups,
            )
        )


    # =========================================================================
    # DECOMPOSE
    # =========================================================================

    def decompose_embeddings(
        self,
        x,
    ):

        if (
            x.ndim != 3
            or
            x.shape[1] % self.ratio != 0
            or
            x.shape[2] != self.byte_dim
        ):

            raise ValueError(
                "Expected embeddings [B,8L,96]."
            )


        details = []

        coarse = x


        for stage in self.stages:

            (
                coarse,
                detail,
            ) = stage(
                coarse
            )

            details.append(
                detail
            )


        return (
            coarse,
            details,
        )


    def decompose_ids(
        self,
        byte_ids,
    ):

        if (
            byte_ids.ndim != 2
            or
            byte_ids.shape[1] % self.ratio != 0
        ):

            raise ValueError(
                "Expected byte IDs [B,8L]."
            )


        if byte_ids.dtype != torch.long:

            byte_ids = (
                byte_ids.long()
            )


        embeddings = (
            self.byte_embedding(
                byte_ids
            )
        )


        return (
            self.decompose_embeddings(
                embeddings
            )
        )


    # =========================================================================
    # PACK ALL LEVELS WITHOUT THROWING AWAY INFORMATION
    # =========================================================================

    def pack_scales(
        self,
        coarse,
        details,
    ):

        if len(
            details
        ) != self.levels:

            raise ValueError(
                "Wrong number of detail levels."
            )


        b, l, d = (
            coarse.shape
        )


        slots = []


        for detail in details:

            if (
                detail.shape[0] != b
                or
                detail.shape[2] != d
                or
                detail.shape[1] % l != 0
            ):

                raise ValueError(
                    "Invalid lifting geometry."
                )


            q = (
                detail.shape[1]
                //
                l
            )


            slots.append(
                detail.reshape(
                    b,
                    l,
                    q,
                    d,
                )
            )


        slots.append(
            coarse.unsqueeze(
                2
            )
        )


        z = torch.cat(
            slots,
            dim=2,
        )


        if (
            z.shape[2]
            !=
            self.ratio
        ):

            raise RuntimeError(
                "Packed scale count mismatch."
            )


        return z


    # =========================================================================
    # EXACT LIFTING INVERSE — FOR TESTING/ANALYSIS
    # =========================================================================

    def reconstruct_embeddings(
        self,
        coarse,
        details,
    ):

        x = coarse


        for stage, detail in zip(

            reversed(
                self.stages
            ),

            reversed(
                details
            ),

        ):

            x = stage.inverse(
                x,
                detail,
            )


        return x


    # =========================================================================
    # FORWARD
    # =========================================================================

    def forward(
        self,
        byte_ids,
        return_structured=False,
    ):

        (
            coarse,
            details,
        ) = self.decompose_ids(
            byte_ids
        )


        z = self.pack_scales(
            coarse,
            details,
        )


        # Actual learned exchange between resolutions.
        z = self.cross_scale(
            z
        )


        if return_structured:

            # [B,L,8,96]
            return z


        # Standard [B,L,768] interface.
        return z.flatten(
            2,
            3,
        )


# =============================================================================
# EXACT 15% GPU BYTE-BLOCK MLM MASKING
# =============================================================================

def mask_byte_blocks(
    byte_ids,
    mask_probability=0.15,
    generator=None,
):

    """
    Input:
        byte_ids [B,8L]

    We mask complete 8-byte coarse positions.

    For L=4096:
        K = round(0.15 * 4096)
          = 614 masked coarse positions per sequence

    Targets:
        [B,K,8]

    This prevents leakage where unmasked bytes inside the same
    eight-byte group reveal a masked byte.

    Everything here can execute on the GPU.
    """


    if (
        byte_ids.ndim != 2
        or
        byte_ids.shape[1]
        %
        BYTES_PER_MODEL_TOKEN
        !=
        0
    ):

        raise ValueError(
            "Expected byte IDs [B,8L]."
        )


    b = int(
        byte_ids.shape[0]
    )


    l = (
        int(
            byte_ids.shape[1]
        )
        //
        BYTES_PER_MODEL_TOKEN
    )


    k = max(
        1,
        int(
            round(
                mask_probability
                *
                l
            )
        ),
    )


    blocks = (
        byte_ids
        .long()
        .reshape(
            b,
            l,
            BYTES_PER_MODEL_TOKEN,
        )
    )


    # One random score per coarse position.
    # top-k gives exactly K masked positions per sequence.

    score = torch.rand(
        (
            b,
            l,
        ),
        device=byte_ids.device,
        generator=generator,
    )


    positions = torch.topk(
        score,
        k=k,
        dim=1,
        largest=False,
        sorted=False,
    ).indices


    gather_idx = (
        positions
        .unsqueeze(
            -1
        )
        .expand(
            -1,
            -1,
            BYTES_PER_MODEL_TOKEN,
        )
    )


    target_bytes = torch.gather(
        blocks,
        dim=1,
        index=gather_idx,
    )


    masked = blocks.clone()


    masked.scatter_(
        dim=1,
        index=gather_idx,
        src=torch.full_like(
            target_bytes,
            MASK_BYTE_ID,
        ),
    )


    return (
        masked.reshape(
            b,
            l
            *
            BYTES_PER_MODEL_TOKEN,
        ),

        positions,

        target_bytes,
    )


# =============================================================================
# ZERO-COPY BYTE MEMMAP
# =============================================================================

def open_byte_memmap(
    path,
):

    """
    Maps a raw text file directly as uint8 tokens.

    Cell 4 should:
      1. copy train.txt/validation.txt once from Drive to /content;
      2. memory-map the LOCAL copy;
      3. transfer batches non-blockingly to the GPU.

    No tokenized intermediate is needed.
    """

    path = Path(
        path
    )


    if (
        not path.is_file()
        or
        path.stat().st_size <= 0
    ):

        raise FileNotFoundError(
            path
        )


    return np.memmap(
        path,
        dtype=np.uint8,
        mode="r",
    )


# =============================================================================
# PARAMETER COUNT
# =============================================================================

probe_model = (
    LearnedLiftingTokenizer()
)


TOKENIZER_PARAMETERS = sum(
    p.numel()
    for p in probe_model.parameters()
)


del probe_model


# =============================================================================
# REPORT
# =============================================================================

print(
    "\n"
    +
    "=" * 108
)


print(
    "TRANSWAVE LEARNED LIFTING TOKENIZER READY"
)


print(
    f"corpus                    : {CORPUS}"
)


print(
    "atomic representation     : raw UTF-8 bytes"
)


print(
    "BPE / SentencePiece       : NONE"
)


print(
    "offline tokenizer train   : NONE"
)


print(
    f"lifting levels            : {LIFTING_LEVELS}"
)


print(
    f"compression               : "
    f"{BYTES_PER_MODEL_TOKEN} bytes "
    f"-> one TransWave position"
)


print(
    f"byte embedding width      : {BYTE_DIM}"
)


print(
    f"packed model width        : {MODEL_DIM}"
)


print(
    f"bytes / L=4096 sequence   : "
    f"{RAW_BYTES_PER_SEQUENCE:,}"
)


print(
    f"learned tokenizer params  : "
    f"{TOKENIZER_PARAMETERS:,}"
)


print(
    f"train raw bytes           : "
    f"{TRAIN_BYTES:,}"
)


print(
    f"train usable bytes        : "
    f"{USABLE_TRAIN_BYTES:,}"
)


print(
    f"train L=4096 sequences    : "
    f"{TRAIN_SEQUENCES:,}"
)


print(
    f"validation raw bytes      : "
    f"{VALID_BYTES:,}"
)


print(
    f"validation sequences      : "
    f"{VALID_SEQUENCES:,}"
)


print(
    "new token shards          : NONE"
)


print(
    "obsolete long-token shards: REMOVED"
)


print(
    f"manifest                  : {MANIFEST_PATH}"
)


print(
    "=" * 108
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Removing obsolete shard tree: /content/drive/MyDrive/TransWave_Meyer_Context/streams_more_tokens_transwave_v1
Removing obsolete shard tree: /content/drive/MyDrive/TransWave_Meyer_Context/streams_more_tokens_v1
Removing obsolete local scratch: /content/TransWave_Meyer_Context_more_tokens_transwave_v1

TRANSWAVE LEARNED LIFTING TOKENIZER READY
corpus                    : arxiv
atomic representation     : raw UTF-8 bytes
BPE / SentencePiece       : NONE
offline tokenizer train   : NONE
lifting levels            : 3
compression               : 8 bytes -> one TransWave position
byte embedding width      : 96
packed model width        : 768
bytes / L=4096 sequence   : 32,768
learned tokenizer params  : 48,864
train raw bytes           : 545,262,781
train usable bytes        : 545,259,520
train L=4096 sequences    : 16,640
validation raw bytes      : 16,790,635
vali

In [13]:
# =============================================================================
# CELL 4 V3 — COMPLETE REPLACEMENT
#
# arXiv ONLY
#
# IMPORTANT FIX:
#   Every class/function in this cell has a NEW V3 name.
#   It CANNOT call the stale ComplexCrossScaleMeyer from the earlier Cell 4.
#
# Architecture:
#   learned lifting byte tokenizer from Cell 3
#   + structured [B,L,8,96] multiscale representation
#   + learned COMPLEX within-scale Meyer response
#   + learned COMPLEX low-rank cross-scale mixing
#   + learned scale residual gates
#   + FFN
#   + learned Haar-initialized multiscale synthesis
#   + tied 256-byte prediction head
#
# Dtypes:
#   tokenizer / residual / FFN = BF16 autocast
#   FFT / Meyer / complex math = FP32 + complex64 ONLY
#
# Includes:
#   - dtype preflight BEFORE expensive memory probe
#   - full B=155 training-memory probe including Adam moments
#   - automatic fallback if B=155 genuinely OOMs
#   - restart-safe training
# =============================================================================

from __future__ import annotations

import os
import gc
import json
import math
import time
import random
import shutil
import hashlib

from pathlib import Path

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.checkpoint import checkpoint as torch_checkpoint


# =============================================================================
# REQUIRE CELL 3
# =============================================================================

_REQUIRED_V3 = (
    "LearnedLiftingTokenizer",
    "mask_byte_blocks",
    "BYTE_STREAM_ROOT",
    "BYTE_DIM",
    "MODEL_DIM",
    "BYTES_PER_MODEL_TOKEN",
    "MODEL_CONTEXT",
    "RAW_BYTES_PER_SEQUENCE",
    "BYTE_VOCAB",
)

_missing_v3 = [
    x
    for x in _REQUIRED_V3
    if x not in globals()
]

if _missing_v3:
    raise RuntimeError(
        "Run learned-lifting Cell 3 first. Missing: "
        + ", ".join(_missing_v3)
    )


# =============================================================================
# HARDWARE
# =============================================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required.")


TWV3_DEVICE = torch.device("cuda")

TWV3_GPU_NAME = torch.cuda.get_device_name(0)

TWV3_VRAM_GB = (
    torch.cuda.get_device_properties(0).total_memory
    /
    1024**3
)

TWV3_AMP_DTYPE = torch.bfloat16


torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

torch.set_float32_matmul_precision("high")


# =============================================================================
# SEED
# =============================================================================

TWV3_SEED = 1337


def twv3_seed_all(seed=TWV3_SEED):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)


twv3_seed_all()


# =============================================================================
# ARCHITECTURE
# =============================================================================

TWV3_L = int(MODEL_CONTEXT)

TWV3_SCALES = int(BYTES_PER_MODEL_TOKEN)

TWV3_D_SCALE = int(BYTE_DIM)

TWV3_D_MODEL = int(MODEL_DIM)


assert TWV3_L == 4096
assert TWV3_SCALES == 8
assert TWV3_D_SCALE == 96
assert TWV3_D_MODEL == 768


TWV3_LAYERS = 12

TWV3_GROUPS = 12

TWV3_CHANNELS_PER_GROUP = (
    TWV3_D_SCALE
    //
    TWV3_GROUPS
)

assert TWV3_CHANNELS_PER_GROUP == 8


TWV3_FFN_DIM = 4480

TWV3_CROSS_RANK = 2

TWV3_MEYER_TRANSITION = 0.25


TWV3_LR = 1.0e-4

TWV3_BETAS = (
    0.9,
    0.999,
)

TWV3_EPS = 1.0e-8

TWV3_WEIGHT_DECAY = 0.0


TWV3_BYTE_CE_CHUNK = 131_072


# Prior hardware sweep showed B=155 was the throughput sweet spot.
# We retain that first choice and only fall back on REAL OOM.

TWV3_BATCH_CANDIDATES = (
    155,
    148,
    144,
    138,
    128,
)


TWV3_ARCH = (
    "transwave_lifting_"
    "complexmeyer_rank2_"
    "bytehead_v3_uniqueclasses"
)


# =============================================================================
# PATHS
# =============================================================================

TWV3_PROJECT = Path(
    "/content/drive/MyDrive/"
    "TransWave_Meyer_Context"
)


TWV3_BYTE_MANIFEST = (
    Path(BYTE_STREAM_ROOT)
    /
    "arxiv.json"
)


if not TWV3_BYTE_MANIFEST.is_file():
    raise FileNotFoundError(
        TWV3_BYTE_MANIFEST
    )


TWV3_META = json.loads(
    TWV3_BYTE_MANIFEST.read_text(
        encoding="utf-8"
    )
)


TWV3_RAW_TRAIN = Path(
    TWV3_META[
        "train_path"
    ]
)


TWV3_RAW_VALID = Path(
    TWV3_META[
        "validation_path"
    ]
)


for _twv3_p in (
    TWV3_RAW_TRAIN,
    TWV3_RAW_VALID,
):

    if not _twv3_p.is_file():
        raise FileNotFoundError(
            _twv3_p
        )


TWV3_RUN_ROOT = (
    TWV3_PROJECT
    /
    "checkpoints"
    /
    TWV3_ARCH
    /
    "arxiv"
)


TWV3_RESULT_ROOT = (
    TWV3_PROJECT
    /
    "results"
    /
    TWV3_ARCH
)


TWV3_RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


TWV3_RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


TWV3_LOCAL = Path(
    "/content/"
    "TransWave_Lifting_Arxiv"
)


TWV3_LOCAL.mkdir(
    parents=True,
    exist_ok=True,
)


TWV3_LATEST = (
    TWV3_RUN_ROOT
    /
    "training_latest.pt"
)


TWV3_FINAL_MODEL = (
    TWV3_RUN_ROOT
    /
    "trained_model_final.pt"
)


TWV3_RUN_CONFIG = (
    TWV3_RUN_ROOT
    /
    "run_config.json"
)


TWV3_RESULT_JSON = (
    TWV3_RESULT_ROOT
    /
    "arxiv.json"
)


TWV3_COMPLETE = (
    TWV3_RUN_ROOT
    /
    "TRAINING_COMPLETE.json"
)


# =============================================================================
# ATOMIC WRITES
# =============================================================================

def twv3_atomic_json(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )

    tmp.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    os.replace(
        tmp,
        path,
    )


def twv3_atomic_torch_save(
    obj,
    path,
):

    path = Path(path)

    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )

    torch.save(
        obj,
        tmp,
    )

    os.replace(
        tmp,
        path,
    )


# =============================================================================
# LOCAL SSD — SHOULD ALREADY EXIST
# =============================================================================

def twv3_stage(
    source,
):

    destination = (
        TWV3_LOCAL
        /
        source.name
    )


    if (
        destination.is_file()
        and
        destination.stat().st_size
        ==
        source.stat().st_size
    ):

        print(
            "local SSD already staged:",
            destination,
        )

        return destination


    tmp = destination.with_suffix(
        ".tmp"
    )


    tmp.unlink(
        missing_ok=True
    )


    print(
        "Drive -> local SSD:",
        source,
    )


    shutil.copyfile(
        source,
        tmp,
    )


    if (
        tmp.stat().st_size
        !=
        source.stat().st_size
    ):

        tmp.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "Incomplete local SSD copy."
        )


    os.replace(
        tmp,
        destination,
    )


    return destination


TWV3_TRAIN_LOCAL = twv3_stage(
    TWV3_RAW_TRAIN
)


TWV3_VALID_LOCAL = twv3_stage(
    TWV3_RAW_VALID
)


# =============================================================================
# RAW BYTE VIEWS
# =============================================================================

TWV3_TRAIN_MEM = np.memmap(
    TWV3_TRAIN_LOCAL,
    dtype=np.uint8,
    mode="r",
)


TWV3_VALID_MEM = np.memmap(
    TWV3_VALID_LOCAL,
    dtype=np.uint8,
    mode="r",
)


TWV3_TRAIN_USABLE = (
    len(TWV3_TRAIN_MEM)
    //
    RAW_BYTES_PER_SEQUENCE
    *
    RAW_BYTES_PER_SEQUENCE
)


TWV3_VALID_USABLE = (
    len(TWV3_VALID_MEM)
    //
    RAW_BYTES_PER_SEQUENCE
    *
    RAW_BYTES_PER_SEQUENCE
)


TWV3_TRAIN_BLOCKS = (
    TWV3_TRAIN_MEM[
        :TWV3_TRAIN_USABLE
    ]
    .reshape(
        -1,
        RAW_BYTES_PER_SEQUENCE,
    )
)


TWV3_VALID_BLOCKS = (
    TWV3_VALID_MEM[
        :TWV3_VALID_USABLE
    ]
    .reshape(
        -1,
        RAW_BYTES_PER_SEQUENCE,
    )
)


TWV3_N_TRAIN = int(
    len(TWV3_TRAIN_BLOCKS)
)


TWV3_N_VALID = int(
    len(TWV3_VALID_BLOCKS)
)


if TWV3_N_TRAIN < 128:
    raise RuntimeError(
        "Insufficient arXiv training data."
    )


if TWV3_N_VALID < 1:
    raise RuntimeError(
        "Insufficient validation data."
    )


# =============================================================================
# FIXED MEYER PARTITION
# =============================================================================

def twv3_meyer_nu(x):

    x = x.clamp(
        0.0,
        1.0,
    )

    return (
        x**4
        *
        (
            35.0
            -
            84.0*x
            +
            70.0*x**2
            -
            20.0*x**3
        )
    )


def twv3_lowpass(
    f,
    cutoff,
):

    lo = max(
        0.0,
        cutoff
        *
        (
            1.0
            -
            TWV3_MEYER_TRANSITION
        ),
    )


    hi = min(
        1.0,
        cutoff
        *
        (
            1.0
            +
            TWV3_MEYER_TRANSITION
        ),
    )


    out = torch.ones_like(f)


    out = torch.where(
        f >= hi,
        torch.zeros_like(out),
        out,
    )


    if hi > lo:

        u = (
            (
                f
                -
                lo
            )
            /
            (
                hi
                -
                lo
            )
        ).clamp(
            0.0,
            1.0,
        )


        middle = torch.cos(
            0.5
            *
            math.pi
            *
            twv3_meyer_nu(u)
        ).square()


        mask = (
            (f > lo)
            &
            (f < hi)
        )


        out = torch.where(
            mask,
            middle,
            out,
        )


    return out


def twv3_build_meyer(
    length,
):

    freq = torch.fft.rfftfreq(
        length,
        d=1.0,
    ).float()


    f = (
        freq
        /
        0.5
    ).clamp(
        0.0,
        1.0,
    )


    num_scales = (
        int(
            math.log2(
                length
            )
        )
        -
        1
    )


    cumulative = torch.ones_like(f)


    windows = []


    for j in range(
        1,
        num_scales
        +
        1,
    ):

        low = twv3_lowpass(
            f,
            2.0 ** (-j),
        )


        windows.append(
            (
                cumulative
                -
                low
            ).clamp_min(
                0.0
            )
        )


        cumulative = low


    windows.append(
        cumulative.clamp_min(
            0.0
        )
    )


    q = torch.stack(
        windows,
        dim=0,
    )


    q = (
        q
        /
        q.sum(
            dim=0
        )
        .clamp_min(
            1.0e-12
        )
        .unsqueeze(
            0
        )
    )


    error = float(
        (
            q.sum(
                dim=0
            )
            -
            1.0
        )
        .abs()
        .max()
    )


    return (
        q.contiguous(),
        error,
    )


(
    TWV3_MEYER_Q,
    TWV3_MEYER_ERROR,
) = twv3_build_meyer(
    TWV3_L
)


TWV3_MEYER_BANDS = int(
    TWV3_MEYER_Q.shape[0]
)


assert TWV3_MEYER_BANDS == 12


# =============================================================================
# UNIQUE V3 COMPLEX MEYER CLASS
# =============================================================================

class TWV3ComplexMeyer(
    nn.Module
):

    """
    THIS CLASS NAME DID NOT EXIST IN THE FAILED CELLS.

    Input/output:
        [B,4096,8,96]

    real FFT input = float32
    FFT output     = complex64
    real/imag response tensors = float32

    BF16 cannot reach torch.complex().
    """


    def __init__(
        self,
    ):

        super().__init__()


        self.length = TWV3_L

        self.scales = TWV3_SCALES

        self.dim = TWV3_D_SCALE

        self.groups = TWV3_GROUPS

        self.cpg = TWV3_CHANNELS_PER_GROUP

        self.rank = TWV3_CROSS_RANK


        self.register_buffer(
            "meyer_q_v3",
            TWV3_MEYER_Q.clone(),
            persistent=True,
        )


        j = TWV3_MEYER_BANDS


        # -------------------------------------------------------------
        # WITHIN-SCALE COMPLEX RESPONSE
        # -------------------------------------------------------------

        self.diag_re_v3 = nn.Parameter(
            torch.zeros(
                self.groups,
                self.scales,
                j,
                dtype=torch.float32,
            )
        )


        self.diag_im_v3 = nn.Parameter(
            torch.zeros(
                self.groups,
                self.scales,
                j,
                dtype=torch.float32,
            )
        )


        # -------------------------------------------------------------
        # LOW-RANK CROSS-SCALE COMPLEX RESPONSE
        # -------------------------------------------------------------

        shape = (
            self.groups,
            self.scales,
            self.rank,
            j,
        )


        self.u_re_v3 = nn.Parameter(
            torch.zeros(
                shape,
                dtype=torch.float32,
            )
        )


        self.u_im_v3 = nn.Parameter(
            torch.zeros(
                shape,
                dtype=torch.float32,
            )
        )


        self.v_re_v3 = nn.Parameter(
            torch.empty(
                shape,
                dtype=torch.float32,
            )
        )


        self.v_im_v3 = nn.Parameter(
            torch.empty(
                shape,
                dtype=torch.float32,
            )
        )


        nn.init.normal_(
            self.v_re_v3,
            mean=0.0,
            std=0.02,
        )


        nn.init.normal_(
            self.v_im_v3,
            mean=0.0,
            std=0.02,
        )


    def expand_v3(
        self,
        p,
    ):

        # Explicit FP32 output regardless of surrounding autocast.

        with torch.autocast(
            device_type="cuda",
            enabled=False,
        ):

            out = torch.einsum(
                "...j,jf->...f",
                p.float(),
                self.meyer_q_v3.float(),
            )


        if out.dtype != torch.float32:

            raise RuntimeError(
                "V3 Meyer expansion is not FP32: "
                f"{out.dtype}"
            )


        return out


    def forward(
        self,
        z,
    ):

        original_dtype = z.dtype


        b, l, s, d = z.shape


        if (
            l != self.length
            or
            s != self.scales
            or
            d != self.dim
        ):

            raise RuntimeError(
                "TWV3 Meyer shape mismatch: "
                f"{tuple(z.shape)}"
            )


        # =====================================================================
        # THE ENTIRE COMPLEX SECTION IS OUTSIDE AUTOCAST.
        # =====================================================================

        with torch.autocast(
            device_type="cuda",
            enabled=False,
        ):

            x = (
                z.float()
                .reshape(
                    b,
                    l,
                    s,
                    self.groups,
                    self.cpg,
                )
                .permute(
                    0,
                    3,
                    4,
                    2,
                    1,
                )
                .contiguous()
            )


            if x.dtype != torch.float32:

                raise RuntimeError(
                    f"TWV3 FFT input is {x.dtype}"
                )


            # -------------------------------------------------------------
            # RFFT: float32 -> complex64
            # -------------------------------------------------------------

            X = torch.fft.rfft(
                x,
                n=self.length,
                dim=-1,
            )


            if X.dtype != torch.complex64:

                raise RuntimeError(
                    f"TWV3 FFT result is {X.dtype}, "
                    "expected complex64"
                )


            # -------------------------------------------------------------
            # DIAGONAL RESPONSE
            # -------------------------------------------------------------

            dr = self.expand_v3(
                self.diag_re_v3
            )


            di = self.expand_v3(
                self.diag_im_v3
            )


            # HARD ASSERT BEFORE torch.complex

            if (
                dr.dtype != torch.float32
                or
                di.dtype != torch.float32
            ):

                raise RuntimeError(
                    "TWV3 diag components "
                    "are not FP32."
                )


            diag = torch.complex(
                1.0 + dr,
                di,
            )


            # -------------------------------------------------------------
            # CROSS-SCALE FACTORS
            # -------------------------------------------------------------

            ur = self.expand_v3(
                self.u_re_v3
            )


            ui = self.expand_v3(
                self.u_im_v3
            )


            vr = self.expand_v3(
                self.v_re_v3
            )


            vi = self.expand_v3(
                self.v_im_v3
            )


            # HARD ASSERT BEFORE torch.complex

            for name, value in (
                ("ur", ur),
                ("ui", ui),
                ("vr", vr),
                ("vi", vi),
            ):

                if value.dtype != torch.float32:

                    raise RuntimeError(
                        f"TWV3 {name} is "
                        f"{value.dtype}, expected FP32."
                    )


            U = torch.complex(
                ur,
                ui,
            )


            V = torch.complex(
                vr,
                vi,
            )


            if (
                U.dtype != torch.complex64
                or
                V.dtype != torch.complex64
            ):

                raise RuntimeError(
                    "TWV3 complex factors "
                    "are not complex64."
                )


            # -------------------------------------------------------------
            # ALL-SCALE LOW-RANK EXCHANGE
            #
            # X   B,G,C,S,F
            # V   G,S,R,F
            # tmp B,G,C,R,F
            # -------------------------------------------------------------

            tmp = torch.einsum(
                "bgcsf,gsrf->bgcrf",
                X,
                V,
            )


            cross = torch.einsum(
                "bgcrf,gsrf->bgcsf",
                tmp,
                U,
            )


            Y = (
                X
                *
                diag[
                    None,
                    :,
                    None,
                    :,
                    :,
                ]
                +
                cross
            )


            y = torch.fft.irfft(
                Y,
                n=self.length,
                dim=-1,
            )


            y = (
                y.permute(
                    0,
                    4,
                    3,
                    1,
                    2,
                )
                .contiguous()
                .reshape(
                    b,
                    l,
                    s,
                    d,
                )
            )


        return y.to(
            dtype=original_dtype
        )


# =============================================================================
# UNIQUE V3 TRANSWAVE LAYER
# =============================================================================

class TWV3Layer(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()


        self.norm_meyer_v3 = nn.LayerNorm(
            TWV3_D_MODEL,
            eps=1.0e-5,
        )


        self.meyer_v3 = (
            TWV3ComplexMeyer()
        )


        # ~10% Meyer residual initially.

        self.scale_gate_v3 = nn.Parameter(
            torch.full(
                (
                    TWV3_SCALES,
                ),
                -2.1972246,
            )
        )


        self.norm_ffn_v3 = nn.LayerNorm(
            TWV3_D_MODEL,
            eps=1.0e-5,
        )


        self.ffn_v3 = nn.Sequential(

            nn.Linear(
                TWV3_D_MODEL,
                TWV3_FFN_DIM,
            ),

            nn.GELU(
                approximate="tanh"
            ),

            nn.Linear(
                TWV3_FFN_DIM,
                TWV3_D_MODEL,
            ),
        )


    def forward(
        self,
        z,
    ):

        b, l, s, d = z.shape


        flat = z.flatten(
            2,
            3,
        )


        h = (
            self.norm_meyer_v3(
                flat
            )
            .reshape(
                b,
                l,
                s,
                d,
            )
        )


        mixed = self.meyer_v3(
            h
        )


        gate = torch.sigmoid(
            self.scale_gate_v3
        ).view(
            1,
            1,
            s,
            1,
        )


        z = (
            z
            +
            gate
            *
            mixed
        )


        flat = z.flatten(
            2,
            3,
        )


        flat = (
            flat
            +
            self.ffn_v3(
                self.norm_ffn_v3(
                    flat
                )
            )
        )


        return flat.reshape(
            b,
            l,
            s,
            d,
        )


# =============================================================================
# HAAR-INVERSE INITIAL SYNTHESIS
# =============================================================================

def twv3_haar_matrix():

    return torch.tensor(

        [
            [-0.5,  0.0,  0.0,  0.0, -0.5,  0.0, -0.5, 1.0],
            [ 0.5,  0.0,  0.0,  0.0, -0.5,  0.0, -0.5, 1.0],

            [ 0.0, -0.5,  0.0,  0.0,  0.5,  0.0, -0.5, 1.0],
            [ 0.0,  0.5,  0.0,  0.0,  0.5,  0.0, -0.5, 1.0],

            [ 0.0,  0.0, -0.5,  0.0,  0.0, -0.5,  0.5, 1.0],
            [ 0.0,  0.0,  0.5,  0.0,  0.0, -0.5,  0.5, 1.0],

            [ 0.0,  0.0,  0.0, -0.5,  0.0,  0.5,  0.5, 1.0],
            [ 0.0,  0.0,  0.0,  0.5,  0.0,  0.5,  0.5, 1.0],
        ],

        dtype=torch.float32,
    )


class TWV3Synthesis(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()


        self.register_buffer(
            "haar_v3",
            twv3_haar_matrix(),
            persistent=True,
        )


        self.delta_v3 = nn.Parameter(
            torch.zeros(
                TWV3_GROUPS,
                TWV3_SCALES,
                TWV3_SCALES,
            )
        )


    def forward(
        self,
        z,
    ):

        leading = z.shape[
            :-2
        ]


        zg = z.reshape(
            *leading,
            TWV3_SCALES,
            TWV3_GROUPS,
            TWV3_CHANNELS_PER_GROUP,
        )


        matrix = (
            self.haar_v3[
                None,
                :,
                :,
            ]
            +
            self.delta_v3
        )


        y = torch.einsum(
            "...igc,goi->...ogc",
            zg,
            matrix.to(
                dtype=zg.dtype
            ),
        )


        return y.reshape(
            *leading,
            TWV3_SCALES,
            TWV3_D_SCALE,
        )


# =============================================================================
# UNIQUE V3 COMPLETE MODEL
# =============================================================================

class TWV3Model(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()


        # Cell-3 class is intentional.
        # It contains no complex arithmetic.

        self.tokenizer_v3 = (
            LearnedLiftingTokenizer()
        )


        self.layers_v3 = nn.ModuleList(
            [
                TWV3Layer()
                for _ in range(
                    TWV3_LAYERS
                )
            ]
        )


        self.final_norm_v3 = nn.LayerNorm(
            TWV3_D_MODEL,
            eps=1.0e-5,
        )


        self.synthesis_v3 = (
            TWV3Synthesis()
        )


    def encode_v3(
        self,
        ids,
        use_checkpoint=True,
    ):

        if (
            use_checkpoint
            and
            self.training
        ):

            def tokenizer_fn(
                x,
            ):

                return self.tokenizer_v3(
                    x,
                    return_structured=True,
                )


            z = torch_checkpoint(
                tokenizer_fn,
                ids,
                use_reentrant=False,
            )


        else:

            z = self.tokenizer_v3(
                ids,
                return_structured=True,
            )


        if (
            use_checkpoint
            and
            self.training
        ):

            for layer in self.layers_v3:

                z = torch_checkpoint(
                    layer,
                    z,
                    use_reentrant=False,
                )


        else:

            for layer in self.layers_v3:

                z = layer(
                    z
                )


        b, l, s, d = z.shape


        z = (
            self.final_norm_v3(
                z.flatten(
                    2,
                    3,
                )
            )
            .reshape(
                b,
                l,
                s,
                d,
            )
        )


        return z


    def masked_hidden_v3(
        self,
        ids,
        positions,
        use_checkpoint=True,
    ):

        z = self.encode_v3(
            ids,
            use_checkpoint=use_checkpoint,
        )


        b, _, s, d = z.shape

        k = positions.shape[1]


        gather = (
            positions[
                :,
                :,
                None,
                None,
            ]
            .expand(
                b,
                k,
                s,
                d,
            )
        )


        selected = torch.gather(
            z,
            dim=1,
            index=gather,
        )


        hidden = self.synthesis_v3(
            selected
        )


        return hidden.reshape(
            -1,
            TWV3_D_SCALE,
        )


    def logits_v3(
        self,
        h,
    ):

        # Only 256 real bytes.
        # MASK/PAD are never output classes.

        weight = (
            self.tokenizer_v3
            .byte_embedding
            .weight[
                :BYTE_VOCAB
            ]
        )


        return F.linear(
            h,
            weight,
        )


# =============================================================================
# UNIQUE V3 BYTE LOSS
# =============================================================================

def twv3_byte_loss(
    model,
    hidden,
    targets,
    training,
):

    targets = (
        targets.reshape(
            -1
        )
        .long()
    )


    if (
        hidden.shape[0]
        !=
        targets.numel()
    ):

        raise RuntimeError(
            "V3 hidden/target mismatch."
        )


    n = int(
        targets.numel()
    )


    total = None


    for start in range(
        0,
        n,
        TWV3_BYTE_CE_CHUNK,
    ):

        end = min(
            start
            +
            TWV3_BYTE_CE_CHUNK,
            n,
        )


        h = hidden[
            start:end
        ]


        t = targets[
            start:end
        ]


        if training:

            def ce_fn(
                hh,
                tt=t,
            ):

                with torch.autocast(
                    "cuda",
                    dtype=TWV3_AMP_DTYPE,
                ):

                    logits = model.logits_v3(
                        hh
                    )


                    return F.cross_entropy(
                        logits,
                        tt,
                        reduction="sum",
                    )


            piece = torch_checkpoint(
                ce_fn,
                h,
                use_reentrant=False,
            )


        else:

            with torch.autocast(
                "cuda",
                dtype=TWV3_AMP_DTYPE,
            ):

                logits = model.logits_v3(
                    h
                )


                piece = F.cross_entropy(
                    logits,
                    t,
                    reduction="sum",
                )


            del logits


        total = (
            piece
            if total is None
            else
            total
            +
            piece
        )


    return (
        total
        /
        float(n),
        n,
    )


# =============================================================================
# DATA
# =============================================================================

def twv3_numpy_batch(
    blocks,
    indices,
):

    arr = np.asarray(
        blocks[
            indices
        ],
        dtype=np.uint8,
    )


    arr = np.ascontiguousarray(
        arr
    )


    return (
        torch
        .from_numpy(
            arr
        )
        .pin_memory()
    )


# =============================================================================
# CLEANUP
# =============================================================================

def twv3_cleanup():

    gc.collect()


    try:

        torch.cuda.synchronize()

    except Exception:

        pass


    torch.cuda.empty_cache()


    try:

        torch.cuda.ipc_collect()

    except Exception:

        pass


# =============================================================================
# OPTIMIZER
# =============================================================================

def twv3_optimizer(
    model,
):

    kwargs = dict(

        lr=TWV3_LR,

        betas=TWV3_BETAS,

        eps=TWV3_EPS,

        weight_decay=TWV3_WEIGHT_DECAY,
    )


    try:

        return (
            torch.optim.Adam(
                model.parameters(),
                fused=True,
                **kwargs,
            ),
            True,
        )


    except Exception:

        return (
            torch.optim.Adam(
                model.parameters(),
                **kwargs,
            ),
            False,
        )


# =============================================================================
# VERY SMALL DTYPE PREFLIGHT
#
# THIS RUNS BEFORE B=155.
#
# If a stale class were somehow called, the filename printed below
# would expose it immediately.
# =============================================================================

print(
    "\n"
    +
    "=" * 108
)

print(
    "V3 COMPLEX-PATH PREFLIGHT"
)

print(
    "V3 forward source:",
    TWV3ComplexMeyer
    .forward
    .__code__
    .co_filename,
)

print(
    "=" * 108
)


twv3_cleanup()


_twv3_test_mixer = (
    TWV3ComplexMeyer()
    .to(
        TWV3_DEVICE
    )
)


_twv3_test_z = torch.randn(

    (
        1,
        TWV3_L,
        TWV3_SCALES,
        TWV3_D_SCALE,
    ),

    device=TWV3_DEVICE,

    dtype=torch.bfloat16,

    requires_grad=True,
)


with torch.autocast(
    "cuda",
    dtype=TWV3_AMP_DTYPE,
):

    _twv3_test_y = (
        _twv3_test_mixer(
            _twv3_test_z
        )
    )


if (
    _twv3_test_y.dtype
    !=
    torch.bfloat16
):

    raise RuntimeError(
        "V3 preflight output dtype incorrect."
    )


_twv3_test_loss = (
    _twv3_test_y
    .float()
    .square()
    .mean()
)


_twv3_test_loss.backward()


if (
    _twv3_test_mixer
    .diag_re_v3
    .grad
    is None
):

    raise RuntimeError(
        "No V3 within-scale gradient."
    )


if (
    _twv3_test_mixer
    .u_re_v3
    .grad
    is None
):

    raise RuntimeError(
        "No V3 cross-scale gradient."
    )


print(
    "V3 dtype preflight: PASS"
)

print(
    "input       : bfloat16"
)

print(
    "FFT real    : float32"
)

print(
    "FFT complex : complex64"
)

print(
    "output      : bfloat16"
)

print(
    "gradients   : within-scale + cross-scale PASS"
)


del (
    _twv3_test_mixer,
    _twv3_test_z,
    _twv3_test_y,
    _twv3_test_loss,
)


twv3_cleanup()


# =============================================================================
# FULL TRAINING MEMORY PROBE
# =============================================================================

def twv3_probe_batch(
    batch_size,
):

    twv3_cleanup()

    twv3_seed_all()


    model = (
        TWV3Model()
        .to(
            TWV3_DEVICE
        )
    )


    optimizer, _ = twv3_optimizer(
        model
    )


    model.train()


    idx = (
        np.arange(
            batch_size,
            dtype=np.int64,
        )
        %
        TWV3_N_TRAIN
    )


    cpu = twv3_numpy_batch(
        TWV3_TRAIN_BLOCKS,
        idx,
    )


    ids = cpu.to(
        TWV3_DEVICE,
        non_blocking=True,
    )


    gen = torch.Generator(
        device=TWV3_DEVICE
    )


    gen.manual_seed(
        500_000
        +
        batch_size
    )


    (
        masked,
        positions,
        targets,
    ) = mask_byte_blocks(

        ids,

        mask_probability=0.15,

        generator=gen,
    )


    optimizer.zero_grad(
        set_to_none=True
    )


    with torch.autocast(
        "cuda",
        dtype=TWV3_AMP_DTYPE,
    ):

        hidden = model.masked_hidden_v3(

            masked,

            positions,

            use_checkpoint=True,
        )


        loss, _ = twv3_byte_loss(

            model,

            hidden,

            targets,

            training=True,
        )


    loss.backward()


    # Adam moments included in peak.

    optimizer.step()


    torch.cuda.synchronize()


    alloc = (
        torch.cuda
        .max_memory_allocated()
        /
        1024**3
    )


    reserved = (
        torch.cuda
        .max_memory_reserved()
        /
        1024**3
    )


    value = float(
        loss.detach()
    )


    del (
        model,
        optimizer,
        idx,
        cpu,
        ids,
        masked,
        positions,
        targets,
        hidden,
        loss,
    )


    twv3_cleanup()


    return (
        alloc,
        reserved,
        value,
    )


# =============================================================================
# BATCH SELECTION
# =============================================================================

if TWV3_RUN_CONFIG.is_file():

    _twv3_cfg = json.loads(
        TWV3_RUN_CONFIG.read_text(
            encoding="utf-8"
        )
    )


    if (
        _twv3_cfg.get(
            "architecture_version"
        )
        !=
        TWV3_ARCH
    ):

        raise RuntimeError(
            "V3 run config mismatch."
        )


    TWV3_BATCH = int(
        _twv3_cfg[
            "batch_size"
        ]
    )


    print(
        f"Using validated B="
        f"{TWV3_BATCH}"
    )


else:

    TWV3_BATCH = None


    print(
        "\n"
        +
        "=" * 108
    )

    print(
        "V3 G4 FULL-TRAINING MEMORY PROBE"
    )

    print(
        "forward + backward + Adam moments"
    )

    print(
        "complex path verified FP32/complex64"
    )

    print(
        "=" * 108
    )


    for candidate in (
        TWV3_BATCH_CANDIDATES
    ):

        try:

            twv3_cleanup()

            torch.cuda.reset_peak_memory_stats()


            (
                alloc,
                reserved,
                p_loss,
            ) = twv3_probe_batch(
                candidate
            )


            TWV3_BATCH = int(
                candidate
            )


            print(
                f"B={candidate}: FIT | "
                f"alloc={alloc:.2f} GiB | "
                f"reserved={reserved:.2f} GiB | "
                f"loss={p_loss:.4f}"
            )


            break


        except (
            torch.cuda.OutOfMemoryError
        ):

            print(
                f"B={candidate}: OOM -> "
                "trying lower batch"
            )

            twv3_cleanup()


        except RuntimeError as exc:

            if (
                "out of memory"
                in
                str(exc).lower()
            ):

                print(
                    f"B={candidate}: OOM -> "
                    "trying lower batch"
                )

                twv3_cleanup()

            else:

                raise


    if TWV3_BATCH is None:

        raise RuntimeError(
            "V3: B=128 also failed."
        )


    twv3_atomic_json(

        {
            "architecture_version":
                TWV3_ARCH,

            "batch_size":
                TWV3_BATCH,

            "gpu":
                TWV3_GPU_NAME,

            "vram_gb":
                TWV3_VRAM_GB,

            "amp":
                "bfloat16",

            "fft_real":
                "float32",

            "fft_complex":
                "complex64",
        },

        TWV3_RUN_CONFIG,
    )


# =============================================================================
# ONE RAW-DATA PASS
# =============================================================================

TWV3_TARGET_UPDATES = int(
    math.ceil(
        TWV3_N_TRAIN
        /
        TWV3_BATCH
    )
)


TWV3_WARMUP = max(

    1,

    int(
        math.ceil(
            0.01
            *
            TWV3_TARGET_UPDATES
        )
    ),
)


TWV3_RAW_BYTES_UPDATE = (
    TWV3_BATCH
    *
    RAW_BYTES_PER_SEQUENCE
)


TWV3_POSITIONS_UPDATE = (
    TWV3_BATCH
    *
    TWV3_L
)


TWV3_MASKED_BLOCKS = int(
    round(
        0.15
        *
        TWV3_L
    )
)


assert TWV3_MASKED_BLOCKS == 614


TWV3_PERM = (
    np.random
    .default_rng(
        TWV3_SEED
        +
        1000
    )
    .permutation(
        TWV3_N_TRAIN
    )
)


def twv3_train_indices(
    update_index,
):

    start = (
        update_index
        *
        TWV3_BATCH
    )


    p = (
        np.arange(
            start,
            start
            +
            TWV3_BATCH,
            dtype=np.int64,
        )
        %
        TWV3_N_TRAIN
    )


    return TWV3_PERM[
        p
    ]


# =============================================================================
# VALIDATION
# =============================================================================

@torch.no_grad()
def twv3_evaluate(
    model,
):

    model.eval()


    total_loss = 0.0

    total_n = 0


    for start in range(
        0,
        TWV3_N_VALID,
        TWV3_BATCH,
    ):

        count = min(
            TWV3_BATCH,
            TWV3_N_VALID
            -
            start,
        )


        idx = (
            np.arange(
                start,
                start
                +
                count,
                dtype=np.int64,
            )
        )


        cpu = twv3_numpy_batch(
            TWV3_VALID_BLOCKS,
            idx,
        )


        ids = cpu.to(
            TWV3_DEVICE,
            non_blocking=True,
        )


        gen = torch.Generator(
            device=TWV3_DEVICE
        )


        gen.manual_seed(
            70_000_000
            +
            start
        )


        (
            masked,
            positions,
            targets,
        ) = mask_byte_blocks(

            ids,

            mask_probability=0.15,

            generator=gen,
        )


        with torch.autocast(
            "cuda",
            dtype=TWV3_AMP_DTYPE,
        ):

            hidden = model.masked_hidden_v3(

                masked,

                positions,

                use_checkpoint=False,
            )


            loss, n = twv3_byte_loss(

                model,

                hidden,

                targets,

                training=False,
            )


        total_loss += (
            float(
                loss.detach()
            )
            *
            n
        )


        total_n += n


        del (
            idx,
            cpu,
            ids,
            masked,
            positions,
            targets,
            hidden,
            loss,
        )


    model.train()


    return (
        total_loss
        /
        total_n,
        total_n,
    )


# =============================================================================
# REAL MODEL
# =============================================================================

twv3_cleanup()

twv3_seed_all()


TWV3_MODEL = (
    TWV3Model()
    .to(
        TWV3_DEVICE
    )
)


(
    TWV3_OPTIMIZER,
    TWV3_FUSED_ADAM,
) = twv3_optimizer(
    TWV3_MODEL
)


TWV3_PARAMS = sum(
    p.numel()
    for p in TWV3_MODEL.parameters()
)


TWV3_TRAINABLE = sum(
    p.numel()
    for p in TWV3_MODEL.parameters()
    if p.requires_grad
)


# =============================================================================
# FINGERPRINT
# =============================================================================

TWV3_FINGERPRINT = hashlib.sha256(
    (
        f"{TWV3_ARCH}|"
        f"{TWV3_BATCH}|"
        f"{TWV3_L}|"
        f"{TWV3_LAYERS}|"
        f"{TWV3_FFN_DIM}|"
        f"{TWV3_CROSS_RANK}|"
        f"{TWV3_N_TRAIN}|"
        f"{TWV3_N_VALID}"
    )
    .encode()
).hexdigest()


# =============================================================================
# RESUME
# =============================================================================

TWV3_UPDATE = 0

TWV3_PRIOR_SECONDS = 0.0

TWV3_MASKED_BYTES_SEEN = 0


if TWV3_LATEST.is_file():

    state = torch.load(
        TWV3_LATEST,
        map_location="cpu",
        weights_only=False,
    )


    if (
        state.get(
            "fingerprint"
        )
        !=
        TWV3_FINGERPRINT
    ):

        raise RuntimeError(
            "V3 recovery fingerprint mismatch."
        )


    TWV3_MODEL.load_state_dict(
        state[
            "model_state"
        ],
        strict=True,
    )


    TWV3_OPTIMIZER.load_state_dict(
        state[
            "optimizer_state"
        ]
    )


    for opt_state in (
        TWV3_OPTIMIZER
        .state
        .values()
    ):

        for key, value in list(
            opt_state.items()
        ):

            if torch.is_tensor(
                value
            ):

                opt_state[
                    key
                ] = value.to(
                    TWV3_DEVICE
                )


    TWV3_UPDATE = int(
        state[
            "update"
        ]
    )


    TWV3_PRIOR_SECONDS = float(
        state.get(
            "training_seconds",
            0.0,
        )
    )


    TWV3_MASKED_BYTES_SEEN = int(
        state.get(
            "masked_bytes_seen",
            0,
        )
    )


    del state


    print(
        f"RESUME V3: "
        f"{TWV3_UPDATE}/"
        f"{TWV3_TARGET_UPDATES}"
    )


# =============================================================================
# TIMING / SAVE
# =============================================================================

TWV3_SESSION_START = (
    time.perf_counter()
)


def twv3_elapsed():

    return (
        TWV3_PRIOR_SECONDS
        +
        time.perf_counter()
        -
        TWV3_SESSION_START
    )


def twv3_lr(
    update,
):

    step = update + 1


    return (
        TWV3_LR
        *
        min(
            1.0,
            step
            /
            TWV3_WARMUP,
        )
    )


def twv3_save():

    twv3_atomic_torch_save(

        {
            "fingerprint":
                TWV3_FINGERPRINT,

            "architecture":
                TWV3_ARCH,

            "batch_size":
                TWV3_BATCH,

            "update":
                TWV3_UPDATE,

            "masked_bytes_seen":
                TWV3_MASKED_BYTES_SEEN,

            "training_seconds":
                twv3_elapsed(),

            "model_state":
                TWV3_MODEL.state_dict(),

            "optimizer_state":
                TWV3_OPTIMIZER.state_dict(),
        },

        TWV3_LATEST,
    )


# =============================================================================
# HEADER
# =============================================================================

print(
    "\n"
    +
    "=" * 116
)

print(
    "TRANSWAVE V3 — "
    "LEARNED LIFTING + "
    "COMPLEX CROSS-SCALE MEYER"
)

print(
    f"GPU                  : "
    f"{TWV3_GPU_NAME}"
)

print(
    f"VRAM                 : "
    f"{TWV3_VRAM_GB:.2f} GiB"
)

print(
    f"batch                : "
    f"{TWV3_BATCH}"
)

print(
    f"context              : "
    f"{TWV3_L:,} model positions"
)

print(
    f"bytes/context        : "
    f"{RAW_BYTES_PER_SEQUENCE:,}"
)

print(
    f"train sequences      : "
    f"{TWV3_N_TRAIN:,}"
)

print(
    f"updates              : "
    f"{TWV3_TARGET_UPDATES:,}"
)

print(
    f"parameters           : "
    f"{TWV3_PARAMS:,}"
)

print(
    f"trainable            : "
    f"{TWV3_TRAINABLE:,}"
)

print(
    f"Meyer bands          : "
    f"{TWV3_MEYER_BANDS}"
)

print(
    f"Meyer partition err  : "
    f"{TWV3_MEYER_ERROR:.3e}"
)

print(
    "network AMP           : BF16"
)

print(
    "FFT/Meyer real        : FP32"
)

print(
    "FFT/Meyer complex     : complex64"
)

print(
    f"fused Adam           : "
    f"{TWV3_FUSED_ADAM}"
)

print(
    "=" * 116
)


# =============================================================================
# TRAIN
# =============================================================================

TWV3_MODEL.train()

torch.cuda.reset_peak_memory_stats()


TWV3_LAST_SAVE = (
    time.monotonic()
)


try:

    while (
        TWV3_UPDATE
        <
        TWV3_TARGET_UPDATES
    ):

        indices = twv3_train_indices(
            TWV3_UPDATE
        )


        cpu = twv3_numpy_batch(
            TWV3_TRAIN_BLOCKS,
            indices,
        )


        ids = cpu.to(
            TWV3_DEVICE,
            non_blocking=True,
        )


        gen = torch.Generator(
            device=TWV3_DEVICE
        )


        gen.manual_seed(
            10_000_000
            +
            TWV3_UPDATE
        )


        (
            masked,
            positions,
            targets,
        ) = mask_byte_blocks(

            ids,

            mask_probability=0.15,

            generator=gen,
        )


        lr = twv3_lr(
            TWV3_UPDATE
        )


        for group in (
            TWV3_OPTIMIZER.param_groups
        ):

            group[
                "lr"
            ] = lr


        TWV3_OPTIMIZER.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            "cuda",
            dtype=TWV3_AMP_DTYPE,
        ):

            hidden = TWV3_MODEL.masked_hidden_v3(

                masked,

                positions,

                use_checkpoint=True,
            )


            loss, n_targets = twv3_byte_loss(

                TWV3_MODEL,

                hidden,

                targets,

                training=True,
            )


        loss.backward()


        TWV3_OPTIMIZER.step()


        TWV3_UPDATE += 1


        TWV3_MASKED_BYTES_SEEN += (
            n_targets
        )


        if (
            TWV3_UPDATE == 1
            or
            TWV3_UPDATE % 10 == 0
            or
            TWV3_UPDATE
            ==
            TWV3_TARGET_UPDATES
        ):

            torch.cuda.synchronize()


            seconds = twv3_elapsed()


            raw_seen = (
                TWV3_UPDATE
                *
                TWV3_RAW_BYTES_UPDATE
            )


            raw_rate = (
                raw_seen
                /
                max(
                    seconds,
                    1.0e-9,
                )
            )


            peak_alloc = (
                torch.cuda
                .max_memory_allocated()
                /
                1024**3
            )


            peak_res = (
                torch.cuda
                .max_memory_reserved()
                /
                1024**3
            )


            eta = (
                (
                    TWV3_TARGET_UPDATES
                    -
                    TWV3_UPDATE
                )
                *
                TWV3_RAW_BYTES_UPDATE
                /
                max(
                    raw_rate,
                    1.0e-9,
                )
                /
                3600
            )


            bpb = (
                float(
                    loss.detach()
                )
                /
                math.log(2.0)
            )


            print(
                f"arxiv | "
                f"{TWV3_UPDATE:4d}/"
                f"{TWV3_TARGET_UPDATES} | "
                f"loss={float(loss.detach()):.4f} | "
                f"BPB={bpb:.4f} | "
                f"lr={lr:.2e} | "
                f"{raw_rate/1e6:.2f} MB/s | "
                f"VRAM="
                f"{peak_alloc:.2f}/"
                f"{peak_res:.2f} GiB | "
                f"ETA={eta:.2f}h"
            )


        del (
            indices,
            cpu,
            ids,
            masked,
            positions,
            targets,
            hidden,
            loss,
        )


        if (
            TWV3_UPDATE % 50 == 0
            or
            (
                time.monotonic()
                -
                TWV3_LAST_SAVE
            )
            >=
            1200
        ):

            twv3_save()


            TWV3_LAST_SAVE = (
                time.monotonic()
            )


            print(
                f"  recovery saved @ "
                f"{TWV3_UPDATE}"
            )


except BaseException:

    try:

        twv3_save()

        print(
            "Emergency V3 checkpoint saved."
        )

    except Exception as e:

        print(
            "Emergency save failed:",
            e,
        )


    raise


# =============================================================================
# VALIDATE
# =============================================================================

torch.cuda.synchronize()

TWV3_TRAINING_SECONDS = (
    twv3_elapsed()
)


twv3_save()


print(
    "\nRunning V3 byte validation ..."
)


(
    TWV3_VAL_LOSS,
    TWV3_VAL_TARGETS,
) = twv3_evaluate(
    TWV3_MODEL
)


TWV3_VAL_BPB = (
    TWV3_VAL_LOSS
    /
    math.log(2.0)
)


TWV3_VAL_PPL = math.exp(
    min(
        TWV3_VAL_LOSS,
        20.0,
    )
)


# =============================================================================
# RESULT
# =============================================================================

TWV3_RESULT = {

    "status":
        "complete",

    "fingerprint":
        TWV3_FINGERPRINT,

    "architecture":
        TWV3_ARCH,

    "model":
        "TransWave-Lifting-ComplexMeyer",

    "corpus":
        "arxiv",

    "input":
        "raw_utf8_bytes",

    "bpe":
        False,

    "batch_size":
        TWV3_BATCH,

    "model_context":
        TWV3_L,

    "bytes_per_model_position":
        BYTES_PER_MODEL_TOKEN,

    "bytes_per_sequence":
        RAW_BYTES_PER_SEQUENCE,

    "optimizer_updates":
        TWV3_TARGET_UPDATES,

    "training_raw_byte_exposures":
        (
            TWV3_TARGET_UPDATES
            *
            TWV3_RAW_BYTES_UPDATE
        ),

    "masked_byte_targets":
        TWV3_MASKED_BYTES_SEEN,

    "parameters":
        TWV3_PARAMS,

    "trainable_parameters":
        TWV3_TRAINABLE,

    "meyer_bands":
        TWV3_MEYER_BANDS,

    "meyer_real_dtype":
        "float32",

    "meyer_complex_dtype":
        "complex64",

    "network_amp":
        "bfloat16",

    "training_seconds":
        TWV3_TRAINING_SECONDS,

    "validation_byte_nll":
        TWV3_VAL_LOSS,

    "validation_bits_per_byte":
        TWV3_VAL_BPB,

    "validation_byte_perplexity":
        TWV3_VAL_PPL,

    "validation_byte_targets":
        TWV3_VAL_TARGETS,

    "peak_allocated_vram_gb":
        (
            torch.cuda
            .max_memory_allocated()
            /
            1024**3
        ),

    "peak_reserved_vram_gb":
        (
            torch.cuda
            .max_memory_reserved()
            /
            1024**3
        ),

    "gpu":
        TWV3_GPU_NAME,
}


twv3_atomic_torch_save(

    {
        **TWV3_RESULT,

        "model_state":
            TWV3_MODEL.state_dict(),
    },

    TWV3_FINAL_MODEL,
)


twv3_atomic_json(
    TWV3_RESULT,
    TWV3_RESULT_JSON,
)


twv3_atomic_json(
    TWV3_RESULT,
    TWV3_COMPLETE,
)


TWV3_LATEST.unlink(
    missing_ok=True
)


# =============================================================================
# FINAL
# =============================================================================

print(
    "\n"
    +
    "=" * 116
)

print(
    "TRANSWAVE V3 arXiv COMPLETE"
)

print(
    f"validation byte NLL : "
    f"{TWV3_VAL_LOSS:.6f}"
)

print(
    f"validation BPB      : "
    f"{TWV3_VAL_BPB:.6f}"
)

print(
    f"byte perplexity     : "
    f"{TWV3_VAL_PPL:.6f}"
)

print(
    f"peak VRAM allocated : "
    f"{TWV3_RESULT['peak_allocated_vram_gb']:.2f} GiB"
)

print(
    "result:",
    TWV3_RESULT_JSON,
)

print(
    "model:",
    TWV3_FINAL_MODEL,
)

print(
    "=" * 116
)

local SSD already staged: /content/TransWave_Lifting_Arxiv/train.txt
local SSD already staged: /content/TransWave_Lifting_Arxiv/validation.txt

V3 COMPLEX-PATH PREFLIGHT
V3 forward source: /tmp/ipykernel_1753/2357041828.py
V3 dtype preflight: PASS
input       : bfloat16
FFT real    : float32
FFT complex : complex64
output      : bfloat16
gradients   : within-scale + cross-scale PASS

V3 G4 FULL-TRAINING MEMORY PROBE
forward + backward + Adam moments
complex path verified FP32/complex64
B=155: FIT | alloc=59.53 GiB | reserved=64.88 GiB | loss=6.6705

TRANSWAVE V3 — LEARNED LIFTING + COMPLEX CROSS-SCALE MEYER
GPU                  : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM                 : 94.97 GiB
batch                : 155
context              : 4,096 model positions
bytes/context        : 32,768
train sequences      : 16,640
updates              : 108
parameters           : 82,864,704
trainable            : 82,864,704
Meyer bands          : 12
Meyer partition err  : 0.000e+0

In [14]:
import numpy as np
import math

train = np.memmap(
    "/content/TransWave_Lifting_Arxiv/train.txt",
    dtype=np.uint8,
    mode="r",
)

valid = np.memmap(
    "/content/TransWave_Lifting_Arxiv/validation.txt",
    dtype=np.uint8,
    mode="r",
)

# Train byte distribution.
counts = np.bincount(
    train,
    minlength=256,
).astype(np.float64)

# Tiny smoothing only to avoid log(0).
p = (
    counts + 1.0
) / (
    counts.sum() + 256.0
)

# Validation cross entropy under zero-context byte unigram model.
valid_counts = np.bincount(
    valid,
    minlength=256,
).astype(np.float64)

nll = -(
    valid_counts
    *
    np.log(p)
).sum() / valid_counts.sum()

bpb = nll / math.log(2.0)

ppl = math.exp(nll)

print(f"Unigram byte NLL : {nll:.6f}")
print(f"Unigram BPB      : {bpb:.6f}")
print(f"Unigram byte PPL : {ppl:.6f}")

print()
print(f"TransWave BPB    : {4.579127:.6f}")
print(f"BPB improvement  : {bpb - 4.579127:.6f}")

Unigram byte NLL : 3.143694
Unigram BPB      : 4.535391
Unigram byte PPL : 23.189362

TransWave BPB    : 4.579127
BPB improvement  : -0.043736


In [19]:
# =============================================================================
# TRANSWAVE V4 — CHECKPOINTED MAX-GPU CONTINUATION
#
# CURRENT LIVE STATE:
#       update = 474
#       examples seen = 15,168
#
# NO TOKENIZATION.
# NO CELL 3.
# NO BPE.
# NO SHARDS.
#
# IMPORTANT FIX:
#   V4 activations do NOT fit without checkpointing, even at B=32.
#
# Therefore:
#   activation checkpointing = ON
#   physical batch           = probe upward to max B=155
#   CE checkpointing         = OFF
#
# UNIVERSAL VALIDATION TARGET:
#
#       PPL <= 3.000000
#       NLL <= ln(3) = 1.098612289
#
# TRAINING PPL IS NOT A STOPPING CONDITION.
# Only validation PPL/NLL determines success.
# =============================================================================

from __future__ import annotations

import gc
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F


# =============================================================================
# TARGET
# =============================================================================

TARGET_PPL = 3.0
TARGET_NLL = math.log(TARGET_PPL)

ORIGINAL_BATCH = 32

MAX_PHYSICAL_BATCH = 155

BATCH_CANDIDATES = (
    155,
    152,
    148,
    144,
    138,
    128,
    120,
    112,
    104,
    96,
    88,
    80,
    72,
    64,
    56,
    48,
    40,
    32,
)

# Validate frequently because each large-batch step sees many samples.
VALIDATE_EVERY_STEPS = 5

# Original V4 experiment = at most 3 complete corpus passes.
MAX_TOTAL_EXAMPLES = 3 * TWV4_N_TRAIN

FIXED_VALIDATION_SEED = 777_000_000

# Leave enough room for fused Adam and allocator fluctuations.
VRAM_HEADROOM_GB = 5.0


print("=" * 120)
print("TRANSWAVE V4 — CHECKPOINTED MAX-GPU CONTINUATION")
print("tokenization                : NONE")
print("activation checkpointing    : ON")
print("CE checkpointing            : OFF")
print(f"physical batch ceiling      : {MAX_PHYSICAL_BATCH}")
print(f"VALIDATION TARGET PPL       : {TARGET_PPL:.6f}")
print(f"VALIDATION TARGET NLL       : {TARGET_NLL:.9f}")
print("training PPL stop           : NONE")
print("=" * 120)


# =============================================================================
# VERIFY LIVE STATE
# =============================================================================

_required = (
    "TWV4_MODEL",
    "TWV4_OPTIMIZER",
    "TWV4_UPDATE",
    "TWV4_TRAIN_BLOCKS",
    "TWV4_VALID_BLOCKS",
    "TWV4_N_TRAIN",
    "TWV4_N_VALID",
    "TWV4_DEVICE",
    "TWV4_AMP",
    "TWV4_CE_CHUNK",
    "TWV4_RUN_ROOT",
    "TWV4_RESULTS_ROOT",
    "TWV4_ARCH",
    "TWV4_FINGERPRINT",
    "twv4_batch",
    "twv4_permutation",
    "twv4_mask_individual_bytes",
    "twv4_learning_rate",
    "twv4_atomic_torch_save",
    "twv4_atomic_json",
)

missing = [
    x for x in _required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Live V4 state is missing. DO NOT TOKENIZE AGAIN.\n"
        "Missing: " + ", ".join(missing)
    )


# =============================================================================
# REMOVE STALE OBJECTS LEFT BY FAILED NO-CHECKPOINT PROBES
# =============================================================================

# Do NOT delete TWV4_MODEL or TWV4_OPTIMIZER.
for _name in (
    "MODEL",
    "OPTIMIZER",
    "hidden",
    "loss",
    "ids",
    "masked",
    "targets",
    "selected_positions",
    "cpu",
    "indices",
    "SOURCE_STATE",
):
    globals().pop(_name, None)


gc.collect()

try:
    torch.cuda.synchronize()
except Exception:
    pass

torch.cuda.empty_cache()

try:
    torch.cuda.ipc_collect()
except Exception:
    pass


MODEL = TWV4_MODEL
OPTIMIZER = TWV4_OPTIMIZER

OPTIMIZER.zero_grad(set_to_none=True)


SOURCE_UPDATE = int(TWV4_UPDATE)

EXAMPLES_SEEN = (
    SOURCE_UPDATE
    *
    ORIGINAL_BATCH
)

NEW_STEPS = 0


print()
print(f"resuming live update       : {SOURCE_UPDATE}")
print(f"examples already seen      : {EXAMPLES_SEEN:,}")
print(f"maximum total exposure     : {MAX_TOTAL_EXAMPLES:,}")


# =============================================================================
# OUTPUT
# =============================================================================

RUN_ROOT = (
    Path(TWV4_RUN_ROOT)
    /
    "ppl3_checkpointed_maxgpu"
)

RESULT_ROOT = (
    Path(TWV4_RESULTS_ROOT)
    /
    "ppl3_checkpointed_maxgpu"
)

RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

LATEST_FILE = (
    RUN_ROOT
    /
    "training_latest.pt"
)

BEST_FILE = (
    RUN_ROOT
    /
    "best_model.pt"
)

FINAL_FILE = (
    RUN_ROOT
    /
    "trained_model_final.pt"
)

RESULT_FILE = (
    RESULT_ROOT
    /
    "arxiv.json"
)


# =============================================================================
# CLEANUP
# =============================================================================

def clean_cuda():

    gc.collect()

    try:
        torch.cuda.synchronize()
    except Exception:
        pass

    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


# =============================================================================
# NON-CHECKPOINTED CE
#
# Layer activations are checkpointed.
# Tiny 256-way CE is not.
# =============================================================================

def exact_loss(
    model,
    hidden,
    targets,
):

    targets = (
        targets
        .reshape(-1)
        .long()
    )

    n = int(
        targets.numel()
    )

    if hidden.shape[0] != n:
        raise RuntimeError(
            "hidden/target mismatch"
        )

    total = None

    for start in range(
        0,
        n,
        TWV4_CE_CHUNK,
    ):

        end = min(
            start + TWV4_CE_CHUNK,
            n,
        )

        h = hidden[start:end]
        t = targets[start:end]

        with torch.autocast(
            "cuda",
            dtype=TWV4_AMP,
        ):

            logits = model.logits_v4(h)

            part = F.cross_entropy(
                logits,
                t,
                reduction="sum",
            )

        total = (
            part
            if total is None
            else total + part
        )

    return (
        total / float(n),
        n,
    )


# =============================================================================
# DATA ORDER BASED ON NUMBER OF EXAMPLES ALREADY SEEN
# =============================================================================

def get_indices(
    example_offset,
    batch_size,
):

    cursor = int(
        example_offset
    )

    remaining = int(
        batch_size
    )

    pieces = []

    while remaining > 0:

        pass_index = (
            cursor
            //
            TWV4_N_TRAIN
        )

        offset = (
            cursor
            %
            TWV4_N_TRAIN
        )

        permutation = twv4_permutation(
            pass_index
        )

        take = min(
            remaining,
            TWV4_N_TRAIN - offset,
        )

        pieces.append(
            permutation[
                offset:
                offset + take
            ]
        )

        cursor += take
        remaining -= take

    return np.concatenate(
        pieces
    )


# =============================================================================
# CHECKPOINTED TRAINING-BATCH PROBE
#
# NO optimizer step.
# Therefore model/Adam state is not altered.
# =============================================================================

def probe_batch(
    batch_size,
):

    clean_cuda()

    MODEL.train()

    OPTIMIZER.zero_grad(
        set_to_none=True
    )

    torch.cuda.reset_peak_memory_stats()

    indices = get_indices(
        EXAMPLES_SEEN,
        batch_size,
    )

    cpu = twv4_batch(
        TWV4_TRAIN_BLOCKS,
        indices,
    )

    ids = cpu.to(
        TWV4_DEVICE,
        non_blocking=True,
    )

    (
        masked,
        selected_positions,
        targets,
    ) = twv4_mask_individual_bytes(
        ids,
        seed=(
            810_000_000
            +
            batch_size
        ),
    )

    with torch.autocast(
        "cuda",
        dtype=TWV4_AMP,
    ):

        hidden = MODEL.selected_hidden_v4(
            masked,
            selected_positions,

            # =========================================================
            # CORRECT SETTING FOR V4
            # =========================================================
            checkpoint_layers=True,
        )

        loss, _ = exact_loss(
            MODEL,
            hidden,
            targets,
        )

    loss.backward()

    torch.cuda.synchronize()

    allocated = (
        torch.cuda.max_memory_allocated()
        /
        1024**3
    )

    reserved = (
        torch.cuda.max_memory_reserved()
        /
        1024**3
    )

    nll = float(
        loss.detach()
    )

    ppl = math.exp(
        min(nll, 20.0)
    )

    OPTIMIZER.zero_grad(
        set_to_none=True
    )

    del (
        indices,
        cpu,
        ids,
        masked,
        selected_positions,
        targets,
        hidden,
        loss,
    )

    clean_cuda()

    return (
        allocated,
        reserved,
        nll,
        ppl,
    )


# =============================================================================
# PROBE LARGEST BATCH
# =============================================================================

TOTAL_VRAM_GB = (
    torch.cuda
    .get_device_properties(0)
    .total_memory
    /
    1024**3
)


print()
print("=" * 120)
print("CHECKPOINTED PHYSICAL-BATCH PROBE")
print(f"total GPU VRAM         : {TOTAL_VRAM_GB:.2f} GiB")
print(f"required free headroom : {VRAM_HEADROOM_GB:.2f} GiB")
print("=" * 120)


CHOSEN_BATCH = None


for candidate in BATCH_CANDIDATES:

    try:

        (
            allocated,
            reserved,
            probe_nll,
            probe_ppl,
        ) = probe_batch(
            candidate
        )

        free_gb = (
            TOTAL_VRAM_GB
            -
            reserved
        )

        if free_gb < VRAM_HEADROOM_GB:

            print(
                f"B={candidate}: FIT BUT TOO TIGHT | "
                f"alloc={allocated:.2f} GiB | "
                f"reserved={reserved:.2f} GiB | "
                f"free={free_gb:.2f} GiB"
            )

            continue

        CHOSEN_BATCH = candidate

        print(
            f"B={candidate}: SELECTED | "
            f"alloc={allocated:.2f} GiB | "
            f"reserved={reserved:.2f} GiB | "
            f"free={free_gb:.2f} GiB | "
            f"NLL={probe_nll:.4f} | "
            f"PPL={probe_ppl:.3f}"
        )

        break

    except torch.cuda.OutOfMemoryError:

        print(
            f"B={candidate}: OOM"
        )

        OPTIMIZER.zero_grad(
            set_to_none=True
        )

        clean_cuda()

    except RuntimeError as exc:

        if "out of memory" in str(exc).lower():

            print(
                f"B={candidate}: OOM"
            )

            OPTIMIZER.zero_grad(
                set_to_none=True
            )

            clean_cuda()

        else:
            raise


if CHOSEN_BATCH is None:

    raise RuntimeError(
        "Even checkpointed B=32 did not fit. "
        "This would indicate stale GPU allocations rather than "
        "the V4 architecture itself."
    )


print()
print(
    f"SELECTED PHYSICAL BATCH = "
    f"{CHOSEN_BATCH}"
)


# =============================================================================
# FIXED VALIDATION
#
# No gradient graph => activation checkpointing is unnecessary here.
# =============================================================================

@torch.no_grad()
def validate():

    MODEL.eval()

    total_loss = 0.0
    total_targets = 0

    for start in range(
        0,
        TWV4_N_VALID,
        CHOSEN_BATCH,
    ):

        count = min(
            CHOSEN_BATCH,
            TWV4_N_VALID - start,
        )

        indices = np.arange(
            start,
            start + count,
            dtype=np.int64,
        )

        cpu = twv4_batch(
            TWV4_VALID_BLOCKS,
            indices,
        )

        ids = cpu.to(
            TWV4_DEVICE,
            non_blocking=True,
        )

        (
            masked,
            selected_positions,
            targets,
        ) = twv4_mask_individual_bytes(
            ids,
            seed=(
                FIXED_VALIDATION_SEED
                +
                start
            ),
        )

        with torch.autocast(
            "cuda",
            dtype=TWV4_AMP,
        ):

            hidden = MODEL.selected_hidden_v4(
                masked,
                selected_positions,
                checkpoint_layers=False,
            )

            loss, n_targets = exact_loss(
                MODEL,
                hidden,
                targets,
            )

        total_loss += (
            float(loss.detach())
            *
            n_targets
        )

        total_targets += (
            n_targets
        )

        del (
            indices,
            cpu,
            ids,
            masked,
            selected_positions,
            targets,
            hidden,
            loss,
        )

    MODEL.train()

    mean_nll = (
        total_loss
        /
        total_targets
    )

    mean_ppl = math.exp(
        min(mean_nll, 20.0)
    )

    return (
        mean_nll,
        mean_ppl,
        total_targets,
    )


# =============================================================================
# HISTORY
# =============================================================================

VALIDATION_HISTORY = list(
    globals().get(
        "TWV4_VALIDATION_HISTORY",
        [],
    )
)

BEST_NLL = float(
    globals().get(
        "TWV4_BEST_NLL",
        float("inf"),
    )
)


# =============================================================================
# SAVE HELPERS
# =============================================================================

def save_latest():

    twv4_atomic_torch_save(
        {
            "fingerprint":
                TWV4_FINGERPRINT,

            "architecture":
                TWV4_ARCH,

            "source_update":
                SOURCE_UPDATE,

            "examples_seen":
                EXAMPLES_SEEN,

            "new_steps":
                NEW_STEPS,

            "physical_batch":
                CHOSEN_BATCH,

            "target_ppl":
                TARGET_PPL,

            "target_nll":
                TARGET_NLL,

            "validation_history":
                VALIDATION_HISTORY,

            "best_nll":
                BEST_NLL,

            "model_state":
                MODEL.state_dict(),

            "optimizer_state":
                OPTIMIZER.state_dict(),
        },

        LATEST_FILE,
    )


def save_best(
    val_nll,
    val_ppl,
):

    twv4_atomic_torch_save(
        {
            "fingerprint":
                TWV4_FINGERPRINT,

            "architecture":
                TWV4_ARCH,

            "examples_seen":
                EXAMPLES_SEEN,

            "new_steps":
                NEW_STEPS,

            "physical_batch":
                CHOSEN_BATCH,

            "validation_nll":
                float(val_nll),

            "validation_ppl":
                float(val_ppl),

            "model_state":
                MODEL.state_dict(),
        },

        BEST_FILE,
    )


# =============================================================================
# IMMEDIATE VALIDATION
# =============================================================================

print()
print("=" * 120)
print("IMMEDIATE VALIDATION OF UPDATE", SOURCE_UPDATE)
print("=" * 120)


(
    val_nll,
    val_ppl,
    val_targets,
) = validate()


print(
    f"validation NLL : {val_nll:.6f}"
)

print(
    f"validation PPL : {val_ppl:.6f}"
)


VALIDATION_HISTORY.append(
    {
        "examples_seen":
            EXAMPLES_SEEN,

        "new_steps":
            0,

        "validation_nll":
            float(val_nll),

        "validation_ppl":
            float(val_ppl),

        "reason":
            "immediate_validation",
    }
)


if val_nll < BEST_NLL:

    BEST_NLL = float(
        val_nll
    )

    save_best(
        val_nll,
        val_ppl,
    )


STOP_REASON = None


if val_ppl <= TARGET_PPL:

    STOP_REASON = (
        "validation_target_reached"
    )


# =============================================================================
# TRAIN
# =============================================================================

MODEL.train()

torch.cuda.reset_peak_memory_stats()


LAST_TRAIN_NLL = None
LAST_TRAIN_PPL = None


try:

    while STOP_REASON is None:

        # -------------------------------------------------------------
        # THREE-PASS EXPOSURE LIMIT
        # -------------------------------------------------------------

        if EXAMPLES_SEEN >= MAX_TOTAL_EXAMPLES:

            STOP_REASON = (
                "three_pass_exposure_limit"
            )

            break


        this_batch = min(
            CHOSEN_BATCH,
            MAX_TOTAL_EXAMPLES
            -
            EXAMPLES_SEEN,
        )


        # -------------------------------------------------------------
        # DATA
        # -------------------------------------------------------------

        indices = get_indices(
            EXAMPLES_SEEN,
            this_batch,
        )


        cpu = twv4_batch(
            TWV4_TRAIN_BLOCKS,
            indices,
        )


        ids = cpu.to(
            TWV4_DEVICE,
            non_blocking=True,
        )


        (
            masked,
            selected_positions,
            targets,
        ) = twv4_mask_individual_bytes(
            ids,
            seed=(
                920_000_000
                +
                EXAMPLES_SEEN
            ),
        )


        # -------------------------------------------------------------
        # LR FROM EQUIVALENT B=32 SAMPLE EXPOSURE
        # -------------------------------------------------------------

        equivalent_b32_update = (
            EXAMPLES_SEEN
            //
            ORIGINAL_BATCH
        )


        lr = twv4_learning_rate(
            equivalent_b32_update
        )


        for group in OPTIMIZER.param_groups:

            group["lr"] = lr


        OPTIMIZER.zero_grad(
            set_to_none=True
        )


        # -------------------------------------------------------------
        # FORWARD
        #
        # ACTIVATION CHECKPOINTING ON.
        # -------------------------------------------------------------

        with torch.autocast(
            "cuda",
            dtype=TWV4_AMP,
        ):

            hidden = MODEL.selected_hidden_v4(
                masked,
                selected_positions,
                checkpoint_layers=True,
            )


            loss, n_targets = exact_loss(
                MODEL,
                hidden,
                targets,
            )


        # -------------------------------------------------------------
        # BACKWARD
        # -------------------------------------------------------------

        loss.backward()

        OPTIMIZER.step()


        NEW_STEPS += 1

        EXAMPLES_SEEN += int(
            this_batch
        )


        LAST_TRAIN_NLL = float(
            loss.detach()
        )


        LAST_TRAIN_PPL = math.exp(
            min(
                LAST_TRAIN_NLL,
                20.0,
            )
        )


        torch.cuda.synchronize()


        alloc = (
            torch.cuda
            .max_memory_allocated()
            /
            1024**3
        )


        reserved = (
            torch.cuda
            .max_memory_reserved()
            /
            1024**3
        )


        print(
            f"arxiv | "
            f"B={this_batch} | "
            f"step={NEW_STEPS:3d} | "
            f"NLL={LAST_TRAIN_NLL:.4f} | "
            f"PPL={LAST_TRAIN_PPL:.3f} | "
            f"lr={lr:.2e} | "
            f"VRAM={alloc:.2f}/{reserved:.2f} GiB"
        )


        del (
            indices,
            cpu,
            ids,
            masked,
            selected_positions,
            targets,
            hidden,
            loss,
        )


        # -------------------------------------------------------------
        # VALIDATION
        #
        # DO NOT STOP BECAUSE TRAIN PPL < 3.
        # VALIDATION IS THE TARGET.
        # -------------------------------------------------------------

        if (
            NEW_STEPS
            %
            VALIDATE_EVERY_STEPS
            ==
            0
        ):

            print()
            print(
                f"VALIDATING @ NEW STEP {NEW_STEPS}"
            )


            (
                val_nll,
                val_ppl,
                val_targets,
            ) = validate()


            print(
                f"validation NLL : "
                f"{val_nll:.6f}"
            )


            print(
                f"validation PPL : "
                f"{val_ppl:.6f}"
            )


            VALIDATION_HISTORY.append(
                {
                    "examples_seen":
                        EXAMPLES_SEEN,

                    "new_steps":
                        NEW_STEPS,

                    "training_nll":
                        LAST_TRAIN_NLL,

                    "training_ppl":
                        LAST_TRAIN_PPL,

                    "validation_nll":
                        float(val_nll),

                    "validation_ppl":
                        float(val_ppl),

                    "reason":
                        "periodic_validation",
                }
            )


            if val_nll < BEST_NLL:

                BEST_NLL = float(
                    val_nll
                )

                save_best(
                    val_nll,
                    val_ppl,
                )


            save_latest()


            if val_ppl <= TARGET_PPL:

                STOP_REASON = (
                    "validation_ppl_3_reached"
                )

                print()
                print("=" * 120)
                print(
                    "VALIDATION PPL <= 3 REACHED"
                )
                print(
                    f"NLL = {val_nll:.6f}"
                )
                print(
                    f"PPL = {val_ppl:.6f}"
                )
                print("=" * 120)

                break


        if NEW_STEPS % 10 == 0:

            save_latest()


except BaseException:

    try:

        save_latest()

        print(
            "Emergency checkpoint saved."
        )

    except Exception as save_error:

        print(
            "Emergency save failed:",
            save_error,
        )

    raise


# =============================================================================
# FINAL VALIDATION IF LOOP ENDED ON EXPOSURE CAP
# =============================================================================

if (
    STOP_REASON
    ==
    "three_pass_exposure_limit"
):

    print()
    print(
        "Three-pass exposure limit reached. "
        "Running final validation."
    )

    (
        val_nll,
        val_ppl,
        val_targets,
    ) = validate()

    VALIDATION_HISTORY.append(
        {
            "examples_seen":
                EXAMPLES_SEEN,

            "new_steps":
                NEW_STEPS,

            "training_nll":
                LAST_TRAIN_NLL,

            "training_ppl":
                LAST_TRAIN_PPL,

            "validation_nll":
                float(val_nll),

            "validation_ppl":
                float(val_ppl),

            "reason":
                "final_exposure_limit_validation",
        }
    )

    if val_nll < BEST_NLL:

        BEST_NLL = float(
            val_nll
        )

        save_best(
            val_nll,
            val_ppl,
        )


# =============================================================================
# FINAL RESULT
# =============================================================================

BEST_ENTRY = min(
    VALIDATION_HISTORY,
    key=lambda x:
        x["validation_nll"],
)


RESULT = {
    "status":
        "complete",

    "architecture":
        TWV4_ARCH,

    "dataset":
        "arxiv",

    "tokenization":
        "NONE",

    "activation_checkpointing":
        True,

    "ce_checkpointing":
        False,

    "physical_batch":
        CHOSEN_BATCH,

    "batch_ceiling":
        MAX_PHYSICAL_BATCH,

    "source_update":
        SOURCE_UPDATE,

    "initial_examples_seen":
        SOURCE_UPDATE * ORIGINAL_BATCH,

    "final_examples_seen":
        EXAMPLES_SEEN,

    "new_optimizer_steps":
        NEW_STEPS,

    "target_ppl":
        TARGET_PPL,

    "target_nll":
        TARGET_NLL,

    "best_validation_nll":
        float(
            BEST_ENTRY[
                "validation_nll"
            ]
        ),

    "best_validation_ppl":
        float(
            BEST_ENTRY[
                "validation_ppl"
            ]
        ),

    "target_reached":
        bool(
            BEST_ENTRY[
                "validation_ppl"
            ]
            <=
            TARGET_PPL
        ),

    "last_training_nll":
        LAST_TRAIN_NLL,

    "last_training_ppl":
        LAST_TRAIN_PPL,

    "stop_reason":
        STOP_REASON,

    "validation_history":
        VALIDATION_HISTORY,

    "peak_allocated_vram_gb":
        float(
            torch.cuda
            .max_memory_allocated()
            /
            1024**3
        ),

    "peak_reserved_vram_gb":
        float(
            torch.cuda
            .max_memory_reserved()
            /
            1024**3
        ),
}


twv4_atomic_torch_save(
    {
        **RESULT,
        "model_state":
            MODEL.state_dict(),
    },
    FINAL_FILE,
)


twv4_atomic_json(
    RESULT,
    RESULT_FILE,
)


LATEST_FILE.unlink(
    missing_ok=True
)


print()
print("=" * 120)
print("TRANSWAVE V4 FINISHED")
print(f"physical batch          : {CHOSEN_BATCH}")
print(f"new optimizer steps     : {NEW_STEPS}")
print(
    f"best validation NLL     : "
    f"{RESULT['best_validation_nll']:.6f}"
)
print(
    f"best validation PPL     : "
    f"{RESULT['best_validation_ppl']:.6f}"
)
print(
    f"target NLL              : "
    f"{TARGET_NLL:.9f}"
)
print(
    f"target PPL              : "
    f"{TARGET_PPL:.6f}"
)
print(f"target reached          : {RESULT['target_reached']}")
print(f"stop reason             : {STOP_REASON}")
print(
    f"peak allocated VRAM     : "
    f"{RESULT['peak_allocated_vram_gb']:.2f} GiB"
)
print(
    f"peak reserved VRAM      : "
    f"{RESULT['peak_reserved_vram_gb']:.2f} GiB"
)
print(f"result                  : {RESULT_FILE}")
print(f"final model             : {FINAL_FILE}")
print("=" * 120)

TRANSWAVE V4 — CHECKPOINTED MAX-GPU CONTINUATION
tokenization                : NONE
activation checkpointing    : ON
CE checkpointing            : OFF
physical batch ceiling      : 155
VALIDATION TARGET PPL       : 3.000000
VALIDATION TARGET NLL       : 1.098612289
training PPL stop           : NONE

resuming live update       : 474
examples already seen      : 15,168
maximum total exposure     : 49,920

CHECKPOINTED PHYSICAL-BATCH PROBE
total GPU VRAM         : 94.97 GiB
required free headroom : 5.00 GiB
B=155: SELECTED | alloc=61.60 GiB | reserved=67.32 GiB | free=27.65 GiB | NLL=0.9281 | PPL=2.530

SELECTED PHYSICAL BATCH = 155

IMMEDIATE VALIDATION OF UPDATE 474
validation NLL : 0.932117
validation PPL : 2.539880

TRANSWAVE V4 FINISHED
physical batch          : 155
new optimizer steps     : 0
best validation NLL     : 0.929433
best validation PPL     : 2.533072
target NLL              : 1.098612289
target PPL              : 3.000000
target reached          : True
stop reason       

In [21]:
# =============================================================================
# RECOVER THE EXACT DELETED TRANSWAVE V4 ARCHITECTURE CELL
#
# This DOES NOT reconstruct anything.
# It reads the actual executed input history of the current Colab kernel.
# =============================================================================

from pathlib import Path

ip = get_ipython()

if ip is None:
    raise RuntimeError("This must be run in the same live Colab/IPython runtime.")

# Distinctive strings from the REAL V4 architecture/run.
needles = (
    "TWV4Model",
    "transwave_v4_lifting_selective_meyer96_fullrank_scale8_localconv7_swiglu_byte801010",
)

matches = []

# Raw input history: exact code that was executed, including deleted notebook cells.
for session, line_no, source in ip.history_manager.get_range(
    session=0,
    start=1,
    stop=None,
    raw=True,
    output=False,
):

    if source and any(n in source for n in needles):

        # Exclude the later continuation/recovery cells that merely USE TWV4Model.
        is_architecture = (
            "class TWV4Model" in source
            or
            (
                "TWV4Model" in source
                and "TWV4_MODEL =" in source
                and "selected_hidden_v4" in source
                and "logits_v4" in source
            )
        )

        if is_architecture:
            matches.append(
                (session, line_no, source)
            )


# Also check normal In[] history in case history_manager filtering differs.
for line_no, source in enumerate(In):

    if not source:
        continue

    if (
        "class TWV4Model" in source
        and
        "selected_hidden_v4" in source
        and
        "logits_v4" in source
    ):
        key = source.strip()

        if not any(
            old_source.strip() == key
            for _, _, old_source in matches
        ):
            matches.append(
                (0, line_no, source)
            )


if not matches:

    # Last fallback: inspect the live class itself.
    # This is still the REAL runtime class, not a reconstruction.
    import inspect

    if "TWV4_MODEL" not in globals():
        raise RuntimeError(
            "TWV4_MODEL is no longer live and the deleted cell "
            "was not found in IPython history."
        )

    cls = type(TWV4_MODEL)

    try:
        source = inspect.getsource(cls)
    except Exception as exc:
        raise RuntimeError(
            "The live V4 object exists, but its full deleted source "
            "could not be recovered automatically."
        ) from exc

    matches.append(
        (0, -1, source)
    )


print(f"Recovered {len(matches)} candidate exact V4 source cell(s).")

for i, (session, line_no, source) in enumerate(matches, 1):

    print()
    print("=" * 120)
    print(
        f"CANDIDATE {i} | session={session} | "
        f"history line={line_no} | chars={len(source):,}"
    )
    print("=" * 120)

    print(source[:2000])

    if len(source) > 2000:
        print("\n... [preview truncated; full source is saved] ...")


# Pick the largest matching cell: architecture/training cell should dominate
# any tiny helper/redefinition cell.
session, line_no, recovered_source = max(
    matches,
    key=lambda x: len(x[2]),
)


OUT = Path(
    "/content/drive/MyDrive/TransWave_Meyer_Context/"
    "RECOVERED_EXACT_TRANSWAVE_V4_CELL.py"
)

OUT.write_text(
    recovered_source,
    encoding="utf-8",
)


print()
print("=" * 120)
print("EXACT V4 SOURCE RECOVERED")
print(f"history line : {line_no}")
print(f"characters   : {len(recovered_source):,}")
print(f"saved to     : {OUT}")
print("=" * 120)

Recovered 3 candidate exact V4 source cell(s).

CANDIDATE 1 | session=0 | history line=15 | chars=73,712
# =============================================================================
# CELL 4 V4 — SELECTIVE TRANSWAVE
#
# DO NOT RERUN CELL 3.
#
# NO OFFLINE TOKENIZATION.
# NO BPE.
# NO SentencePiece.
# NO NEW SHARDS.
#
# Existing raw UTF-8 bytes are read directly.
# The Cell-3 learned lifting tokenizer is trained jointly inside the network.
#
# V4 changes:
#   1. individual-byte exact 15% MLM
#   2. BERT 80% MASK / 10% random / 10% unchanged corruption
#   3. content-selective Meyer input gate
#   4. local depthwise convolution
#   5. 12 Meyer bands x 8 smooth within-band controls = 96 controls
#   6. full-rank 8x8 cross-scale mixing
#   7. content-selective Meyer output gate
#   8. SwiGLU FFN
#   9. B=32, 3 passes = ~1560 optimizer updates
#  10. validate after every pass
#
# Target:
#       validation NLL <= 2.30
#       validation PPL <= ~10
# ======================================

In [28]:
# =============================================================================
# CAUSAL TRANSWAVE — COMPLETE REPLACEMENT CELL
#
# Fresh experiment. No offline tokenization, BPE, SentencePiece, MLM or masking.
# Exact autoregressive byte likelihood:
#       NLL = -mean log p(x_t | x_<t)
#       PPL = exp(NLL)
#       BPB = NLL / ln(2)
#
# Causality:
#   * lifting is performed independently inside each completed 8-byte block;
#   * global Meyer mixing is one-sided causal linear convolution;
#   * local convolution is left-causal;
#   * the global block representation is shifted by one block;
#   * the 8 bytes of the current block are decoded autoregressively.
#
# G4:
#   * BF16 network, FP32/complex64 FFT path;
#   * activation checkpointing ON;
#   * batch probe includes forward + backward + grad clip + actual AdamW.step();
#   * choose the largest safe batch with 3 GiB reserved-memory headroom.
#
# Data:
#   existing /content/drive/MyDrive/spliNet/raw_text/<dataset>/*.txt only.
#   No download. No token shards. No re-tokenization.
#
# Runs independent models on:
#   arXiv, FineWeb-Edu, PG19, WikiText103
#
# Maximum exposure: 3 passes.
# Stop only if genuine validation PPL <= 3 (NLL <= ln 3).
# =============================================================================

from __future__ import annotations

import os
import gc
import json
import math
import time
import random
import shutil
import hashlib
from pathlib import Path

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.checkpoint import checkpoint as torch_checkpoint


# =============================================================================
# HARDWARE
# =============================================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU required."
    )


DEVICE = torch.device(
    "cuda"
)


AMP_DTYPE = torch.bfloat16


torch.backends.cuda.matmul.allow_tf32 = True

torch.backends.cudnn.allow_tf32 = True


try:

    torch.set_float32_matmul_precision(
        "high"
    )

except Exception:

    pass


GPU_NAME = torch.cuda.get_device_name(
    0
)


GPU_TOTAL_GB = (
    torch.cuda
    .get_device_properties(
        0
    )
    .total_memory
    /
    1024**3
)


# =============================================================================
# FIXED ARCHITECTURE
# =============================================================================

ARCH = (
    "causal_transwave_blocklifting_meyer_byte_lm_g4_v2"
)


BYTE_VOCAB = 256

BYTE_DIM = 96

BYTES_PER_BLOCK = 8

N_SCALES = 8

MODEL_DIM = 768

MODEL_CONTEXT = 4096


RAW_BYTES_PER_SEQUENCE = (
    MODEL_CONTEXT
    *
    BYTES_PER_BLOCK
)


N_LAYERS = 12

N_GROUPS = 12


CHANNELS_PER_GROUP = (
    BYTE_DIM
    //
    N_GROUPS
)


N_MEYER_BANDS = 12

N_BAND_CONTROLS = 8


# Causal kernel in eight-byte block positions.
MEYER_KERNEL = 512


LOCAL_KERNEL = 7


# Keep V4 capacity initially.
FFN_DIM = 2816


BYTE_DECODER_DIM = 256


HEAD_CHUNK = 32768


assert (
    BYTE_DIM
    *
    BYTES_PER_BLOCK
    ==
    MODEL_DIM
)


assert (
    RAW_BYTES_PER_SEQUENCE
    ==
    32768
)


assert (
    BYTE_DIM
    %
    N_GROUPS
    ==
    0
)


# =============================================================================
# TRAINING
# =============================================================================

MAX_PASSES = 3


BASE_LR = 1.0e-4

MIN_LR = 1.0e-5


WEIGHT_DECAY = 0.0


ADAM_BETAS = (
    0.9,
    0.999,
)


ADAM_EPS = 1.0e-8


GRAD_CLIP = 1.0


TARGET_PPL = 3.0


TARGET_NLL = math.log(
    TARGET_PPL
)


VALIDATE_EVERY_STEPS = 50

SAVE_EVERY_STEPS = 25


# Probe includes actual AdamW state.
VRAM_HEADROOM_GB = 3.0


BATCH_CEILING = 512


MODEL_SEED = 20261010


ORDER_SEEDS = {

    "arxiv":
        8101,

    "fineweb_edu":
        8201,

    "pg19":
        8301,

    "wikitext103":
        8401,
}


DATASETS = (
    "arxiv",
    "fineweb_edu",
    "pg19",
    "wikitext103",
)


DISPLAY_NAMES = {

    "arxiv":
        "arXiv",

    "fineweb_edu":
        "FineWeb-Edu",

    "pg19":
        "PG19",

    "wikitext103":
        "WikiText103",
}


# =============================================================================
# PATHS
# =============================================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)


RAW_ROOT = (
    DRIVE_ROOT
    /
    "spliNet"
    /
    "raw_text"
)


PROJECT_ROOT = (
    DRIVE_ROOT
    /
    "TransWave_Meyer_Context"
)


CHECKPOINT_ROOT = (
    PROJECT_ROOT
    /
    "checkpoints"
    /
    ARCH
)


RESULT_ROOT = (
    PROJECT_ROOT
    /
    "results"
    /
    ARCH
)


LOCAL_RAW_ROOT = Path(
    "/content/causal_transwave_raw"
)


for p in (

    CHECKPOINT_ROOT,
    RESULT_ROOT,
    LOCAL_RAW_ROOT,

):

    p.mkdir(
        parents=True,
        exist_ok=True,
    )


CONFIG = {

    "architecture":
        ARCH,

    "objective":
        "exact_causal_next_byte_nll",

    "byte_vocab":
        BYTE_VOCAB,

    "byte_dim":
        BYTE_DIM,

    "bytes_per_block":
        BYTES_PER_BLOCK,

    "model_dim":
        MODEL_DIM,

    "model_context":
        MODEL_CONTEXT,

    "raw_bytes_per_sequence":
        RAW_BYTES_PER_SEQUENCE,

    "layers":
        N_LAYERS,

    "groups":
        N_GROUPS,

    "meyer_bands":
        N_MEYER_BANDS,

    "band_controls":
        N_BAND_CONTROLS,

    "meyer_kernel":
        MEYER_KERNEL,

    "local_kernel":
        LOCAL_KERNEL,

    "ffn_dim":
        FFN_DIM,

    "byte_decoder_dim":
        BYTE_DECODER_DIM,

    "maximum_passes":
        MAX_PASSES,
}


CONFIG_FINGERPRINT = hashlib.sha256(

    json.dumps(
        CONFIG,
        sort_keys=True,
    ).encode()

).hexdigest()


# =============================================================================
# UTILITIES
# =============================================================================

def cleanup_cuda():

    gc.collect()


    try:

        torch.cuda.synchronize()

    except Exception:

        pass


    torch.cuda.empty_cache()


    try:

        torch.cuda.ipc_collect()

    except Exception:

        pass


def seed_all(
    seed,
):

    random.seed(
        seed
    )


    np.random.seed(
        seed
    )


    torch.manual_seed(
        seed
    )


    torch.cuda.manual_seed_all(
        seed
    )


def atomic_torch_save(
    obj,
    path,
):

    path = Path(
        path
    )


    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )


    tmp.unlink(
        missing_ok=True
    )


    torch.save(
        obj,
        tmp,
    )


    os.replace(
        tmp,
        path,
    )


def atomic_json(
    obj,
    path,
):

    path = Path(
        path
    )


    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    tmp = path.with_suffix(
        path.suffix
        +
        ".tmp"
    )


    tmp.unlink(
        missing_ok=True
    )


    tmp.write_text(

        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
        ),

        encoding="utf-8",
    )


    os.replace(
        tmp,
        path,
    )


# =============================================================================
# RAW BYTE DATA
# =============================================================================

def stage_raw_file(
    dataset,
    split,
):

    src = (
        RAW_ROOT
        /
        dataset
        /
        f"{split}.txt"
    )


    if not src.is_file():

        raise FileNotFoundError(

            "Existing raw file missing; "
            "this cell will not download/tokenize:\n"
            f"{src}"
        )


    dst_dir = (
        LOCAL_RAW_ROOT
        /
        dataset
    )


    dst_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    dst = (
        dst_dir
        /
        f"{split}.txt"
    )


    if (
        dst.is_file()
        and
        dst.stat().st_size
        ==
        src.stat().st_size
    ):

        print(

            f"  {dataset}/{split}: "
            f"local raw copy present "
            f"({dst.stat().st_size / 1024**2:.1f} MiB)"
        )


        return dst


    print(

        f"  {dataset}/{split}: "
        f"copying raw bytes to local SSD "
        f"({src.stat().st_size / 1024**2:.1f} MiB)"
    )


    tmp = dst.with_suffix(
        ".txt.tmp"
    )


    tmp.unlink(
        missing_ok=True
    )


    shutil.copyfile(
        src,
        tmp,
    )


    if (
        tmp.stat().st_size
        !=
        src.stat().st_size
    ):

        tmp.unlink(
            missing_ok=True
        )


        raise RuntimeError(
            f"Incomplete raw copy for "
            f"{dataset}/{split}"
        )


    os.replace(
        tmp,
        dst,
    )


    return dst


def open_blocks(
    path,
):

    raw = np.memmap(
        path,
        dtype=np.uint8,
        mode="r",
    )


    usable = (
        len(
            raw
        )
        //
        RAW_BYTES_PER_SEQUENCE
        *
        RAW_BYTES_PER_SEQUENCE
    )


    blocks = (
        raw[
            :usable
        ]
        .reshape(
            -1,
            RAW_BYTES_PER_SEQUENCE,
        )
    )


    return (
        raw,
        blocks,
    )


def make_cpu_batch(
    blocks,
    indices,
):

    arr = np.asarray(
        blocks[
            indices
        ],
        dtype=np.uint8,
    )


    arr = np.ascontiguousarray(
        arr
    )


    return (
        torch
        .from_numpy(
            arr
        )
        .pin_memory()
    )


# =============================================================================
# CAUSAL BLOCK-LOCAL LEARNED LIFTING
#
# Same Cell-3 LocalBlockDelta architecture, but applied independently to each
# completed eight-byte block. Therefore no convolution can cross from block b
# into block b+1.
# =============================================================================

class CTWLocalBlockDelta(
    nn.Module
):

    def __init__(
        self,
        dim=BYTE_DIM,
        groups=12,
        expansion=2,
    ):

        super().__init__()


        if (
            dim
            %
            groups
            !=
            0
        ):

            raise ValueError(
                "dim must be divisible by groups"
            )


        hidden = (
            dim
            *
            expansion
        )


        if (
            hidden
            %
            groups
            !=
            0
        ):

            raise ValueError(
                "expanded dimension must be divisible by groups"
            )


        self.depthwise = nn.Conv1d(
            dim,
            dim,
            kernel_size=3,
            padding=1,
            groups=dim,
            bias=True,
        )


        self.mix_in = nn.Conv1d(
            dim,
            hidden,
            kernel_size=1,
            groups=groups,
            bias=True,
        )


        self.mix_out = nn.Conv1d(
            hidden,
            dim,
            kernel_size=1,
            groups=groups,
            bias=True,
        )


        # Haar-like initialization:
        # residual branch starts exactly at zero.

        nn.init.zeros_(
            self.mix_out.weight
        )


        nn.init.zeros_(
            self.mix_out.bias
        )


    def forward(
        self,
        x,
    ):

        y = x.transpose(
            1,
            2,
        )


        y = self.depthwise(
            y
        )


        y = F.silu(
            y
        )


        y = self.mix_in(
            y
        )


        y = F.silu(
            y
        )


        y = self.mix_out(
            y
        )


        return y.transpose(
            1,
            2,
        )


class CTWLiftingStage(
    nn.Module
):

    def __init__(
        self,
        dim=BYTE_DIM,
        groups=12,
        expansion=2,
    ):

        super().__init__()


        self.predict_delta = (
            CTWLocalBlockDelta(
                dim=dim,
                groups=groups,
                expansion=expansion,
            )
        )


        self.update_delta = (
            CTWLocalBlockDelta(
                dim=dim,
                groups=groups,
                expansion=expansion,
            )
        )


    def predict(
        self,
        even,
    ):

        return (
            even
            +
            self.predict_delta(
                even
            )
        )


    def update(
        self,
        detail,
    ):

        return (
            0.5
            *
            detail
            +
            self.update_delta(
                detail
            )
        )


    def forward(
        self,
        x,
    ):

        if (
            x.ndim != 3
            or
            x.shape[1] % 2 != 0
        ):

            raise ValueError(
                "Expected [N,T,C] with even T."
            )


        even = x[
            :,
            0::2,
            :
        ]


        odd = x[
            :,
            1::2,
            :
        ]


        detail = (
            odd
            -
            self.predict(
                even
            )
        )


        coarse = (
            even
            +
            self.update(
                detail
            )
        )


        return (
            coarse,
            detail,
        )


class CTWCrossScaleMixer(
    nn.Module
):

    def __init__(
        self,
        scales=N_SCALES,
        dim=BYTE_DIM,
        groups=N_GROUPS,
    ):

        super().__init__()


        if (
            dim
            %
            groups
            !=
            0
        ):

            raise ValueError(
                "dim must be divisible by groups"
            )


        self.scales = int(
            scales
        )


        self.dim = int(
            dim
        )


        self.groups = int(
            groups
        )


        self.channels_per_group = (
            self.dim
            //
            self.groups
        )


        # M_g = I + Delta_g

        self.scale_delta = nn.Parameter(

            torch.zeros(
                self.groups,
                self.scales,
                self.scales,
            )
        )


        # Content-dependent gain.

        self.gate_weight = nn.Parameter(

            torch.zeros(
                self.groups,
                self.scales,
                self.channels_per_group,
            )
        )


        self.gate_bias = nn.Parameter(

            torch.zeros(
                self.groups,
                self.scales,
            )
        )


    def forward(
        self,
        z,
    ):

        if (
            z.ndim != 4
            or
            z.shape[2]
            !=
            self.scales
            or
            z.shape[3]
            !=
            self.dim
        ):

            raise ValueError(

                f"Expected "
                f"[B,L,{self.scales},{self.dim}]"
            )


        b, l, s, _ = z.shape


        zg = z.reshape(
            b,
            l,
            s,
            self.groups,
            self.channels_per_group,
        )


        gate_logits = torch.einsum(
            "blsgc,gsc->blsg",
            zg,
            self.gate_weight,
        )


        gate_logits = (
            gate_logits
            /
            math.sqrt(
                self.channels_per_group
            )
            +
            self.gate_bias.transpose(
                0,
                1,
            )
        )


        # Zero logits -> gain exactly 1.

        gain = (
            2.0
            *
            torch.sigmoid(
                gate_logits
            )
        )


        zg = (
            zg
            *
            gain
            .to(
                zg.dtype
            )
            .unsqueeze(
                -1
            )
        )


        eye = torch.eye(
            self.scales,
            device=z.device,
            dtype=z.dtype,
        ).unsqueeze(
            0
        )


        mix = (
            eye
            +
            self.scale_delta.to(
                z.dtype
            )
        )


        mixed = torch.einsum(
            "blsgc,gts->bltgc",
            zg,
            mix,
        )


        return mixed.reshape(
            b,
            l,
            s,
            self.dim,
        )


class CTWBlockLiftingTokenizer(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()


        self.byte_embedding = nn.Embedding(
            BYTE_VOCAB,
            BYTE_DIM,
        )


        nn.init.normal_(

            self.byte_embedding.weight,

            mean=0.0,

            std=(
                BYTE_DIM
                **
                -0.5
            ),
        )


        self.stages = nn.ModuleList(

            [

                CTWLiftingStage(
                    dim=BYTE_DIM,
                    groups=N_GROUPS,
                    expansion=2,
                )

                for _ in range(
                    3
                )
            ]
        )


        self.cross_scale = (
            CTWCrossScaleMixer(
                scales=N_SCALES,
                dim=BYTE_DIM,
                groups=N_GROUPS,
            )
        )


    def forward(
        self,
        byte_ids,
    ):

        if (
            byte_ids.ndim
            !=
            2
        ):

            raise ValueError(
                "Expected byte IDs [B,8L]."
            )


        b, t = byte_ids.shape


        if (
            t
            %
            BYTES_PER_BLOCK
            !=
            0
        ):

            raise ValueError(
                "Byte length must be divisible by 8."
            )


        l = (
            t
            //
            BYTES_PER_BLOCK
        )


        # -------------------------------------------------------------
        # CRITICAL CAUSAL CHANGE:
        #
        # each completed eight-byte block becomes an independent item
        # before ANY lifting convolution is applied.
        # -------------------------------------------------------------

        blocks = (
            byte_ids
            .long()
            .reshape(
                b,
                l,
                BYTES_PER_BLOCK,
            )
        )


        x = self.byte_embedding(

            blocks.reshape(
                b
                *
                l,
                BYTES_PER_BLOCK,
            )
        )


        details = []


        coarse = x


        for stage in self.stages:

            (
                coarse,
                detail,
            ) = stage(
                coarse
            )


            details.append(
                detail
            )


        # D0=4, D1=2, D2=1, S3=1.

        packed = torch.cat(

            [
                details[0],
                details[1],
                details[2],
                coarse,
            ],

            dim=1,
        )


        if (
            packed.shape[1]
            !=
            N_SCALES
        ):

            raise RuntimeError(
                "Lifting scale packing mismatch."
            )


        z = packed.reshape(
            b,
            l,
            N_SCALES,
            BYTE_DIM,
        )


        return self.cross_scale(
            z
        )


# =============================================================================
# RMS NORM / SWIGLU
# =============================================================================

class CTWRMSNorm(
    nn.Module
):

    def __init__(
        self,
        dim,
        eps=1e-6,
    ):

        super().__init__()


        self.weight = nn.Parameter(
            torch.ones(
                dim
            )
        )


        self.eps = eps


    def forward(
        self,
        x,
    ):

        xf = x.float()


        y = (
            xf
            *
            torch.rsqrt(

                xf.pow(2).mean(
                    dim=-1,
                    keepdim=True,
                )
                +
                self.eps
            )
        )


        return (
            y.to(
                x.dtype
            )
            *
            self.weight.to(
                x.dtype
            )
        )


class CTWSwiGLU(
    nn.Module
):

    def __init__(
        self,
        dim,
        hidden,
    ):

        super().__init__()


        self.gate = nn.Linear(
            dim,
            hidden,
        )


        self.value = nn.Linear(
            dim,
            hidden,
        )


        self.out = nn.Linear(
            hidden,
            dim,
        )


        for m in (
            self.gate,
            self.value,
            self.out,
        ):

            nn.init.xavier_uniform_(
                m.weight
            )


            nn.init.zeros_(
                m.bias
            )


    def forward(
        self,
        x,
    ):

        return self.out(

            F.silu(
                self.gate(
                    x
                )
            )

            *

            self.value(
                x
            )
        )


# =============================================================================
# MEYER BASIS
# =============================================================================

def ctw_meyer_nu(
    x,
):

    x = x.clamp(
        0.0,
        1.0,
    )


    return (
        x**4
        *
        (
            35.0
            -
            84.0 * x
            +
            70.0 * x**2
            -
            20.0 * x**3
        )
    )


def ctw_build_meyer_basis(
    kernel_size,
    bands,
    controls,
):

    nf = (
        kernel_size
        //
        2
        +
        1
    )


    f = torch.linspace(
        0.0,
        1.0,
        nf,
        dtype=torch.float32,
    )


    edges = torch.linspace(
        0.0,
        1.0,
        bands + 1,
        dtype=torch.float32,
    )


    windows = []


    for j in range(
        bands
    ):

        left = float(
            edges[j]
        )


        right = float(
            edges[j + 1]
        )


        delta = (
            0.35
            *
            (
                right
                -
                left
            )
        )


        if j == 0:

            left_gate = torch.ones_like(
                f
            )

        else:

            a = (
                left
                -
                delta
            )


            b = (
                left
                +
                delta
            )


            u = (
                f
                -
                a
            ) / max(
                b - a,
                1e-8,
            )


            left_gate = torch.sin(

                0.5
                *
                math.pi
                *
                ctw_meyer_nu(
                    u
                )
            )


            left_gate = torch.where(
                f <= a,
                torch.zeros_like(
                    f
                ),
                left_gate,
            )


            left_gate = torch.where(
                f >= b,
                torch.ones_like(
                    f
                ),
                left_gate,
            )


        if j == bands - 1:

            right_gate = torch.ones_like(
                f
            )

        else:

            a = (
                right
                -
                delta
            )


            b = (
                right
                +
                delta
            )


            u = (
                f
                -
                a
            ) / max(
                b - a,
                1e-8,
            )


            right_gate = torch.cos(

                0.5
                *
                math.pi
                *
                ctw_meyer_nu(
                    u
                )
            )


            right_gate = torch.where(
                f <= a,
                torch.ones_like(
                    f
                ),
                right_gate,
            )


            right_gate = torch.where(
                f >= b,
                torch.zeros_like(
                    f
                ),
                right_gate,
            )


        windows.append(
            left_gate
            *
            right_gate
        )


    windows = torch.stack(
        windows,
        dim=0,
    )


    windows = (
        windows
        /
        torch.sqrt(

            windows.pow(2).sum(
                dim=0,
                keepdim=True,
            )
            +
            1e-12
        )
    )


    degree = (
        controls
        -
        1
    )


    basis = []


    for j in range(
        bands
    ):

        left = edges[j]


        right = edges[j + 1]


        u = (
            (
                f
                -
                left
            )
            /
            (
                right
                -
                left
                +
                1e-12
            )
        ).clamp(
            0.0,
            1.0,
        )


        row = []


        for m in range(
            controls
        ):

            bern = (

                math.comb(
                    degree,
                    m,
                )

                *

                u.pow(
                    m
                )

                *

                (
                    1.0
                    -
                    u
                ).pow(
                    degree
                    -
                    m
                )
            )


            row.append(
                windows[j]
                *
                bern
            )


        basis.append(

            torch.stack(
                row,
                dim=0,
            )
        )


    return torch.stack(
        basis,
        dim=0,
    )


# =============================================================================
# SAME-TIME FULL 8x8 SCALE MIXING
# =============================================================================

class CTWScaleMixer(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()


        eye = torch.eye(
            N_SCALES
        )


        self.weight = nn.Parameter(

            eye
            .unsqueeze(
                0
            )
            .repeat(
                N_GROUPS,
                1,
                1,
            )
        )


    def forward(
        self,
        x,
    ):

        b, l, s, c = x.shape


        xg = x.reshape(
            b,
            l,
            s,
            N_GROUPS,
            CHANNELS_PER_GROUP,
        )


        y = torch.einsum(
            "blsgc,gts->bltgc",
            xg,
            self.weight,
        )


        return y.reshape(
            b,
            l,
            s,
            c,
        )


# =============================================================================
# STRICTLY CAUSAL MEYER FIR
#
# h[k] is a nonnegative lag:
#
# y[t] = SUM_{k=0}^{K-1} h[k] x[t-k]
#
# FFT only implements this zero-padded LINEAR convolution efficiently.
# =============================================================================

class CTWCausalMeyerFIR(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()


        basis = ctw_build_meyer_basis(
            MEYER_KERNEL,
            N_MEYER_BANDS,
            N_BAND_CONTROLS,
        )


        self.register_buffer(
            "basis",
            basis,
            persistent=False,
        )


        imag_mask = torch.ones(
            MEYER_KERNEL
            //
            2
            +
            1,
            dtype=torch.float32,
        )


        imag_mask[0] = 0.0


        if (
            MEYER_KERNEL
            %
            2
            ==
            0
        ):

            imag_mask[-1] = 0.0


        self.register_buffer(
            "imag_mask",
            imag_mask,
            persistent=False,
        )


        shape = (
            N_GROUPS,
            N_SCALES,
            N_MEYER_BANDS,
            N_BAND_CONTROLS,
        )


        self.coeff_re = nn.Parameter(
            torch.empty(
                shape
            )
        )


        self.coeff_im = nn.Parameter(
            torch.empty(
                shape
            )
        )


        nn.init.normal_(
            self.coeff_re,
            mean=0.0,
            std=1e-3,
        )


        nn.init.normal_(
            self.coeff_im,
            mean=0.0,
            std=1e-3,
        )


    def causal_kernel(
        self,
    ):

        with torch.autocast(
            "cuda",
            enabled=False,
        ):

            basis = self.basis.float()


            re = torch.einsum(
                "gsbm,bmf->gsf",
                self.coeff_re.float(),
                basis,
            )


            im = torch.einsum(
                "gsbm,bmf->gsf",
                self.coeff_im.float(),
                basis,
            )


            im = (
                im
                *
                self.imag_mask[
                    None,
                    None,
                    :
                ]
            )


            spectrum = torch.complex(
                re,
                im,
            )


            h = torch.fft.irfft(
                spectrum,
                n=MEYER_KERNEL,
                dim=-1,
            )


        return h


    def forward(
        self,
        x,
    ):

        # [B,L,S,96]

        b, l, s, c = x.shape


        # [B,G,S,Cg,L]

        xg = (
            x
            .reshape(
                b,
                l,
                s,
                N_GROUPS,
                CHANNELS_PER_GROUP,
            )
            .permute(
                0,
                3,
                2,
                4,
                1,
            )
            .contiguous()
        )


        linear_length = (
            l
            +
            MEYER_KERNEL
            -
            1
        )


        nfft = (
            1
            <<
            (
                linear_length
                -
                1
            ).bit_length()
        )


        with torch.autocast(
            "cuda",
            enabled=False,
        ):

            xf = xg.float()


            h = self.causal_kernel()


            X = torch.fft.rfft(
                xf,
                n=nfft,
                dim=-1,
            )


            H = torch.fft.rfft(
                h,
                n=nfft,
                dim=-1,
            )


            Y = torch.fft.irfft(

                X
                *
                H[
                    None,
                    :,
                    :,
                    None,
                    :
                ],

                n=nfft,
                dim=-1,
            )


            # First L outputs of LINEAR convolution.

            Y = Y[
                ...,
                :l
            ]


        return (

            Y
            .permute(
                0,
                4,
                2,
                1,
                3,
            )
            .contiguous()
            .reshape(
                b,
                l,
                s,
                c,
            )
            .to(
                x.dtype
            )
        )


# =============================================================================
# STRICTLY LEFT-CAUSAL LOCAL CONVOLUTION
# =============================================================================

class CTWCausalLocalConv(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()


        self.conv = nn.Conv1d(
            MODEL_DIM,
            MODEL_DIM,
            kernel_size=LOCAL_KERNEL,
            groups=MODEL_DIM,
            bias=True,
        )


        nn.init.normal_(
            self.conv.weight,
            mean=0.0,
            std=0.02,
        )


        nn.init.zeros_(
            self.conv.bias
        )


    def forward(
        self,
        x,
    ):

        xt = x.transpose(
            1,
            2,
        )


        xt = F.pad(
            xt,
            (
                LOCAL_KERNEL
                -
                1,
                0,
            ),
        )


        return (
            self.conv(
                xt
            )
            .transpose(
                1,
                2,
            )
        )


# =============================================================================
# ONE CAUSAL TRANSWAVE LAYER
# =============================================================================

class CTWLayer(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()


        self.norm1 = CTWRMSNorm(
            MODEL_DIM
        )


        self.input_gate = nn.Linear(
            MODEL_DIM,
            MODEL_DIM,
        )


        self.output_gate = nn.Linear(
            MODEL_DIM,
            MODEL_DIM,
        )


        nn.init.zeros_(
            self.input_gate.weight
        )


        nn.init.constant_(
            self.input_gate.bias,
            2.0,
        )


        nn.init.zeros_(
            self.output_gate.weight
        )


        nn.init.constant_(
            self.output_gate.bias,
            2.0,
        )


        self.scale_in = (
            CTWScaleMixer()
        )


        self.meyer = (
            CTWCausalMeyerFIR()
        )


        self.scale_out = (
            CTWScaleMixer()
        )


        self.local = (
            CTWCausalLocalConv()
        )


        self.alpha_meyer = nn.Parameter(

            torch.tensor(
                0.10,
                dtype=torch.float32,
            )
        )


        self.alpha_local = nn.Parameter(

            torch.tensor(
                0.10,
                dtype=torch.float32,
            )
        )


        self.norm2 = CTWRMSNorm(
            MODEL_DIM
        )


        self.ffn = CTWSwiGLU(
            MODEL_DIM,
            FFN_DIM,
        )


    def forward(
        self,
        x,
    ):

        b, l, s, c = x.shape


        flat = x.reshape(
            b,
            l,
            MODEL_DIM,
        )


        n = self.norm1(
            flat
        )


        gate_in = torch.sigmoid(

            self.input_gate(
                n
            )

        ).reshape(
            b,
            l,
            s,
            c,
        )


        gate_out = torch.sigmoid(

            self.output_gate(
                n
            )

        ).reshape(
            b,
            l,
            s,
            c,
        )


        spectral = self.meyer(

            self.scale_in(
                x
                *
                gate_in
            )
        )


        spectral = self.scale_out(
            spectral
        )


        local = self.local(
            n
        ).reshape(
            b,
            l,
            s,
            c,
        )


        x = (

            x

            +

            self.alpha_meyer.to(
                x.dtype
            )
            *
            spectral
            *
            gate_out

            +

            self.alpha_local.to(
                x.dtype
            )
            *
            local
        )


        flat = x.reshape(
            b,
            l,
            MODEL_DIM,
        )


        flat = (

            flat

            +

            self.ffn(
                self.norm2(
                    flat
                )
            )
        )


        return flat.reshape(
            b,
            l,
            s,
            c,
        )


# =============================================================================
# EXACT EIGHT-BYTE AUTOREGRESSIVE DECODER
# =============================================================================

class CTWByteDecoder(
    nn.Module
):

    def __init__(
        self,
        shared_embedding,
    ):

        super().__init__()


        self.shared_embedding = (
            shared_embedding
        )


        self.context_proj = nn.Linear(
            MODEL_DIM,
            BYTE_DECODER_DIM,
        )


        self.gru = nn.GRUCell(
            BYTE_DIM,
            BYTE_DECODER_DIM,
        )


        self.norm = CTWRMSNorm(
            BYTE_DECODER_DIM
        )


        self.to_byte_space = nn.Linear(
            BYTE_DECODER_DIM,
            BYTE_DIM,
        )


        self.bos = nn.Parameter(
            torch.zeros(
                BYTE_DIM
            )
        )


        self.byte_position = nn.Parameter(

            torch.zeros(
                BYTES_PER_BLOCK,
                BYTE_DIM,
            )
        )


        nn.init.normal_(
            self.bos,
            mean=0.0,
            std=0.02,
        )


        nn.init.normal_(
            self.byte_position,
            mean=0.0,
            std=0.01,
        )


    def nll(
        self,
        context,
        target_blocks,
    ):

        b, l, _ = context.shape


        n = (
            b
            *
            l
        )


        hidden = torch.tanh(

            self.context_proj(
                context
            )

        ).reshape(
            n,
            BYTE_DECODER_DIM,
        )


        previous = (
            self.bos[
                None,
                :
            ]
            .expand(
                n,
                -1,
            )
        )


        total_loss = None


        for r in range(
            BYTES_PER_BLOCK
        ):

            decoder_input = (

                previous

                +

                self.byte_position[
                    r
                ][
                    None,
                    :
                ]
            )


            hidden = self.gru(
                decoder_input,
                hidden,
            )


            z = self.to_byte_space(
                self.norm(
                    hidden
                )
            )


            target = (
                target_blocks[
                    :,
                    :,
                    r
                ]
                .reshape(
                    -1
                )
                .long()
            )


            position_loss = None


            for start in range(
                0,
                n,
                HEAD_CHUNK,
            ):

                end = min(
                    start
                    +
                    HEAD_CHUNK,
                    n,
                )


                logits = F.linear(
                    z[
                        start:end
                    ],
                    self.shared_embedding.weight,
                )


                piece = F.cross_entropy(
                    logits.float(),
                    target[
                        start:end
                    ],
                    reduction="sum",
                )


                position_loss = (
                    piece
                    if position_loss is None
                    else
                    position_loss
                    +
                    piece
                )


            total_loss = (
                position_loss
                if total_loss is None
                else
                total_loss
                +
                position_loss
            )


            # Exact teacher forcing:
            # byte r+1 may condition on true byte r.

            previous = self.shared_embedding(
                target
            )


        return (
            total_loss
            /
            float(
                n
                *
                BYTES_PER_BLOCK
            )
        )


# =============================================================================
# COMPLETE CAUSAL MODEL
# =============================================================================

class CTWModel(
    nn.Module
):

    def __init__(
        self,
    ):

        super().__init__()


        self.tokenizer = (
            CTWBlockLiftingTokenizer()
        )


        self.position = nn.Parameter(

            torch.zeros(
                1,
                MODEL_CONTEXT,
                MODEL_DIM,
            )
        )


        nn.init.normal_(
            self.position,
            mean=0.0,
            std=0.01,
        )


        self.layers = nn.ModuleList(

            [

                CTWLayer()

                for _ in range(
                    N_LAYERS
                )
            ]
        )


        self.final_norm = CTWRMSNorm(
            MODEL_DIM
        )


        self.bos_context = nn.Parameter(
            torch.zeros(
                MODEL_DIM
            )
        )


        nn.init.normal_(
            self.bos_context,
            mean=0.0,
            std=0.02,
        )


        self.decoder = CTWByteDecoder(
            self.tokenizer.byte_embedding
        )


    def encode_completed_blocks(
        self,
        raw_bytes,
        checkpoint_layers,
    ):

        if (
            raw_bytes.ndim
            !=
            2
        ):

            raise ValueError(
                "Expected raw byte IDs [B,T]."
            )


        b, t = raw_bytes.shape


        if (
            t
            %
            BYTES_PER_BLOCK
            !=
            0
        ):

            raise ValueError(
                "Raw byte length must be divisible by 8."
            )


        l = (
            t
            //
            BYTES_PER_BLOCK
        )


        if (
            l
            >
            MODEL_CONTEXT
        ):

            raise ValueError(

                f"Context {l} exceeds "
                f"MODEL_CONTEXT={MODEL_CONTEXT}."
            )


        x = self.tokenizer(
            raw_bytes.long()
        )


        pos = (
            self.position[
                :,
                :l,
                :
            ]
            .reshape(
                1,
                l,
                N_SCALES,
                BYTE_DIM,
            )
        )


        x = (
            x
            +
            pos.to(
                x.dtype
            )
        )


        for layer in self.layers:

            if (
                checkpoint_layers
                and
                self.training
            ):

                x = torch_checkpoint(
                    layer,
                    x,
                    use_reentrant=False,
                    preserve_rng_state=False,
                )

            else:

                x = layer(
                    x
                )


        h = self.final_norm(

            x.reshape(
                b,
                l,
                MODEL_DIM,
            )
        )


        blocks = (
            raw_bytes
            .long()
            .reshape(
                b,
                l,
                BYTES_PER_BLOCK,
            )
        )


        return (
            h,
            blocks,
        )


    def contexts(
        self,
        raw_bytes,
        checkpoint_layers=False,
    ):

        (
            h,
            blocks,
        ) = self.encode_completed_blocks(
            raw_bytes,
            checkpoint_layers,
        )


        b, l, _ = h.shape


        context = torch.empty_like(
            h
        )


        # First block has no preceding block.

        context[
            :,
            0,
            :
        ] = self.bos_context.to(
            h.dtype
        )


        # -------------------------------------------------------------
        # CRITICAL CAUSAL SHIFT
        #
        # prediction for block b receives h[b-1], never h[b].
        # -------------------------------------------------------------

        if l > 1:

            context[
                :,
                1:,
                :
            ] = h[
                :,
                :-1,
                :
            ]


        return (
            context,
            blocks,
        )


    def nll(
        self,
        raw_bytes,
        checkpoint_layers=True,
    ):

        (
            context,
            blocks,
        ) = self.contexts(
            raw_bytes,
            checkpoint_layers,
        )


        return self.decoder.nll(
            context,
            blocks,
        )


# =============================================================================
# PARAMETERS / OPTIMIZER
# =============================================================================

def count_parameters(
    model,
):

    total = sum(
        p.numel()
        for p in model.parameters()
    )


    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


    return (
        total,
        trainable,
    )


def make_optimizer(
    model,
):

    kwargs = dict(

        lr=BASE_LR,

        betas=ADAM_BETAS,

        eps=ADAM_EPS,

        weight_decay=WEIGHT_DECAY,
    )


    try:

        opt = torch.optim.AdamW(
            model.parameters(),
            fused=True,
            **kwargs,
        )


        return (
            opt,
            True,
        )


    except Exception:

        opt = torch.optim.AdamW(
            model.parameters(),
            **kwargs,
        )


        return (
            opt,
            False,
        )


# =============================================================================
# STRICT CAUSALITY TEST
# =============================================================================

@torch.no_grad()
def causality_test(
    model,
):

    print(
        "\nSTRICT CAUSALITY TEST"
    )


    model.eval()


    n_blocks = 64

    cut = 30


    a = torch.randint(
        0,
        BYTE_VOCAB,
        (
            1,
            n_blocks
            *
            BYTES_PER_BLOCK,
        ),
        device=DEVICE,
        dtype=torch.long,
    )


    b = a.clone()


    # Change block cut+1 and everything after it.

    mutation_start = (
        cut
        +
        1
    ) * BYTES_PER_BLOCK


    b[
        :,
        mutation_start:
    ] = torch.randint(
        0,
        BYTE_VOCAB,
        b[
            :,
            mutation_start:
        ].shape,
        device=DEVICE,
        dtype=torch.long,
    )


    ca, _ = model.contexts(
        a,
        checkpoint_layers=False,
    )


    cb, _ = model.contexts(
        b,
        checkpoint_layers=False,
    )


    # Context for block cut+1 must still be identical.

    diff = float(

        (
            ca[
                :,
                :
                cut
                +
                2,
                :
            ]
            -
            cb[
                :,
                :
                cut
                +
                2,
                :
            ]
        )
        .abs()
        .max()
    )


    print(

        f"max future->past difference : "
        f"{diff:.3e}"
    )


    if (
        diff
        >
        2e-4
    ):

        raise RuntimeError(
            "CAUSALITY TEST FAILED."
        )


    print(
        "causality                    : PASS"
    )


    model.train()


# =============================================================================
# COMPLETE-STEP G4 VRAM PROBE
# =============================================================================

def full_step_probe(
    batch,
):

    cleanup_cuda()


    seed_all(
        MODEL_SEED
    )


    model = (
        CTWModel()
        .to(
            DEVICE
        )
    )


    optimizer, fused = make_optimizer(
        model
    )


    model.train()


    torch.cuda.reset_peak_memory_stats()


    try:

        dummy = torch.randint(
            0,
            BYTE_VOCAB,
            (
                batch,
                RAW_BYTES_PER_SEQUENCE,
            ),
            device=DEVICE,
            dtype=torch.long,
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            "cuda",
            dtype=AMP_DTYPE,
        ):

            loss = model.nll(
                dummy,
                checkpoint_layers=True,
            )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP,
        )


        # Important:
        # allocate/exercise actual AdamW moments.

        optimizer.step()


        torch.cuda.synchronize()


        peak_alloc = (
            torch.cuda
            .max_memory_allocated()
            /
            1024**3
        )


        peak_reserved = (
            torch.cuda
            .max_memory_reserved()
            /
            1024**3
        )


        result = {

            "fits":
                True,

            "batch":
                batch,

            "peak_alloc":
                peak_alloc,

            "peak_reserved":
                peak_reserved,

            "free":
                GPU_TOTAL_GB
                -
                peak_reserved,

            "nll":
                float(
                    loss.detach()
                ),

            "fused":
                fused,
        }


        del (
            dummy,
            loss,
        )


    except torch.cuda.OutOfMemoryError:

        result = {

            "fits":
                False,

            "batch":
                batch,
        }


    except RuntimeError as exc:

        if (
            "out of memory"
            in
            str(
                exc
            ).lower()
        ):

            result = {

                "fits":
                    False,

                "batch":
                    batch,
            }

        else:

            raise


    finally:

        try:

            optimizer.zero_grad(
                set_to_none=True
            )

        except Exception:

            pass


        del (
            optimizer,
            model,
        )


        cleanup_cuda()


    return result


def choose_batch_size():

    print(
        "=" * 120
    )


    print(
        "G4 COMPLETE-STEP VRAM AUTOTUNER"
    )


    print(
        f"GPU                     : "
        f"{GPU_NAME}"
    )


    print(
        f"VRAM                    : "
        f"{GPU_TOTAL_GB:.2f} GiB"
    )


    print(
        f"required headroom       : "
        f"{VRAM_HEADROOM_GB:.2f} GiB"
    )


    print(
        "probe                   : "
        "forward + backward + AdamW.step()"
    )


    print(
        "activation checkpointing: ON"
    )


    print(
        "=" * 120
    )


    seed_all(
        MODEL_SEED
    )


    test_model = (
        CTWModel()
        .to(
            DEVICE
        )
    )


    (
        total,
        trainable,
    ) = count_parameters(
        test_model
    )


    print(
        f"parameters              : "
        f"{total:,}"
    )


    print(
        f"trainable               : "
        f"{trainable:,}"
    )


    causality_test(
        test_model
    )


    del test_model


    cleanup_cuda()


    coarse = (

        8,
        16,
        24,
        32,
        48,
        64,
        80,
        96,
        112,
        128,
        144,
        160,
        176,
        192,
        224,
        256,
        288,
        320,
        352,
        384,
        416,
        448,
        480,
        512,
    )


    best_safe = None

    first_unsafe = None


    for batch in coarse:

        if (
            batch
            >
            BATCH_CEILING
        ):

            break


        r = full_step_probe(
            batch
        )


        if not r[
            "fits"
        ]:

            print(
                f"B={batch:<3d} OOM"
            )


            first_unsafe = batch


            break


        print(

            f"B={batch:<3d} FIT | "

            f"alloc="
            f"{r['peak_alloc']:6.2f} GiB | "

            f"reserved="
            f"{r['peak_reserved']:6.2f} GiB | "

            f"free="
            f"{r['free']:6.2f} GiB | "

            f"NLL="
            f"{r['nll']:.4f}"
        )


        if (
            r[
                "free"
            ]
            >=
            VRAM_HEADROOM_GB
        ):

            best_safe = batch


        else:

            first_unsafe = batch


            print(

                f"B={batch}: "
                f"fits but violates "
                f"{VRAM_HEADROOM_GB:.1f} GiB headroom."
            )


            break


    if best_safe is None:

        for batch in (
            6,
            4,
            2,
            1,
        ):

            r = full_step_probe(
                batch
            )


            if not r[
                "fits"
            ]:

                print(
                    f"B={batch:<3d} OOM"
                )


                continue


            print(

                f"B={batch:<3d} FIT | "

                f"alloc="
                f"{r['peak_alloc']:6.2f} GiB | "

                f"reserved="
                f"{r['peak_reserved']:6.2f} GiB | "

                f"free="
                f"{r['free']:6.2f} GiB"
            )


            if (
                r[
                    "free"
                ]
                >=
                VRAM_HEADROOM_GB
            ):

                best_safe = batch


                break


    if best_safe is None:

        raise RuntimeError(
            "No batch fits a complete AdamW step."
        )


    # Refine between safe and unsafe in multiples of 4.

    if (
        first_unsafe is not None
        and
        first_unsafe
        >
        best_safe
        +
        4
    ):

        low = best_safe

        high = first_unsafe


        while (
            high
            -
            low
            >
            4
        ):

            mid = (
                (
                    low
                    +
                    high
                )
                //
                8
                *
                4
            )


            if mid <= low:

                mid = (
                    low
                    +
                    4
                )


            if mid >= high:

                break


            r = full_step_probe(
                mid
            )


            safe = (
                r[
                    "fits"
                ]
                and
                r[
                    "free"
                ]
                >=
                VRAM_HEADROOM_GB
            )


            if safe:

                print(

                    f"B={mid:<3d} SAFE | "

                    f"alloc="
                    f"{r['peak_alloc']:6.2f} GiB | "

                    f"reserved="
                    f"{r['peak_reserved']:6.2f} GiB | "

                    f"free="
                    f"{r['free']:6.2f} GiB"
                )


                low = mid


                best_safe = mid


            else:

                if r[
                    "fits"
                ]:

                    print(

                        f"B={mid:<3d} TOO TIGHT | "

                        f"reserved="
                        f"{r['peak_reserved']:6.2f} GiB | "

                        f"free="
                        f"{r['free']:6.2f} GiB"
                    )

                else:

                    print(
                        f"B={mid:<3d} OOM"
                    )


                high = mid


    print(
        "=" * 120
    )


    print(
        f"SELECTED PHYSICAL BATCH : "
        f"{best_safe}"
    )


    print(
        "selection includes actual "
        "AdamW state/step memory"
    )


    print(
        "=" * 120
    )


    return best_safe


# =============================================================================
# LEARNING RATE / DATA ORDER
# =============================================================================

def learning_rate(
    update,
    total_updates,
):

    warmup = max(
        10,
        int(
            0.01
            *
            total_updates
        ),
    )


    if update < warmup:

        return (
            BASE_LR
            *
            float(
                update
                +
                1
            )
            /
            float(
                warmup
            )
        )


    progress = (
        float(
            update
            -
            warmup
        )
        /
        float(
            max(
                1,
                total_updates
                -
                warmup,
            )
        )
    )


    progress = max(
        0.0,
        min(
            1.0,
            progress,
        ),
    )


    cosine = (
        0.5
        *
        (
            1.0
            +
            math.cos(
                math.pi
                *
                progress
            )
        )
    )


    return (
        MIN_LR
        +
        (
            BASE_LR
            -
            MIN_LR
        )
        *
        cosine
    )


def dataset_indices(
    n_train,
    examples_seen,
    batch_size,
    seed,
):

    cursor = int(
        examples_seen
    )


    remaining = int(
        batch_size
    )


    pieces = []


    while remaining > 0:

        pass_id = (
            cursor
            //
            n_train
        )


        offset = (
            cursor
            %
            n_train
        )


        permutation = (

            np.random
            .default_rng(
                seed
                +
                pass_id
            )
            .permutation(
                n_train
            )
        )


        take = min(
            remaining,
            n_train
            -
            offset,
        )


        pieces.append(

            permutation[
                offset:
                offset
                +
                take
            ]
        )


        cursor += take


        remaining -= take


    return np.concatenate(
        pieces
    )


# =============================================================================
# GENUINE CAUSAL VALIDATION
# =============================================================================

@torch.no_grad()
def validate_model(
    model,
    blocks,
    batch_size,
):

    model.eval()


    total_loss = 0.0

    total_bytes = 0


    for start in range(
        0,
        len(
            blocks
        ),
        batch_size,
    ):

        count = min(
            batch_size,
            len(
                blocks
            )
            -
            start,
        )


        indices = np.arange(
            start,
            start
            +
            count,
            dtype=np.int64,
        )


        cpu = make_cpu_batch(
            blocks,
            indices,
        )


        ids = cpu.to(
            DEVICE,
            dtype=torch.long,
            non_blocking=True,
        )


        with torch.autocast(
            "cuda",
            dtype=AMP_DTYPE,
        ):

            nll = model.nll(
                ids,
                checkpoint_layers=False,
            )


        nbytes = (
            count
            *
            RAW_BYTES_PER_SEQUENCE
        )


        total_loss += (
            float(
                nll.detach()
            )
            *
            nbytes
        )


        total_bytes += nbytes


        del (
            indices,
            cpu,
            ids,
            nll,
        )


    mean_nll = (
        total_loss
        /
        total_bytes
    )


    ppl = math.exp(
        min(
            mean_nll,
            20.0,
        )
    )


    bpb = (
        mean_nll
        /
        math.log(
            2.0
        )
    )


    model.train()


    return (
        mean_nll,
        ppl,
        bpb,
        total_bytes,
    )


# =============================================================================
# TRAIN ONE DATASET
# =============================================================================

def train_dataset(
    dataset,
    default_physical_batch,
):

    name = DISPLAY_NAMES[
        dataset
    ]


    print(
        "\n"
        +
        "#" * 120
    )


    print(
        f"# CAUSAL TRANSWAVE — {name}"
    )


    print(
        "#" * 120
    )


    train_path = stage_raw_file(
        dataset,
        "train",
    )


    valid_path = stage_raw_file(
        dataset,
        "validation",
    )


    (
        train_mem,
        train_blocks,
    ) = open_blocks(
        train_path
    )


    (
        valid_mem,
        valid_blocks,
    ) = open_blocks(
        valid_path
    )


    n_train = int(
        len(
            train_blocks
        )
    )


    n_valid = int(
        len(
            valid_blocks
        )
    )


    if (
        n_train < 1
        or
        n_valid < 1
    ):

        raise RuntimeError(
            f"{dataset}: empty train/validation stream."
        )


    run_root = (
        CHECKPOINT_ROOT
        /
        dataset
    )


    run_root.mkdir(
        parents=True,
        exist_ok=True,
    )


    latest_path = (
        run_root
        /
        "training_latest.pt"
    )


    best_path = (
        run_root
        /
        "best_model.pt"
    )


    final_path = (
        run_root
        /
        "trained_model_final.pt"
    )


    result_path = (
        RESULT_ROOT
        /
        f"{dataset}.json"
    )


    run_fingerprint = hashlib.sha256(

        (
            CONFIG_FINGERPRINT
            +
            "|"
            +
            dataset
        ).encode()

    ).hexdigest()


    # A completed 3-pass run is already complete,
    # regardless of whether target was reached.

    if result_path.is_file():

        try:

            old = json.loads(

                result_path.read_text(
                    encoding="utf-8"
                )
            )


            if (
                old.get(
                    "fingerprint"
                )
                ==
                run_fingerprint
                and
                old.get(
                    "status"
                )
                ==
                "complete"
            ):

                print(
                    f"{name}: "
                    f"completed result already exists."
                )


                print(

                    f"NLL="
                    f"{old['best_validation_nll']:.6f} | "

                    f"PPL="
                    f"{old['best_validation_ppl']:.6f} | "

                    f"BPB="
                    f"{old['best_validation_bpb']:.6f}"
                )


                del (
                    train_mem,
                    train_blocks,
                    valid_mem,
                    valid_blocks,
                )


                cleanup_cuda()


                return old


        except Exception:

            pass


    cleanup_cuda()


    seed_all(
        MODEL_SEED
    )


    model = (
        CTWModel()
        .to(
            DEVICE
        )
    )


    optimizer, fused_adam = (
        make_optimizer(
            model
        )
    )


    (
        total_params,
        trainable_params,
    ) = count_parameters(
        model
    )


    examples_seen = 0

    update = 0

    best_nll = float(
        "inf"
    )

    history = []

    prior_seconds = 0.0

    stop_reason = None


    physical_batch = int(
        default_physical_batch
    )


    # -------------------------------------------------------------------------
    # RESUME
    # -------------------------------------------------------------------------

    if latest_path.is_file():

        state = torch.load(
            latest_path,
            map_location="cpu",
            weights_only=False,
        )


        if (
            state.get(
                "fingerprint"
            )
            !=
            run_fingerprint
        ):

            raise RuntimeError(
                f"{dataset}: "
                f"checkpoint fingerprint mismatch."
            )


        model.load_state_dict(
            state[
                "model_state"
            ],
            strict=True,
        )


        optimizer.load_state_dict(
            state[
                "optimizer_state"
            ]
        )


        for item in optimizer.state.values():

            for key, value in list(
                item.items()
            ):

                if torch.is_tensor(
                    value
                ):

                    item[
                        key
                    ] = value.to(
                        DEVICE
                    )


        examples_seen = int(
            state.get(
                "examples_seen",
                0,
            )
        )


        update = int(
            state.get(
                "update",
                0,
            )
        )


        best_nll = float(
            state.get(
                "best_nll",
                float(
                    "inf"
                ),
            )
        )


        history = list(
            state.get(
                "history",
                [],
            )
        )


        prior_seconds = float(
            state.get(
                "training_seconds",
                0.0,
            )
        )


        physical_batch = int(
            state.get(
                "physical_batch",
                physical_batch,
            )
        )


        del state


        print(

            f"RESUME | "

            f"update="
            f"{update:,} | "

            f"examples="
            f"{examples_seen:,} | "

            f"B="
            f"{physical_batch}"
        )


    max_examples = (
        MAX_PASSES
        *
        n_train
    )


    max_updates = max(

        1,

        math.ceil(
            max_examples
            /
            physical_batch
        ),
    )


    print(
        f"train sequences          : "
        f"{n_train:,}"
    )


    print(
        f"validation sequences     : "
        f"{n_valid:,}"
    )


    print(
        f"bytes / sequence         : "
        f"{RAW_BYTES_PER_SEQUENCE:,}"
    )


    print(
        f"physical batch           : "
        f"{physical_batch}"
    )


    print(
        f"parameters               : "
        f"{total_params:,}"
    )


    print(
        f"trainable parameters     : "
        f"{trainable_params:,}"
    )


    print(
        f"fused AdamW              : "
        f"{fused_adam}"
    )


    print(
        f"maximum passes           : "
        f"{MAX_PASSES}"
    )


    print(
        f"maximum updates          : "
        f"{max_updates:,}"
    )


    session_start = (
        time.perf_counter()
    )


    def elapsed():

        return (
            prior_seconds
            +
            time.perf_counter()
            -
            session_start
        )


    def save_latest():

        atomic_torch_save(

            {
                "fingerprint":
                    run_fingerprint,

                "config":
                    CONFIG,

                "dataset":
                    dataset,

                "physical_batch":
                    physical_batch,

                "examples_seen":
                    examples_seen,

                "update":
                    update,

                "best_nll":
                    best_nll,

                "history":
                    history,

                "training_seconds":
                    elapsed(),

                "model_state":
                    model.state_dict(),

                "optimizer_state":
                    optimizer.state_dict(),
            },

            latest_path,
        )


    def save_best(
        val_nll,
        val_ppl,
        val_bpb,
    ):

        atomic_torch_save(

            {
                "fingerprint":
                    run_fingerprint,

                "config":
                    CONFIG,

                "dataset":
                    dataset,

                "physical_batch":
                    physical_batch,

                "examples_seen":
                    examples_seen,

                "update":
                    update,

                "validation_nll":
                    float(
                        val_nll
                    ),

                "validation_ppl":
                    float(
                        val_ppl
                    ),

                "validation_bpb":
                    float(
                        val_bpb
                    ),

                "model_state":
                    model.state_dict(),
            },

            best_path,
        )


    # -------------------------------------------------------------------------
    # IMMEDIATE VALIDATION AFTER RESUME
    # -------------------------------------------------------------------------

    if update > 0:

        print(

            f"\n{name}: "
            f"immediate genuine causal validation"
        )


        (
            val_nll,
            val_ppl,
            val_bpb,
            val_bytes,
        ) = validate_model(
            model,
            valid_blocks,
            physical_batch,
        )


        print(

            f"VALID | "

            f"NLL="
            f"{val_nll:.6f} | "

            f"PPL="
            f"{val_ppl:.6f} | "

            f"BPB="
            f"{val_bpb:.6f} | "

            f"bytes="
            f"{val_bytes:,}"
        )


        history.append(

            {
                "update":
                    update,

                "examples_seen":
                    examples_seen,

                "nll":
                    float(
                        val_nll
                    ),

                "ppl":
                    float(
                        val_ppl
                    ),

                "bpb":
                    float(
                        val_bpb
                    ),

                "reason":
                    "resume_validation",
            }
        )


        if val_nll < best_nll:

            best_nll = float(
                val_nll
            )


            save_best(
                val_nll,
                val_ppl,
                val_bpb,
            )


        if (
            val_nll
            <=
            TARGET_NLL
        ):

            stop_reason = (
                "validation_ppl_3_reached"
            )


    # -------------------------------------------------------------------------
    # TRAIN
    # -------------------------------------------------------------------------

    model.train()


    torch.cuda.reset_peak_memory_stats()


    last_train_nll = None

    last_train_ppl = None

    last_train_bpb = None


    last_completed_pass = (
        examples_seen
        //
        n_train
    )


    try:

        while (
            stop_reason is None
            and
            examples_seen
            <
            max_examples
        ):

            this_batch = min(
                physical_batch,
                max_examples
                -
                examples_seen,
            )


            indices = dataset_indices(
                n_train,
                examples_seen,
                this_batch,
                ORDER_SEEDS[
                    dataset
                ],
            )


            cpu = make_cpu_batch(
                train_blocks,
                indices,
            )


            ids = cpu.to(
                DEVICE,
                dtype=torch.long,
                non_blocking=True,
            )


            lr = learning_rate(
                update,
                max_updates,
            )


            for group in optimizer.param_groups:

                group[
                    "lr"
                ] = lr


            optimizer.zero_grad(
                set_to_none=True
            )


            step_start = (
                time.perf_counter()
            )


            with torch.autocast(
                "cuda",
                dtype=AMP_DTYPE,
            ):

                loss = model.nll(
                    ids,
                    checkpoint_layers=True,
                )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )


            optimizer.step()


            torch.cuda.synchronize()


            update += 1


            examples_seen += int(
                this_batch
            )


            last_train_nll = float(
                loss.detach()
            )


            last_train_ppl = math.exp(
                min(
                    last_train_nll,
                    20.0,
                )
            )


            last_train_bpb = (
                last_train_nll
                /
                math.log(
                    2.0
                )
            )


            peak_alloc = (
                torch.cuda
                .max_memory_allocated()
                /
                1024**3
            )


            peak_reserved = (
                torch.cuda
                .max_memory_reserved()
                /
                1024**3
            )


            pass_fraction = (
                float(
                    examples_seen
                )
                /
                float(
                    n_train
                )
            )


            step_seconds = (
                time.perf_counter()
                -
                step_start
            )


            bytes_per_second = (
                this_batch
                *
                RAW_BYTES_PER_SEQUENCE
                /
                max(
                    step_seconds,
                    1e-9,
                )
            )


            print(

                f"{dataset:<12} | "

                f"step="
                f"{update:5d} | "

                f"pass="
                f"{pass_fraction:5.3f}/"
                f"{MAX_PASSES} | "

                f"B="
                f"{this_batch:<3d} | "

                f"NLL="
                f"{last_train_nll:7.4f} | "

                f"PPL="
                f"{last_train_ppl:8.3f} | "

                f"BPB="
                f"{last_train_bpb:6.3f} | "

                f"lr="
                f"{lr:.2e} | "

                f"bytes/s="
                f"{bytes_per_second:,.0f} | "

                f"VRAM="
                f"{peak_alloc:.2f}/"
                f"{peak_reserved:.2f} GiB"
            )


            del (
                indices,
                cpu,
                ids,
                loss,
            )


            if (
                update
                %
                SAVE_EVERY_STEPS
                ==
                0
            ):

                save_latest()


            completed_pass = (
                examples_seen
                //
                n_train
            )


            crossed_pass = (
                completed_pass
                >
                last_completed_pass
            )


            if crossed_pass:

                last_completed_pass = (
                    completed_pass
                )


            if (
                update
                %
                VALIDATE_EVERY_STEPS
                ==
                0

                or

                crossed_pass
            ):

                print(

                    f"\n{name}: "
                    f"FULL GENUINE CAUSAL VALIDATION "
                    f"@ update {update}"
                )


                (
                    val_nll,
                    val_ppl,
                    val_bpb,
                    val_bytes,
                ) = validate_model(
                    model,
                    valid_blocks,
                    physical_batch,
                )


                print(

                    f"VALID | "

                    f"NLL="
                    f"{val_nll:.6f} | "

                    f"PPL="
                    f"{val_ppl:.6f} | "

                    f"BPB="
                    f"{val_bpb:.6f} | "

                    f"bytes="
                    f"{val_bytes:,}"
                )


                history.append(

                    {
                        "update":
                            update,

                        "examples_seen":
                            examples_seen,

                        "training_nll":
                            last_train_nll,

                        "training_ppl":
                            last_train_ppl,

                        "training_bpb":
                            last_train_bpb,

                        "nll":
                            float(
                                val_nll
                            ),

                        "ppl":
                            float(
                                val_ppl
                            ),

                        "bpb":
                            float(
                                val_bpb
                            ),

                        "reason":
                            (
                                "pass_boundary"
                                if crossed_pass
                                else
                                "periodic"
                            ),
                    }
                )


                if (
                    val_nll
                    <
                    best_nll
                ):

                    best_nll = float(
                        val_nll
                    )


                    save_best(
                        val_nll,
                        val_ppl,
                        val_bpb,
                    )


                    print(
                        "NEW BEST GENUINE CAUSAL MODEL"
                    )


                save_latest()


                if (
                    val_nll
                    <=
                    TARGET_NLL
                ):

                    stop_reason = (
                        "validation_ppl_3_reached"
                    )


                    print(
                        "=" * 100
                    )


                    print(
                        f"{name}: "
                        f"GENUINE PPL <= 3 REACHED"
                    )


                    print(

                        f"NLL="
                        f"{val_nll:.6f} | "

                        f"PPL="
                        f"{val_ppl:.6f} | "

                        f"BPB="
                        f"{val_bpb:.6f}"
                    )


                    print(
                        "=" * 100
                    )


                    break


    except BaseException:

        try:

            save_latest()


            print(
                f"{name}: "
                f"emergency checkpoint saved."
            )


        except Exception as save_error:

            print(

                f"{name}: "
                f"emergency save failed: "
                f"{save_error}"
            )


        raise


    # -------------------------------------------------------------------------
    # FINAL VALIDATION
    # -------------------------------------------------------------------------

    if stop_reason is None:

        stop_reason = (
            "three_pass_limit"
        )


        print(

            f"\n{name}: "
            f"FINAL GENUINE CAUSAL VALIDATION"
        )


        (
            val_nll,
            val_ppl,
            val_bpb,
            val_bytes,
        ) = validate_model(
            model,
            valid_blocks,
            physical_batch,
        )


        print(

            f"FINAL | "

            f"NLL="
            f"{val_nll:.6f} | "

            f"PPL="
            f"{val_ppl:.6f} | "

            f"BPB="
            f"{val_bpb:.6f} | "

            f"bytes="
            f"{val_bytes:,}"
        )


        history.append(

            {
                "update":
                    update,

                "examples_seen":
                    examples_seen,

                "training_nll":
                    last_train_nll,

                "training_ppl":
                    last_train_ppl,

                "training_bpb":
                    last_train_bpb,

                "nll":
                    float(
                        val_nll
                    ),

                "ppl":
                    float(
                        val_ppl
                    ),

                "bpb":
                    float(
                        val_bpb
                    ),

                "reason":
                    "final_validation",
            }
        )


        if (
            val_nll
            <
            best_nll
        ):

            best_nll = float(
                val_nll
            )


            save_best(
                val_nll,
                val_ppl,
                val_bpb,
            )


        if (
            val_nll
            <=
            TARGET_NLL
        ):

            stop_reason = (
                "validation_ppl_3_reached"
            )


    if not history:

        raise RuntimeError(
            f"{dataset}: "
            f"no validation result generated."
        )


    best = min(

        history,

        key=lambda row:
            row[
                "nll"
            ],
    )


    target_reached = bool(
        best[
            "nll"
        ]
        <=
        TARGET_NLL
    )


    result = {

        "status":
            "complete",

        "fingerprint":
            run_fingerprint,

        "architecture":
            ARCH,

        "dataset":
            dataset,

        "display_name":
            name,

        "objective":
            "exact_causal_next_byte_nll",

        "tokenization":
            "NONE_raw_utf8_bytes_learned_block_lifting",

        "future_leakage":
            False,

        "physical_batch":
            physical_batch,

        "maximum_passes":
            MAX_PASSES,

        "updates":
            update,

        "examples_seen":
            examples_seen,

        "effective_passes":
            float(
                examples_seen
            )
            /
            float(
                n_train
            ),

        "train_sequences":
            n_train,

        "validation_sequences":
            n_valid,

        "raw_context_bytes":
            RAW_BYTES_PER_SEQUENCE,

        "model_context_blocks":
            MODEL_CONTEXT,

        "parameter_count":
            total_params,

        "trainable_parameter_count":
            trainable_params,

        "target_nll":
            TARGET_NLL,

        "target_ppl":
            TARGET_PPL,

        "target_reached":
            target_reached,

        "best_validation_nll":
            float(
                best[
                    "nll"
                ]
            ),

        "best_validation_ppl":
            float(
                best[
                    "ppl"
                ]
            ),

        "best_validation_bpb":
            float(
                best[
                    "bpb"
                ]
            ),

        "best_validation_update":
            int(
                best[
                    "update"
                ]
            ),

        "last_training_nll":
            last_train_nll,

        "last_training_ppl":
            last_train_ppl,

        "last_training_bpb":
            last_train_bpb,

        "stop_reason":
            stop_reason,

        "training_seconds":
            elapsed(),

        "peak_allocated_vram_gb":
            float(
                torch.cuda
                .max_memory_allocated()
                /
                1024**3
            ),

        "peak_reserved_vram_gb":
            float(
                torch.cuda
                .max_memory_reserved()
                /
                1024**3
            ),

        "validation_history":
            history,
    }


    atomic_torch_save(

        {
            **result,

            "model_state":
                model.state_dict(),
        },

        final_path,
    )


    atomic_json(
        result,
        result_path,
    )


    latest_path.unlink(
        missing_ok=True
    )


    print(
        "=" * 120
    )


    print(
        f"{name} FINISHED"
    )


    print(

        f"best genuine NLL : "
        f"{result['best_validation_nll']:.6f}"
    )


    print(

        f"best genuine PPL : "
        f"{result['best_validation_ppl']:.6f}"
    )


    print(

        f"best BPB         : "
        f"{result['best_validation_bpb']:.6f}"
    )


    print(
        f"target reached   : "
        f"{target_reached}"
    )


    print(
        f"stop reason      : "
        f"{stop_reason}"
    )


    print(

        f"peak VRAM        : "

        f"{result['peak_allocated_vram_gb']:.2f}/"

        f"{result['peak_reserved_vram_gb']:.2f} GiB"
    )


    print(
        f"model            : "
        f"{final_path}"
    )


    print(
        f"result           : "
        f"{result_path}"
    )


    print(
        "=" * 120
    )


    del (
        model,
        optimizer,
        train_mem,
        train_blocks,
        valid_mem,
        valid_blocks,
    )


    cleanup_cuda()


    return result


# =============================================================================
# START
# =============================================================================

print(
    "=" * 120
)


print(
    "CAUSAL TRANSWAVE — REAL AUTOREGRESSIVE BYTE LANGUAGE MODEL"
)


print(
    f"GPU                     : "
    f"{GPU_NAME}"
)


print(
    f"VRAM                    : "
    f"{GPU_TOTAL_GB:.2f} GiB"
)


print(
    "offline tokenization    : NONE"
)


print(
    "BPE / SentencePiece     : NONE"
)


print(
    "MLM / MASK corruption   : NONE"
)


print(
    "objective               : exact p(x_t | x_<t)"
)


print(
    "lifting                 : "
    "learned 3-level block-local Haar-initialized lifting"
)


print(
    "Meyer temporal operator : "
    "strictly causal zero-padded FFT linear FIR"
)


print(
    "local convolution       : "
    "strictly left-causal"
)


print(
    "within-block decoder    : "
    "exact 8-byte autoregressive factorization"
)


print(

    f"raw context             : "
    f"{RAW_BYTES_PER_SEQUENCE:,} bytes"
)


print(

    f"model context           : "
    f"{MODEL_CONTEXT:,} eight-byte blocks"
)


print(
    f"layers                  : "
    f"{N_LAYERS}"
)


print(
    f"dimension               : "
    f"{MODEL_DIM}"
)


print(
    f"Meyer bands             : "
    f"{N_MEYER_BANDS}"
)


print(
    f"controls / band         : "
    f"{N_BAND_CONTROLS}"
)


print(

    f"Meyer causal FIR length : "
    f"{MEYER_KERNEL} block positions"
)


print(

    f"REAL TARGET NLL         : "
    f"{TARGET_NLL:.9f}"
)


print(

    f"REAL TARGET PPL         : "
    f"{TARGET_PPL:.6f}"
)


print(
    "=" * 120
)


# =============================================================================
# MAXIMIZE G4 VRAM
# =============================================================================

PHYSICAL_BATCH = choose_batch_size()


# =============================================================================
# FOUR INDEPENDENT CORPUS RUNS
# =============================================================================

ALL_RESULTS = []


for dataset in DATASETS:

    ALL_RESULTS.append(

        train_dataset(
            dataset,
            PHYSICAL_BATCH,
        )
    )


# =============================================================================
# FINAL SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 120
)


print(
    "CAUSAL TRANSWAVE — GENUINE HELD-OUT AUTOREGRESSIVE RESULTS"
)


print(
    "=" * 120
)


print(

    f"{'DATASET':<18}"

    f"{'NLL':>14}"

    f"{'PPL':>14}"

    f"{'BPB':>14}"

    f"{'PASSES':>12}"

    f"{'TARGET':>12}"
)


print(
    "-" * 84
)


for row in ALL_RESULTS:

    print(

        f"{row['display_name']:<18}"

        f"{row['best_validation_nll']:>14.6f}"

        f"{row['best_validation_ppl']:>14.6f}"

        f"{row['best_validation_bpb']:>14.6f}"

        f"{row['effective_passes']:>12.3f}"

        f"{('YES' if row['target_reached'] else 'NO'):>12}"
    )


print(
    "-" * 84
)


print(

    f"Stopping target: "
    f"genuine NLL <= "
    f"{TARGET_NLL:.9f} | "

    f"genuine PPL <= "
    f"{TARGET_PPL:.6f}"
)


print(

    "PPL = exp(exact held-out causal byte NLL). "
    "It is not an MLM reconstruction score."
)


print(
    "=" * 120
)

CAUSAL TRANSWAVE — REAL AUTOREGRESSIVE BYTE LANGUAGE MODEL
GPU                     : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM                    : 94.97 GiB
offline tokenization    : NONE
BPE / SentencePiece     : NONE
MLM / MASK corruption   : NONE
objective               : exact p(x_t | x_<t)
lifting                 : learned 3-level block-local Haar-initialized lifting
Meyer temporal operator : strictly causal zero-padded FFT linear FIR
local convolution       : strictly left-causal
within-block decoder    : exact 8-byte autoregressive factorization
raw context             : 32,768 bytes
model context           : 4,096 eight-byte blocks
layers                  : 12
dimension               : 768
Meyer bands             : 12
controls / band         : 8
Meyer causal FIR length : 512 block positions
REAL TARGET NLL         : 1.098612289
REAL TARGET PPL         : 3.000000
G4 COMPLETE-STEP VRAM AUTOTUNER
GPU                     : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM 

In [30]:
# CELL 1 — NATIVE TOKENIZATION, NO GATED REPOSITORIES
from __future__ import annotations

import os
import json
import hashlib
from pathlib import Path

import numpy as np

from transformers import AutoTokenizer


# =============================================================================
# PATHS
# =============================================================================

RAW_ROOT = Path(
    "/content/drive/MyDrive/spliNet/raw_text"
)

ROOT = Path(
    "/content/drive/MyDrive/"
    "TransWave_Meyer_Context/"
    "commercial_transformer_baselines_v2"
)

TOKEN_ROOT = (
    ROOT
    /
    "native_tokens"
)

TOKEN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# =============================================================================
# DATASETS
# =============================================================================

DATASETS = (
    "arxiv",
    "fineweb_edu",
    "pg19",
    "wikitext103",
)

SPLITS = (
    "train",
    "validation",
)


# =============================================================================
# PUBLIC REPOSITORIES
#
# We use these only for:
#   tokenizer
#   architecture config
#
# NO pretrained weights are loaded.
# =============================================================================

MODEL_SPECS = {

    "llama": {

        "display":
            "LLaMA-3.2",

        "family":
            "meta-llama/Llama-3.2-1B",

        "repo":
            "unsloth/Llama-3.2-1B",
    },

    "gemma": {

        "display":
            "Gemma-2",

        "family":
            "google/gemma-2-2b",

        "repo":
            "unsloth/gemma-2-2b",
    },

    "qwen": {

        "display":
            "Qwen2.5",

        "family":
            "Qwen/Qwen2.5-1.5B",

        "repo":
            "Qwen/Qwen2.5-1.5B",
    },
}


# =============================================================================
# STREAMING TOKENIZATION
# =============================================================================

CHARS_PER_CHUNK = 1_000_000

TOKEN_DTYPE = np.uint32


def tokenizer_signature(
    tokenizer,
    repo,
):

    payload = {

        "repo":
            repo,

        "tokenizer_class":
            tokenizer.__class__.__name__,

        "vocab_size":
            len(
                tokenizer
            ),

        "bos_token_id":
            tokenizer.bos_token_id,

        "eos_token_id":
            tokenizer.eos_token_id,

        "pad_token_id":
            tokenizer.pad_token_id,

        "add_bos_token":
            bool(
                getattr(
                    tokenizer,
                    "add_bos_token",
                    False,
                )
            ),

        "add_eos_token":
            bool(
                getattr(
                    tokenizer,
                    "add_eos_token",
                    False,
                )
            ),
    }


    return hashlib.sha256(

        json.dumps(
            payload,
            sort_keys=True,
        ).encode()

    ).hexdigest()


def native_prefix_suffix(
    tokenizer,
):

    prefix = []

    suffix = []


    if (
        bool(
            getattr(
                tokenizer,
                "add_bos_token",
                False,
            )
        )
        and
        tokenizer.bos_token_id
        is not None
    ):

        prefix.append(
            int(
                tokenizer.bos_token_id
            )
        )


    if (
        bool(
            getattr(
                tokenizer,
                "add_eos_token",
                False,
            )
        )
        and
        tokenizer.eos_token_id
        is not None
    ):

        suffix.append(
            int(
                tokenizer.eos_token_id
            )
        )


    return (
        prefix,
        suffix,
    )


def tokenize_file(
    model_key,
    dataset,
    split,
    tokenizer,
    repo,
):

    src = (
        RAW_ROOT
        /
        dataset
        /
        f"{split}.txt"
    )


    if not src.is_file():

        raise FileNotFoundError(
            src
        )


    out_dir = (
        TOKEN_ROOT
        /
        model_key
        /
        dataset
    )


    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    bin_path = (
        out_dir
        /
        f"{split}.uint32.bin"
    )


    meta_path = (
        out_dir
        /
        f"{split}.json"
    )


    signature = tokenizer_signature(
        tokenizer,
        repo,
    )


    source_bytes = (
        src.stat().st_size
    )


    # -------------------------------------------------------------------------
    # RESTART SAFE
    # -------------------------------------------------------------------------

    if (
        bin_path.is_file()
        and
        meta_path.is_file()
    ):

        try:

            old = json.loads(
                meta_path.read_text(
                    encoding="utf-8"
                )
            )


            expected_size = (
                int(
                    old[
                        "token_count"
                    ]
                )
                *
                np.dtype(
                    TOKEN_DTYPE
                ).itemsize
            )


            if (
                old.get(
                    "status"
                )
                ==
                "complete"

                and

                old.get(
                    "tokenizer_signature"
                )
                ==
                signature

                and

                int(
                    old.get(
                        "source_bytes",
                        -1,
                    )
                )
                ==
                source_bytes

                and

                bin_path.stat().st_size
                ==
                expected_size
            ):

                print(

                    f"{model_key:<6} | "
                    f"{dataset:<12} | "
                    f"{split:<10} | "
                    f"CACHED | "
                    f"{old['token_count']:,} tokens"
                )


                return old


        except Exception:

            pass


    tmp_bin = bin_path.with_suffix(
        ".uint32.bin.tmp"
    )


    tmp_meta = meta_path.with_suffix(
        ".json.tmp"
    )


    tmp_bin.unlink(
        missing_ok=True
    )


    tmp_meta.unlink(
        missing_ok=True
    )


    prefix, suffix = (
        native_prefix_suffix(
            tokenizer
        )
    )


    token_count = 0


    print()

    print(
        f"{MODEL_SPECS[model_key]['display']} | "
        f"{dataset} | {split}"
    )


    print(
        f"  tokenizer : {repo}"
    )


    print(
        f"  raw       : "
        f"{source_bytes / 1024**2:.1f} MiB"
    )


    print(
        f"  BOS       : "
        f"{prefix if prefix else 'none'}"
    )


    print(
        f"  EOS       : "
        f"{suffix if suffix else 'none'}"
    )


    old_model_max_length = (
        tokenizer.model_max_length
    )


    tokenizer.model_max_length = (
        10**12
    )


    try:

        with open(
            src,
            "r",
            encoding="utf-8",
            errors="strict",
        ) as fin, open(
            tmp_bin,
            "wb",
        ) as fout:


            if prefix:

                a = np.asarray(
                    prefix,
                    dtype=TOKEN_DTYPE,
                )


                a.tofile(
                    fout
                )


                token_count += int(
                    a.size
                )


            chars_done = 0


            while True:

                text = fin.read(
                    CHARS_PER_CHUNK
                )


                if not text:

                    break


                ids = tokenizer(

                    text,

                    add_special_tokens=False,

                    truncation=False,

                    return_attention_mask=False,

                )[
                    "input_ids"
                ]


                a = np.asarray(
                    ids,
                    dtype=TOKEN_DTYPE,
                )


                if a.size:

                    if (
                        int(
                            a.max()
                        )
                        >=
                        2**32
                    ):

                        raise RuntimeError(
                            "Token ID exceeds uint32."
                        )


                    a.tofile(
                        fout
                    )


                    token_count += int(
                        a.size
                    )


                chars_done += len(
                    text
                )


                if (
                    chars_done
                    %
                    20_000_000
                    <
                    CHARS_PER_CHUNK
                ):

                    print(

                        f"  ~"
                        f"{chars_done / 1e6:.0f}M chars | "
                        f"{token_count:,} tokens"
                    )


            if suffix:

                a = np.asarray(
                    suffix,
                    dtype=TOKEN_DTYPE,
                )


                a.tofile(
                    fout
                )


                token_count += int(
                    a.size
                )


    finally:

        tokenizer.model_max_length = (
            old_model_max_length
        )


    os.replace(
        tmp_bin,
        bin_path,
    )


    meta = {

        "status":
            "complete",

        "model_key":
            model_key,

        "display":
            MODEL_SPECS[
                model_key
            ][
                "display"
            ],

        "family":
            MODEL_SPECS[
                model_key
            ][
                "family"
            ],

        "tokenizer_repo":
            repo,

        "tokenizer_class":
            tokenizer.__class__.__name__,

        "tokenizer_signature":
            signature,

        "vocab_size":
            len(
                tokenizer
            ),

        "source_path":
            str(
                src
            ),

        "source_bytes":
            source_bytes,

        "token_count":
            token_count,

        "dtype":
            "uint32",

        "bos_token_id":
            tokenizer.bos_token_id,

        "eos_token_id":
            tokenizer.eos_token_id,

        "pad_token_id":
            tokenizer.pad_token_id,

        "add_bos_token":
            bool(
                getattr(
                    tokenizer,
                    "add_bos_token",
                    False,
                )
            ),

        "add_eos_token":
            bool(
                getattr(
                    tokenizer,
                    "add_eos_token",
                    False,
                )
            ),
    }


    tmp_meta.write_text(

        json.dumps(
            meta,
            indent=2,
            sort_keys=True,
        ),

        encoding="utf-8",
    )


    os.replace(
        tmp_meta,
        meta_path,
    )


    print(

        f"  DONE      : "
        f"{token_count:,} tokens | "

        f"{token_count/source_bytes:.4f} "
        f"tokens/raw-byte"
    )


    return meta


# =============================================================================
# LOAD AND VERIFY THE THREE TOKENIZERS FIRST
#
# Fail before processing 12 huge streams if any repository is unavailable.
# =============================================================================

TOKENIZERS = {}


print(
    "=" * 110
)


print(
    "VERIFYING ALL THREE PUBLIC NATIVE TOKENIZERS"
)


print(
    "=" * 110
)


for model_key, spec in MODEL_SPECS.items():

    print(

        f"{spec['display']:<12} | "
        f"{spec['repo']}"
    )


    tok = AutoTokenizer.from_pretrained(
        spec[
            "repo"
        ],
        use_fast=True,
    )


    TOKENIZERS[
        model_key
    ] = tok


    probe = tok(
        "The quick brown fox jumps over the lazy dog.",
        add_special_tokens=False,
    )[
        "input_ids"
    ]


    print(

        f"  class="
        f"{tok.__class__.__name__} | "

        f"vocab="
        f"{len(tok):,} | "

        f"probe tokens="
        f"{len(probe)}"
    )


print(
    "=" * 110
)


print(
    "ALL THREE TOKENIZERS ACCESSIBLE"
)


print(
    "=" * 110
)


# =============================================================================
# TOKENIZE 3 x 4 x 2 STREAMS
# =============================================================================

TOKEN_META = {}


for model_key, spec in MODEL_SPECS.items():

    tokenizer = TOKENIZERS[
        model_key
    ]


    TOKEN_META[
        model_key
    ] = {}


    print()

    print(
        "=" * 110
    )


    print(

        f"NATIVE TOKENIZATION: "
        f"{spec['display']} | "
        f"{spec['repo']}"
    )


    print(
        "=" * 110
    )


    for dataset in DATASETS:

        TOKEN_META[
            model_key
        ][
            dataset
        ] = {}


        for split in SPLITS:

            TOKEN_META[
                model_key
            ][
                dataset
            ][
                split
            ] = tokenize_file(

                model_key,

                dataset,

                split,

                tokenizer,

                spec[
                    "repo"
                ],
            )


# =============================================================================
# SUMMARY
# =============================================================================

print()

print(
    "=" * 110
)


print(
    "NATIVE TOKENIZATION COMPLETE"
)


print(
    "=" * 110
)


for model_key in MODEL_SPECS:

    for dataset in DATASETS:

        train_n = (

            TOKEN_META[
                model_key
            ][
                dataset
            ][
                "train"
            ][
                "token_count"
            ]
        )


        valid_n = (

            TOKEN_META[
                model_key
            ][
                dataset
            ][
                "validation"
            ][
                "token_count"
            ]
        )


        print(

            f"{model_key:<6} | "
            f"{dataset:<12} | "

            f"train="
            f"{train_n:,} | "

            f"validation="
            f"{valid_n:,}"
        )

VERIFYING ALL THREE PUBLIC NATIVE TOKENIZERS
LLaMA-3.2    | unsloth/Llama-3.2-1B


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

  class=Qwen2Tokenizer | vocab=151,665 | probe tokens=10
ALL THREE TOKENIZERS ACCESSIBLE

NATIVE TOKENIZATION: LLaMA-3.2 | unsloth/Llama-3.2-1B

LLaMA-3.2 | arxiv | train
  tokenizer : unsloth/Llama-3.2-1B
  raw       : 520.0 MiB
  BOS       : none
  EOS       : none
  ~20M chars | 5,003,209 tokens
  ~40M chars | 10,022,529 tokens
  ~60M chars | 14,976,838 tokens
  ~80M chars | 20,028,336 tokens
  ~100M chars | 24,994,847 tokens
  ~120M chars | 29,974,940 tokens
  ~140M chars | 35,015,702 tokens
  ~160M chars | 40,052,079 tokens
  ~180M chars | 45,021,856 tokens
  ~200M chars | 50,044,789 tokens
  ~220M chars | 55,029,398 tokens
  ~240M chars | 60,096,502 tokens
  ~260M chars | 65,063,730 tokens
  ~280M chars | 70,042,005 tokens
  ~300M chars | 75,077,368 tokens
  ~320M chars | 80,070,764 tokens
  ~340M chars | 85,065,419 tokens
  ~360M chars | 90,060,039 tokens
  ~380M chars | 95,110,014 tokens
  ~400M chars | 100,074,399 tokens
  ~420M chars | 105,109,155 tokens
  ~440M chars | 110,1